In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import random

PROJECT_ROOT = Path('/content/drive/MyDrive/CAPSTONE_ASL')
KEYPOINT_DIR = PROJECT_ROOT / 'keypoints'
SUBSET_DIR   = PROJECT_ROOT / 'subset_top100'

sub_train = pd.read_csv(SUBSET_DIR / 'train.csv')
sub_val   = pd.read_csv(SUBSET_DIR / 'val.csv')
sub_test  = pd.read_csv(SUBSET_DIR / 'test.csv')

# Confirm every CSV row has a matching .npz file
all_files = pd.concat([sub_train, sub_val, sub_test])['Video file'].tolist()
missing = []
sizes = []
for f in all_files:
    p = KEYPOINT_DIR / f"{Path(f).stem}.npz"
    if not p.exists():
        missing.append(f)
    else:
        sizes.append(p.stat().st_size)

print(f"Total expected: {len(all_files)}")
print(f"Missing:        {len(missing)}")
print(f"File sizes:     min={min(sizes)/1024:.1f}KB  "
      f"max={max(sizes)/1024:.1f}KB  "
      f"mean={sum(sizes)/len(sizes)/1024:.1f}KB  "
      f"total={sum(sizes)/1e6:.1f}MB")

# Load 5 random files and check structure
print("\nSpot-check 5 random keypoint files:")
random.seed(0)
for f in random.sample(all_files, 5):
    p = KEYPOINT_DIR / f"{Path(f).stem}.npz"
    data = np.load(p)
    T = data['pose'].shape[0]
    pose_rate = data['pose_mask'].mean()
    either = (data['lh_mask'] | data['rh_mask']).mean()
    print(f"  {p.name[:45]:45s}  T={T:3d} pose={pose_rate:.0%} either-hand={either:.0%} fps={data['fps']:.1f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total expected: 3453
Missing:        0
File sizes:     min=1.4KB  max=124.9KB  mean=21.2KB  total=75.0MB

Spot-check 5 random keypoint files:
  6151843197692557-IMPOSSIBLE.npz                T= 40 pose=100% either-hand=48% fps=15.0
  25322510507863427-TAIL.npz                     T= 44 pose=100% either-hand=89% fps=15.0
  14322213163115816-seedPATIENT 2.npz            T=125 pose=100% either-hand=52% fps=29.8
  2978383554033186-LETTUCE.npz                   T= 96 pose=100% either-hand=60% fps=30.0
  34668296948798183-DEMAND.npz                   T= 98 pose=100% either-hand=48% fps=30.0


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

PROJECT_ROOT = Path('/content/drive/MyDrive/CAPSTONE_ASL')
KEYPOINT_DIR = PROJECT_ROOT / 'keypoints'
SUBSET_DIR   = PROJECT_ROOT / 'subset_top100'
FEATURES_DIR = PROJECT_ROOT / 'features'
FEATURES_DIR.mkdir(exist_ok=True)

sub_train = pd.read_csv(SUBSET_DIR / 'train.csv')
sub_val   = pd.read_csv(SUBSET_DIR / 'val.csv')
sub_test  = pd.read_csv(SUBSET_DIR / 'test.csv')
print(f"Loaded: train={len(sub_train)} val={len(sub_val)} test={len(sub_test)}")

In [ ]:
# Pose landmark indices in our saved arrays (we kept 7 from the original 33)
# Order: [0]=nose, [1]=left_shoulder, [2]=right_shoulder,
#        [3]=left_elbow, [4]=right_elbow, [5]=left_wrist, [6]=right_wrist
P_NOSE       = 0
P_LSHOULDER  = 1
P_RSHOULDER  = 2
P_LELBOW     = 3
P_RELBOW     = 4
P_LWRIST     = 5
P_RWRIST     = 6

TARGET_FPS = 30.0  # reference framerate for velocity normalization


def normalize_pose(pose):
    """Center on shoulder midpoint, scale by shoulder width.
    pose: (T, 7, 3)
    returns: (T, 7, 3) normalized, plus the scale factor used."""
    # Shoulder midpoint per frame
    mid = (pose[:, P_LSHOULDER] + pose[:, P_RSHOULDER]) / 2  # (T, 3)
    # Shoulder distance per frame
    shoulder_dist = np.linalg.norm(
        pose[:, P_LSHOULDER, :2] - pose[:, P_RSHOULDER, :2], axis=-1
    )  # (T,)
    # Use the median shoulder distance as the scale (robust to bad frames)
    scale = np.median(shoulder_dist[shoulder_dist > 1e-6])
    if scale < 1e-6:
        scale = 1.0
    centered = pose - mid[:, None, :]
    return centered / scale, scale, mid


def normalize_hand(hand, anchor_xy_per_frame, scale):
    """Center hand on per-frame anchor (the wrist from pose), scale by body scale.
    hand: (T, 21, 3)
    anchor_xy_per_frame: (T, 3) — the wrist landmark for this hand
    """
    centered = hand - anchor_xy_per_frame[:, None, :]
    return centered / scale


def interpolate_missing(features, mask):
    """Linear interpolation across frames where mask is False.
    features: (T, D) or (T, N, D)
    mask: (T,) bool — True where features are valid
    """
    if not mask.any():
        return features
    if mask.all():
        return features

    T = len(mask)
    valid_idx = np.where(mask)[0]
    result = features.copy()
    # Forward-fill from earliest valid frame
    first = valid_idx[0]
    if first > 0:
        result[:first] = features[first]
    # Backward-fill from last valid frame
    last = valid_idx[-1]
    if last < T - 1:
        result[last+1:] = features[last]
    # Linear interp in between
    for i in range(len(valid_idx) - 1):
        a, b = valid_idx[i], valid_idx[i+1]
        if b - a > 1:
            for t in range(a+1, b):
                alpha = (t - a) / (b - a)
                result[t] = (1 - alpha) * features[a] + alpha * features[b]
    return result


def build_features(kp):
    """Turn one keypoint file into normalized, motion-augmented features.

    Returns dict with:
      'positions': (T, F) flat per-frame coordinates
      'velocity':  (T, F) frame-to-frame difference (normalized to target FPS)
      'speed':     (T,)   summary speed of active hand(s) per frame
      'fps':       original FPS
      'pose_mask', 'lh_mask', 'rh_mask': original detection masks
    """
    pose = kp['pose']      # (T, 7, 3)
    lh   = kp['lh']        # (T, 21, 3)
    rh   = kp['rh']        # (T, 21, 3)
    pose_mask = kp['pose_mask']
    lh_mask   = kp['lh_mask']
    rh_mask   = kp['rh_mask']
    fps = float(kp['fps'])

    T = pose.shape[0]
    if T < 2:
        return None  # too short to compute velocity

    # --- Normalize pose ---
    pose_n, scale, mid = normalize_pose(pose)  # (T, 7, 3)

    # --- Normalize hands to pose wrists ---
    # Wrist coordinates in original (pre-normalization) space
    lwrist = pose[:, P_LWRIST]   # (T, 3)
    rwrist = pose[:, P_RWRIST]   # (T, 3)
    lh_n = normalize_hand(lh, lwrist, scale)  # (T, 21, 3)
    rh_n = normalize_hand(rh, rwrist, scale)  # (T, 21, 3)

    # --- Interpolate missing detections ---
    # For frames where the hand wasn't detected, fill with interpolation
    # so the model sees continuous trajectories
    lh_n = interpolate_missing(lh_n, lh_mask)
    rh_n = interpolate_missing(rh_n, rh_mask)
    pose_n = interpolate_missing(pose_n, pose_mask)

    # --- Flatten into per-frame feature vectors ---
    # pose: 7*3=21, lh: 21*3=63, rh: 21*3=63 → total 147 features per frame
    positions = np.concatenate([
        pose_n.reshape(T, -1),
        lh_n.reshape(T, -1),
        rh_n.reshape(T, -1),
    ], axis=-1).astype(np.float32)  # (T, 147)

    # --- Velocity, normalized to target FPS ---
    # Raw velocity is position[t] - position[t-1], which depends on FPS.
    # Multiply by (fps / TARGET_FPS) so that 1 frame at 15 FPS counts the
    # same as 2 frames at 30 FPS — i.e., motion per "target frame".
    velocity = np.diff(positions, axis=0, prepend=positions[:1])
    velocity = velocity * (TARGET_FPS / fps)

    # --- Smooth velocity to reduce frame-to-frame noise ---
    velocity = uniform_filter1d(velocity, size=3, axis=0)

    # --- Per-frame speed: how fast are the active hands moving? ---
    # Use the wrist landmarks (most stable hand indicator)
    # Wrist is landmark index 0 inside each hand
    lh_wrist_vel = velocity[:, 21:24]      # left hand landmark 0
    rh_wrist_vel = velocity[:, 84:87]      # right hand landmark 0
    # Speed = magnitude; use whichever hand is more active per frame
    lh_speed = np.linalg.norm(lh_wrist_vel, axis=-1)
    rh_speed = np.linalg.norm(rh_wrist_vel, axis=-1)
    # If only one hand is signing, the other's speed will be small; max() picks the active one
    speed = np.maximum(lh_speed, rh_speed)

    return {
        'positions': positions,
        'velocity':  velocity,
        'speed':     speed.astype(np.float32),
        'fps':       np.float32(fps),
        'pose_mask': pose_mask,
        'lh_mask':   lh_mask,
        'rh_mask':   rh_mask,
    }

In [ ]:
# Pick one clip with high hand detection so we can see clean motion
import random
random.seed(0)

# Find a clip from the training set with decent hand detection
sample_file = None
for f in random.sample(sub_train['Video file'].tolist(), 50):
    kp = np.load(KEYPOINT_DIR / f"{Path(f).stem}.npz")
    either = (kp['lh_mask'] | kp['rh_mask']).mean()
    if either > 0.7 and len(kp['pose']) > 30:
        sample_file = f
        sample_kp = kp
        break

print(f"Using: {sample_file}")
print(f"Frames: {len(sample_kp['pose'])}  FPS: {float(sample_kp['fps']):.1f}")
print(f"Either-hand: {(sample_kp['lh_mask'] | sample_kp['rh_mask']).mean():.0%}")

# Build features
feats = build_features(sample_kp)

print(f"\nPosition features shape: {feats['positions'].shape}")
print(f"Velocity features shape: {feats['velocity'].shape}")
print(f"Speed shape:             {feats['speed'].shape}")
print(f"Speed range:             {feats['speed'].min():.4f} to {feats['speed'].max():.4f}")

In [ ]:
T = len(feats['speed'])
t = np.arange(T)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: speed (this is the key signal for phase detection)
axes[0].plot(t, feats['speed'], linewidth=2, color='steelblue')
axes[0].fill_between(t, 0, feats['speed'], alpha=0.2, color='steelblue')
axes[0].set_ylabel('Wrist speed\n(normalized)', fontsize=11)
axes[0].set_title(f"{sample_file}  —  speed profile")
axes[0].grid(alpha=0.3)

# Bottom: hand detection masks (when was hand actually visible?)
axes[1].fill_between(t, 0, feats['rh_mask'].astype(int), alpha=0.5,
                     label='Right hand detected', color='tomato', step='mid')
axes[1].fill_between(t, 0, -feats['lh_mask'].astype(int), alpha=0.5,
                     label='Left hand detected', color='seagreen', step='mid')
axes[1].set_ylabel('Detection')
axes[1].set_xlabel('Frame')
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['L hand', '', 'R hand'])
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
compound_signs = [g for g in sub_train['Gloss'].unique()
                  if ' ' in g or
                     any(c.isdigit() for c in g.rstrip('0123456789'))]
top100 = sub_train['Gloss'].value_counts().head(100).index.tolist()
top100_compound = [g for g in top100 if ' ' in g]
print(f"All compound signs in top 100 (containing space):")
for g in top100_compound:
    n = sub_train['Gloss'].value_counts()[g]
    print(f"  {g:30s}  ({n} train samples)")
print(f"\nTotal: {len(top100_compound)} / 100 classes are compound")

In [ ]:
import random
random.seed(42)

# Pick 8 diverse clips: mix of one-word and compound, varied lengths
candidates = []
for f in sub_train['Video file'].tolist():
    kp = np.load(KEYPOINT_DIR / f"{Path(f).stem}.npz")
    if (kp['lh_mask'] | kp['rh_mask']).mean() > 0.5:
        gloss = sub_train.loc[sub_train['Video file'] == f, 'Gloss'].iloc[0]
        candidates.append((f, gloss, len(kp['pose'])))

# Get a mix
single = [c for c in candidates if ' ' not in c[1]]
compound = [c for c in candidates if ' ' in c[1]]
sample_set = random.sample(single, 6) + random.sample(compound, min(2, len(compound)))

fig, axes = plt.subplots(len(sample_set), 1, figsize=(14, 2*len(sample_set)), sharex=False)
for ax, (f, gloss, T) in zip(axes, sample_set):
    kp = np.load(KEYPOINT_DIR / f"{Path(f).stem}.npz")
    feats = build_features(kp)
    if feats is None:
        continue
    ax.plot(feats['speed'], linewidth=2, color='steelblue')
    ax.fill_between(np.arange(len(feats['speed'])), 0, feats['speed'], alpha=0.2)
    is_compound = ' ' in gloss
    title_color = 'firebrick' if is_compound else 'darkgreen'
    ax.set_title(f"{gloss}  ({T} frames, fps={float(kp['fps']):.0f})",
                 fontsize=10, color=title_color)
    ax.grid(alpha=0.3)
    ax.set_ylabel('speed', fontsize=9)
axes[-1].set_xlabel('Frame')
plt.tight_layout()
plt.show()

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

random.seed(42)

# Sample 100 files from train (small enough to be fast on Drive)
sample_files = random.sample(sub_train['Video file'].tolist(), 100)

# Pick 8 diverse ones based on length
candidates = []
for f in sample_files:
    kp = np.load(KEYPOINT_DIR / f"{Path(f).stem}.npz")
    either = (kp['lh_mask'] | kp['rh_mask']).mean()
    T = len(kp['pose'])
    gloss = sub_train.loc[sub_train['Video file'] == f, 'Gloss'].iloc[0]
    if either > 0.4 and T >= 30:
        candidates.append((f, gloss, T))

# Spread across length buckets
candidates.sort(key=lambda c: c[2])
n = len(candidates)
indices = [int(i * (n-1) / 7) for i in range(8)]
sample_set = [candidates[i] for i in indices]

print(f"Visualizing 8 clips spanning lengths {sample_set[0][2]} to {sample_set[-1][2]} frames")
print()

fig, axes = plt.subplots(len(sample_set), 1, figsize=(14, 2*len(sample_set)), sharex=False)
for ax, (f, gloss, T) in zip(axes, sample_set):
    kp = np.load(KEYPOINT_DIR / f"{Path(f).stem}.npz")
    feats = build_features(kp)
    if feats is None:
        continue
    ax.plot(feats['speed'], linewidth=2, color='steelblue')
    ax.fill_between(np.arange(len(feats['speed'])), 0, feats['speed'], alpha=0.2)
    ax.set_title(f"{gloss}  —  {T} frames @ {float(kp['fps']):.0f}fps",
                 fontsize=10, color='darkgreen')
    ax.grid(alpha=0.3)
    ax.set_ylabel('speed', fontsize=9)
axes[-1].set_xlabel('Frame')
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import random

PROJECT_ROOT = Path('/content/drive/MyDrive/CAPSTONE_ASL')
KEYPOINT_DIR = PROJECT_ROOT / 'keypoints'
SUBSET_DIR   = PROJECT_ROOT / 'subset_top100'
ANNOT_DIR    = PROJECT_ROOT / 'annotations'
ANNOT_DIR.mkdir(exist_ok=True)

sub_train = pd.read_csv(SUBSET_DIR / 'train.csv')

# Pick 50 clips: stratified by duration bucket, varied glosses and signers
random.seed(42)

# Get per-clip metadata
clip_meta = []
for f in random.sample(sub_train['Video file'].tolist(), 300):  # sample for speed
    kp = np.load(KEYPOINT_DIR / f"{Path(f).stem}.npz")
    T = len(kp['pose'])
    either = (kp['lh_mask'] | kp['rh_mask']).mean()
    row = sub_train[sub_train['Video file'] == f].iloc[0]
    clip_meta.append({
        'video_file': f,
        'gloss': row['Gloss'],
        'signer': row['Participant ID'],
        'n_frames': T,
        'either_hand_rate': either,
    })

clip_meta = pd.DataFrame(clip_meta)
clip_meta = clip_meta[(clip_meta['n_frames'] >= 30) &
                      (clip_meta['n_frames'] <= 200) &
                      (clip_meta['either_hand_rate'] > 0.3)]

# Stratify by frame-count bucket and pick across buckets
clip_meta['bucket'] = pd.cut(clip_meta['n_frames'], bins=[0, 50, 80, 120, 200])
to_annotate = (clip_meta.groupby('bucket', observed=True)
                        .apply(lambda g: g.sample(min(13, len(g)), random_state=42))
                        .reset_index(drop=True))
to_annotate = to_annotate.head(50).reset_index(drop=True)

print(f"Selected {len(to_annotate)} clips to annotate")
print(f"Duration distribution:")
print(to_annotate['n_frames'].describe())
print(f"\nUnique glosses: {to_annotate['gloss'].nunique()}")
print(f"Unique signers: {to_annotate['signer'].nunique()}")

# Save the list so we can resume across sessions
to_annotate.to_csv(ANNOT_DIR / 'clips_to_annotate.csv', index=False)
print(f"\nSaved list to {ANNOT_DIR / 'clips_to_annotate.csv'}")

In [ ]:
import zipfile

LOCAL_BASE = Path('/content/asl_local')
LOCAL_BASE.mkdir(exist_ok=True)
LOCAL_VIDEO_DIR = LOCAL_BASE / 'ASL_Citizen' / 'videos'

zip_path = PROJECT_ROOT / 'download' / 'ASL_Citizen.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    print(f"Extracting {len(to_annotate)} clips for annotation...")
    for f in to_annotate['video_file']:
        member = f'ASL_Citizen/videos/{f}'
        out = LOCAL_BASE / member
        if not out.exists():
            z.extract(member, path=LOCAL_BASE)
print(f"Done. {len(list(LOCAL_VIDEO_DIR.glob('*.mp4')))} clips on local disk")

In [ ]:
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import base64
from scipy.ndimage import uniform_filter1d
import json

# Load the build_features function from our earlier work
# (paste it here if not in memory — defined in the previous motion-features cells)


def compute_speed(kp):
    """Quick speed computation just for visualization during annotation."""
    pose = kp['pose']
    fps = float(kp['fps'])
    T = pose.shape[0]
    if T < 2:
        return np.zeros(T)

    # Just use raw wrist motion from pose (faster than full pipeline)
    lwrist = pose[:, 5, :2]  # left wrist, x/y
    rwrist = pose[:, 6, :2]  # right wrist
    lv = np.linalg.norm(np.diff(lwrist, axis=0, prepend=lwrist[:1]), axis=-1)
    rv = np.linalg.norm(np.diff(rwrist, axis=0, prepend=rwrist[:1]), axis=-1)
    speed = np.maximum(lv, rv) * (30.0 / fps)
    return uniform_filter1d(speed, size=3)


def load_existing_annotations():
    out_path = ANNOT_DIR / 'annotations.json'
    if out_path.exists():
        with open(out_path) as fp:
            return json.load(fp)
    return {}


def save_annotation(video_file, segments, note=""):
    annotations = load_existing_annotations()
    annotations[video_file] = {
        'segments': segments,  # list of [start, end] pairs
        'note': note,
    }
    out_path = ANNOT_DIR / 'annotations.json'
    with open(out_path, 'w') as fp:
        json.dump(annotations, fp, indent=2)


def video_to_data_url(video_path):
    with open(video_path, 'rb') as fp:
        data = base64.b64encode(fp.read()).decode('ascii')
    return f"data:video/mp4;base64,{data}"


class Annotator:
    def __init__(self, clips_df):
        self.clips = clips_df.reset_index(drop=True)
        self.idx = self._find_next_unannotated()
        self.annotations = load_existing_annotations()

    def _find_next_unannotated(self):
        existing = load_existing_annotations()
        for i, row in self.clips.iterrows():
            if row['video_file'] not in existing:
                return i
        return len(self.clips)  # all done

    def show(self):
        clear_output(wait=True)
        if self.idx >= len(self.clips):
            print(f"DONE! All {len(self.clips)} clips annotated.")
            return

        row = self.clips.iloc[self.idx]
        video_file = row['video_file']
        gloss = row['gloss']
        signer = row['signer']
        T = row['n_frames']

        existing = load_existing_annotations()
        n_done = len([k for k in existing if k in self.clips['video_file'].values])

        print(f"Clip {self.idx + 1} / {len(self.clips)}  "
              f"({n_done} already done)")
        print(f"  Gloss:    {gloss}")
        print(f"  Signer:   {signer}")
        print(f"  Frames:   {T}")
        print()

        # Show video
        video_path = LOCAL_VIDEO_DIR / video_file
        data_url = video_to_data_url(video_path)
        display(HTML(f"""
        <video controls width="500" autoplay loop>
          <source src="{data_url}" type="video/mp4">
        </video>
        """))

        # Show speed profile
        kp = np.load(KEYPOINT_DIR / f"{Path(video_file).stem}.npz")
        speed = compute_speed(kp)

        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(speed, linewidth=2, color='steelblue')
        ax.fill_between(np.arange(len(speed)), 0, speed, alpha=0.2)
        ax.set_xlabel('Frame')
        ax.set_ylabel('Speed')
        ax.set_title(f"{gloss} — speed profile")
        ax.grid(alpha=0.3)
        # Add frame number gridlines every 10 frames
        ax.set_xticks(np.arange(0, len(speed), 10))
        plt.tight_layout()
        plt.show()

        # Input widgets
        print("\n--- Annotation ---")
        print("Enter stroke segments as comma-separated pairs: 'start-end, start-end'")
        print("Example: '5-25' for one stroke from frame 5 to 25")
        print("Example: '5-15, 22-30' for two strokes")
        print("Enter 'skip' to skip this clip, or 'undo' to redo the previous one.")

        self.input_box = widgets.Text(
            value='',
            placeholder='e.g. 5-25',
            description='Strokes:',
            layout=widgets.Layout(width='400px')
        )
        self.note_box = widgets.Text(
            value='',
            placeholder='Optional note (e.g. "unclear ending")',
            description='Note:',
            layout=widgets.Layout(width='400px')
        )
        submit_btn = widgets.Button(description='Save & next', button_style='success')
        skip_btn = widgets.Button(description='Skip', button_style='warning')
        prev_btn = widgets.Button(description='Previous', button_style='info')

        submit_btn.on_click(self._on_submit)
        skip_btn.on_click(self._on_skip)
        prev_btn.on_click(self._on_prev)

        display(self.input_box, self.note_box,
                widgets.HBox([submit_btn, skip_btn, prev_btn]))

    def _on_submit(self, b):
        text = self.input_box.value.strip()
        if not text:
            print("Empty input — please enter segments or click Skip")
            return
        try:
            segments = []
            for part in text.split(','):
                part = part.strip()
                if not part:
                    continue
                a, c = part.split('-')
                segments.append([int(a.strip()), int(c.strip())])
            # Sanity check
            row = self.clips.iloc[self.idx]
            T = row['n_frames']
            for s, e in segments:
                if s < 0 or e > T or s >= e:
                    print(f"Invalid segment {s}-{e} for clip of {T} frames. Re-enter.")
                    return
            save_annotation(row['video_file'], segments, self.note_box.value.strip())
            self.idx += 1
            self.show()
        except Exception as e:
            print(f"Parse error: {e}. Format is 'start-end' or 'start-end, start-end'")

    def _on_skip(self, b):
        # Mark as skipped (empty segments, note explains why)
        row = self.clips.iloc[self.idx]
        save_annotation(row['video_file'], [], "SKIPPED")
        self.idx += 1
        self.show()

    def _on_prev(self, b):
        if self.idx > 0:
            self.idx -= 1
            self.show()


# Start the annotator
ann = Annotator(to_annotate)
ann.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Change this number to view any clip (0 = RAZOR2, 1 = next, etc.)
CLIP_INDEX = 49

row = to_annotate.iloc[CLIP_INDEX]
video_file = row['video_file']
video_path = LOCAL_VIDEO_DIR / video_file
gloss = row['gloss']

cap = cv2.VideoCapture(str(video_path))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_indices = [int(i * (n_frames - 1) / 11) for i in range(12)]

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for ax, idx in zip(axes.flat, frame_indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {idx}", fontsize=11)
    ax.axis('off')
cap.release()
plt.suptitle(f"[{CLIP_INDEX}] {gloss}  —  {video_file}", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ANNOT_DIR = Path('/content/drive/MyDrive/CAPSTONE_ASL/annotations')

with open(ANNOT_DIR / 'annotations.json') as fp:
    annotations = json.load(fp)

to_annotate = pd.read_csv(ANNOT_DIR / 'clips_to_annotate.csv')

# Summary stats
n_total = len(annotations)
n_skipped = sum(1 for v in annotations.values() if v.get('note') == 'SKIPPED' or not v.get('segments'))
n_single = sum(1 for v in annotations.values() if len(v.get('segments', [])) == 1)
n_multi  = sum(1 for v in annotations.values() if len(v.get('segments', [])) >= 2)

print(f"Total annotated: {n_total}")
print(f"  Skipped:       {n_skipped}")
print(f"  Single-stroke: {n_single}")
print(f"  Multi-stroke:  {n_multi}")
print()

# Stroke duration and coverage stats
stroke_durations = []
stroke_coverage = []  # fraction of clip that is stroke
for video_file, ann in annotations.items():
    if not ann.get('segments'):
        continue
    row = to_annotate[to_annotate['video_file'] == video_file]
    if len(row) == 0:
        continue
    T = row.iloc[0]['n_frames']
    total_stroke = sum(e - s for s, e in ann['segments'])
    stroke_coverage.append(total_stroke / T)
    for s, e in ann['segments']:
        stroke_durations.append(e - s)

print(f"Stroke segment durations:")
print(f"  count:  {len(stroke_durations)}")
print(f"  min:    {min(stroke_durations)} frames")
print(f"  max:    {max(stroke_durations)} frames")
print(f"  median: {int(np.median(stroke_durations))} frames")
print(f"  mean:   {np.mean(stroke_durations):.1f} frames")
print()
print(f"Per-clip stroke coverage (fraction of frames marked as stroke):")
print(f"  min:    {min(stroke_coverage):.1%}")
print(f"  max:    {max(stroke_coverage):.1%}")
print(f"  median: {np.median(stroke_coverage):.1%}")
print(f"  mean:   {np.mean(stroke_coverage):.1%}")
print()

# Flag suspicious annotations
print("Suspicious cases worth re-checking:")
flagged = []
for video_file, ann in annotations.items():
    if not ann.get('segments'):
        continue
    row = to_annotate[to_annotate['video_file'] == video_file]
    if len(row) == 0:
        continue
    T = row.iloc[0]['n_frames']
    gloss = row.iloc[0]['gloss']
    total_stroke = sum(e - s for s, e in ann['segments'])
    coverage = total_stroke / T
    # Suspicious: very short total stroke (< 5 frames) or > 90% of clip is stroke
    if total_stroke < 5:
        flagged.append((video_file, gloss, T, ann['segments'], "very short stroke"))
    if coverage > 0.9:
        flagged.append((video_file, gloss, T, ann['segments'], "nearly entire clip"))
    # Stroke starts at frame 0 or ends at last frame — likely missed padding
    for s, e in ann['segments']:
        if s == 0:
            flagged.append((video_file, gloss, T, ann['segments'], "stroke starts at frame 0"))
        if e >= T - 1:
            flagged.append((video_file, gloss, T, ann['segments'], "stroke ends at last frame"))

if flagged:
    for video_file, gloss, T, segs, reason in flagged[:15]:
        print(f"  [{reason}]  {gloss}  T={T}  segs={segs}")
    if len(flagged) > 15:
        print(f"  ... and {len(flagged) - 15} more")
else:
    print("  None flagged.")

In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter1d, binary_dilation, binary_erosion

def compute_speed_for_pseudolabels(kp):
    """Same speed computation used during annotation."""
    pose = kp['pose']
    fps = float(kp['fps'])
    T = pose.shape[0]
    if T < 2:
        return np.zeros(T)
    lwrist = pose[:, 5, :2]
    rwrist = pose[:, 6, :2]
    lv = np.linalg.norm(np.diff(lwrist, axis=0, prepend=lwrist[:1]), axis=-1)
    rv = np.linalg.norm(np.diff(rwrist, axis=0, prepend=rwrist[:1]), axis=-1)
    speed = np.maximum(lv, rv) * (30.0 / fps)
    return uniform_filter1d(speed, size=3)


def generate_pseudolabels(speed, threshold_quantile=0.5, min_stroke_frames=5,
                          min_rest_frames=3):
    """Heuristic: stroke = frames where speed > clip-adaptive threshold.

    Parameters:
        threshold_quantile: percentile of speed values used as threshold.
            0.5 = median. Higher = stricter (smaller stroke regions).
        min_stroke_frames: stroke segments shorter than this are deleted.
        min_rest_frames: gaps shorter than this between strokes are merged.

    Returns: (T,) bool array where True = stroke.
    """
    if len(speed) < 2:
        return np.zeros(len(speed), dtype=bool)

    # Clip-adaptive threshold: use a quantile of the speed distribution
    # This handles clips with different overall motion levels
    threshold = np.quantile(speed, threshold_quantile)
    stroke = speed > threshold

    # Merge short rest gaps (close holes in stroke regions)
    if min_rest_frames > 0:
        stroke = binary_dilation(stroke, iterations=min_rest_frames)
        stroke = binary_erosion(stroke, iterations=min_rest_frames)

    # Remove short stroke segments (open small islands)
    if min_stroke_frames > 0:
        stroke = binary_erosion(stroke, iterations=min_stroke_frames // 2)
        stroke = binary_dilation(stroke, iterations=min_stroke_frames // 2)

    return stroke


def segments_to_mask(segments, T):
    """Convert annotation segment list to per-frame bool mask."""
    mask = np.zeros(T, dtype=bool)
    for s, e in segments:
        mask[s:e+1] = True
    return mask


def f1_score(pred, true):
    """Frame-level F1 between predicted and ground-truth stroke masks."""
    tp = (pred & true).sum()
    fp = (pred & ~true).sum()
    fn = (~pred & true).sum()
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)

print("Pseudo-label generator ready")

In [ ]:
# Load all hand annotations
import json
with open(ANNOT_DIR / 'annotations.json') as fp:
    annotations = json.load(fp)

# Load keypoints + ground truth for each annotated clip
annotated_clips = []
for video_file, ann in annotations.items():
    if not ann.get('segments'):
        continue
    kp_path = KEYPOINT_DIR / f"{Path(video_file).stem}.npz"
    if not kp_path.exists():
        continue
    kp = dict(np.load(kp_path))
    T = len(kp['pose'])
    gt = segments_to_mask(ann['segments'], T)
    annotated_clips.append({'kp': kp, 'gt': gt, 'name': video_file})

print(f"Loaded {len(annotated_clips)} clips with ground truth")

# Grid search over heuristic parameters
results = []
for q in [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7]:
    for min_stroke in [3, 5, 8, 12]:
        for min_rest in [0, 2, 3, 5]:
            f1s = []
            for c in annotated_clips:
                speed = compute_speed_for_pseudolabels(c['kp'])
                pred = generate_pseudolabels(speed, q, min_stroke, min_rest)
                f1s.append(f1_score(pred, c['gt']))
            results.append({
                'q': q,
                'min_stroke': min_stroke,
                'min_rest': min_rest,
                'mean_f1': np.mean(f1s),
                'median_f1': np.median(f1s),
            })

results_df = pd.DataFrame(results).sort_values('mean_f1', ascending=False)
print("\nTop 10 parameter combinations by mean F1:")
print(results_df.head(10).to_string(index=False))
print(f"\nBest: q={results_df.iloc[0]['q']}, min_stroke={int(results_df.iloc[0]['min_stroke'])}, "
      f"min_rest={int(results_df.iloc[0]['min_rest'])}, "
      f"mean F1 = {results_df.iloc[0]['mean_f1']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

best = results_df.iloc[0]
q, min_stroke, min_rest = best['q'], int(best['min_stroke']), int(best['min_rest'])

# Show 6 clips: best 3 (highest F1) and worst 3 (lowest F1) under these params
per_clip_f1 = []
for c in annotated_clips:
    speed = compute_speed_for_pseudolabels(c['kp'])
    pred = generate_pseudolabels(speed, q, min_stroke, min_rest)
    per_clip_f1.append((c, pred, speed, f1_score(pred, c['gt'])))

per_clip_f1.sort(key=lambda x: -x[3])
show = per_clip_f1[:3] + per_clip_f1[-3:]

fig, axes = plt.subplots(len(show), 1, figsize=(14, 2.5 * len(show)))
for ax, (c, pred, speed, f1) in zip(axes, show):
    t = np.arange(len(speed))
    ax.plot(t, speed, color='steelblue', linewidth=1.5, label='speed')
    # Ground truth (red shaded)
    ax.fill_between(t, 0, speed.max(), where=c['gt'], alpha=0.2, color='red', label='hand-annotated stroke')
    # Heuristic prediction (blue outline)
    ax.fill_between(t, 0, speed.max() * 0.5, where=pred, alpha=0.25, color='green', label='heuristic stroke')
    ax.set_title(f"{c['name'][:50]}  F1={f1:.2f}", fontsize=10)
    ax.set_xlabel('Frame')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def generate_pseudolabels_v2(speed, abs_threshold=0.05, min_stroke_frames=5, min_rest_frames=3):
    """Absolute threshold version. Speed values are already FPS-normalized."""
    stroke = speed > abs_threshold

    if min_rest_frames > 0:
        stroke = binary_dilation(stroke, iterations=min_rest_frames)
        stroke = binary_erosion(stroke, iterations=min_rest_frames)
    if min_stroke_frames > 0:
        stroke = binary_erosion(stroke, iterations=min_stroke_frames // 2)
        stroke = binary_dilation(stroke, iterations=min_stroke_frames // 2)
    return stroke


# Grid search over absolute threshold
results_v2 = []
for thr in [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.1, 0.12, 0.15]:
    for min_stroke in [3, 5, 8, 12]:
        for min_rest in [0, 2, 3, 5]:
            f1s = []
            for c in annotated_clips:
                speed = compute_speed_for_pseudolabels(c['kp'])
                pred = generate_pseudolabels_v2(speed, thr, min_stroke, min_rest)
                f1s.append(f1_score(pred, c['gt']))
            results_v2.append({
                'threshold': thr,
                'min_stroke': min_stroke,
                'min_rest': min_rest,
                'mean_f1': np.mean(f1s),
                'median_f1': np.median(f1s),
                'min_f1':  np.min(f1s),
            })

results_v2_df = pd.DataFrame(results_v2).sort_values('mean_f1', ascending=False)
print("Top 10 (absolute threshold):")
print(results_v2_df.head(10).to_string(index=False))

In [ ]:
best = results_v2_df.iloc[0]
thr, min_stroke, min_rest = best['threshold'], int(best['min_stroke']), int(best['min_rest'])
print(f"Using: threshold={thr}, min_stroke={min_stroke}, min_rest={min_rest}")

per_clip = []
for c in annotated_clips:
    speed = compute_speed_for_pseudolabels(c['kp'])
    pred = generate_pseudolabels_v2(speed, thr, min_stroke, min_rest)
    per_clip.append((c, pred, speed, f1_score(pred, c['gt'])))

per_clip.sort(key=lambda x: -x[3])
show = per_clip[:3] + per_clip[-3:]

fig, axes = plt.subplots(len(show), 1, figsize=(14, 2.5 * len(show)))
for ax, (c, pred, speed, f1) in zip(axes, show):
    t = np.arange(len(speed))
    ax.plot(t, speed, color='steelblue', linewidth=1.5)
    ax.fill_between(t, 0, speed.max(), where=c['gt'], alpha=0.2, color='red', label='hand-annotated')
    ax.fill_between(t, 0, speed.max() * 0.5, where=pred, alpha=0.25, color='green', label='heuristic')
    ax.set_title(f"{c['name'][:50]}  F1={f1:.2f}", fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

flagged = []
for c in annotated_clips:
    speed = compute_speed_for_pseudolabels(c['kp'])
    gt = c['gt']
    if not gt.any():
        continue

    # Inside the labeled stroke region, what fraction of frames are below
    # 25% of peak speed (basically "flat")?
    stroke_indices = np.where(gt)[0]
    stroke_speed = speed[stroke_indices]
    flat_threshold = 0.25 * speed.max()
    frac_flat = (stroke_speed < flat_threshold).mean()

    if frac_flat > 0.4:
        flagged.append({
            'name': c['name'],
            'flat_fraction': frac_flat,
            'stroke_len': int(gt.sum()),
        })

flagged_df = pd.DataFrame(flagged).sort_values('flat_fraction', ascending=False)
print(f"Clips where stroke includes >40% flat frames: {len(flagged_df)} / {len(annotated_clips)}")
print()
if len(flagged_df) > 0:
    print(flagged_df.to_string(index=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/content/drive/MyDrive/CAPSTONE_ASL')
KEYPOINT_DIR = PROJECT_ROOT / 'keypoints'
SUBSET_DIR = PROJECT_ROOT / 'subset_top100'

sub_train = pd.read_csv(SUBSET_DIR / 'train.csv')
sub_val = pd.read_csv(SUBSET_DIR / 'val.csv')
sub_test = pd.read_csv(SUBSET_DIR / 'test.csv')

total_expected = len(sub_train) + len(sub_val) + len(sub_test)
total_done = len(list(KEYPOINT_DIR.glob('*.npz')))

print("Expected:", total_expected)
print("Keypoints done:", total_done)
print("Missing:", total_expected - total_done)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/CAPSTONE_ASL')
KEYPOINT_DIR = PROJECT_ROOT / 'keypoints'
SUBSET_DIR   = PROJECT_ROOT / 'subset_top100'

sub_train = pd.read_csv(SUBSET_DIR / 'train.csv')
sub_val   = pd.read_csv(SUBSET_DIR / 'val.csv')
sub_test  = pd.read_csv(SUBSET_DIR / 'test.csv')

# Build label mapping (gloss → integer)
all_glosses = sorted(sub_train['Gloss'].unique())
GLOSS_TO_IDX = {g: i for i, g in enumerate(all_glosses)}
IDX_TO_GLOSS = {i: g for g, i in GLOSS_TO_IDX.items()}
NUM_CLASSES = len(all_glosses)
print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(sub_train)}  Val: {len(sub_val)}  Test: {len(sub_test)}")

In [ ]:
from scipy.ndimage import uniform_filter1d

# MediaPipe pose subset we kept: nose, l_shoulder, r_shoulder, l_elbow, r_elbow, l_wrist, r_wrist
P_NOSE, P_LSH, P_RSH, P_LEL, P_REL, P_LWR, P_RWR = 0, 1, 2, 3, 4, 5, 6
TARGET_FPS = 30.0


def linear_interp_missing(features, mask):
    """Fill False-mask frames by linearly interpolating from neighbors."""
    T = len(mask)
    if mask.all() or not mask.any():
        return features
    valid = np.where(mask)[0]
    result = features.copy()
    # Forward-fill before first valid
    if valid[0] > 0:
        result[:valid[0]] = features[valid[0]]
    # Backward-fill after last valid
    if valid[-1] < T - 1:
        result[valid[-1]+1:] = features[valid[-1]]
    # Linear interp between valid frames
    for i in range(len(valid) - 1):
        a, b = valid[i], valid[i+1]
        if b - a > 1:
            for t in range(a+1, b):
                alpha = (t - a) / (b - a)
                result[t] = (1 - alpha) * features[a] + alpha * features[b]
    return result


def normalize_clip(kp_dict):
    """Normalize one clip's keypoints to body-relative coordinates.

    Returns (T, 49, 3) float32 with:
        - Origin at shoulder midpoint
        - Scaled by shoulder distance
        - Missing detections interpolated
    """
    pose = kp_dict['pose']        # (T, 7, 3)
    lh   = kp_dict['lh']          # (T, 21, 3)
    rh   = kp_dict['rh']          # (T, 21, 3)
    pose_mask = kp_dict['pose_mask']
    lh_mask = kp_dict['lh_mask']
    rh_mask = kp_dict['rh_mask']

    # Interpolate missing detections (pose is ~100%, hands are often missing)
    pose = linear_interp_missing(pose, pose_mask)
    lh = linear_interp_missing(lh, lh_mask)
    rh = linear_interp_missing(rh, rh_mask)

    # Body-relative normalization
    mid = (pose[:, P_LSH] + pose[:, P_RSH]) / 2     # (T, 3)
    shoulder_dist = np.linalg.norm(
        pose[:, P_LSH, :2] - pose[:, P_RSH, :2], axis=-1
    )
    scale = np.median(shoulder_dist[shoulder_dist > 1e-6])
    if scale < 1e-6:
        scale = 1.0

    pose_n = (pose - mid[:, None, :]) / scale
    lh_n   = (lh   - mid[:, None, :]) / scale
    rh_n   = (rh   - mid[:, None, :]) / scale

    # Concatenate into a single (T, 49, 3) tensor
    return np.concatenate([pose_n, lh_n, rh_n], axis=1).astype(np.float32)


def load_clip_features(video_file):
    """Load keypoints and return normalized (T, 49, 3) array."""
    kp_path = KEYPOINT_DIR / f"{Path(video_file).stem}.npz"
    kp = dict(np.load(kp_path))
    return normalize_clip(kp), float(kp['fps'])


# Sanity check on one clip
sample_file = sub_train['Video file'].iloc[0]
features, fps = load_clip_features(sample_file)
print(f"Sample clip: {sample_file}")
print(f"  features shape: {features.shape}  (T, 49 joints, 3 dims)")
print(f"  fps: {fps}")
print(f"  position range: x=[{features[..., 0].min():.2f}, {features[..., 0].max():.2f}], "
      f"y=[{features[..., 1].min():.2f}, {features[..., 1].max():.2f}]")

In [ ]:
from torch.utils.data import Dataset, DataLoader

MAX_FRAMES = 150   # pad/truncate target. Covers 95th-percentile clip length.
N_JOINTS = 49
N_DIMS = 3


class ASLCitizenKeypointDataset(Dataset):
    """Loads keypoint features for ASL Citizen subset clips.

    Each item is a dict with:
        features: (MAX_FRAMES, 49, 3) float32, zero-padded
        length:   int, true number of frames (before padding)
        label:    int, class index
        video_file: str, original filename (for debugging)
    """
    def __init__(self, df, max_frames=MAX_FRAMES):
        self.df = df.reset_index(drop=True)
        self.max_frames = max_frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_file = row['Video file']
        gloss = row['Gloss']
        label = GLOSS_TO_IDX[gloss]

        features, _ = load_clip_features(video_file)
        T = features.shape[0]

        if T >= self.max_frames:
            # Truncate to max_frames
            features = features[:self.max_frames]
            true_length = self.max_frames
        else:
            # Zero-pad to max_frames
            pad = np.zeros((self.max_frames - T, N_JOINTS, N_DIMS), dtype=np.float32)
            features = np.concatenate([features, pad], axis=0)
            true_length = T

        return {
            'features': torch.from_numpy(features),         # (MAX_FRAMES, 49, 3)
            'length':   torch.tensor(true_length, dtype=torch.long),
            'label':    torch.tensor(label, dtype=torch.long),
            'video_file': video_file,
        }


# Build datasets
train_ds = ASLCitizenKeypointDataset(sub_train)
val_ds   = ASLCitizenKeypointDataset(sub_val)
test_ds  = ASLCitizenKeypointDataset(sub_test)

print(f"Datasets built:")
print(f"  Train: {len(train_ds)}")
print(f"  Val:   {len(val_ds)}")
print(f"  Test:  {len(test_ds)}")

# Sanity check: pull one item
sample = train_ds[0]
print(f"\nSample item:")
print(f"  features: {sample['features'].shape}  dtype={sample['features'].dtype}")
print(f"  length:   {sample['length'].item()}")
print(f"  label:    {sample['label'].item()}  ({IDX_TO_GLOSS[sample['label'].item()]})")
print(f"  file:     {sample['video_file']}")

In [ ]:
def collate_fn(batch):
    """Stack a list of dataset items into a batched dict."""
    return {
        'features':   torch.stack([b['features'] for b in batch]),
        'length':     torch.stack([b['length']   for b in batch]),
        'label':      torch.stack([b['label']    for b in batch]),
        'video_file': [b['video_file'] for b in batch],
    }


train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

# Get one batch
import time
t0 = time.time()
batch = next(iter(train_loader))
dt = time.time() - t0

print(f"One batch loaded in {dt:.2f}s")
print(f"  features: {batch['features'].shape}  ({batch['features'].dtype})")
print(f"  length:   {batch['length'].shape}  min={batch['length'].min().item()} max={batch['length'].max().item()}")
print(f"  label:    {batch['label'].shape}")
print(f"  labels in batch: {sorted(set(batch['label'].tolist()))[:10]}...")

In [ ]:
import shutil
from pathlib import Path
from tqdm.auto import tqdm

LOCAL_KEYPOINT_DIR = Path('/content/keypoints_local')
LOCAL_KEYPOINT_DIR.mkdir(exist_ok=True)

# Copy all keypoint files from Drive to local
drive_files = list(KEYPOINT_DIR.glob('*.npz'))
print(f"Caching {len(drive_files)} keypoint files to local disk...")

copied = 0
skipped = 0
for src in tqdm(drive_files):
    dst = LOCAL_KEYPOINT_DIR / src.name
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        skipped += 1
        continue
    shutil.copy(src, dst)
    copied += 1

local_size_mb = sum(f.stat().st_size for f in LOCAL_KEYPOINT_DIR.glob('*.npz')) / 1e6
print(f"\nCopied: {copied}  Skipped (already cached): {skipped}")
print(f"Local cache: {len(list(LOCAL_KEYPOINT_DIR.glob('*.npz')))} files, {local_size_mb:.1f} MB")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
from pathlib import Path
LOCAL_KEYPOINT_DIR = Path('/content/keypoints_local')
n = len(list(LOCAL_KEYPOINT_DIR.glob('*.npz'))) if LOCAL_KEYPOINT_DIR.exists() else 0
print(f"Local cache: {n} files")

In [ ]:
# Sanity check before retrying Cell 7
print(f"NUM_CLASSES: {NUM_CLASSES}")
print(f"train_loader exists: {'train_loader' in dir()}")
print(f"Local cache: {len(list(Path('/content/keypoints_local').glob('*.npz')))} files")

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
from torch.utils.data import Dataset, DataLoader

MAX_FRAMES = 150
N_JOINTS = 49
N_DIMS = 3


class ASLCitizenKeypointDataset(Dataset):
    def __init__(self, df, max_frames=MAX_FRAMES):
        self.df = df.reset_index(drop=True)
        self.max_frames = max_frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_file = row['Video file']
        gloss = row['Gloss']
        label = GLOSS_TO_IDX[gloss]

        features, _ = load_clip_features(video_file)
        T = features.shape[0]

        if T >= self.max_frames:
            features = features[:self.max_frames]
            true_length = self.max_frames
        else:
            pad = np.zeros((self.max_frames - T, N_JOINTS, N_DIMS), dtype=np.float32)
            features = np.concatenate([features, pad], axis=0)
            true_length = T

        return {
            'features': torch.from_numpy(features),
            'length':   torch.tensor(true_length, dtype=torch.long),
            'label':    torch.tensor(label, dtype=torch.long),
            'video_file': video_file,
        }


train_ds = ASLCitizenKeypointDataset(sub_train)
val_ds   = ASLCitizenKeypointDataset(sub_val)
test_ds  = ASLCitizenKeypointDataset(sub_test)
print(f"Datasets: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

In [ ]:
from scipy.ndimage import uniform_filter1d

P_NOSE, P_LSH, P_RSH, P_LEL, P_REL, P_LWR, P_RWR = 0, 1, 2, 3, 4, 5, 6
TARGET_FPS = 30.0


def linear_interp_missing(features, mask):
    T = len(mask)
    if mask.all() or not mask.any():
        return features
    valid = np.where(mask)[0]
    result = features.copy()
    if valid[0] > 0:
        result[:valid[0]] = features[valid[0]]
    if valid[-1] < T - 1:
        result[valid[-1]+1:] = features[valid[-1]]
    for i in range(len(valid) - 1):
        a, b = valid[i], valid[i+1]
        if b - a > 1:
            for t in range(a+1, b):
                alpha = (t - a) / (b - a)
                result[t] = (1 - alpha) * features[a] + alpha * features[b]
    return result


def normalize_clip(kp_dict):
    pose = kp_dict['pose']
    lh   = kp_dict['lh']
    rh   = kp_dict['rh']
    pose_mask = kp_dict['pose_mask']
    lh_mask   = kp_dict['lh_mask']
    rh_mask   = kp_dict['rh_mask']

    pose = linear_interp_missing(pose, pose_mask)
    lh   = linear_interp_missing(lh, lh_mask)
    rh   = linear_interp_missing(rh, rh_mask)

    mid = (pose[:, P_LSH] + pose[:, P_RSH]) / 2
    shoulder_dist = np.linalg.norm(
        pose[:, P_LSH, :2] - pose[:, P_RSH, :2], axis=-1
    )
    scale = np.median(shoulder_dist[shoulder_dist > 1e-6])
    if scale < 1e-6:
        scale = 1.0

    pose_n = (pose - mid[:, None, :]) / scale
    lh_n   = (lh   - mid[:, None, :]) / scale
    rh_n   = (rh   - mid[:, None, :]) / scale

    return np.concatenate([pose_n, lh_n, rh_n], axis=1).astype(np.float32)


print("normalize_clip and linear_interp_missing defined")

In [ ]:
need = ['KEYPOINT_DIR', 'GLOSS_TO_IDX', 'normalize_clip', 'linear_interp_missing', 'np', 'torch']
for name in need:
    print(f"  {name}: {'OK' if name in dir() else 'MISSING'}")

In [ ]:
LOCAL_KEYPOINT_DIR = Path('/content/keypoints_local')

def load_clip_features(video_file):
    kp_path = LOCAL_KEYPOINT_DIR / f"{Path(video_file).stem}.npz"
    if not kp_path.exists():
        kp_path = KEYPOINT_DIR / f"{Path(video_file).stem}.npz"
    kp = dict(np.load(kp_path))
    return normalize_clip(kp), float(kp['fps'])


def collate_fn(batch):
    return {
        'features':   torch.stack([b['features'] for b in batch]),
        'length':     torch.stack([b['length']   for b in batch]),
        'label':      torch.stack([b['label']    for b in batch]),
        'video_file': [b['video_file'] for b in batch],
    }


train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

# Verify
batch = next(iter(train_loader))
print(f"Batch features: {batch['features'].shape}")
print(f"Batch length range: min={batch['length'].min().item()} max={batch['length'].max().item()}")

In [ ]:
print(f"train_loader exists: {'train_loader' in dir()}")
print(f"val_loader exists:   {'val_loader' in dir()}")
# Quick batch test
batch = next(iter(train_loader))
print(f"Batch features: {batch['features'].shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TCNBlock(nn.Module):
    """One dilated 1D conv block with residual connection."""
    def __init__(self, in_ch, out_ch, kernel_size=5, dilation=1, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2  # 'same' padding
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              padding=padding, dilation=dilation)
        self.bn = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)
        # Residual: project input if channels don't match
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x: (B, C, T)
        out = self.conv(x)
        out = self.bn(out)
        out = F.relu(out)
        out = self.dropout(out)
        return out + self.residual(x)


class KeypointTCN(nn.Module):
    """Small dilated TCN classifier for keypoint sequences.

    Input:  (B, T, J, D) keypoint positions  -- e.g. (B, 150, 49, 3)
    Output: (B, num_classes) logits
    """
    def __init__(self, n_joints=49, n_dims=3, num_classes=100,
                 hidden=128, n_blocks=4, kernel_size=5, dropout=0.3):
        super().__init__()
        in_features = n_joints * n_dims  # 147

        # Initial projection from raw features to hidden dim
        self.input_proj = nn.Conv1d(in_features, hidden, 1)

        # Stack of dilated TCN blocks: dilation doubles per block
        # With kernel=5 and 4 blocks at dilations [1,2,4,8], receptive field ~80 frames
        self.blocks = nn.ModuleList([
            TCNBlock(hidden, hidden, kernel_size=kernel_size,
                     dilation=2**i, dropout=dropout)
            for i in range(n_blocks)
        ])

        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, features, length):
        """
        features: (B, T, J, D) — keypoint positions
        length:   (B,)         — true number of frames before padding
        Returns:  (B, num_classes) logits
        """
        B, T, J, D = features.shape

        # (B, T, J, D) -> (B, T, J*D) -> (B, J*D, T) for Conv1d
        x = features.reshape(B, T, J * D).transpose(1, 2)  # (B, 147, T)

        # Apply blocks
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        # x: (B, hidden, T)

        # Length-aware pooling: avg over valid frames only
        # Build a mask: (B, 1, T) with 1.0 for valid frames, 0.0 for padding
        idx = torch.arange(T, device=features.device).unsqueeze(0)  # (1, T)
        mask = (idx < length.unsqueeze(1)).float().unsqueeze(1)     # (B, 1, T)
        x_masked = x * mask
        pooled = x_masked.sum(dim=2) / mask.sum(dim=2).clamp(min=1.0)
        # pooled: (B, hidden)

        return self.classifier(pooled)


# Build and check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KeypointTCN(num_classes=NUM_CLASSES).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Model parameters: {n_params:,}")
print(model)

In [ ]:
batch = next(iter(train_loader))
features = batch['features'].to(device)
length   = batch['length'].to(device)
label    = batch['label'].to(device)

with torch.no_grad():
    logits = model(features, length)

print(f"Input:  features {features.shape}, length {length.shape}")
print(f"Output: logits {logits.shape}  expected ({features.shape[0]}, {NUM_CLASSES})")
print(f"Output range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")

loss_fn = nn.CrossEntropyLoss()
initial_loss = loss_fn(logits, label)
expected_random_loss = np.log(NUM_CLASSES)
print(f"Initial loss: {initial_loss.item():.3f}  (random would be ~{expected_random_loss:.3f})")

In [ ]:
import time

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    n_correct_top1 = 0
    n_correct_top5 = 0
    n_total = 0
    loss_fn = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            features = batch['features'].to(device)
            length   = batch['length'].to(device)
            label    = batch['label'].to(device)
            logits = model(features, length)
            loss = loss_fn(logits, label)
            total_loss += loss.item() * features.size(0)
            top5 = logits.topk(5, dim=1).indices
            n_correct_top1 += (top5[:, 0] == label).sum().item()
            n_correct_top5 += (top5 == label.unsqueeze(1)).any(dim=1).sum().item()
            n_total += features.size(0)
    return {'loss': total_loss/n_total, 'top1': n_correct_top1/n_total, 'top5': n_correct_top5/n_total}


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    n_correct_top1 = 0
    n_correct_top5 = 0
    n_total = 0
    loss_fn = nn.CrossEntropyLoss()
    for batch in loader:
        features = batch['features'].to(device)
        length   = batch['length'].to(device)
        label    = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(features, length)
        loss = loss_fn(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * features.size(0)
        top5 = logits.topk(5, dim=1).indices
        n_correct_top1 += (top5[:, 0] == label).sum().item()
        n_correct_top5 += (top5 == label.unsqueeze(1)).any(dim=1).sum().item()
        n_total += features.size(0)
    return {'loss': total_loss/n_total, 'top1': n_correct_top1/n_total, 'top5': n_correct_top5/n_total}


def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, device=device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = []
    best_val_top1 = 0.0
    best_state = None
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_stats = train_one_epoch(model, train_loader, optimizer, device)
        val_stats   = evaluate(model, val_loader, device)
        scheduler.step()
        dt = time.time() - t0
        history.append({
            'epoch': epoch,
            'train_loss': train_stats['loss'], 'train_top1': train_stats['top1'], 'train_top5': train_stats['top5'],
            'val_loss':   val_stats['loss'],   'val_top1':   val_stats['top1'],   'val_top5':   val_stats['top5'],
            'lr': scheduler.get_last_lr()[0],
        })
        marker = ''
        if val_stats['top1'] > best_val_top1:
            best_val_top1 = val_stats['top1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            marker = '  *best'
        print(f"Epoch {epoch:2d}/{epochs}  "
              f"train: loss={train_stats['loss']:.3f} top1={train_stats['top1']:.1%} top5={train_stats['top5']:.1%}  |  "
              f"val: loss={val_stats['loss']:.3f} top1={val_stats['top1']:.1%} top5={val_stats['top5']:.1%}  "
              f"({dt:.1f}s){marker}")
    print(f"\nBest val top-1: {best_val_top1:.1%}")
    return history, best_state

print("Training functions ready")

In [ ]:
history, best_state = train_model(model, train_loader, val_loader, epochs=30, lr=1e-3)

In [ ]:
import json
from pathlib import Path

CKPT_DIR = Path('/content/drive/MyDrive/CAPSTONE_ASL/checkpoints')
CKPT_DIR.mkdir(exist_ok=True)

# Save best model state (the one with 59.7% val top-1)
torch.save({
    'model_state_dict': best_state,
    'num_classes': NUM_CLASSES,
    'config': {
        'hidden': 128,
        'n_blocks': 4,
        'kernel_size': 5,
        'dropout': 0.3,
    },
    'val_top1': 0.597,
    'val_top5': 0.853,
}, CKPT_DIR / 'baseline_tcn_best.pt')

# Save training history
with open(CKPT_DIR / 'baseline_tcn_history.json', 'w') as fp:
    json.dump(history, fp, indent=2)

# Save the gloss mapping so we can interpret predictions later
with open(CKPT_DIR / 'gloss_mapping.json', 'w') as fp:
    json.dump({'gloss_to_idx': GLOSS_TO_IDX, 'idx_to_gloss': {str(k): v for k, v in IDX_TO_GLOSS.items()}}, fp, indent=2)

# Flush Drive to ensure it actually persists
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

# Verify
print(f"Saved files:")
for f in CKPT_DIR.iterdir():
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

In [ ]:
# Load best weights into model
model.load_state_dict(best_state)
model.to(device)

test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)
test_stats = evaluate(model, test_loader, device)
print(f"TEST set results (signer-independent, 1286 clips):")
print(f"  Loss:   {test_stats['loss']:.3f}")
print(f"  Top-1:  {test_stats['top1']:.1%}")
print(f"  Top-5:  {test_stats['top5']:.1%}")

In [ ]:
# ============================================================
# PHASE 2A — FIXED SETUP + KEYPOINT MANIFEST / DATA AUDIT
# Run this cell only, then paste the printed output back.
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import re
import numpy as np
import pandas as pd

# ----------------------------
# 1. Project paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
SUBSET_DIR = BASE_DIR / "subset_top100"
KEYPOINT_DIR = BASE_DIR / "keypoints"
MANIFEST_DIR = BASE_DIR / "manifests"

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

KEYPOINT_MANIFEST_PATH = MANIFEST_DIR / "keypoint_manifest.csv"

print("=" * 80)
print("PHASE 2A — FIXED KEYPOINT MANIFEST / DATA AUDIT")
print("=" * 80)
print(f"BASE_DIR:      {BASE_DIR}")
print(f"SUBSET_DIR:    {SUBSET_DIR}")
print(f"KEYPOINT_DIR:  {KEYPOINT_DIR}")
print(f"MANIFEST_DIR:  {MANIFEST_DIR}")
print()

# ----------------------------
# 2. Folder checks
# ----------------------------

for name, path in {
    "BASE_DIR": BASE_DIR,
    "SUBSET_DIR": SUBSET_DIR,
    "KEYPOINT_DIR": KEYPOINT_DIR,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")

print("Required folders exist.")
print()

# ----------------------------
# 3. Load split CSVs
# ----------------------------

csv_paths = {
    "train": SUBSET_DIR / "train.csv",
    "val": SUBSET_DIR / "val.csv",
    "test": SUBSET_DIR / "test.csv",
}

dfs = []

for split, csv_path in csv_paths.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    df["split"] = split
    df["source_csv"] = str(csv_path)
    dfs.append(df)

    print(f"Loaded {split}.csv: {len(df)} rows")
    print(f"Columns: {list(df.columns)}")
    print()

meta_df = pd.concat(dfs, ignore_index=True)

print("-" * 80)
print("Combined dataset rows:")
print(meta_df["split"].value_counts())
print(f"TOTAL rows: {len(meta_df)}")
print("-" * 80)
print()

# ----------------------------
# 4. Robust column detection
# ----------------------------

def normalize_col_name(c):
    """
    Makes column names comparable:
    'Video file' -> 'videofile'
    'video_file' -> 'videofile'
    'VIDEO-FILE' -> 'videofile'
    """
    return re.sub(r"[^a-z0-9]", "", str(c).lower())

def find_column(df, candidates):
    normalized_map = {normalize_col_name(c): c for c in df.columns}

    for cand in candidates:
        key = normalize_col_name(cand)
        if key in normalized_map:
            return normalized_map[key]

    return None

video_col = find_column(
    meta_df,
    [
        "Video file",
        "video_file",
        "video file",
        "video",
        "filename",
        "file",
        "path",
        "video_path",
        "clip",
        "clip_name",
        "id",
    ],
)

gloss_col = find_column(
    meta_df,
    [
        "Gloss",
        "gloss",
        "label",
        "sign",
        "word",
        "class",
        "class_name",
        "clean_text",
        "text",
    ],
)

participant_col = find_column(
    meta_df,
    [
        "Participant ID",
        "participant_id",
        "participant",
        "signer",
        "signer_id",
    ],
)

asl_lex_col = find_column(
    meta_df,
    [
        "ASL-LEX Code",
        "asl_lex_code",
        "asl lex code",
        "asllex",
    ],
)

print("Detected columns:")
print(f"video_col:       {video_col}")
print(f"gloss_col:       {gloss_col}")
print(f"participant_col: {participant_col}")
print(f"asl_lex_col:     {asl_lex_col}")
print()

if video_col is None:
    print("Available columns:")
    print(list(meta_df.columns))
    raise ValueError("Could not detect video filename column.")

if gloss_col is None:
    print("Available columns:")
    print(list(meta_df.columns))
    raise ValueError("Could not detect gloss/label column.")

# Create stable class IDs from gloss names.
glosses_sorted = sorted(meta_df[gloss_col].astype(str).unique())
gloss_to_label_id = {g: i for i, g in enumerate(glosses_sorted)}

print(f"Unique gloss/classes: {len(glosses_sorted)}")
print("First 10 glosses:", glosses_sorted[:10])
print()

# ----------------------------
# 5. Helper functions
# ----------------------------

def clean_stem(x):
    """
    Converts video filename/path into matching stem.
    Example:
    'abc.mp4' -> 'abc'
    '/folder/abc.mp4' -> 'abc'
    """
    if pd.isna(x):
        return ""

    s = str(x).strip()
    s = s.replace("\\", "/")
    s = s.split("/")[-1]
    s = re.sub(r"\.(mp4|mov|avi|mkv|webm|npz|npy)$", "", s, flags=re.IGNORECASE)
    return s

def safe_shape(arr):
    try:
        return tuple(arr.shape)
    except Exception:
        return None

def arr_has_nan_inf(arr):
    try:
        if not np.issubdtype(arr.dtype, np.number):
            return False
        return bool(np.isnan(arr).any() or np.isinf(arr).any())
    except Exception:
        return True

def get_frame_count_from_npz(npz):
    preferred_keys = [
        "pose", "lh", "rh",
        "left_hand", "right_hand",
        "pose_landmarks",
        "left_hand_landmarks",
        "right_hand_landmarks",
        "keypoints", "landmarks", "data",
        "arr_0", "x", "kp", "skeleton", "joints",
    ]

    for k in preferred_keys:
        if k in npz.files:
            arr = npz[k]
            if hasattr(arr, "shape") and len(arr.shape) >= 1:
                return int(arr.shape[0])

    for k in npz.files:
        arr = npz[k]
        if hasattr(arr, "shape") and len(arr.shape) >= 1:
            return int(arr.shape[0])

    return np.nan

def get_mask_rate(npz, key):
    if key not in npz.files:
        return np.nan

    try:
        m = np.asarray(npz[key])
        if m.size == 0:
            return np.nan
        return float(np.mean(m > 0))
    except Exception:
        return np.nan

def inspect_npz(npz_path):
    info = {
        "keypoint_path": str(npz_path),
        "exists": npz_path.exists(),
        "status": "unknown",
        "keys": "",
        "n_frames": np.nan,
        "fps": np.nan,
        "pose_shape": "",
        "lh_shape": "",
        "rh_shape": "",
        "pose_mask_rate": np.nan,
        "lh_mask_rate": np.nan,
        "rh_mask_rate": np.nan,
        "either_hand_mask_rate": np.nan,
        "has_nan_inf": False,
        "file_size_kb": np.nan,
        "error": "",
    }

    if not npz_path.exists():
        info["status"] = "missing"
        return info

    try:
        info["file_size_kb"] = round(npz_path.stat().st_size / 1024, 2)

        with np.load(npz_path, allow_pickle=False) as npz:
            keys = list(npz.files)
            info["keys"] = "|".join(keys)
            info["n_frames"] = get_frame_count_from_npz(npz)

            if "fps" in keys:
                try:
                    info["fps"] = float(np.asarray(npz["fps"]).item())
                except Exception:
                    info["fps"] = np.nan

            if "pose" in keys:
                info["pose_shape"] = str(safe_shape(npz["pose"]))
            if "lh" in keys:
                info["lh_shape"] = str(safe_shape(npz["lh"]))
            if "rh" in keys:
                info["rh_shape"] = str(safe_shape(npz["rh"]))

            info["pose_mask_rate"] = get_mask_rate(npz, "pose_mask")
            info["lh_mask_rate"] = get_mask_rate(npz, "lh_mask")
            info["rh_mask_rate"] = get_mask_rate(npz, "rh_mask")

            lh_rate = info["lh_mask_rate"]
            rh_rate = info["rh_mask_rate"]
            if not np.isnan(lh_rate) and not np.isnan(rh_rate):
                info["either_hand_mask_rate"] = max(lh_rate, rh_rate)

            any_bad = False
            for k in keys:
                arr = npz[k]
                if arr_has_nan_inf(arr):
                    any_bad = True
                    break

            info["has_nan_inf"] = bool(any_bad)

            if pd.isna(info["n_frames"]):
                info["status"] = "bad_no_frame_count"
            elif int(info["n_frames"]) < 5:
                info["status"] = "bad_too_short"
            elif info["has_nan_inf"]:
                info["status"] = "bad_nan_inf"
            else:
                info["status"] = "ok"

    except Exception as e:
        info["status"] = "load_error"
        info["error"] = repr(e)

    return info

# ----------------------------
# 6. Index keypoint files
# ----------------------------

npz_files = sorted(KEYPOINT_DIR.rglob("*.npz"))

print(f"Found .npz files under KEYPOINT_DIR: {len(npz_files)}")

if len(npz_files) == 0:
    raise FileNotFoundError(f"No .npz files found under {KEYPOINT_DIR}")

stem_to_npz = {}
lower_stem_to_npz = {}
duplicate_stems = []

for p in npz_files:
    stem = p.stem
    lower = stem.lower()

    if stem in stem_to_npz:
        duplicate_stems.append(stem)
    else:
        stem_to_npz[stem] = p

    if lower not in lower_stem_to_npz:
        lower_stem_to_npz[lower] = p

print(f"Unique .npz stems indexed: {len(stem_to_npz)}")
print(f"Duplicate stems: {len(set(duplicate_stems))}")
print()

# ----------------------------
# 7. Match metadata rows to keypoints
# ----------------------------

records = []

for idx, row in meta_df.iterrows():
    video_raw = row[video_col]
    video_stem = clean_stem(video_raw)

    kp_path = None

    # Exact stem match.
    if video_stem in stem_to_npz:
        kp_path = stem_to_npz[video_stem]

    # Case-insensitive match.
    elif video_stem.lower() in lower_stem_to_npz:
        kp_path = lower_stem_to_npz[video_stem.lower()]

    # Fallback: contains match.
    else:
        matches = [
            p for p in npz_files
            if video_stem and video_stem.lower() in p.stem.lower()
        ]
        if len(matches) == 1:
            kp_path = matches[0]

    gloss = str(row[gloss_col])

    base = {
        "row_id": idx,
        "split": row["split"],
        "source_csv": row["source_csv"],
        "participant_id": str(row[participant_col]) if participant_col else "",
        "asl_lex_code": str(row[asl_lex_col]) if asl_lex_col else "",
        "video_value_raw": str(video_raw),
        "video_stem": video_stem,
        "gloss": gloss,
        "label_id": gloss_to_label_id[gloss],
    }

    if kp_path is None:
        inspect_info = {
            "keypoint_path": "",
            "exists": False,
            "status": "missing",
            "keys": "",
            "n_frames": np.nan,
            "fps": np.nan,
            "pose_shape": "",
            "lh_shape": "",
            "rh_shape": "",
            "pose_mask_rate": np.nan,
            "lh_mask_rate": np.nan,
            "rh_mask_rate": np.nan,
            "either_hand_mask_rate": np.nan,
            "has_nan_inf": False,
            "file_size_kb": np.nan,
            "error": "No matching .npz found for this CSV row.",
        }
    else:
        inspect_info = inspect_npz(kp_path)

    records.append({**base, **inspect_info})

manifest_df = pd.DataFrame(records)

# ----------------------------
# 8. Save manifest
# ----------------------------

manifest_df.to_csv(KEYPOINT_MANIFEST_PATH, index=False)

print("=" * 80)
print("KEYPOINT MANIFEST SAVED")
print("=" * 80)
print(KEYPOINT_MANIFEST_PATH)
print()

# ----------------------------
# 9. Summary
# ----------------------------

print("=" * 80)
print("SUMMARY")
print("=" * 80)

print("Rows by split:")
print(manifest_df["split"].value_counts(dropna=False))
print()

print("Status counts:")
print(manifest_df["status"].value_counts(dropna=False))
print()

print("Frame count summary:")
print(manifest_df["n_frames"].describe())
print()

print("FPS summary:")
print(manifest_df["fps"].describe())
print()

print("Detection-rate summaries:")
for col in ["pose_mask_rate", "lh_mask_rate", "rh_mask_rate", "either_hand_mask_rate"]:
    print()
    print(col)
    print(manifest_df[col].describe())

print()
print("Missing files:", int((manifest_df["status"] == "missing").sum()))
print("Load errors:", int((manifest_df["status"] == "load_error").sum()))
print("NaN/Inf files:", int((manifest_df["has_nan_inf"] == True).sum()))
print("Too-short files:", int((manifest_df["status"] == "bad_too_short").sum()))
print()

# ----------------------------
# 10. First 3 ok inspections
# ----------------------------

print("=" * 80)
print("FIRST 3 OK FILE INSPECTIONS")
print("=" * 80)

ok_examples = manifest_df[manifest_df["status"] == "ok"].head(3)

if len(ok_examples) == 0:
    print("No OK examples found.")
else:
    for _, r in ok_examples.iterrows():
        print("-" * 80)
        print(f"split:       {r['split']}")
        print(f"gloss:       {r['gloss']}")
        print(f"label_id:    {r['label_id']}")
        print(f"video_stem:  {r['video_stem']}")
        print(f"path:        {r['keypoint_path']}")
        print(f"keys:        {r['keys']}")
        print(f"n_frames:    {r['n_frames']}")
        print(f"fps:         {r['fps']}")
        print(f"pose_shape:  {r['pose_shape']}")
        print(f"lh_shape:    {r['lh_shape']}")
        print(f"rh_shape:    {r['rh_shape']}")
        print(f"pose_rate:   {r['pose_mask_rate']}")
        print(f"lh_rate:     {r['lh_mask_rate']}")
        print(f"rh_rate:     {r['rh_mask_rate']}")

# ----------------------------
# 11. Bad/suspicious preview
# ----------------------------

bad_df = manifest_df[manifest_df["status"] != "ok"].copy()

print()
print("=" * 80)
print("SUSPICIOUS / BAD FILES PREVIEW")
print("=" * 80)

if len(bad_df) == 0:
    print("No suspicious files found by this audit.")
else:
    print(f"Found {len(bad_df)} suspicious rows. Showing first 20:")
    display_cols = [
        "row_id", "split", "gloss", "video_stem",
        "status", "n_frames", "keys", "keypoint_path", "error"
    ]
    display(bad_df[display_cols].head(20))

print()
print("=" * 80)
print("PHASE 2A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Send these sections:")
print("1. Rows by split")
print("2. Status counts")
print("3. Frame count summary")
print("4. FPS summary")
print("5. Missing / Load errors / NaN/Inf / Too-short counts")
print("6. First 3 OK file inspections")
print("7. Suspicious / bad files preview")
print()
print("Do not run motion features yet.")

In [ ]:
# ============================================================
# PHASE 2B — CLEAN MANIFEST + NORMALIZATION SAFETY SCAN
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

KEYPOINT_MANIFEST_PATH = MANIFEST_DIR / "keypoint_manifest.csv"
KEYPOINT_CLEAN_MANIFEST_PATH = MANIFEST_DIR / "keypoint_manifest_clean.csv"
NORMALIZATION_AUDIT_PATH = MANIFEST_DIR / "normalization_audit.csv"

print("=" * 80)
print("PHASE 2B — CLEAN MANIFEST + NORMALIZATION SAFETY SCAN")
print("=" * 80)

if not KEYPOINT_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing keypoint manifest: {KEYPOINT_MANIFEST_PATH}")

manifest_df = pd.read_csv(KEYPOINT_MANIFEST_PATH)

print(f"Loaded keypoint manifest: {KEYPOINT_MANIFEST_PATH}")
print(f"Rows: {len(manifest_df)}")
print()

# ----------------------------
# 2. Safety thresholds
# ----------------------------

MIN_FRAMES_FOR_MOTION = 8
MIN_FRAMES_FOR_TRAINING = 16
LOW_HAND_RATE_THRESHOLD = 0.05
EXTREME_NORMALIZED_ABS_MAX = 100.0

print("Safety thresholds:")
print(f"MIN_FRAMES_FOR_MOTION:    {MIN_FRAMES_FOR_MOTION}")
print(f"MIN_FRAMES_FOR_TRAINING:  {MIN_FRAMES_FOR_TRAINING}")
print(f"LOW_HAND_RATE_THRESHOLD:  {LOW_HAND_RATE_THRESHOLD}")
print(f"EXTREME_NORMALIZED_ABS_MAX: {EXTREME_NORMALIZED_ABS_MAX}")
print()

# ----------------------------
# 3. Helper functions
# ----------------------------

def safe_float_array(x):
    arr = np.asarray(x, dtype=np.float32)
    arr[~np.isfinite(arr)] = 0.0
    return arr

def ensure_time_mask(mask, T):
    """
    Converts mask into shape (T,) boolean.
    Handles masks like (T,), (T,1), or weird shapes.
    """
    if mask is None:
        return np.ones(T, dtype=bool)

    m = np.asarray(mask)

    if m.ndim == 0:
        return np.ones(T, dtype=bool) * bool(m)

    if m.shape[0] != T:
        return np.ones(T, dtype=bool)

    if m.ndim == 1:
        return m > 0

    # If mask has extra dimensions, compress per frame.
    return np.any(m > 0, axis=tuple(range(1, m.ndim)))

def interpolate_missing_sequence(arr, valid_mask):
    """
    arr: (T, N, C)
    valid_mask: (T,)

    If a whole frame for a body part is missing, interpolate that body part over time.
    If no valid frames exist, return zeros.
    """
    arr = safe_float_array(arr)
    T = arr.shape[0]

    valid_mask = ensure_time_mask(valid_mask, T)

    if T == 0:
        return arr

    if valid_mask.sum() == 0:
        return np.zeros_like(arr, dtype=np.float32)

    out = arr.copy()

    # Set invalid frames to NaN temporarily.
    out[~valid_mask] = np.nan

    flat = out.reshape(T, -1)
    x = np.arange(T)

    for j in range(flat.shape[1]):
        col = flat[:, j]
        good = np.isfinite(col)

        if good.sum() == 0:
            flat[:, j] = 0.0
        elif good.sum() == 1:
            flat[:, j] = col[good][0]
        else:
            flat[:, j] = np.interp(x, x[good], col[good])

    out = flat.reshape(arr.shape).astype(np.float32)
    out[~np.isfinite(out)] = 0.0
    return out

def robust_normalize_keypoints(pose, lh, rh, pose_mask=None, lh_mask=None, rh_mask=None):
    """
    Robust normalization for your confirmed format:
      pose: (T, 7, 3)
      lh:   (T, 21, 3)
      rh:   (T, 21, 3)

    Main idea:
    - interpolate missing pose/hand frames
    - use pose center as sequence origin
    - use median pose spread as sequence scale
    - avoid division by zero
    - return positions as (T, 147)
    """
    pose = safe_float_array(pose)
    lh = safe_float_array(lh)
    rh = safe_float_array(rh)

    T = pose.shape[0]

    pose_mask = ensure_time_mask(pose_mask, T)
    lh_mask = ensure_time_mask(lh_mask, T)
    rh_mask = ensure_time_mask(rh_mask, T)

    # Interpolate missing detections.
    pose_i = interpolate_missing_sequence(pose, pose_mask)
    lh_i = interpolate_missing_sequence(lh, lh_mask)
    rh_i = interpolate_missing_sequence(rh, rh_mask)

    # Pose center per frame: mean of available pose landmarks.
    # Since pose has already been interpolated, this is stable.
    pose_xy = pose_i[:, :, :2]  # (T, 7, 2)
    center_xy = np.mean(pose_xy, axis=1, keepdims=True)  # (T, 1, 2)

    # Sequence-level scale from pose spread.
    # This avoids frame-by-frame scale jitter.
    spread_xy = np.max(pose_xy, axis=1) - np.min(pose_xy, axis=1)  # (T, 2)
    frame_scales = np.maximum(spread_xy[:, 0], spread_xy[:, 1])
    frame_scales = frame_scales[np.isfinite(frame_scales)]
    frame_scales = frame_scales[frame_scales > 1e-6]

    if len(frame_scales) == 0:
        scale = 1.0
        scale_status = "fallback_scale_1"
    else:
        scale = float(np.median(frame_scales))
        if not np.isfinite(scale) or scale < 1e-6:
            scale = 1.0
            scale_status = "fallback_scale_1"
        else:
            scale_status = "ok"

    # Normalize x/y by pose center and scale.
    # Normalize z by same scale, but no z-centering for simplicity.
    def normalize_part(part):
        out = part.copy().astype(np.float32)
        out[:, :, :2] = (out[:, :, :2] - center_xy) / scale
        out[:, :, 2] = out[:, :, 2] / scale
        out[~np.isfinite(out)] = 0.0
        return out

    pose_n = normalize_part(pose_i)
    lh_n = normalize_part(lh_i)
    rh_n = normalize_part(rh_i)

    # Concatenate into one vector per frame.
    all_points = np.concatenate([pose_n, lh_n, rh_n], axis=1)  # (T, 49, 3)
    positions = all_points.reshape(T, -1).astype(np.float32)  # (T, 147)

    positions[~np.isfinite(positions)] = 0.0

    # Basic motion features for safety check only.
    velocity = np.zeros_like(positions, dtype=np.float32)
    acceleration = np.zeros_like(positions, dtype=np.float32)

    if T >= 2:
        velocity[1:] = positions[1:] - positions[:-1]
    if T >= 3:
        acceleration[1:] = velocity[1:] - velocity[:-1]

    # Global speed: frame-level magnitude.
    global_speed = np.sqrt(np.mean(velocity ** 2, axis=1)).astype(np.float32)
    global_speed[~np.isfinite(global_speed)] = 0.0

    stats = {
        "T": int(T),
        "feature_dim": int(positions.shape[1]),
        "scale": float(scale),
        "scale_status": scale_status,
        "positions_abs_max": float(np.max(np.abs(positions))) if positions.size else np.nan,
        "velocity_abs_max": float(np.max(np.abs(velocity))) if velocity.size else np.nan,
        "acceleration_abs_max": float(np.max(np.abs(acceleration))) if acceleration.size else np.nan,
        "speed_min": float(np.min(global_speed)) if global_speed.size else np.nan,
        "speed_mean": float(np.mean(global_speed)) if global_speed.size else np.nan,
        "speed_max": float(np.max(global_speed)) if global_speed.size else np.nan,
        "has_nan_inf_after_norm": bool(
            np.isnan(positions).any()
            or np.isinf(positions).any()
            or np.isnan(velocity).any()
            or np.isinf(velocity).any()
            or np.isnan(acceleration).any()
            or np.isinf(acceleration).any()
            or np.isnan(global_speed).any()
            or np.isinf(global_speed).any()
        ),
    }

    return positions, velocity, acceleration, global_speed, stats

def audit_one_file(row):
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "keypoint_path": row["keypoint_path"],
        "original_status": row["status"],
        "n_frames": row["n_frames"],
        "fps": row["fps"],
        "either_hand_mask_rate": row["either_hand_mask_rate"],
        "norm_status": "unknown",
        "include_for_motion": False,
        "include_for_phase_training": False,
        "quality_flag": "",
        "error": "",
    }

    try:
        # Exclude files already marked bad.
        if row["status"] != "ok":
            result["norm_status"] = "skipped_bad_original_status"
            result["quality_flag"] = str(row["status"])
            return result

        if int(row["n_frames"]) < MIN_FRAMES_FOR_MOTION:
            result["norm_status"] = "skipped_too_short_for_motion"
            result["quality_flag"] = "too_short_for_motion"
            return result

        path = Path(row["keypoint_path"])
        if not path.exists():
            result["norm_status"] = "missing_file"
            result["quality_flag"] = "missing_file"
            return result

        with np.load(path, allow_pickle=False) as npz:
            pose = npz["pose"]
            lh = npz["lh"]
            rh = npz["rh"]
            pose_mask = npz["pose_mask"] if "pose_mask" in npz.files else None
            lh_mask = npz["lh_mask"] if "lh_mask" in npz.files else None
            rh_mask = npz["rh_mask"] if "rh_mask" in npz.files else None

            positions, velocity, acceleration, global_speed, stats = robust_normalize_keypoints(
                pose, lh, rh, pose_mask, lh_mask, rh_mask
            )

        result.update(stats)

        # Decide quality.
        if stats["has_nan_inf_after_norm"]:
            result["norm_status"] = "bad_nan_inf_after_norm"
            result["quality_flag"] = "nan_inf_after_norm"
            return result

        if stats["positions_abs_max"] > EXTREME_NORMALIZED_ABS_MAX:
            result["norm_status"] = "bad_extreme_normalized_values"
            result["quality_flag"] = "extreme_normalized_values"
            return result

        hand_rate = row["either_hand_mask_rate"]
        if pd.isna(hand_rate):
            hand_rate = 0.0

        result["include_for_motion"] = True

        if int(row["n_frames"]) >= MIN_FRAMES_FOR_TRAINING and float(hand_rate) >= LOW_HAND_RATE_THRESHOLD:
            result["include_for_phase_training"] = True
            result["norm_status"] = "ok"
            result["quality_flag"] = "good"
        elif int(row["n_frames"]) < MIN_FRAMES_FOR_TRAINING:
            result["include_for_phase_training"] = False
            result["norm_status"] = "ok_but_short_for_training"
            result["quality_flag"] = "short_for_training"
        elif float(hand_rate) < LOW_HAND_RATE_THRESHOLD:
            result["include_for_phase_training"] = False
            result["norm_status"] = "ok_but_low_hand_detection"
            result["quality_flag"] = "low_hand_detection"
        else:
            result["norm_status"] = "ok"
            result["quality_flag"] = "good"

        return result

    except Exception as e:
        result["norm_status"] = "norm_error"
        result["quality_flag"] = "norm_error"
        result["error"] = repr(e)
        return result

# ----------------------------
# 4. Run normalization audit
# ----------------------------

records = []

print("Running normalization safety scan...")
print("This may take a few minutes.")
print()

for i, row in manifest_df.iterrows():
    records.append(audit_one_file(row))

    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1}/{len(manifest_df)} files...")

audit_df = pd.DataFrame(records)

# ----------------------------
# 5. Merge audit back into manifest
# ----------------------------

audit_keep_cols = [
    "row_id",
    "norm_status",
    "include_for_motion",
    "include_for_phase_training",
    "quality_flag",
    "scale",
    "scale_status",
    "positions_abs_max",
    "velocity_abs_max",
    "acceleration_abs_max",
    "speed_min",
    "speed_mean",
    "speed_max",
    "has_nan_inf_after_norm",
    "error",
]

merged_df = manifest_df.merge(
    audit_df[audit_keep_cols],
    on="row_id",
    how="left",
    suffixes=("", "_norm")
)

# Save full audit and clean manifest.
audit_df.to_csv(NORMALIZATION_AUDIT_PATH, index=False)
merged_df.to_csv(KEYPOINT_CLEAN_MANIFEST_PATH, index=False)

# ----------------------------
# 6. Print summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 2B OUTPUT FILES SAVED")
print("=" * 80)
print(f"Normalization audit: {NORMALIZATION_AUDIT_PATH}")
print(f"Clean manifest:      {KEYPOINT_CLEAN_MANIFEST_PATH}")
print()

print("=" * 80)
print("NORMALIZATION AUDIT SUMMARY")
print("=" * 80)

print("Original keypoint status counts:")
print(merged_df["status"].value_counts(dropna=False))
print()

print("Normalization status counts:")
print(merged_df["norm_status"].value_counts(dropna=False))
print()

print("Quality flag counts:")
print(merged_df["quality_flag"].value_counts(dropna=False))
print()

print("Include for motion:")
print(merged_df["include_for_motion"].value_counts(dropna=False))
print()

print("Include for phase training:")
print(merged_df["include_for_phase_training"].value_counts(dropna=False))
print()

print("Scale summary:")
print(merged_df["scale"].describe())
print()

print("Positions abs max summary:")
print(merged_df["positions_abs_max"].describe())
print()

print("Speed max summary:")
print(merged_df["speed_max"].describe())
print()

print("NaN/Inf after normalization:")
print(merged_df["has_nan_inf_after_norm"].value_counts(dropna=False))
print()

# ----------------------------
# 7. Show excluded / suspicious rows
# ----------------------------

suspicious = merged_df[
    (merged_df["include_for_motion"] != True)
    | (merged_df["include_for_phase_training"] != True)
    | (merged_df["norm_status"] != "ok")
].copy()

print("=" * 80)
print("SUSPICIOUS / EXCLUDED PREVIEW")
print("=" * 80)

if len(suspicious) == 0:
    print("No suspicious rows found.")
else:
    print(f"Suspicious/excluded rows: {len(suspicious)}")
    display_cols = [
        "row_id",
        "split",
        "gloss",
        "video_stem",
        "status",
        "n_frames",
        "either_hand_mask_rate",
        "norm_status",
        "quality_flag",
        "include_for_motion",
        "include_for_phase_training",
        "positions_abs_max",
        "speed_max",
        "keypoint_path",
        "error",
    ]
    display(suspicious[display_cols].head(30))

print()
print("=" * 80)
print("PHASE 2B COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Normalization status counts")
print("2. Quality flag counts")
print("3. Include for motion")
print("4. Include for phase training")
print("5. Scale summary")
print("6. Positions abs max summary")
print("7. Speed max summary")
print("8. NaN/Inf after normalization")
print("9. Suspicious / excluded preview")
print()
print("Do NOT run motion feature generation yet.")

In [ ]:
# ============================================================
# PHASE 3A — BATCH MOTION FEATURE GENERATION
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"
MOTION_DIR = BASE_DIR / "motion_features"

KEYPOINT_CLEAN_MANIFEST_PATH = MANIFEST_DIR / "keypoint_manifest_clean.csv"
MOTION_MANIFEST_PATH = MANIFEST_DIR / "motion_manifest.csv"

MOTION_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 3A — BATCH MOTION FEATURE GENERATION")
print("=" * 80)

if not KEYPOINT_CLEAN_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing clean manifest: {KEYPOINT_CLEAN_MANIFEST_PATH}")

manifest_df = pd.read_csv(KEYPOINT_CLEAN_MANIFEST_PATH)

print(f"Loaded clean manifest: {KEYPOINT_CLEAN_MANIFEST_PATH}")
print(f"Rows: {len(manifest_df)}")
print(f"Motion output directory: {MOTION_DIR}")
print()

# ----------------------------
# 2. Select files for motion generation
# ----------------------------

motion_df = manifest_df[manifest_df["include_for_motion"] == True].copy()

print("Motion generation input:")
print(f"Total manifest rows:       {len(manifest_df)}")
print(f"Include for motion rows:   {len(motion_df)}")
print(f"Excluded from motion rows: {len(manifest_df) - len(motion_df)}")
print()

if len(motion_df) == 0:
    raise ValueError("No rows are marked include_for_motion=True. Stop.")

# ----------------------------
# 3. Helper functions
# ----------------------------

def safe_filename_stem(s):
    """
    Creates a safe filename stem while preserving readability.
    """
    s = str(s)
    s = s.replace("/", "_").replace("\\", "_")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_float_array(x):
    arr = np.asarray(x, dtype=np.float32)
    arr[~np.isfinite(arr)] = 0.0
    return arr

def ensure_time_mask(mask, T):
    if mask is None:
        return np.ones(T, dtype=bool)

    m = np.asarray(mask)

    if m.ndim == 0:
        return np.ones(T, dtype=bool) * bool(m)

    if m.shape[0] != T:
        return np.ones(T, dtype=bool)

    if m.ndim == 1:
        return m > 0

    return np.any(m > 0, axis=tuple(range(1, m.ndim)))

def interpolate_missing_sequence(arr, valid_mask):
    """
    Interpolate missing frames for one body part.
    arr shape: (T, N, C)
    valid_mask shape: (T,)
    """
    arr = safe_float_array(arr)
    T = arr.shape[0]
    valid_mask = ensure_time_mask(valid_mask, T)

    if T == 0:
        return arr

    if valid_mask.sum() == 0:
        return np.zeros_like(arr, dtype=np.float32)

    out = arr.copy()
    out[~valid_mask] = np.nan

    flat = out.reshape(T, -1)
    x = np.arange(T)

    for j in range(flat.shape[1]):
        col = flat[:, j]
        good = np.isfinite(col)

        if good.sum() == 0:
            flat[:, j] = 0.0
        elif good.sum() == 1:
            flat[:, j] = col[good][0]
        else:
            flat[:, j] = np.interp(x, x[good], col[good])

    out = flat.reshape(arr.shape).astype(np.float32)
    out[~np.isfinite(out)] = 0.0
    return out

def robust_normalize_keypoints(pose, lh, rh, pose_mask=None, lh_mask=None, rh_mask=None):
    """
    Confirmed input:
      pose: (T, 7, 3)
      lh:   (T, 21, 3)
      rh:   (T, 21, 3)

    Output:
      normalized_points: (T, 49, 3)
      positions:         (T, 147)
    """
    pose = safe_float_array(pose)
    lh = safe_float_array(lh)
    rh = safe_float_array(rh)

    T = pose.shape[0]

    pose_mask = ensure_time_mask(pose_mask, T)
    lh_mask = ensure_time_mask(lh_mask, T)
    rh_mask = ensure_time_mask(rh_mask, T)

    pose_i = interpolate_missing_sequence(pose, pose_mask)
    lh_i = interpolate_missing_sequence(lh, lh_mask)
    rh_i = interpolate_missing_sequence(rh, rh_mask)

    pose_xy = pose_i[:, :, :2]
    center_xy = np.mean(pose_xy, axis=1, keepdims=True)

    spread_xy = np.max(pose_xy, axis=1) - np.min(pose_xy, axis=1)
    frame_scales = np.maximum(spread_xy[:, 0], spread_xy[:, 1])
    frame_scales = frame_scales[np.isfinite(frame_scales)]
    frame_scales = frame_scales[frame_scales > 1e-6]

    if len(frame_scales) == 0:
        scale = 1.0
        scale_status = "fallback_scale_1"
    else:
        scale = float(np.median(frame_scales))
        if not np.isfinite(scale) or scale < 1e-6:
            scale = 1.0
            scale_status = "fallback_scale_1"
        else:
            scale_status = "ok"

    def normalize_part(part):
        out = part.copy().astype(np.float32)
        out[:, :, :2] = (out[:, :, :2] - center_xy) / scale
        out[:, :, 2] = out[:, :, 2] / scale
        out[~np.isfinite(out)] = 0.0
        return out

    pose_n = normalize_part(pose_i)
    lh_n = normalize_part(lh_i)
    rh_n = normalize_part(rh_i)

    normalized_points = np.concatenate([pose_n, lh_n, rh_n], axis=1).astype(np.float32)
    positions = normalized_points.reshape(T, -1).astype(np.float32)

    positions[~np.isfinite(positions)] = 0.0

    return normalized_points, positions, scale, scale_status

def compute_motion_features(normalized_points, positions):
    """
    Computes:
      velocity:      (T, 147)
      acceleration:  (T, 147)
      global_speed:  (T,)
      hand_speed:    (T,)
    """
    T = positions.shape[0]

    velocity = np.zeros_like(positions, dtype=np.float32)
    acceleration = np.zeros_like(positions, dtype=np.float32)

    if T >= 2:
        velocity[1:] = positions[1:] - positions[:-1]

    if T >= 3:
        acceleration[1:] = velocity[1:] - velocity[:-1]

    # Global speed across all 49 landmarks.
    global_speed = np.sqrt(np.mean(velocity ** 2, axis=1)).astype(np.float32)

    # Hand speed only: landmarks 7:49 are left + right hands.
    # normalized_points shape: (T, 49, 3)
    hand_points = normalized_points[:, 7:, :]  # (T, 42, 3)

    hand_velocity = np.zeros_like(hand_points, dtype=np.float32)
    if T >= 2:
        hand_velocity[1:] = hand_points[1:] - hand_points[:-1]

    hand_speed = np.sqrt(np.mean(hand_velocity ** 2, axis=(1, 2))).astype(np.float32)

    for arr in [velocity, acceleration]:
        arr[~np.isfinite(arr)] = 0.0

    global_speed[~np.isfinite(global_speed)] = 0.0
    hand_speed[~np.isfinite(hand_speed)] = 0.0

    return velocity, acceleration, global_speed, hand_speed

def smooth_1d(x, window=7):
    """
    Simple moving-average smoothing.
    No scipy dependency.
    """
    x = np.asarray(x, dtype=np.float32)

    if len(x) == 0:
        return x

    if window <= 1 or len(x) < 3:
        return x.copy()

    # Make window odd and not longer than the sequence.
    window = int(window)
    if window % 2 == 0:
        window += 1

    window = min(window, len(x))
    if window % 2 == 0:
        window -= 1

    if window < 3:
        return x.copy()

    pad = window // 2
    padded = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=np.float32) / window
    y = np.convolve(padded, kernel, mode="valid").astype(np.float32)
    y[~np.isfinite(y)] = 0.0
    return y

def process_one_row(row):
    """
    Load one keypoint file, generate motion features, save .npz.
    """
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "keypoint_path": row["keypoint_path"],
        "motion_path": "",
        "n_frames": row["n_frames"],
        "fps": row["fps"],
        "status": "unknown",
        "feature_dim": np.nan,
        "scale": np.nan,
        "scale_status": "",
        "global_speed_mean": np.nan,
        "global_speed_max": np.nan,
        "hand_speed_mean": np.nan,
        "hand_speed_max": np.nan,
        "smoothed_speed_mean": np.nan,
        "smoothed_speed_max": np.nan,
        "has_nan_inf": False,
        "error": "",
    }

    try:
        keypoint_path = Path(row["keypoint_path"])
        if not keypoint_path.exists():
            result["status"] = "missing_keypoint_file"
            result["error"] = f"Missing: {keypoint_path}"
            return result

        with np.load(keypoint_path, allow_pickle=False) as npz:
            pose = npz["pose"]
            lh = npz["lh"]
            rh = npz["rh"]
            pose_mask = npz["pose_mask"] if "pose_mask" in npz.files else None
            lh_mask = npz["lh_mask"] if "lh_mask" in npz.files else None
            rh_mask = npz["rh_mask"] if "rh_mask" in npz.files else None
            fps = float(np.asarray(npz["fps"]).item()) if "fps" in npz.files else float(row["fps"])

        normalized_points, positions, scale, scale_status = robust_normalize_keypoints(
            pose, lh, rh, pose_mask, lh_mask, rh_mask
        )

        velocity, acceleration, global_speed, hand_speed = compute_motion_features(
            normalized_points, positions
        )

        # For phase labels, hand speed is the most relevant motion curve.
        smoothed_speed = smooth_1d(hand_speed, window=7)

        # Final NaN/Inf check.
        has_bad = (
            np.isnan(positions).any() or np.isinf(positions).any()
            or np.isnan(velocity).any() or np.isinf(velocity).any()
            or np.isnan(acceleration).any() or np.isinf(acceleration).any()
            or np.isnan(global_speed).any() or np.isinf(global_speed).any()
            or np.isnan(hand_speed).any() or np.isinf(hand_speed).any()
            or np.isnan(smoothed_speed).any() or np.isinf(smoothed_speed).any()
        )

        if has_bad:
            result["status"] = "bad_nan_inf"
            result["has_nan_inf"] = True
            return result

        motion_stem = safe_filename_stem(row["video_stem"])
        motion_path = MOTION_DIR / f"{motion_stem}.npz"

        np.savez_compressed(
            motion_path,
            positions=positions.astype(np.float32),
            velocity=velocity.astype(np.float32),
            acceleration=acceleration.astype(np.float32),
            global_speed=global_speed.astype(np.float32),
            hand_speed=hand_speed.astype(np.float32),
            smoothed_speed=smoothed_speed.astype(np.float32),
            fps=np.float32(fps),
            scale=np.float32(scale),
            video_stem=str(row["video_stem"]),
            gloss=str(row["gloss"]),
            split=str(row["split"]),
            label_id=np.int64(row["label_id"]),
        )

        result["motion_path"] = str(motion_path)
        result["status"] = "ok"
        result["feature_dim"] = int(positions.shape[1])
        result["scale"] = float(scale)
        result["scale_status"] = scale_status
        result["global_speed_mean"] = float(np.mean(global_speed))
        result["global_speed_max"] = float(np.max(global_speed))
        result["hand_speed_mean"] = float(np.mean(hand_speed))
        result["hand_speed_max"] = float(np.max(hand_speed))
        result["smoothed_speed_mean"] = float(np.mean(smoothed_speed))
        result["smoothed_speed_max"] = float(np.max(smoothed_speed))
        result["has_nan_inf"] = False

        return result

    except Exception as e:
        result["status"] = "error"
        result["error"] = repr(e)
        return result

# ----------------------------
# 4. Generate motion features
# ----------------------------

records = []

print("Generating motion feature files...")
print("This may take several minutes because it reads and writes many Google Drive files.")
print()

for i, row in motion_df.iterrows():
    records.append(process_one_row(row))

    if len(records) % 500 == 0:
        print(f"Processed {len(records)}/{len(motion_df)} motion files...")

motion_manifest_df = pd.DataFrame(records)
motion_manifest_df.to_csv(MOTION_MANIFEST_PATH, index=False)

# ----------------------------
# 5. Summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 3A OUTPUT SAVED")
print("=" * 80)
print(f"Motion feature directory: {MOTION_DIR}")
print(f"Motion manifest:          {MOTION_MANIFEST_PATH}")
print()

print("=" * 80)
print("MOTION GENERATION SUMMARY")
print("=" * 80)

print("Status counts:")
print(motion_manifest_df["status"].value_counts(dropna=False))
print()

print("Rows by split:")
print(motion_manifest_df["split"].value_counts(dropna=False))
print()

print("Feature dimension summary:")
print(motion_manifest_df["feature_dim"].describe())
print()

print("Frame count summary:")
print(motion_manifest_df["n_frames"].describe())
print()

print("Scale summary:")
print(motion_manifest_df["scale"].describe())
print()

print("Global speed max summary:")
print(motion_manifest_df["global_speed_max"].describe())
print()

print("Hand speed max summary:")
print(motion_manifest_df["hand_speed_max"].describe())
print()

print("Smoothed speed max summary:")
print(motion_manifest_df["smoothed_speed_max"].describe())
print()

print("NaN/Inf status:")
print(motion_manifest_df["has_nan_inf"].value_counts(dropna=False))
print()

# ----------------------------
# 6. First 3 OK motion files
# ----------------------------

print("=" * 80)
print("FIRST 3 OK MOTION FILES")
print("=" * 80)

ok_motion = motion_manifest_df[motion_manifest_df["status"] == "ok"].head(3)

if len(ok_motion) == 0:
    print("No OK motion files found. Something is wrong.")
else:
    for _, r in ok_motion.iterrows():
        print("-" * 80)
        print(f"split:              {r['split']}")
        print(f"gloss:              {r['gloss']}")
        print(f"video_stem:         {r['video_stem']}")
        print(f"motion_path:        {r['motion_path']}")
        print(f"n_frames:           {r['n_frames']}")
        print(f"feature_dim:        {r['feature_dim']}")
        print(f"global_speed_max:   {r['global_speed_max']}")
        print(f"hand_speed_max:     {r['hand_speed_max']}")
        print(f"smoothed_speed_max: {r['smoothed_speed_max']}")

# ----------------------------
# 7. Bad preview
# ----------------------------

bad_motion = motion_manifest_df[motion_manifest_df["status"] != "ok"].copy()

print()
print("=" * 80)
print("BAD / FAILED MOTION FILES PREVIEW")
print("=" * 80)

if len(bad_motion) == 0:
    print("No failed motion files.")
else:
    print(f"Failed rows: {len(bad_motion)}")
    display_cols = [
        "row_id",
        "split",
        "gloss",
        "video_stem",
        "status",
        "n_frames",
        "keypoint_path",
        "motion_path",
        "error",
    ]
    display(bad_motion[display_cols].head(30))

print()
print("=" * 80)
print("PHASE 3A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Status counts")
print("2. Rows by split")
print("3. Feature dimension summary")
print("4. Frame count summary")
print("5. Global speed max summary")
print("6. Hand speed max summary")
print("7. Smoothed speed max summary")
print("8. NaN/Inf status")
print("9. First 3 OK motion files")
print("10. Bad / failed motion files preview")
print()
print("Do NOT generate phase labels yet.")

In [ ]:
# ============================================================
# PHASE 4A — MOTION CURVE VISUALIZATION
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"
PLOTS_DIR = BASE_DIR / "plots"
MOTION_CURVE_DIR = PLOTS_DIR / "motion_curves"

MOTION_MANIFEST_PATH = MANIFEST_DIR / "motion_manifest.csv"
MOTION_CURVE_AUDIT_PATH = MANIFEST_DIR / "motion_curve_audit.csv"

MOTION_CURVE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 4A — MOTION CURVE VISUALIZATION")
print("=" * 80)

if not MOTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing motion manifest: {MOTION_MANIFEST_PATH}")

motion_df = pd.read_csv(MOTION_MANIFEST_PATH)

print(f"Loaded motion manifest: {MOTION_MANIFEST_PATH}")
print(f"Rows: {len(motion_df)}")
print(f"Plot output directory: {MOTION_CURVE_DIR}")
print()

# ----------------------------
# 2. Keep only successful motion rows
# ----------------------------

ok_df = motion_df[motion_df["status"] == "ok"].copy()

print("Motion status counts:")
print(motion_df["status"].value_counts(dropna=False))
print()

if len(ok_df) == 0:
    raise ValueError("No OK motion files found.")

# ----------------------------
# 3. Basic motion curve audit
# ----------------------------

def count_simple_peaks(x, min_prominence_ratio=0.20):
    """
    Simple local peak counter.
    This is only for curve quality inspection, not final phase labels.
    """
    x = np.asarray(x, dtype=np.float32)
    if len(x) < 5:
        return 0

    x_max = float(np.max(x))
    if x_max <= 1e-8:
        return 0

    min_height = x_max * min_prominence_ratio
    peaks = 0

    for i in range(1, len(x) - 1):
        if x[i] > x[i - 1] and x[i] >= x[i + 1] and x[i] >= min_height:
            peaks += 1

    return peaks

def audit_motion_curve(row):
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "motion_path": row["motion_path"],
        "n_frames": row["n_frames"],
        "smoothed_speed_max": np.nan,
        "smoothed_speed_mean": np.nan,
        "smoothed_speed_std": np.nan,
        "active_ratio_20pct": np.nan,
        "simple_peak_count": np.nan,
        "curve_quality_flag": "unknown",
        "error": "",
    }

    try:
        path = Path(row["motion_path"])
        if not path.exists():
            result["curve_quality_flag"] = "missing_motion_file"
            result["error"] = f"Missing: {path}"
            return result

        with np.load(path, allow_pickle=False) as npz:
            speed = np.asarray(npz["smoothed_speed"], dtype=np.float32)

        speed[~np.isfinite(speed)] = 0.0

        speed_max = float(np.max(speed)) if len(speed) else 0.0
        speed_mean = float(np.mean(speed)) if len(speed) else 0.0
        speed_std = float(np.std(speed)) if len(speed) else 0.0

        if speed_max > 1e-8:
            active_ratio = float(np.mean(speed >= 0.20 * speed_max))
        else:
            active_ratio = 0.0

        peak_count = count_simple_peaks(speed)

        result["smoothed_speed_max"] = speed_max
        result["smoothed_speed_mean"] = speed_mean
        result["smoothed_speed_std"] = speed_std
        result["active_ratio_20pct"] = active_ratio
        result["simple_peak_count"] = peak_count

        # Rough quality flags.
        # These are only to guide visual inspection.
        if len(speed) < 16:
            flag = "too_short"
        elif speed_max <= 1e-8:
            flag = "flat_zero"
        elif active_ratio > 0.85:
            flag = "too_flat_or_always_active"
        elif peak_count >= 8:
            flag = "many_peaks_noisy"
        elif peak_count == 0:
            flag = "no_clear_peak"
        else:
            flag = "visually_check"

        result["curve_quality_flag"] = flag
        return result

    except Exception as e:
        result["curve_quality_flag"] = "error"
        result["error"] = repr(e)
        return result

print("Building motion curve audit...")
audit_records = []

for i, row in ok_df.iterrows():
    audit_records.append(audit_motion_curve(row))

    if len(audit_records) % 500 == 0:
        print(f"Audited {len(audit_records)}/{len(ok_df)} curves...")

curve_audit_df = pd.DataFrame(audit_records)
curve_audit_df.to_csv(MOTION_CURVE_AUDIT_PATH, index=False)

print()
print(f"Saved motion curve audit: {MOTION_CURVE_AUDIT_PATH}")
print()

# ----------------------------
# 4. Select examples to plot
# ----------------------------

plot_df = curve_audit_df.copy()

# Merge in original manifest columns if needed.
plot_df = plot_df[plot_df["curve_quality_flag"] != "missing_motion_file"].copy()
plot_df = plot_df[plot_df["curve_quality_flag"] != "error"].copy()

random.seed(42)
np.random.seed(42)

selected_rows = []

def add_sample_rows(df, name, n):
    if len(df) == 0:
        print(f"Warning: no rows available for bucket: {name}")
        return

    n = min(n, len(df))
    sampled = df.sample(n=n, random_state=42).copy()
    sampled["plot_bucket"] = name
    selected_rows.append(sampled)

# Random normal examples.
add_sample_rows(plot_df, "random", 8)

# Low-motion examples, excluding totally flat if possible.
low_df = plot_df.sort_values("smoothed_speed_max", ascending=True).head(50)
add_sample_rows(low_df, "low_motion", 5)

# Medium-motion examples around the median.
median_speed = plot_df["smoothed_speed_max"].median()
mid_df = plot_df.iloc[(plot_df["smoothed_speed_max"] - median_speed).abs().sort_values().index].head(50)
add_sample_rows(mid_df, "medium_motion", 5)

# High-motion examples.
high_df = plot_df.sort_values("smoothed_speed_max", ascending=False).head(50)
add_sample_rows(high_df, "high_motion", 5)

# Noisy/many-peaks examples if any.
noisy_df = plot_df[plot_df["curve_quality_flag"] == "many_peaks_noisy"]
add_sample_rows(noisy_df, "many_peaks_noisy", 5)

# Flat or no clear peak examples if any.
flat_df = plot_df[plot_df["curve_quality_flag"].isin(["flat_zero", "no_clear_peak", "too_flat_or_always_active"])]
add_sample_rows(flat_df, "weak_or_flat", 5)

selected_df = pd.concat(selected_rows, ignore_index=True)
selected_df = selected_df.drop_duplicates(subset=["motion_path"]).reset_index(drop=True)

print(f"Selected plots to generate: {len(selected_df)}")
print()

# ----------------------------
# 5. Plot helper
# ----------------------------

def plot_motion_curve(row, save_dir, show=False):
    path = Path(row["motion_path"])

    with np.load(path, allow_pickle=False) as npz:
        global_speed = np.asarray(npz["global_speed"], dtype=np.float32)
        hand_speed = np.asarray(npz["hand_speed"], dtype=np.float32)
        smoothed_speed = np.asarray(npz["smoothed_speed"], dtype=np.float32)

    T = len(smoothed_speed)
    frames = np.arange(T)

    global_speed[~np.isfinite(global_speed)] = 0.0
    hand_speed[~np.isfinite(hand_speed)] = 0.0
    smoothed_speed[~np.isfinite(smoothed_speed)] = 0.0

    plt.figure(figsize=(12, 4))
    plt.plot(frames, global_speed, label="global_speed", alpha=0.45)
    plt.plot(frames, hand_speed, label="hand_speed", alpha=0.45)
    plt.plot(frames, smoothed_speed, label="smoothed_speed", linewidth=2.5)

    if float(np.max(smoothed_speed)) > 1e-8:
        threshold = 0.20 * float(np.max(smoothed_speed))
        plt.axhline(threshold, linestyle="--", linewidth=1.2, label="20% of smoothed max")

    title = (
        f"{row['plot_bucket']} | {row['split']} | {row['gloss']} | "
        f"{row['video_stem']} | T={T} | flag={row['curve_quality_flag']}"
    )

    plt.title(title)
    plt.xlabel("Frame")
    plt.ylabel("Normalized speed")
    plt.legend(loc="upper right")
    plt.tight_layout()

    safe_name = str(row["video_stem"]).replace("/", "_").replace("\\", "_")
    safe_name = safe_name.replace(" ", "_")
    save_path = save_dir / f"{row['plot_bucket']}__{safe_name}.png"

    plt.savefig(save_path, dpi=140, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return save_path

# ----------------------------
# 6. Generate plots
# ----------------------------

saved_paths = []

print("Generating and saving motion curve plots...")
for idx, row in selected_df.iterrows():
    # Show only first 6 in notebook to avoid huge output.
    show_now = idx < 6
    saved_path = plot_motion_curve(row, MOTION_CURVE_DIR, show=show_now)
    saved_paths.append(str(saved_path))

print()
print(f"Saved {len(saved_paths)} motion curve plots.")
print()

# ----------------------------
# 7. Summary output
# ----------------------------

print("=" * 80)
print("PHASE 4A OUTPUT SAVED")
print("=" * 80)
print(f"Motion curve audit: {MOTION_CURVE_AUDIT_PATH}")
print(f"Motion curve plots: {MOTION_CURVE_DIR}")
print()

print("=" * 80)
print("MOTION CURVE AUDIT SUMMARY")
print("=" * 80)

print("Curve quality flag counts:")
print(curve_audit_df["curve_quality_flag"].value_counts(dropna=False))
print()

print("Smoothed speed max summary:")
print(curve_audit_df["smoothed_speed_max"].describe())
print()

print("Active ratio summary:")
print(curve_audit_df["active_ratio_20pct"].describe())
print()

print("Simple peak count summary:")
print(curve_audit_df["simple_peak_count"].describe())
print()

print("=" * 80)
print("PLOTTED EXAMPLES")
print("=" * 80)

display_cols = [
    "plot_bucket",
    "split",
    "gloss",
    "video_stem",
    "n_frames",
    "smoothed_speed_max",
    "active_ratio_20pct",
    "simple_peak_count",
    "curve_quality_flag",
    "motion_path",
]

display(selected_df[display_cols])

print()
print("First 10 saved plot paths:")
for p in saved_paths[:10]:
    print(p)

print()
print("=" * 80)
print("PHASE 4A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Curve quality flag counts")
print("2. Smoothed speed max summary")
print("3. Active ratio summary")
print("4. Simple peak count summary")
print("5. Plotted examples table")
print()
print("Also visually check the first 6 plots shown above and tell me:")
print("- Do the curves rise and fall like actual movement?")
print("- Are they mostly flat?")
print("- Are they extremely noisy with too many peaks?")
print()
print("Do NOT generate phase labels yet.")

In [ ]:
# ============================================================
# PHASE 4B — BUILD IMPROVED PHASE-SPEED SIGNAL
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import random

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

MOTION_MANIFEST_PATH = MANIFEST_DIR / "motion_manifest.csv"
OLD_CURVE_AUDIT_PATH = MANIFEST_DIR / "motion_curve_audit.csv"

PHASE_SIGNAL_DIR = BASE_DIR / "phase_signals"
PHASE_SIGNAL_PLOT_DIR = BASE_DIR / "plots" / "phase_speed_curves"

PHASE_SIGNAL_MANIFEST_PATH = MANIFEST_DIR / "phase_signal_manifest.csv"

PHASE_SIGNAL_DIR.mkdir(parents=True, exist_ok=True)
PHASE_SIGNAL_PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 4B — BUILD IMPROVED PHASE-SPEED SIGNAL")
print("=" * 80)

if not MOTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing motion manifest: {MOTION_MANIFEST_PATH}")

motion_df = pd.read_csv(MOTION_MANIFEST_PATH)
ok_df = motion_df[motion_df["status"] == "ok"].copy()

print(f"Loaded motion manifest: {MOTION_MANIFEST_PATH}")
print(f"Total rows: {len(motion_df)}")
print(f"OK motion rows: {len(ok_df)}")
print(f"Phase signal output dir: {PHASE_SIGNAL_DIR}")
print(f"Phase signal plot dir:   {PHASE_SIGNAL_PLOT_DIR}")
print()

# ----------------------------
# 2. Helper functions
# ----------------------------

def safe_stem(s):
    s = str(s)
    s = s.replace("/", "_").replace("\\", "_")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def robust_01(x):
    """
    Robust per-video normalization to [0, 1].
    Uses 5th and 95th percentiles to avoid one spike dominating the curve.
    """
    x = np.asarray(x, dtype=np.float32)
    x[~np.isfinite(x)] = 0.0

    if len(x) == 0:
        return x

    p5 = float(np.percentile(x, 5))
    p95 = float(np.percentile(x, 95))

    denom = p95 - p5

    if not np.isfinite(denom) or denom < 1e-8:
        return np.zeros_like(x, dtype=np.float32)

    y = (x - p5) / denom
    y = np.clip(y, 0.0, 1.0).astype(np.float32)
    y[~np.isfinite(y)] = 0.0

    return y

def smooth_1d(x, window=7):
    """
    Simple moving average smoothing.
    """
    x = np.asarray(x, dtype=np.float32)

    if len(x) == 0:
        return x

    if window <= 1 or len(x) < 3:
        return x.copy()

    window = int(window)

    if window % 2 == 0:
        window += 1

    window = min(window, len(x))

    if window % 2 == 0:
        window -= 1

    if window < 3:
        return x.copy()

    pad = window // 2
    padded = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=np.float32) / window
    y = np.convolve(padded, kernel, mode="valid").astype(np.float32)
    y[~np.isfinite(y)] = 0.0

    return y

def adaptive_window(T):
    """
    Longer clips get slightly stronger smoothing.
    We do not over-smooth short clips.
    """
    T = int(T)

    if T < 40:
        return 5
    elif T < 80:
        return 7
    elif T < 160:
        return 9
    elif T < 300:
        return 13
    else:
        return 17

def count_simple_peaks(x, min_prominence_ratio=0.25):
    """
    Simple peak count for quality inspection only.
    """
    x = np.asarray(x, dtype=np.float32)

    if len(x) < 5:
        return 0

    x_max = float(np.max(x))

    if x_max <= 1e-8:
        return 0

    min_height = x_max * min_prominence_ratio
    peaks = 0

    for i in range(1, len(x) - 1):
        if x[i] > x[i - 1] and x[i] >= x[i + 1] and x[i] >= min_height:
            peaks += 1

    return peaks

def compute_centroid_speed_from_positions(positions):
    """
    positions: (T, 147)
    reshaped as:
      49 landmarks x 3 coordinates
      0:7   = pose
      7:28  = left hand
      28:49 = right hand

    Returns active_hand_centroid_speed: (T,)
    """
    positions = np.asarray(positions, dtype=np.float32)
    positions[~np.isfinite(positions)] = 0.0

    T = positions.shape[0]
    points = positions.reshape(T, 49, 3)

    lh = points[:, 7:28, :]
    rh = points[:, 28:49, :]

    lh_center = np.mean(lh, axis=1)
    rh_center = np.mean(rh, axis=1)

    lh_vel = np.zeros_like(lh_center, dtype=np.float32)
    rh_vel = np.zeros_like(rh_center, dtype=np.float32)

    if T >= 2:
        lh_vel[1:] = lh_center[1:] - lh_center[:-1]
        rh_vel[1:] = rh_center[1:] - rh_center[:-1]

    lh_speed = np.linalg.norm(lh_vel, axis=1).astype(np.float32)
    rh_speed = np.linalg.norm(rh_vel, axis=1).astype(np.float32)

    # Use the faster hand per frame as the main phase signal.
    active_hand_centroid_speed = np.maximum(lh_speed, rh_speed).astype(np.float32)
    active_hand_centroid_speed[~np.isfinite(active_hand_centroid_speed)] = 0.0

    return active_hand_centroid_speed, lh_speed, rh_speed

def quality_flag_from_phase_speed(phase_speed):
    """
    Rough quality flag for the new signal.
    This is not final validation, only signal inspection.
    """
    phase_speed = np.asarray(phase_speed, dtype=np.float32)

    T = len(phase_speed)

    if T < 16:
        return "too_short", 0.0, 0

    max_val = float(np.max(phase_speed)) if T else 0.0

    if max_val <= 1e-8:
        return "flat_zero", 0.0, 0

    active_ratio_30 = float(np.mean(phase_speed >= 0.30 * max_val))
    peak_count = count_simple_peaks(phase_speed, min_prominence_ratio=0.25)

    if active_ratio_30 > 0.80:
        flag = "too_flat_or_always_active"
    elif peak_count >= 8:
        flag = "many_peaks_noisy"
    elif peak_count == 0:
        flag = "no_clear_peak"
    else:
        flag = "phase_signal_ok"

    return flag, active_ratio_30, peak_count

def process_one_motion(row):
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "motion_path": row["motion_path"],
        "phase_signal_path": "",
        "n_frames": row["n_frames"],
        "phase_speed_max": np.nan,
        "phase_speed_mean": np.nan,
        "active_ratio_30pct": np.nan,
        "simple_peak_count_v2": np.nan,
        "phase_signal_flag": "unknown",
        "smoothing_window": np.nan,
        "has_nan_inf": False,
        "error": "",
    }

    try:
        motion_path = Path(row["motion_path"])

        if not motion_path.exists():
            result["phase_signal_flag"] = "missing_motion_file"
            result["error"] = f"Missing: {motion_path}"
            return result

        with np.load(motion_path, allow_pickle=False) as npz:
            positions = np.asarray(npz["positions"], dtype=np.float32)
            hand_speed = np.asarray(npz["hand_speed"], dtype=np.float32)
            old_smoothed_speed = np.asarray(npz["smoothed_speed"], dtype=np.float32)
            global_speed = np.asarray(npz["global_speed"], dtype=np.float32)
            fps = float(np.asarray(npz["fps"]).item()) if "fps" in npz.files else 30.0

        positions[~np.isfinite(positions)] = 0.0
        hand_speed[~np.isfinite(hand_speed)] = 0.0
        old_smoothed_speed[~np.isfinite(old_smoothed_speed)] = 0.0
        global_speed[~np.isfinite(global_speed)] = 0.0

        T = positions.shape[0]

        centroid_speed, lh_centroid_speed, rh_centroid_speed = compute_centroid_speed_from_positions(positions)

        # Normalize signals per clip before blending.
        centroid_norm = robust_01(centroid_speed)
        hand_norm = robust_01(hand_speed)
        global_norm = robust_01(global_speed)

        # Main new phase-speed signal.
        # More weight on centroid motion, less on finger-level jitter.
        phase_speed_raw = (
            0.65 * centroid_norm
            + 0.25 * hand_norm
            + 0.10 * global_norm
        ).astype(np.float32)

        w = adaptive_window(T)
        phase_speed = smooth_1d(phase_speed_raw, window=w)

        # Light second smoothing only for very long/noisy clips.
        if T >= 160:
            phase_speed = smooth_1d(phase_speed, window=5)

        phase_speed = np.clip(phase_speed, 0.0, 1.0).astype(np.float32)
        phase_speed[~np.isfinite(phase_speed)] = 0.0

        has_bad = (
            np.isnan(phase_speed).any()
            or np.isinf(phase_speed).any()
            or np.isnan(phase_speed_raw).any()
            or np.isinf(phase_speed_raw).any()
        )

        if has_bad:
            result["phase_signal_flag"] = "bad_nan_inf"
            result["has_nan_inf"] = True
            return result

        flag, active_ratio_30, peak_count = quality_flag_from_phase_speed(phase_speed)

        phase_signal_path = PHASE_SIGNAL_DIR / f"{safe_stem(row['video_stem'])}.npz"

        np.savez_compressed(
            phase_signal_path,
            phase_speed=phase_speed.astype(np.float32),
            phase_speed_raw=phase_speed_raw.astype(np.float32),
            centroid_speed=centroid_speed.astype(np.float32),
            lh_centroid_speed=lh_centroid_speed.astype(np.float32),
            rh_centroid_speed=rh_centroid_speed.astype(np.float32),
            centroid_norm=centroid_norm.astype(np.float32),
            hand_norm=hand_norm.astype(np.float32),
            global_norm=global_norm.astype(np.float32),
            old_smoothed_speed=old_smoothed_speed.astype(np.float32),
            fps=np.float32(fps),
            video_stem=str(row["video_stem"]),
            gloss=str(row["gloss"]),
            split=str(row["split"]),
            label_id=np.int64(row["label_id"]),
        )

        result["phase_signal_path"] = str(phase_signal_path)
        result["phase_speed_max"] = float(np.max(phase_speed))
        result["phase_speed_mean"] = float(np.mean(phase_speed))
        result["active_ratio_30pct"] = active_ratio_30
        result["simple_peak_count_v2"] = peak_count
        result["phase_signal_flag"] = flag
        result["smoothing_window"] = w
        result["has_nan_inf"] = False

        return result

    except Exception as e:
        result["phase_signal_flag"] = "error"
        result["error"] = repr(e)
        return result

# ----------------------------
# 3. Build phase signals
# ----------------------------

records = []

print("Generating improved phase-speed signals...")
print("This reads motion files and writes smaller phase_signal files.")
print()

for _, row in ok_df.iterrows():
    records.append(process_one_motion(row))

    if len(records) % 500 == 0:
        print(f"Processed {len(records)}/{len(ok_df)} phase signals...")

phase_signal_df = pd.DataFrame(records)
phase_signal_df.to_csv(PHASE_SIGNAL_MANIFEST_PATH, index=False)

print()
print("=" * 80)
print("PHASE 4B OUTPUT SAVED")
print("=" * 80)
print(f"Phase signal directory: {PHASE_SIGNAL_DIR}")
print(f"Phase signal manifest:  {PHASE_SIGNAL_MANIFEST_PATH}")
print()

# ----------------------------
# 4. Plot comparisons
# ----------------------------

plot_df = phase_signal_df[
    phase_signal_df["phase_signal_flag"].isin(
        ["phase_signal_ok", "many_peaks_noisy", "too_flat_or_always_active", "no_clear_peak", "flat_zero"]
    )
].copy()

random.seed(42)
np.random.seed(42)

selected_rows = []

def add_sample_rows(df, bucket, n):
    if len(df) == 0:
        print(f"Warning: no rows for bucket {bucket}")
        return

    n = min(n, len(df))
    sample = df.sample(n=n, random_state=42).copy()
    sample["plot_bucket"] = bucket
    selected_rows.append(sample)

add_sample_rows(plot_df[plot_df["phase_signal_flag"] == "phase_signal_ok"], "phase_signal_ok", 8)
add_sample_rows(plot_df[plot_df["phase_signal_flag"] == "many_peaks_noisy"], "many_peaks_noisy", 5)
add_sample_rows(plot_df[plot_df["phase_signal_flag"] == "too_flat_or_always_active"], "too_flat_or_always_active", 5)
add_sample_rows(plot_df.sort_values("phase_speed_max", ascending=False).head(50), "high_motion", 5)
add_sample_rows(plot_df.sort_values("phase_speed_max", ascending=True).head(50), "low_motion", 5)

selected_df = pd.concat(selected_rows, ignore_index=True)
selected_df = selected_df.drop_duplicates(subset=["phase_signal_path"]).reset_index(drop=True)

def plot_phase_signal(row, show=False):
    path = Path(row["phase_signal_path"])

    with np.load(path, allow_pickle=False) as npz:
        phase_speed = np.asarray(npz["phase_speed"], dtype=np.float32)
        phase_speed_raw = np.asarray(npz["phase_speed_raw"], dtype=np.float32)
        centroid_norm = np.asarray(npz["centroid_norm"], dtype=np.float32)
        hand_norm = np.asarray(npz["hand_norm"], dtype=np.float32)
        old_smoothed = np.asarray(npz["old_smoothed_speed"], dtype=np.float32)

    old_norm = robust_01(old_smoothed)
    frames = np.arange(len(phase_speed))

    plt.figure(figsize=(12, 4))
    plt.plot(frames, old_norm, label="old_smoothed_speed_norm", alpha=0.45)
    plt.plot(frames, centroid_norm, label="centroid_norm", alpha=0.35)
    plt.plot(frames, hand_norm, label="hand_norm", alpha=0.30)
    plt.plot(frames, phase_speed_raw, label="phase_speed_raw_blend", alpha=0.45)
    plt.plot(frames, phase_speed, label="phase_speed_v2", linewidth=2.8)

    if float(np.max(phase_speed)) > 1e-8:
        plt.axhline(0.30 * float(np.max(phase_speed)), linestyle="--", linewidth=1.2, label="30% of v2 max")

    title = (
        f"{row['plot_bucket']} | {row['split']} | {row['gloss']} | "
        f"{row['video_stem']} | T={int(row['n_frames'])} | flag={row['phase_signal_flag']}"
    )

    plt.title(title)
    plt.xlabel("Frame")
    plt.ylabel("Normalized motion")
    plt.legend(loc="upper right")
    plt.tight_layout()

    safe_name = str(row["video_stem"]).replace("/", "_").replace("\\", "_").replace(" ", "_")
    save_path = PHASE_SIGNAL_PLOT_DIR / f"{row['plot_bucket']}__{safe_name}.png"

    plt.savefig(save_path, dpi=140, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return save_path

saved_plot_paths = []

print("Generating comparison plots...")
for idx, row in selected_df.iterrows():
    show_now = idx < 6
    saved_plot_paths.append(str(plot_phase_signal(row, show=show_now)))

print(f"Saved comparison plots: {len(saved_plot_paths)}")
print()

# ----------------------------
# 5. Summary
# ----------------------------

print("=" * 80)
print("PHASE SIGNAL SUMMARY")
print("=" * 80)

print("Phase signal flag counts:")
print(phase_signal_df["phase_signal_flag"].value_counts(dropna=False))
print()

print("Phase speed max summary:")
print(phase_signal_df["phase_speed_max"].describe())
print()

print("Active ratio 30% summary:")
print(phase_signal_df["active_ratio_30pct"].describe())
print()

print("Simple peak count v2 summary:")
print(phase_signal_df["simple_peak_count_v2"].describe())
print()

print("Smoothing window counts:")
print(phase_signal_df["smoothing_window"].value_counts(dropna=False).sort_index())
print()

print("NaN/Inf status:")
print(phase_signal_df["has_nan_inf"].value_counts(dropna=False))
print()

if OLD_CURVE_AUDIT_PATH.exists():
    old_audit = pd.read_csv(OLD_CURVE_AUDIT_PATH)
    print("=" * 80)
    print("OLD VS NEW FLAG COMPARISON")
    print("=" * 80)
    print("Old curve quality flags:")
    print(old_audit["curve_quality_flag"].value_counts(dropna=False))
    print()
    print("New phase signal flags:")
    print(phase_signal_df["phase_signal_flag"].value_counts(dropna=False))
    print()

print("=" * 80)
print("PLOTTED PHASE-SIGNAL EXAMPLES")
print("=" * 80)

display_cols = [
    "plot_bucket",
    "split",
    "gloss",
    "video_stem",
    "n_frames",
    "phase_speed_max",
    "active_ratio_30pct",
    "simple_peak_count_v2",
    "phase_signal_flag",
    "phase_signal_path",
]

display(selected_df[display_cols])

print()
print("First 10 saved comparison plot paths:")
for p in saved_plot_paths[:10]:
    print(p)

print()
print("=" * 80)
print("PHASE 4B COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Phase signal flag counts")
print("2. Phase speed max summary")
print("3. Active ratio 30% summary")
print("4. Simple peak count v2 summary")
print("5. Smoothing window counts")
print("6. NaN/Inf status")
print("7. Old vs new flag comparison")
print("8. Plotted phase-signal examples")
print()
print("Also visually check the first 6 plots and tell me:")
print("- Is phase_speed_v2 smoother than old_smoothed_speed_norm?")
print("- Does it still rise/fall with the signing movement?")
print("- Does it look too flat?")
print()
print("Do NOT generate phase labels yet.")

In [ ]:
# ============================================================
# PHASE 5A — 4-CLASS PHASE PSEUDO-LABEL GENERATION
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import re

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

PHASE_SIGNAL_MANIFEST_PATH = MANIFEST_DIR / "phase_signal_manifest.csv"

PHASE_LABEL_DIR = BASE_DIR / "phase_labels"
PHASE_LABEL_MANIFEST_PATH = MANIFEST_DIR / "phase_label_manifest.csv"

PHASE_LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 5A — 4-CLASS PHASE PSEUDO-LABEL GENERATION")
print("=" * 80)

if not PHASE_SIGNAL_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing phase signal manifest: {PHASE_SIGNAL_MANIFEST_PATH}")

phase_signal_df = pd.read_csv(PHASE_SIGNAL_MANIFEST_PATH)

print(f"Loaded phase signal manifest: {PHASE_SIGNAL_MANIFEST_PATH}")
print(f"Rows: {len(phase_signal_df)}")
print(f"Phase label output directory: {PHASE_LABEL_DIR}")
print()

print("Phase signal flag counts:")
print(phase_signal_df["phase_signal_flag"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Helper functions
# ----------------------------

def safe_stem(s):
    s = str(s)
    s = s.replace("/", "_").replace("\\", "_")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def get_segments(mask):
    """
    Returns list of (start, end) inclusive segments where mask is True.
    """
    mask = np.asarray(mask, dtype=bool)
    segments = []

    if len(mask) == 0:
        return segments

    in_seg = False
    start = 0

    for i, val in enumerate(mask):
        if val and not in_seg:
            start = i
            in_seg = True
        elif not val and in_seg:
            segments.append((start, i - 1))
            in_seg = False

    if in_seg:
        segments.append((start, len(mask) - 1))

    return segments

def fill_small_gaps(mask, max_gap):
    """
    Converts small False gaps between True regions into True.
    """
    mask = np.asarray(mask, dtype=bool).copy()

    if len(mask) == 0:
        return mask

    true_segments = get_segments(mask)

    if len(true_segments) <= 1:
        return mask

    for i in range(len(true_segments) - 1):
        end1 = true_segments[i][1]
        start2 = true_segments[i + 1][0]
        gap_len = start2 - end1 - 1

        if 0 < gap_len <= max_gap:
            mask[end1 + 1:start2] = True

    return mask

def remove_small_islands(mask, min_len):
    """
    Removes very short True regions.
    """
    mask = np.asarray(mask, dtype=bool).copy()
    segments = get_segments(mask)

    for start, end in segments:
        if (end - start + 1) < min_len:
            mask[start:end + 1] = False

    return mask

def clamp_int(x, lo, hi):
    return int(max(lo, min(hi, x)))

def choose_active_region(speed):
    """
    Finds the overall active signing region using adaptive thresholds.
    Returns active_start, active_end, active_mask, active_status.
    """
    speed = np.asarray(speed, dtype=np.float32)
    speed[~np.isfinite(speed)] = 0.0

    T = len(speed)

    if T < 16:
        return None, None, np.zeros(T, dtype=bool), "too_short"

    max_val = float(np.max(speed))

    if max_val <= 1e-8:
        return None, None, np.zeros(T, dtype=bool), "flat_zero"

    # Try several thresholds and pick the one with reasonable active ratio.
    candidate_factors = [0.20, 0.25, 0.30, 0.15, 0.10, 0.35]

    best = None

    for factor in candidate_factors:
        thr = factor * max_val
        mask = speed >= thr

        gap = max(2, int(round(0.04 * T)))
        min_island = max(2, int(round(0.03 * T)))

        mask = fill_small_gaps(mask, max_gap=gap)
        mask = remove_small_islands(mask, min_len=min_island)

        if mask.sum() == 0:
            continue

        segments = get_segments(mask)
        if len(segments) == 0:
            continue

        # For isolated signs, use first to last active movement.
        start = segments[0][0]
        end = segments[-1][1]
        span_len = end - start + 1
        span_ratio = span_len / T

        # Prefer a middle active-span ratio, but still allow broad signs.
        score = abs(span_ratio - 0.55)

        candidate = {
            "factor": factor,
            "start": start,
            "end": end,
            "span_ratio": span_ratio,
            "score": score,
        }

        if best is None:
            best = candidate
        else:
            # Prefer candidates in a reasonable interval.
            best_reasonable = 0.20 <= best["span_ratio"] <= 0.85
            cand_reasonable = 0.20 <= candidate["span_ratio"] <= 0.85

            if cand_reasonable and not best_reasonable:
                best = candidate
            elif cand_reasonable == best_reasonable and candidate["score"] < best["score"]:
                best = candidate

    if best is None:
        # Fallback: use central region around max.
        peak = int(np.argmax(speed))
        half = max(4, int(round(0.25 * T)))
        start = clamp_int(peak - half, 0, T - 1)
        end = clamp_int(peak + half, 0, T - 1)
        mask = np.zeros(T, dtype=bool)
        mask[start:end + 1] = True
        return start, end, mask, "fallback_active_region"

    active_mask = np.zeros(T, dtype=bool)
    active_mask[best["start"]:best["end"] + 1] = True

    status = "active_ok"
    if best["span_ratio"] > 0.85:
        status = "active_very_broad"
    elif best["span_ratio"] < 0.20:
        status = "active_very_narrow"

    return best["start"], best["end"], active_mask, status

def choose_stroke_region(speed, active_start, active_end):
    """
    Finds one stroke region inside the active region.
    The stroke is centered around strongest movement and may include nearby high-motion islands.
    """
    speed = np.asarray(speed, dtype=np.float32)
    speed[~np.isfinite(speed)] = 0.0

    T = len(speed)
    active_len = active_end - active_start + 1

    if active_len < 5:
        return None, None, "active_too_short_for_stroke"

    active_speed = speed[active_start:active_end + 1]
    local_peak = int(np.argmax(active_speed))
    peak_idx = active_start + local_peak
    active_max = float(np.max(active_speed))

    if active_max <= 1e-8:
        return None, None, "no_stroke_peak"

    min_stroke_len = max(3, int(round(0.12 * active_len)))
    max_stroke_len = max(min_stroke_len, int(round(0.70 * active_len)))

    best = None

    # Try high thresholds first, then relax.
    for factor in [0.65, 0.55, 0.45, 0.35]:
        high_mask = np.zeros(T, dtype=bool)
        high_mask[active_start:active_end + 1] = speed[active_start:active_end + 1] >= factor * active_max

        gap = max(1, int(round(0.05 * active_len)))
        min_island = max(2, int(round(0.04 * active_len)))

        high_mask = fill_small_gaps(high_mask, max_gap=gap)
        high_mask = remove_small_islands(high_mask, min_len=min_island)

        segments = get_segments(high_mask)
        segments = [(s, e) for s, e in segments if e >= active_start and s <= active_end]

        if len(segments) == 0:
            continue

        # Keep high-motion islands around the main peak.
        # If no island contains the peak, choose the closest island to the peak.
        containing = [(s, e) for s, e in segments if s <= peak_idx <= e]

        if containing:
            center_seg = containing[0]
        else:
            center_seg = min(
                segments,
                key=lambda se: min(abs(peak_idx - se[0]), abs(peak_idx - se[1]))
            )

        # Merge neighboring high-motion islands if gaps are not huge.
        merge_gap = max(2, int(round(0.10 * active_len)))
        stroke_start, stroke_end = center_seg

        changed = True
        while changed:
            changed = False
            for s, e in segments:
                if e < stroke_start and (stroke_start - e - 1) <= merge_gap:
                    stroke_start = min(stroke_start, s)
                    changed = True
                elif s > stroke_end and (s - stroke_end - 1) <= merge_gap:
                    stroke_end = max(stroke_end, e)
                    changed = True

        stroke_start = max(stroke_start, active_start)
        stroke_end = min(stroke_end, active_end)
        stroke_len = stroke_end - stroke_start + 1

        # If the stroke region is reasonable, use it.
        if stroke_len >= min_stroke_len and stroke_len <= max_stroke_len:
            best = (stroke_start, stroke_end, f"stroke_ok_factor_{factor}")
            break

        # Keep best fallback candidate.
        if best is None:
            best = (stroke_start, stroke_end, f"stroke_fallback_factor_{factor}")

    if best is None:
        # Last fallback: create a peak-centered stroke window.
        stroke_len = max(min_stroke_len, int(round(0.30 * active_len)))
        stroke_len = min(stroke_len, active_len)
        stroke_start = peak_idx - stroke_len // 2
        stroke_start = clamp_int(stroke_start, active_start, active_end - stroke_len + 1)
        stroke_end = stroke_start + stroke_len - 1
        best = (stroke_start, stroke_end, "stroke_peak_window_fallback")

    stroke_start, stroke_end, stroke_status = best

    # Enforce at least one preparation and one retraction frame when possible.
    if active_len >= 8:
        if stroke_start <= active_start:
            stroke_start = active_start + 1
        if stroke_end >= active_end:
            stroke_end = active_end - 1

        if stroke_start > stroke_end:
            peak_idx = clamp_int(peak_idx, active_start + 1, active_end - 1)
            stroke_start = peak_idx
            stroke_end = peak_idx
            stroke_status = "stroke_forced_single_peak"

    return int(stroke_start), int(stroke_end), stroke_status

def generate_phase_labels(speed):
    """
    Generates frame labels:
      0 = background
      1 = preparation
      2 = stroke
      3 = retraction
    """
    speed = np.asarray(speed, dtype=np.float32)
    speed[~np.isfinite(speed)] = 0.0

    T = len(speed)
    labels = np.zeros(T, dtype=np.int64)

    active_start, active_end, active_mask, active_status = choose_active_region(speed)

    if active_start is None or active_end is None:
        return labels, {
            "label_status": "invalid",
            "quality_flag": active_status,
            "active_start": -1,
            "active_end": -1,
            "stroke_start": -1,
            "stroke_end": -1,
            "prep_len": 0,
            "stroke_len": 0,
            "retract_len": 0,
            "background_len": int(T),
            "active_len": 0,
            "active_span_ratio": 0.0,
            "active_status": active_status,
            "stroke_status": "",
        }

    stroke_start, stroke_end, stroke_status = choose_stroke_region(speed, active_start, active_end)

    if stroke_start is None or stroke_end is None:
        return labels, {
            "label_status": "invalid",
            "quality_flag": stroke_status,
            "active_start": active_start,
            "active_end": active_end,
            "stroke_start": -1,
            "stroke_end": -1,
            "prep_len": 0,
            "stroke_len": 0,
            "retract_len": 0,
            "background_len": int(T),
            "active_len": int(active_end - active_start + 1),
            "active_span_ratio": float((active_end - active_start + 1) / max(T, 1)),
            "active_status": active_status,
            "stroke_status": stroke_status,
        }

    # Assign phases.
    labels[active_start:stroke_start] = 1
    labels[stroke_start:stroke_end + 1] = 2
    labels[stroke_end + 1:active_end + 1] = 3

    prep_len = int(np.sum(labels == 1))
    stroke_len = int(np.sum(labels == 2))
    retract_len = int(np.sum(labels == 3))
    background_len = int(np.sum(labels == 0))
    active_len = int(active_end - active_start + 1)
    active_span_ratio = float(active_len / max(T, 1))

    # Quality flag.
    if stroke_len == 0:
        label_status = "invalid"
        quality_flag = "empty_stroke"
    elif prep_len == 0 or retract_len == 0:
        label_status = "generated_warning"
        quality_flag = "missing_prep_or_retract"
    elif active_span_ratio > 0.90:
        label_status = "generated_warning"
        quality_flag = "active_region_very_broad"
    elif stroke_len / max(active_len, 1) > 0.75:
        label_status = "generated_warning"
        quality_flag = "stroke_region_very_broad"
    else:
        label_status = "generated"
        quality_flag = "good"

    stats = {
        "label_status": label_status,
        "quality_flag": quality_flag,
        "active_start": int(active_start),
        "active_end": int(active_end),
        "stroke_start": int(stroke_start),
        "stroke_end": int(stroke_end),
        "prep_len": prep_len,
        "stroke_len": stroke_len,
        "retract_len": retract_len,
        "background_len": background_len,
        "active_len": active_len,
        "active_span_ratio": active_span_ratio,
        "active_status": active_status,
        "stroke_status": stroke_status,
    }

    return labels, stats

def process_one_phase_signal(row):
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "n_frames": row["n_frames"],
        "phase_signal_flag": row["phase_signal_flag"],
        "phase_signal_path": row["phase_signal_path"],
        "phase_label_path": "",
        "label_status": "unknown",
        "quality_flag": "",
        "active_start": -1,
        "active_end": -1,
        "stroke_start": -1,
        "stroke_end": -1,
        "prep_len": 0,
        "stroke_len": 0,
        "retract_len": 0,
        "background_len": 0,
        "active_len": 0,
        "active_span_ratio": 0.0,
        "active_status": "",
        "stroke_status": "",
        "label_0_count": 0,
        "label_1_count": 0,
        "label_2_count": 0,
        "label_3_count": 0,
        "has_valid_order": False,
        "has_nan_inf": False,
        "error": "",
    }

    try:
        signal_path = Path(row["phase_signal_path"])

        if not signal_path.exists():
            result["label_status"] = "error"
            result["quality_flag"] = "missing_phase_signal_file"
            result["error"] = f"Missing: {signal_path}"
            return result

        with np.load(signal_path, allow_pickle=False) as npz:
            phase_speed = np.asarray(npz["phase_speed"], dtype=np.float32)
            fps = float(np.asarray(npz["fps"]).item()) if "fps" in npz.files else 30.0

        phase_speed[~np.isfinite(phase_speed)] = 0.0

        if np.isnan(phase_speed).any() or np.isinf(phase_speed).any():
            result["label_status"] = "invalid"
            result["quality_flag"] = "nan_inf_phase_speed"
            result["has_nan_inf"] = True
            return result

        labels, stats = generate_phase_labels(phase_speed)

        result.update(stats)

        result["label_0_count"] = int(np.sum(labels == 0))
        result["label_1_count"] = int(np.sum(labels == 1))
        result["label_2_count"] = int(np.sum(labels == 2))
        result["label_3_count"] = int(np.sum(labels == 3))

        # Check order: preparation before stroke before retraction.
        non_bg = labels[labels != 0]
        if len(non_bg) > 0:
            # Remove repeated consecutive labels.
            compressed = [int(non_bg[0])]
            for v in non_bg[1:]:
                if int(v) != compressed[-1]:
                    compressed.append(int(v))
            result["has_valid_order"] = compressed == [1, 2, 3]
        else:
            result["has_valid_order"] = False

        # If generated but order invalid, downgrade.
        if result["label_status"] in ["generated", "generated_warning"] and not result["has_valid_order"]:
            result["label_status"] = "invalid"
            result["quality_flag"] = "invalid_phase_order"

        label_path = PHASE_LABEL_DIR / f"{safe_stem(row['video_stem'])}.npz"

        np.savez_compressed(
            label_path,
            labels=labels.astype(np.int64),
            phase_speed=phase_speed.astype(np.float32),
            active_start=np.int64(result["active_start"]),
            active_end=np.int64(result["active_end"]),
            stroke_start=np.int64(result["stroke_start"]),
            stroke_end=np.int64(result["stroke_end"]),
            fps=np.float32(fps),
            video_stem=str(row["video_stem"]),
            gloss=str(row["gloss"]),
            split=str(row["split"]),
            label_id=np.int64(row["label_id"]),
            phase_signal_flag=str(row["phase_signal_flag"]),
            label_status=str(result["label_status"]),
            quality_flag=str(result["quality_flag"]),
        )

        result["phase_label_path"] = str(label_path)

        return result

    except Exception as e:
        result["label_status"] = "error"
        result["quality_flag"] = "exception"
        result["error"] = repr(e)
        return result

# ----------------------------
# 3. Generate phase labels
# ----------------------------

records = []

print("Generating 4-class phase pseudo-labels...")
print("This does not train anything.")
print()

for _, row in phase_signal_df.iterrows():
    records.append(process_one_phase_signal(row))

    if len(records) % 500 == 0:
        print(f"Processed {len(records)}/{len(phase_signal_df)} phase labels...")

phase_label_df = pd.DataFrame(records)
phase_label_df.to_csv(PHASE_LABEL_MANIFEST_PATH, index=False)

# ----------------------------
# 4. Summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 5A OUTPUT SAVED")
print("=" * 80)
print(f"Phase label directory: {PHASE_LABEL_DIR}")
print(f"Phase label manifest:  {PHASE_LABEL_MANIFEST_PATH}")
print()

print("=" * 80)
print("PHASE LABEL GENERATION SUMMARY")
print("=" * 80)

print("Label status counts:")
print(phase_label_df["label_status"].value_counts(dropna=False))
print()

print("Quality flag counts:")
print(phase_label_df["quality_flag"].value_counts(dropna=False))
print()

print("Original phase signal flag counts:")
print(phase_label_df["phase_signal_flag"].value_counts(dropna=False))
print()

print("Valid order counts:")
print(phase_label_df["has_valid_order"].value_counts(dropna=False))
print()

print("Phase length summaries:")
for col in ["background_len", "prep_len", "stroke_len", "retract_len", "active_len", "active_span_ratio"]:
    print()
    print(col)
    print(phase_label_df[col].describe())

print()
print("Total label counts across all generated files:")
print("background / 0:", int(phase_label_df["label_0_count"].sum()))
print("preparation / 1:", int(phase_label_df["label_1_count"].sum()))
print("stroke / 2:", int(phase_label_df["label_2_count"].sum()))
print("retraction / 3:", int(phase_label_df["label_3_count"].sum()))
print()

print("NaN/Inf counts:")
print(phase_label_df["has_nan_inf"].value_counts(dropna=False))
print()

# ----------------------------
# 5. First examples
# ----------------------------

print("=" * 80)
print("FIRST 5 GENERATED LABEL FILES")
print("=" * 80)

generated_examples = phase_label_df[phase_label_df["label_status"].isin(["generated", "generated_warning"])].head(5)

if len(generated_examples) == 0:
    print("No generated labels found. Something is wrong.")
else:
    for _, r in generated_examples.iterrows():
        print("-" * 80)
        print(f"split:             {r['split']}")
        print(f"gloss:             {r['gloss']}")
        print(f"video_stem:        {r['video_stem']}")
        print(f"phase_signal_flag: {r['phase_signal_flag']}")
        print(f"label_status:      {r['label_status']}")
        print(f"quality_flag:      {r['quality_flag']}")
        print(f"phase_label_path:  {r['phase_label_path']}")
        print(f"n_frames:          {r['n_frames']}")
        print(f"active:            {r['active_start']} → {r['active_end']}")
        print(f"stroke:            {r['stroke_start']} → {r['stroke_end']}")
        print(f"lengths:           bg={r['background_len']}, prep={r['prep_len']}, stroke={r['stroke_len']}, retract={r['retract_len']}")
        print(f"valid_order:       {r['has_valid_order']}")

# ----------------------------
# 6. Invalid / warning preview
# ----------------------------

print()
print("=" * 80)
print("INVALID / WARNING PREVIEW")
print("=" * 80)

problem_df = phase_label_df[
    (phase_label_df["label_status"] != "generated")
    | (phase_label_df["quality_flag"] != "good")
].copy()

if len(problem_df) == 0:
    print("No invalid/warning rows.")
else:
    print(f"Problem/warning rows: {len(problem_df)}")
    display_cols = [
        "row_id",
        "split",
        "gloss",
        "video_stem",
        "n_frames",
        "phase_signal_flag",
        "label_status",
        "quality_flag",
        "active_status",
        "stroke_status",
        "prep_len",
        "stroke_len",
        "retract_len",
        "active_span_ratio",
        "has_valid_order",
        "phase_label_path",
        "error",
    ]
    display(problem_df[display_cols].head(40))

print()
print("=" * 80)
print("PHASE 5A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Label status counts")
print("2. Quality flag counts")
print("3. Valid order counts")
print("4. Phase length summaries")
print("5. Total label counts")
print("6. First 5 generated label files")
print("7. Invalid / warning preview")
print()
print("Do NOT train a Phase TCN yet.")
print("Next step will be Phase 6A: visualize phase labels over speed curves.")

In [ ]:
# ============================================================
# PHASE 6A — PHASE LABEL VISUALIZATION
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

PHASE_LABEL_MANIFEST_PATH = MANIFEST_DIR / "phase_label_manifest.csv"

PHASE_LABEL_PLOT_DIR = BASE_DIR / "plots" / "phase_labels"
PHASE_LABEL_VIS_MANIFEST_PATH = MANIFEST_DIR / "phase_label_visualization_manifest.csv"

PHASE_LABEL_PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 6A — PHASE LABEL VISUALIZATION")
print("=" * 80)

if not PHASE_LABEL_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing phase label manifest: {PHASE_LABEL_MANIFEST_PATH}")

phase_df = pd.read_csv(PHASE_LABEL_MANIFEST_PATH)

print(f"Loaded phase label manifest: {PHASE_LABEL_MANIFEST_PATH}")
print(f"Rows: {len(phase_df)}")
print(f"Plot output directory: {PHASE_LABEL_PLOT_DIR}")
print()

print("Label status counts:")
print(phase_df["label_status"].value_counts(dropna=False))
print()

print("Quality flag counts:")
print(phase_df["quality_flag"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Select examples to visualize
# ----------------------------

random.seed(42)
np.random.seed(42)

selected_parts = []

def add_examples(df, bucket_name, n):
    if len(df) == 0:
        print(f"Warning: no rows for bucket: {bucket_name}")
        return

    n = min(n, len(df))
    sample = df.sample(n=n, random_state=42).copy()
    sample["plot_bucket"] = bucket_name
    selected_parts.append(sample)

good_df = phase_df[
    (phase_df["label_status"] == "generated")
    & (phase_df["quality_flag"] == "good")
    & (phase_df["has_valid_order"] == True)
].copy()

warning_active_df = phase_df[
    phase_df["quality_flag"] == "active_region_very_broad"
].copy()

warning_stroke_df = phase_df[
    phase_df["quality_flag"] == "stroke_region_very_broad"
].copy()

invalid_df = phase_df[
    phase_df["label_status"] == "invalid"
].copy()

many_peaks_good_df = phase_df[
    (phase_df["phase_signal_flag"] == "many_peaks_noisy")
    & (phase_df["label_status"] == "generated")
    & (phase_df["quality_flag"] == "good")
].copy()

short_phase_df = phase_df[
    (
        (phase_df["prep_len"] <= 2)
        | (phase_df["retract_len"] <= 2)
        | (phase_df["stroke_len"] <= 3)
    )
    & (phase_df["label_status"].isin(["generated", "generated_warning"]))
].copy()

# Buckets
add_examples(good_df, "good_random", 10)
add_examples(many_peaks_good_df, "good_many_peaks_original", 5)
add_examples(warning_active_df, "warning_active_broad", 6)
add_examples(warning_stroke_df, "warning_stroke_broad", 6)
add_examples(short_phase_df, "short_phase_edges", 6)
add_examples(invalid_df, "invalid", 2)

selected_df = pd.concat(selected_parts, ignore_index=True)
selected_df = selected_df.drop_duplicates(subset=["phase_label_path"]).reset_index(drop=True)

print(f"Selected examples to plot: {len(selected_df)}")
print()

# ----------------------------
# 3. Plot helper
# ----------------------------

PHASE_NAMES = {
    0: "background",
    1: "preparation",
    2: "stroke",
    3: "retraction",
}

PHASE_COLORS = {
    0: "lightgray",
    1: "gold",
    2: "tomato",
    3: "skyblue",
}

def get_label_segments(labels):
    """
    Returns list of (label, start, end) inclusive segments.
    """
    labels = np.asarray(labels, dtype=np.int64)

    if len(labels) == 0:
        return []

    segments = []
    current = int(labels[0])
    start = 0

    for i in range(1, len(labels)):
        val = int(labels[i])
        if val != current:
            segments.append((current, start, i - 1))
            current = val
            start = i

    segments.append((current, start, len(labels) - 1))
    return segments

def plot_phase_labels(row, show=False):
    label_path = Path(row["phase_label_path"])

    if not label_path.exists():
        raise FileNotFoundError(f"Missing phase label file: {label_path}")

    with np.load(label_path, allow_pickle=False) as npz:
        labels = np.asarray(npz["labels"], dtype=np.int64)
        phase_speed = np.asarray(npz["phase_speed"], dtype=np.float32)
        active_start = int(np.asarray(npz["active_start"]).item())
        active_end = int(np.asarray(npz["active_end"]).item())
        stroke_start = int(np.asarray(npz["stroke_start"]).item())
        stroke_end = int(np.asarray(npz["stroke_end"]).item())

    labels[~np.isfinite(labels)] = 0
    phase_speed[~np.isfinite(phase_speed)] = 0.0

    T = len(labels)
    frames = np.arange(T)

    fig, ax = plt.subplots(figsize=(13, 4.5))

    # Background phase regions.
    for phase_id, start, end in get_label_segments(labels):
        ax.axvspan(
            start,
            end + 1,
            alpha=0.22,
            color=PHASE_COLORS.get(phase_id, "white"),
            label=PHASE_NAMES.get(phase_id, str(phase_id))
        )

    ax.plot(frames, phase_speed, linewidth=2.5, label="phase_speed")

    # Boundary lines
    if active_start >= 0:
        ax.axvline(active_start, linestyle="--", linewidth=1.2, label="active_start")
    if active_end >= 0:
        ax.axvline(active_end, linestyle="--", linewidth=1.2, label="active_end")
    if stroke_start >= 0:
        ax.axvline(stroke_start, linestyle="-.", linewidth=1.2, label="stroke_start")
    if stroke_end >= 0:
        ax.axvline(stroke_end, linestyle="-.", linewidth=1.2, label="stroke_end")

    # Avoid duplicate legend labels
    handles, labels_legend = ax.get_legend_handles_labels()
    unique = {}
    for h, l in zip(handles, labels_legend):
        if l not in unique:
            unique[l] = h

    title = (
        f"{row['plot_bucket']} | {row['split']} | {row['gloss']} | {row['video_stem']}\n"
        f"status={row['label_status']} | flag={row['quality_flag']} | "
        f"T={int(row['n_frames'])} | prep={int(row['prep_len'])}, "
        f"stroke={int(row['stroke_len'])}, retract={int(row['retract_len'])}, "
        f"active_ratio={float(row['active_span_ratio']):.2f}"
    )

    ax.set_title(title)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Phase speed")
    ax.set_ylim(bottom=0)
    ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize=8)
    plt.tight_layout()

    safe_name = str(row["video_stem"]).replace("/", "_").replace("\\", "_").replace(" ", "_")
    save_path = PHASE_LABEL_PLOT_DIR / f"{row['plot_bucket']}__{safe_name}.png"

    plt.savefig(save_path, dpi=140, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return save_path

# ----------------------------
# 4. Generate plots
# ----------------------------

saved_paths = []

print("Generating phase label visualizations...")
print("The first 8 plots will be displayed below.")
print()

for idx, row in selected_df.iterrows():
    show_now = idx < 8
    save_path = plot_phase_labels(row, show=show_now)
    saved_paths.append(str(save_path))

selected_df["phase_label_plot_path"] = saved_paths
selected_df.to_csv(PHASE_LABEL_VIS_MANIFEST_PATH, index=False)

# ----------------------------
# 5. Summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 6A OUTPUT SAVED")
print("=" * 80)
print(f"Phase label plots:                  {PHASE_LABEL_PLOT_DIR}")
print(f"Phase label visualization manifest: {PHASE_LABEL_VIS_MANIFEST_PATH}")
print()

print("=" * 80)
print("VISUALIZATION SELECTION SUMMARY")
print("=" * 80)

print("Plot bucket counts:")
print(selected_df["plot_bucket"].value_counts(dropna=False))
print()

print("Selected label status counts:")
print(selected_df["label_status"].value_counts(dropna=False))
print()

print("Selected quality flag counts:")
print(selected_df["quality_flag"].value_counts(dropna=False))
print()

print("=" * 80)
print("SELECTED EXAMPLES")
print("=" * 80)

display_cols = [
    "plot_bucket",
    "split",
    "gloss",
    "video_stem",
    "n_frames",
    "label_status",
    "quality_flag",
    "phase_signal_flag",
    "prep_len",
    "stroke_len",
    "retract_len",
    "active_span_ratio",
    "has_valid_order",
    "phase_label_plot_path",
]

display(selected_df[display_cols])

print()
print("First 15 saved plot paths:")
for p in saved_paths[:15]:
    print(p)

print()
print("=" * 80)
print("PHASE 6A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Plot bucket counts")
print("2. Selected label status counts")
print("3. Selected quality flag counts")
print("4. Selected examples table")
print()
print("Also answer these visual questions:")
print("A. In good_random examples, does stroke usually cover the strongest peak?")
print("B. Are preparation and retraction placed before/after stroke correctly?")
print("C. Are warning_active_broad examples obviously too wide?")
print("D. Are warning_stroke_broad examples obviously over-labeling stroke?")
print()
print("Do NOT train yet.")

In [ ]:
# ============================================================
# PHASE 7A — PHASE LABEL VALIDATION + CLEAN MANIFEST
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

PHASE_LABEL_MANIFEST_PATH = MANIFEST_DIR / "phase_label_manifest.csv"
PHASE_LABEL_VALIDATION_REPORT_PATH = MANIFEST_DIR / "phase_label_validation_report.csv"
PHASE_LABEL_CLEAN_MANIFEST_PATH = MANIFEST_DIR / "phase_label_manifest_clean.csv"

print("=" * 80)
print("PHASE 7A — PHASE LABEL VALIDATION + CLEAN MANIFEST")
print("=" * 80)

if not PHASE_LABEL_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing phase label manifest: {PHASE_LABEL_MANIFEST_PATH}")

phase_df = pd.read_csv(PHASE_LABEL_MANIFEST_PATH)

print(f"Loaded phase label manifest: {PHASE_LABEL_MANIFEST_PATH}")
print(f"Rows: {len(phase_df)}")
print()

# ----------------------------
# 2. Validation thresholds
# ----------------------------

MIN_TOTAL_FRAMES = 16
MIN_PREP_LEN = 2
MIN_STROKE_LEN = 4
MIN_RETRACT_LEN = 2
MAX_ACTIVE_SPAN_RATIO = 0.88
MIN_ACTIVE_SPAN_RATIO = 0.15
MAX_STROKE_ACTIVE_RATIO = 0.70

print("Validation thresholds:")
print(f"MIN_TOTAL_FRAMES:        {MIN_TOTAL_FRAMES}")
print(f"MIN_PREP_LEN:            {MIN_PREP_LEN}")
print(f"MIN_STROKE_LEN:          {MIN_STROKE_LEN}")
print(f"MIN_RETRACT_LEN:         {MIN_RETRACT_LEN}")
print(f"MIN_ACTIVE_SPAN_RATIO:   {MIN_ACTIVE_SPAN_RATIO}")
print(f"MAX_ACTIVE_SPAN_RATIO:   {MAX_ACTIVE_SPAN_RATIO}")
print(f"MAX_STROKE_ACTIVE_RATIO: {MAX_STROKE_ACTIVE_RATIO}")
print()

# ----------------------------
# 3. Helper functions
# ----------------------------

def validate_one_row(row):
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "n_frames": row["n_frames"],
        "phase_label_path": row["phase_label_path"],
        "original_label_status": row["label_status"],
        "original_quality_flag": row["quality_flag"],
        "phase_signal_flag": row["phase_signal_flag"],
        "has_valid_order_original": row["has_valid_order"],
        "validation_status": "unknown",
        "validation_reason": "",
        "usable_for_phase_tcn": False,
        "usable_for_visual_report": False,
        "label_length": np.nan,
        "phase_speed_length": np.nan,
        "unique_labels": "",
        "label_0_count_checked": 0,
        "label_1_count_checked": 0,
        "label_2_count_checked": 0,
        "label_3_count_checked": 0,
        "stroke_active_ratio": np.nan,
        "error": "",
    }

    try:
        # Basic manifest-level checks first.
        if row["label_status"] != "generated":
            result["validation_status"] = "reject"
            result["validation_reason"] = f"label_status_{row['label_status']}"
            return result

        if row["quality_flag"] != "good":
            result["validation_status"] = "reject"
            result["validation_reason"] = f"quality_flag_{row['quality_flag']}"
            return result

        if bool(row["has_valid_order"]) != True:
            result["validation_status"] = "reject"
            result["validation_reason"] = "invalid_phase_order_manifest"
            return result

        if int(row["n_frames"]) < MIN_TOTAL_FRAMES:
            result["validation_status"] = "reject"
            result["validation_reason"] = "too_few_total_frames"
            return result

        label_path = Path(row["phase_label_path"])
        if not label_path.exists():
            result["validation_status"] = "reject"
            result["validation_reason"] = "missing_phase_label_file"
            result["error"] = f"Missing: {label_path}"
            return result

        with np.load(label_path, allow_pickle=False) as npz:
            labels = np.asarray(npz["labels"], dtype=np.int64)
            phase_speed = np.asarray(npz["phase_speed"], dtype=np.float32)

        labels = labels.astype(np.int64)
        phase_speed = phase_speed.astype(np.float32)
        phase_speed[~np.isfinite(phase_speed)] = 0.0

        result["label_length"] = int(len(labels))
        result["phase_speed_length"] = int(len(phase_speed))

        if len(labels) != len(phase_speed):
            result["validation_status"] = "reject"
            result["validation_reason"] = "label_speed_length_mismatch"
            return result

        if len(labels) != int(row["n_frames"]):
            result["validation_status"] = "reject"
            result["validation_reason"] = "label_frame_length_mismatch"
            return result

        unique = sorted([int(x) for x in np.unique(labels)])
        result["unique_labels"] = ",".join(map(str, unique))

        if not set(unique).issubset({0, 1, 2, 3}):
            result["validation_status"] = "reject"
            result["validation_reason"] = "invalid_label_values"
            return result

        # Count labels from file directly.
        c0 = int(np.sum(labels == 0))
        c1 = int(np.sum(labels == 1))
        c2 = int(np.sum(labels == 2))
        c3 = int(np.sum(labels == 3))

        result["label_0_count_checked"] = c0
        result["label_1_count_checked"] = c1
        result["label_2_count_checked"] = c2
        result["label_3_count_checked"] = c3

        if c1 < MIN_PREP_LEN:
            result["validation_status"] = "reject"
            result["validation_reason"] = "prep_too_short"
            return result

        if c2 < MIN_STROKE_LEN:
            result["validation_status"] = "reject"
            result["validation_reason"] = "stroke_too_short"
            return result

        if c3 < MIN_RETRACT_LEN:
            result["validation_status"] = "reject"
            result["validation_reason"] = "retract_too_short"
            return result

        active_len = c1 + c2 + c3
        if active_len <= 0:
            result["validation_status"] = "reject"
            result["validation_reason"] = "empty_active_region"
            return result

        active_span_ratio = active_len / max(len(labels), 1)
        stroke_active_ratio = c2 / max(active_len, 1)

        result["stroke_active_ratio"] = float(stroke_active_ratio)

        if active_span_ratio < MIN_ACTIVE_SPAN_RATIO:
            result["validation_status"] = "reject"
            result["validation_reason"] = "active_region_too_narrow"
            return result

        if active_span_ratio > MAX_ACTIVE_SPAN_RATIO:
            result["validation_status"] = "reject"
            result["validation_reason"] = "active_region_too_broad"
            return result

        if stroke_active_ratio > MAX_STROKE_ACTIVE_RATIO:
            result["validation_status"] = "reject"
            result["validation_reason"] = "stroke_too_broad_inside_active"
            return result

        # Re-check compressed non-background order from file.
        non_bg = labels[labels != 0]
        if len(non_bg) == 0:
            result["validation_status"] = "reject"
            result["validation_reason"] = "no_non_background_labels"
            return result

        compressed = [int(non_bg[0])]
        for v in non_bg[1:]:
            v = int(v)
            if v != compressed[-1]:
                compressed.append(v)

        if compressed != [1, 2, 3]:
            result["validation_status"] = "reject"
            result["validation_reason"] = f"invalid_phase_order_file_{compressed}"
            return result

        # Passed all strict checks.
        result["validation_status"] = "clean"
        result["validation_reason"] = "passed_strict_validation"
        result["usable_for_phase_tcn"] = True
        result["usable_for_visual_report"] = True

        return result

    except Exception as e:
        result["validation_status"] = "reject"
        result["validation_reason"] = "exception"
        result["error"] = repr(e)
        return result

# ----------------------------
# 4. Run validation
# ----------------------------

records = []

print("Validating phase labels...")
print()

for i, row in phase_df.iterrows():
    records.append(validate_one_row(row))

    if (i + 1) % 500 == 0:
        print(f"Validated {i + 1}/{len(phase_df)} labels...")

validation_df = pd.DataFrame(records)

# ----------------------------
# 5. Merge validation back into phase manifest
# ----------------------------

merge_cols = [
    "row_id",
    "validation_status",
    "validation_reason",
    "usable_for_phase_tcn",
    "usable_for_visual_report",
    "label_length",
    "phase_speed_length",
    "unique_labels",
    "label_0_count_checked",
    "label_1_count_checked",
    "label_2_count_checked",
    "label_3_count_checked",
    "stroke_active_ratio",
    "error",
]

phase_validated_df = phase_df.merge(
    validation_df[merge_cols],
    on="row_id",
    how="left",
    suffixes=("", "_validation")
)

clean_df = phase_validated_df[
    phase_validated_df["usable_for_phase_tcn"] == True
].copy()

# Save outputs.
validation_df.to_csv(PHASE_LABEL_VALIDATION_REPORT_PATH, index=False)
clean_df.to_csv(PHASE_LABEL_CLEAN_MANIFEST_PATH, index=False)

# ----------------------------
# 6. Summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 7A OUTPUT SAVED")
print("=" * 80)
print(f"Validation report: {PHASE_LABEL_VALIDATION_REPORT_PATH}")
print(f"Clean manifest:    {PHASE_LABEL_CLEAN_MANIFEST_PATH}")
print()

print("=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

print("Validation status counts:")
print(phase_validated_df["validation_status"].value_counts(dropna=False))
print()

print("Validation reason counts:")
print(phase_validated_df["validation_reason"].value_counts(dropna=False))
print()

print("Usable for Phase TCN:")
print(phase_validated_df["usable_for_phase_tcn"].value_counts(dropna=False))
print()

print("Clean rows by split:")
print(clean_df["split"].value_counts(dropna=False))
print()

print("Clean rows by original phase signal flag:")
print(clean_df["phase_signal_flag"].value_counts(dropna=False))
print()

print("Clean rows by original quality flag:")
print(clean_df["quality_flag"].value_counts(dropna=False))
print()

print("Clean phase length summaries:")
for col in ["background_len", "prep_len", "stroke_len", "retract_len", "active_len", "active_span_ratio", "stroke_active_ratio"]:
    print()
    print(col)
    print(clean_df[col].describe())

print()
print("Clean label totals:")
print("background / 0:", int(clean_df["label_0_count_checked"].sum()))
print("preparation / 1:", int(clean_df["label_1_count_checked"].sum()))
print("stroke / 2:", int(clean_df["label_2_count_checked"].sum()))
print("retraction / 3:", int(clean_df["label_3_count_checked"].sum()))
print()

# ----------------------------
# 7. Rejected preview
# ----------------------------

rejected_df = phase_validated_df[
    phase_validated_df["usable_for_phase_tcn"] != True
].copy()

print("=" * 80)
print("REJECTED PREVIEW")
print("=" * 80)

if len(rejected_df) == 0:
    print("No rejected rows.")
else:
    print(f"Rejected rows: {len(rejected_df)}")
    display_cols = [
        "row_id",
        "split",
        "gloss",
        "video_stem",
        "n_frames",
        "label_status",
        "quality_flag",
        "phase_signal_flag",
        "prep_len",
        "stroke_len",
        "retract_len",
        "active_span_ratio",
        "stroke_active_ratio",
        "validation_status",
        "validation_reason",
        "phase_label_path",
        "error_validation",
    ]

    existing_display_cols = [c for c in display_cols if c in rejected_df.columns]
    display(rejected_df[existing_display_cols].head(50))

print()
print("=" * 80)
print("PHASE 7A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Validation status counts")
print("2. Validation reason counts")
print("3. Usable for Phase TCN")
print("4. Clean rows by split")
print("5. Clean rows by original phase signal flag")
print("6. Clean phase length summaries")
print("7. Clean label totals")
print("8. Rejected preview")
print()
print("Do NOT train yet.")

In [ ]:
# ============================================================
# PHASE 8A — PHASE TCN DATASET / DATALOADER SETUP
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

MOTION_MANIFEST_PATH = MANIFEST_DIR / "motion_manifest.csv"
PHASE_LABEL_CLEAN_MANIFEST_PATH = MANIFEST_DIR / "phase_label_manifest_clean.csv"

PHASE_TCN_DATASET_MANIFEST_PATH = MANIFEST_DIR / "phase_tcn_dataset_manifest.csv"
PHASE_TCN_FEATURE_STATS_PATH = MANIFEST_DIR / "phase_tcn_feature_stats.npz"
PHASE_TCN_CLASS_WEIGHTS_PATH = MANIFEST_DIR / "phase_tcn_class_weights.npy"

print("=" * 80)
print("PHASE 8A — PHASE TCN DATASET / DATALOADER SETUP")
print("=" * 80)

if not MOTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing motion manifest: {MOTION_MANIFEST_PATH}")

if not PHASE_LABEL_CLEAN_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing clean phase label manifest: {PHASE_LABEL_CLEAN_MANIFEST_PATH}")

motion_df = pd.read_csv(MOTION_MANIFEST_PATH)
clean_df = pd.read_csv(PHASE_LABEL_CLEAN_MANIFEST_PATH)

print(f"Loaded motion manifest:      {MOTION_MANIFEST_PATH}")
print(f"Motion rows:                 {len(motion_df)}")
print(f"Loaded clean phase manifest: {PHASE_LABEL_CLEAN_MANIFEST_PATH}")
print(f"Clean phase rows:            {len(clean_df)}")
print()

# ----------------------------
# 2. Merge clean labels with motion files
# ----------------------------

# Keep motion columns needed for training.
motion_keep = motion_df[
    [
        "row_id",
        "video_stem",
        "motion_path",
        "status",
        "feature_dim",
        "n_frames",
        "global_speed_max",
        "hand_speed_max",
        "smoothed_speed_max",
    ]
].copy()

motion_keep = motion_keep.rename(
    columns={
        "status": "motion_status",
        "n_frames": "motion_n_frames",
    }
)

dataset_df = clean_df.merge(
    motion_keep,
    on=["row_id", "video_stem"],
    how="left",
)

print("After merging clean phase labels with motion manifest:")
print(f"Rows: {len(dataset_df)}")
print()

# ----------------------------
# 3. Verify files and lengths
# ----------------------------

def verify_one_row(row):
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "motion_path": row.get("motion_path", ""),
        "phase_label_path": row["phase_label_path"],
        "verify_status": "unknown",
        "T_motion": np.nan,
        "T_label": np.nan,
        "feature_dim_loaded": np.nan,
        "error": "",
    }

    try:
        motion_path = Path(str(row.get("motion_path", "")))
        label_path = Path(str(row["phase_label_path"]))

        if not motion_path.exists():
            result["verify_status"] = "missing_motion_file"
            result["error"] = f"Missing motion: {motion_path}"
            return result

        if not label_path.exists():
            result["verify_status"] = "missing_label_file"
            result["error"] = f"Missing label: {label_path}"
            return result

        with np.load(motion_path, allow_pickle=False) as m:
            positions = np.asarray(m["positions"], dtype=np.float32)
            velocity = np.asarray(m["velocity"], dtype=np.float32)
            acceleration = np.asarray(m["acceleration"], dtype=np.float32)
            global_speed = np.asarray(m["global_speed"], dtype=np.float32)
            hand_speed = np.asarray(m["hand_speed"], dtype=np.float32)

        with np.load(label_path, allow_pickle=False) as l:
            labels = np.asarray(l["labels"], dtype=np.int64)
            phase_speed = np.asarray(l["phase_speed"], dtype=np.float32)

        T = positions.shape[0]

        result["T_motion"] = int(T)
        result["T_label"] = int(len(labels))
        result["feature_dim_loaded"] = int(
            positions.shape[1]
            + velocity.shape[1]
            + acceleration.shape[1]
            + 1
            + 1
            + 1
        )

        # Length checks.
        if len(labels) != T:
            result["verify_status"] = "length_mismatch_labels"
            return result

        if len(phase_speed) != T:
            result["verify_status"] = "length_mismatch_phase_speed"
            return result

        if velocity.shape[0] != T or acceleration.shape[0] != T:
            result["verify_status"] = "length_mismatch_motion_arrays"
            return result

        if len(global_speed) != T or len(hand_speed) != T:
            result["verify_status"] = "length_mismatch_speed_arrays"
            return result

        # NaN checks.
        arrays = [positions, velocity, acceleration, global_speed, hand_speed, phase_speed, labels]
        for arr in arrays:
            if np.isnan(arr).any() or np.isinf(arr).any():
                result["verify_status"] = "nan_inf_found"
                return result

        unique_labels = set(np.unique(labels).tolist())
        if not unique_labels.issubset({0, 1, 2, 3}):
            result["verify_status"] = "invalid_label_values"
            return result

        result["verify_status"] = "ok"
        return result

    except Exception as e:
        result["verify_status"] = "exception"
        result["error"] = repr(e)
        return result

print("Verifying merged dataset rows...")
verify_records = []

for i, row in dataset_df.iterrows():
    verify_records.append(verify_one_row(row))

    if (i + 1) % 500 == 0:
        print(f"Verified {i + 1}/{len(dataset_df)} rows...")

verify_df = pd.DataFrame(verify_records)

dataset_verified_df = dataset_df.merge(
    verify_df[
        [
            "row_id",
            "verify_status",
            "T_motion",
            "T_label",
            "feature_dim_loaded",
            "error",
        ]
    ],
    on="row_id",
    how="left",
    suffixes=("", "_verify"),
)

ok_dataset_df = dataset_verified_df[dataset_verified_df["verify_status"] == "ok"].copy()

dataset_verified_df.to_csv(PHASE_TCN_DATASET_MANIFEST_PATH, index=False)

print()
print("=" * 80)
print("DATASET VERIFICATION SUMMARY")
print("=" * 80)

print("Verify status counts:")
print(dataset_verified_df["verify_status"].value_counts(dropna=False))
print()

print("OK rows by split:")
print(ok_dataset_df["split"].value_counts(dropna=False))
print()

print("Feature dimension loaded summary:")
print(ok_dataset_df["feature_dim_loaded"].describe())
print()

print("Sequence length summary:")
print(ok_dataset_df["T_motion"].describe())
print()

if len(ok_dataset_df) == 0:
    raise ValueError("No valid rows after verification. Stop.")

# ----------------------------
# 4. Feature-building function
# ----------------------------

def load_phase_tcn_features(motion_path, label_path):
    """
    Returns:
      X: (T, 444) float32
      y: (T,) int64
    """
    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(label_path, allow_pickle=False) as l:
        labels = np.asarray(l["labels"], dtype=np.int64)
        phase_speed = np.asarray(l["phase_speed"], dtype=np.float32).reshape(-1, 1)

    X = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X[~np.isfinite(X)] = 0.0

    return X, labels

# ----------------------------
# 5. Compute train feature normalization stats
# ----------------------------

train_df = ok_dataset_df[ok_dataset_df["split"] == "train"].copy()

if len(train_df) == 0:
    raise ValueError("No train rows found in clean dataset.")

print("Computing train-set feature mean/std...")
print("This streams files without loading the whole dataset into RAM.")
print()

feature_sum = None
feature_sq_sum = None
frame_count = 0

for idx, row in train_df.iterrows():
    X, y = load_phase_tcn_features(row["motion_path"], row["phase_label_path"])

    if feature_sum is None:
        feature_sum = np.zeros(X.shape[1], dtype=np.float64)
        feature_sq_sum = np.zeros(X.shape[1], dtype=np.float64)

    feature_sum += X.sum(axis=0)
    feature_sq_sum += (X.astype(np.float64) ** 2).sum(axis=0)
    frame_count += X.shape[0]

feature_mean = feature_sum / max(frame_count, 1)
feature_var = feature_sq_sum / max(frame_count, 1) - feature_mean ** 2
feature_var = np.maximum(feature_var, 1e-8)
feature_std = np.sqrt(feature_var)

feature_mean = feature_mean.astype(np.float32)
feature_std = feature_std.astype(np.float32)

np.savez_compressed(
    PHASE_TCN_FEATURE_STATS_PATH,
    mean=feature_mean,
    std=feature_std,
    frame_count=np.int64(frame_count),
)

print(f"Saved feature stats: {PHASE_TCN_FEATURE_STATS_PATH}")
print(f"Train frames used for stats: {frame_count}")
print(f"Feature dim: {len(feature_mean)}")
print()

print("Feature mean summary:")
print(pd.Series(feature_mean).describe())
print()

print("Feature std summary:")
print(pd.Series(feature_std).describe())
print()

# ----------------------------
# 6. Compute class weights from train labels
# ----------------------------

print("Computing class weights from train labels...")

class_counts = np.zeros(4, dtype=np.int64)

for idx, row in train_df.iterrows():
    _, y = load_phase_tcn_features(row["motion_path"], row["phase_label_path"])
    for c in range(4):
        class_counts[c] += int(np.sum(y == c))

total_labels = int(class_counts.sum())

# Inverse-frequency weights, normalized to mean 1.
class_weights = total_labels / (4.0 * np.maximum(class_counts, 1))
class_weights = class_weights / np.mean(class_weights)
class_weights = class_weights.astype(np.float32)

np.save(PHASE_TCN_CLASS_WEIGHTS_PATH, class_weights)

print("Train class counts:")
print("0 background: ", int(class_counts[0]))
print("1 preparation:", int(class_counts[1]))
print("2 stroke:     ", int(class_counts[2]))
print("3 retraction: ", int(class_counts[3]))
print()

print("Class weights, mean-normalized:")
print("0 background: ", float(class_weights[0]))
print("1 preparation:", float(class_weights[1]))
print("2 stroke:     ", float(class_weights[2]))
print("3 retraction: ", float(class_weights[3]))
print()

print(f"Saved class weights: {PHASE_TCN_CLASS_WEIGHTS_PATH}")
print()

# ----------------------------
# 7. PyTorch Dataset and collate function
# ----------------------------

class PhaseTCNDataset(Dataset):
    def __init__(self, df, feature_mean, feature_std):
        self.df = df.reset_index(drop=True)
        self.feature_mean = feature_mean.astype(np.float32)
        self.feature_std = feature_std.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        X, y = load_phase_tcn_features(row["motion_path"], row["phase_label_path"])

        # Standardize features using train stats.
        X = (X - self.feature_mean) / self.feature_std
        X[~np.isfinite(X)] = 0.0

        return {
            "x": torch.tensor(X, dtype=torch.float32),          # (T, F)
            "y": torch.tensor(y, dtype=torch.long),             # (T,)
            "length": int(X.shape[0]),
            "split": row["split"],
            "gloss": row["gloss"],
            "video_stem": row["video_stem"],
        }

def phase_tcn_collate(batch):
    """
    Pads variable-length sequences.

    Returns:
      x:       (B, T_max, F)
      y:       (B, T_max), padded with -100
      mask:    (B, T_max), True for real frames
      lengths: (B,)
    """
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    y_padded = torch.full((batch_size, max_len), fill_value=-100, dtype=torch.long)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        y_padded[i, :T] = item["y"]
        mask[i, :T] = True

        meta.append(
            {
                "split": item["split"],
                "gloss": item["gloss"],
                "video_stem": item["video_stem"],
            }
        )

    return {
        "x": x_padded,
        "y": y_padded,
        "mask": mask,
        "lengths": lengths,
        "meta": meta,
    }

# ----------------------------
# 8. Build dataloaders and inspect one batch
# ----------------------------

train_df = ok_dataset_df[ok_dataset_df["split"] == "train"].copy()
val_df = ok_dataset_df[ok_dataset_df["split"] == "val"].copy()
test_df = ok_dataset_df[ok_dataset_df["split"] == "test"].copy()

train_dataset = PhaseTCNDataset(train_df, feature_mean, feature_std)
val_dataset = PhaseTCNDataset(val_df, feature_mean, feature_std)
test_dataset = PhaseTCNDataset(test_df, feature_mean, feature_std)

# num_workers=0 is safest in Colab.
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    collate_fn=phase_tcn_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=phase_tcn_collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=phase_tcn_collate,
)

batch = next(iter(train_loader))

print("=" * 80)
print("DATALOADER SANITY CHECK")
print("=" * 80)

print("Dataset sizes:")
print(f"train_dataset: {len(train_dataset)}")
print(f"val_dataset:   {len(val_dataset)}")
print(f"test_dataset:  {len(test_dataset)}")
print()

print("One train batch:")
print(f"x shape:       {tuple(batch['x'].shape)}   # (B, T_max, F)")
print(f"y shape:       {tuple(batch['y'].shape)}   # (B, T_max)")
print(f"mask shape:    {tuple(batch['mask'].shape)}")
print(f"lengths shape: {tuple(batch['lengths'].shape)}")
print(f"lengths:       {batch['lengths'].tolist()}")
print()

real_labels = batch["y"][batch["y"] != -100]
print("Labels in this batch:")
unique, counts = torch.unique(real_labels, return_counts=True)
for u, c in zip(unique.tolist(), counts.tolist()):
    print(f"label {u}: {c}")
print()

print("Example metadata from batch:")
for m in batch["meta"][:5]:
    print(m)

print()
print("=" * 80)
print("PHASE 8A OUTPUT SAVED")
print("=" * 80)
print(f"Dataset manifest: {PHASE_TCN_DATASET_MANIFEST_PATH}")
print(f"Feature stats:    {PHASE_TCN_FEATURE_STATS_PATH}")
print(f"Class weights:    {PHASE_TCN_CLASS_WEIGHTS_PATH}")
print()

print("=" * 80)
print("PHASE 8A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Verify status counts")
print("2. OK rows by split")
print("3. Feature dimension loaded summary")
print("4. Sequence length summary")
print("5. Feature mean/std summaries")
print("6. Train class counts")
print("7. Class weights")
print("8. Dataloader sanity check")
print()
print("Do NOT train yet. Next step is Phase 8B: train Phase Detection TCN.")

In [ ]:
# ============================================================
# PHASE 8B — TRAIN PHASE DETECTION TCN
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import time
import math
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"
MODEL_DIR = BASE_DIR / "models" / "phase_tcn"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PHASE_TCN_DATASET_MANIFEST_PATH = MANIFEST_DIR / "phase_tcn_dataset_manifest.csv"
PHASE_TCN_FEATURE_STATS_PATH = MANIFEST_DIR / "phase_tcn_feature_stats.npz"
PHASE_TCN_CLASS_WEIGHTS_PATH = MANIFEST_DIR / "phase_tcn_class_weights.npy"

BEST_MODEL_PATH = MODEL_DIR / "phase_tcn_best.pt"
LAST_MODEL_PATH = MODEL_DIR / "phase_tcn_last.pt"
TRAINING_LOG_PATH = MODEL_DIR / "phase_tcn_training_log.csv"
CONFIG_PATH = MODEL_DIR / "phase_tcn_config.json"

print("=" * 80)
print("PHASE 8B — TRAIN PHASE DETECTION TCN")
print("=" * 80)

for p in [
    PHASE_TCN_DATASET_MANIFEST_PATH,
    PHASE_TCN_FEATURE_STATS_PATH,
    PHASE_TCN_CLASS_WEIGHTS_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

dataset_df = pd.read_csv(PHASE_TCN_DATASET_MANIFEST_PATH)
dataset_df = dataset_df[dataset_df["verify_status"] == "ok"].copy()

stats = np.load(PHASE_TCN_FEATURE_STATS_PATH)
feature_mean = stats["mean"].astype(np.float32)
feature_std = stats["std"].astype(np.float32)

class_weights_np = np.load(PHASE_TCN_CLASS_WEIGHTS_PATH).astype(np.float32)

print(f"Dataset manifest: {PHASE_TCN_DATASET_MANIFEST_PATH}")
print(f"Rows after verify_status == ok: {len(dataset_df)}")
print(f"Feature dim: {len(feature_mean)}")
print(f"Model directory: {MODEL_DIR}")
print()

print("Rows by split:")
print(dataset_df["split"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Reproducibility and device
# ----------------------------

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# ----------------------------
# 3. Training config
# ----------------------------

CONFIG = {
    "seed": SEED,
    "input_dim": 444,
    "num_classes": 4,
    "hidden_dim": 192,
    "num_blocks": 5,
    "kernel_size": 5,
    "dropout": 0.20,
    "batch_size": 16,
    "epochs": 30,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "patience": 7,
    "ignore_index": -100,
    "num_workers": 0,
}

with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Training config:")
print(json.dumps(CONFIG, indent=2))
print()

# ----------------------------
# 4. Dataset functions
# ----------------------------

def load_phase_tcn_features(motion_path, label_path):
    """
    Returns:
      X: (T, 444) float32
      y: (T,) int64
    """
    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(label_path, allow_pickle=False) as l:
        labels = np.asarray(l["labels"], dtype=np.int64)
        phase_speed = np.asarray(l["phase_speed"], dtype=np.float32).reshape(-1, 1)

    X = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X[~np.isfinite(X)] = 0.0

    return X, labels

class PhaseTCNDataset(Dataset):
    def __init__(self, df, feature_mean, feature_std):
        self.df = df.reset_index(drop=True)
        self.feature_mean = feature_mean.astype(np.float32)
        self.feature_std = feature_std.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        X, y = load_phase_tcn_features(row["motion_path"], row["phase_label_path"])

        X = (X - self.feature_mean) / self.feature_std
        X[~np.isfinite(X)] = 0.0

        return {
            "x": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "length": int(X.shape[0]),
            "split": row["split"],
            "gloss": row["gloss"],
            "video_stem": row["video_stem"],
        }

def phase_tcn_collate(batch):
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    y_padded = torch.full(
        (batch_size, max_len),
        fill_value=CONFIG["ignore_index"],
        dtype=torch.long,
    )
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        y_padded[i, :T] = item["y"]
        mask[i, :T] = True

        meta.append(
            {
                "split": item["split"],
                "gloss": item["gloss"],
                "video_stem": item["video_stem"],
            }
        )

    return {
        "x": x_padded,
        "y": y_padded,
        "mask": mask,
        "lengths": lengths,
        "meta": meta,
    }

# ----------------------------
# 5. Build dataloaders
# ----------------------------

train_df = dataset_df[dataset_df["split"] == "train"].copy()
val_df = dataset_df[dataset_df["split"] == "val"].copy()
test_df = dataset_df[dataset_df["split"] == "test"].copy()

train_dataset = PhaseTCNDataset(train_df, feature_mean, feature_std)
val_dataset = PhaseTCNDataset(val_df, feature_mean, feature_std)
test_dataset = PhaseTCNDataset(test_df, feature_mean, feature_std)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    collate_fn=phase_tcn_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=phase_tcn_collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=phase_tcn_collate,
)

print("Dataset sizes:")
print(f"train: {len(train_dataset)}")
print(f"val:   {len(val_dataset)}")
print(f"test:  {len(test_dataset)}")
print()

# ----------------------------
# 6. Model definition
# ----------------------------

class Chomp1d(nn.Module):
    """
    Removes extra padding from causal-ish Conv1d outputs.
    For this project, we use symmetric-ish temporal modeling via padded Conv1d.
    """
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        # Padding chosen to preserve sequence length after conv.
        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        # x: (B, C, T)

        out = self.conv1(x)

        # In case even-length effects cause small mismatch, crop/pad to input T.
        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class PhaseTCN(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes=4,
        hidden_dim=192,
        num_blocks=5,
        kernel_size=5,
        dropout=0.20,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.classifier = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)

    def forward(self, x):
        """
        x: (B, T, F)
        returns logits: (B, T, num_classes)
        """
        x = x.transpose(1, 2)      # (B, F, T)
        h = self.tcn(x)           # (B, H, T)
        logits = self.classifier(h)  # (B, C, T)
        logits = logits.transpose(1, 2)  # (B, T, C)
        return logits

model = PhaseTCN(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
).to(device)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model:")
print(model)
print()
print(f"Total parameters:     {num_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()

# ----------------------------
# 7. Loss, optimizer, scheduler
# ----------------------------

class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=CONFIG["ignore_index"],
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

# Mixed precision only if CUDA exists.
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print("Class weights on device:")
for i, w in enumerate(class_weights.detach().cpu().numpy().tolist()):
    print(f"class {i}: {w:.4f}")
print()

# ----------------------------
# 8. Metrics
# ----------------------------

def compute_frame_metrics(y_true, y_pred, num_classes=4):
    """
    y_true, y_pred: flat numpy arrays without ignored labels.
    Returns frame accuracy, macro F1, per-class precision/recall/F1.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    if len(y_true) == 0:
        return {
            "accuracy": 0.0,
            "macro_f1": 0.0,
            "per_class": {},
        }

    accuracy = float(np.mean(y_true == y_pred))

    per_class = {}
    f1s = []

    for c in range(num_classes):
        tp = int(np.sum((y_true == c) & (y_pred == c)))
        fp = int(np.sum((y_true != c) & (y_pred == c)))
        fn = int(np.sum((y_true == c) & (y_pred != c)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)

        per_class[c] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": int(np.sum(y_true == c)),
        }

        f1s.append(f1)

    macro_f1 = float(np.mean(f1s))

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "per_class": per_class,
    }

def run_one_epoch(model, loader, optimizer=None):
    """
    If optimizer is provided: train mode.
    Else: eval mode.
    """
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_frames = 0

    all_true = []
    all_pred = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)  # (B, T, C)

                B, T, C = logits.shape
                loss = criterion(
                    logits.reshape(B * T, C),
                    y.reshape(B * T),
                )

            if is_train:
                scaler.scale(loss).backward()

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])

                scaler.step(optimizer)
                scaler.update()

        real_mask = y != CONFIG["ignore_index"]
        num_real = int(real_mask.sum().item())

        total_loss += float(loss.detach().cpu().item()) * max(num_real, 1)
        total_frames += num_real

        preds = torch.argmax(logits, dim=-1)

        y_true_np = y[real_mask].detach().cpu().numpy()
        y_pred_np = preds[real_mask].detach().cpu().numpy()

        all_true.append(y_true_np)
        all_pred.append(y_pred_np)

    if len(all_true) > 0:
        all_true = np.concatenate(all_true)
        all_pred = np.concatenate(all_pred)
    else:
        all_true = np.array([], dtype=np.int64)
        all_pred = np.array([], dtype=np.int64)

    avg_loss = total_loss / max(total_frames, 1)
    metrics = compute_frame_metrics(all_true, all_pred, num_classes=CONFIG["num_classes"])

    return avg_loss, metrics

# ----------------------------
# 9. Training loop
# ----------------------------

print("=" * 80)
print("START TRAINING")
print("=" * 80)

best_val_macro_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0

log_rows = []

start_time = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_start = time.time()

    train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_metrics = run_one_epoch(model, val_loader, optimizer=None)

    scheduler.step(val_metrics["macro_f1"])

    current_lr = optimizer.param_groups[0]["lr"]

    improved = val_metrics["macro_f1"] > best_val_macro_f1

    if improved:
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": CONFIG,
                "feature_mean": feature_mean,
                "feature_std": feature_std,
                "class_weights": class_weights_np,
                "val_loss": val_loss,
                "val_accuracy": val_metrics["accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
                "num_params": num_params,
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    # Save last checkpoint every epoch.
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "feature_mean": feature_mean,
            "feature_std": feature_std,
            "class_weights": class_weights_np,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "num_params": num_params,
        },
        LAST_MODEL_PATH,
    )

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_loss,
        "train_acc": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_loss,
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "best_val_macro_f1": best_val_macro_f1,
        "best_epoch": best_epoch,
        "epoch_seconds": time.time() - epoch_start,
    }

    # Add per-class F1 scores.
    for c in range(CONFIG["num_classes"]):
        row[f"val_class_{c}_f1"] = val_metrics["per_class"][c]["f1"]
        row[f"val_class_{c}_recall"] = val_metrics["per_class"][c]["recall"]
        row[f"val_class_{c}_support"] = val_metrics["per_class"][c]["support"]

    log_rows.append(row)
    pd.DataFrame(log_rows).to_csv(TRAINING_LOG_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
        f"lr={current_lr:.2e} | "
        f"train_loss={train_loss:.4f} | train_acc={train_metrics['accuracy']:.4f} | train_f1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_metrics['accuracy']:.4f} | val_f1={val_metrics['macro_f1']:.4f} | "
        f"best_f1={best_val_macro_f1:.4f}@{best_epoch} | "
        f"{'BEST' if improved else ''}"
    )

    print(
        "  Val per-class F1: "
        f"bg={val_metrics['per_class'][0]['f1']:.3f}, "
        f"prep={val_metrics['per_class'][1]['f1']:.3f}, "
        f"stroke={val_metrics['per_class'][2]['f1']:.3f}, "
        f"retract={val_metrics['per_class'][3]['f1']:.3f}"
    )

    if epochs_without_improvement >= CONFIG["patience"]:
        print()
        print(f"Early stopping triggered after {CONFIG['patience']} epochs without improvement.")
        break

total_time = time.time() - start_time

# ----------------------------
# 10. Final summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 8B TRAINING COMPLETE")
print("=" * 80)

print(f"Best epoch:          {best_epoch}")
print(f"Best val macro F1:   {best_val_macro_f1:.4f}")
print(f"Best model path:     {BEST_MODEL_PATH}")
print(f"Last model path:     {LAST_MODEL_PATH}")
print(f"Training log path:   {TRAINING_LOG_PATH}")
print(f"Config path:         {CONFIG_PATH}")
print(f"Total train time:    {total_time / 60:.2f} minutes")
print()

# Load best checkpoint and evaluate on val once more.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

best_val_loss, best_val_metrics = run_one_epoch(model, val_loader, optimizer=None)

print("=" * 80)
print("BEST MODEL VALIDATION METRICS")
print("=" * 80)
print(f"val_loss:     {best_val_loss:.4f}")
print(f"val_accuracy: {best_val_metrics['accuracy']:.4f}")
print(f"val_macro_f1: {best_val_metrics['macro_f1']:.4f}")
print()

for c, name in enumerate(["background", "preparation", "stroke", "retraction"]):
    m = best_val_metrics["per_class"][c]
    print(
        f"class {c} ({name}): "
        f"precision={m['precision']:.4f}, "
        f"recall={m['recall']:.4f}, "
        f"f1={m['f1']:.4f}, "
        f"support={m['support']}"
    )

print()
print("=" * 80)
print("PHASE 8B COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Training epoch logs")
print("2. Best epoch")
print("3. Best val macro F1")
print("4. Best model validation metrics")
print("5. Per-class validation precision/recall/F1")
print()
print("Do NOT run test evaluation yet. Next step will be Phase 8C: test evaluation + prediction visualization.")

In [ ]:
# ============================================================
# PHASE 8B-FIX — LOAD BEST CHECKPOINT SAFELY + PRINT VAL METRICS
# Run this cell only. Do NOT rerun training.
# ============================================================

from pathlib import Path
import torch
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MODEL_DIR = BASE_DIR / "models" / "phase_tcn"

BEST_MODEL_PATH = MODEL_DIR / "phase_tcn_best.pt"
SAFE_BEST_MODEL_PATH = MODEL_DIR / "phase_tcn_best_safe_state.pt"

print("=" * 80)
print("PHASE 8B-FIX — LOAD BEST CHECKPOINT")
print("=" * 80)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing best checkpoint: {BEST_MODEL_PATH}")

# Check that the previous training objects still exist in the runtime.
needed_objects = ["model", "val_loader", "run_one_epoch", "device"]
missing = [name for name in needed_objects if name not in globals()]

if missing:
    raise RuntimeError(
        "Your Colab runtime no longer has the training objects loaded: "
        f"{missing}\n"
        "Do NOT rerun training yet. Tell ChatGPT this output, and I will give you "
        "a reload-from-scratch evaluation cell."
    )

# Because this checkpoint was created by YOUR OWN training code in this Colab,
# weights_only=False is appropriate here.
# Do not use weights_only=False for random checkpoints downloaded from the internet.
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded successfully.")
print(f"Checkpoint epoch:        {checkpoint.get('epoch')}")
print(f"Saved val loss:          {checkpoint.get('val_loss')}")
print(f"Saved val accuracy:      {checkpoint.get('val_accuracy')}")
print(f"Saved val macro F1:      {checkpoint.get('val_macro_f1')}")
print()

# Load best weights into current model.
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

# Re-evaluate on validation set.
best_val_loss, best_val_metrics = run_one_epoch(model, val_loader, optimizer=None)

print("=" * 80)
print("BEST MODEL VALIDATION METRICS")
print("=" * 80)
print(f"val_loss:     {best_val_loss:.4f}")
print(f"val_accuracy: {best_val_metrics['accuracy']:.4f}")
print(f"val_macro_f1: {best_val_metrics['macro_f1']:.4f}")
print()

for c, name in enumerate(["background", "preparation", "stroke", "retraction"]):
    m = best_val_metrics["per_class"][c]
    print(
        f"class {c} ({name}): "
        f"precision={m['precision']:.4f}, "
        f"recall={m['recall']:.4f}, "
        f"f1={m['f1']:.4f}, "
        f"support={m['support']}"
    )

# Save a safer state-only checkpoint for future loading.
# This avoids the PyTorch 2.6 NumPy unpickling issue.
safe_checkpoint = {
    "epoch": int(checkpoint["epoch"]),
    "model_state_dict": checkpoint["model_state_dict"],
    "config": checkpoint["config"],
    "feature_mean": torch.tensor(checkpoint["feature_mean"], dtype=torch.float32),
    "feature_std": torch.tensor(checkpoint["feature_std"], dtype=torch.float32),
    "class_weights": torch.tensor(checkpoint["class_weights"], dtype=torch.float32),
    "val_loss": float(checkpoint["val_loss"]),
    "val_accuracy": float(checkpoint["val_accuracy"]),
    "val_macro_f1": float(checkpoint["val_macro_f1"]),
    "num_params": int(checkpoint["num_params"]),
}

torch.save(safe_checkpoint, SAFE_BEST_MODEL_PATH)

print()
print("=" * 80)
print("SAFE CHECKPOINT SAVED")
print("=" * 80)
print(f"Original checkpoint: {BEST_MODEL_PATH}")
print(f"Safe checkpoint:     {SAFE_BEST_MODEL_PATH}")
print()
print("PHASE 8B-FIX COMPLETE — send this output back.")

In [ ]:
# ============================================================
# PHASE 8C — TEST EVALUATION + PREDICTION VISUALIZATION
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"
MODEL_DIR = BASE_DIR / "models" / "phase_tcn"

PHASE_TCN_DATASET_MANIFEST_PATH = MANIFEST_DIR / "phase_tcn_dataset_manifest.csv"
SAFE_BEST_MODEL_PATH = MODEL_DIR / "phase_tcn_best_safe_state.pt"

TEST_METRICS_PATH = MODEL_DIR / "phase_tcn_test_metrics.json"
TEST_PREDICTIONS_CSV_PATH = MODEL_DIR / "phase_tcn_test_predictions_summary.csv"

PREDICTION_PLOT_DIR = BASE_DIR / "plots" / "phase_tcn_predictions"
PREDICTION_PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 8C — TEST EVALUATION + PREDICTION VISUALIZATION")
print("=" * 80)

if not PHASE_TCN_DATASET_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing dataset manifest: {PHASE_TCN_DATASET_MANIFEST_PATH}")

if not SAFE_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing safe checkpoint: {SAFE_BEST_MODEL_PATH}")

dataset_df = pd.read_csv(PHASE_TCN_DATASET_MANIFEST_PATH)
dataset_df = dataset_df[dataset_df["verify_status"] == "ok"].copy()

print(f"Dataset manifest: {PHASE_TCN_DATASET_MANIFEST_PATH}")
print(f"Rows: {len(dataset_df)}")
print(f"Safe checkpoint: {SAFE_BEST_MODEL_PATH}")
print(f"Prediction plot dir: {PREDICTION_PLOT_DIR}")
print()

print("Rows by split:")
print(dataset_df["split"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Device + checkpoint
# ----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

checkpoint = torch.load(SAFE_BEST_MODEL_PATH, map_location=device)

CONFIG = checkpoint["config"]
feature_mean = checkpoint["feature_mean"].detach().cpu().numpy().astype(np.float32)
feature_std = checkpoint["feature_std"].detach().cpu().numpy().astype(np.float32)

print("Loaded checkpoint:")
print(f"epoch:        {checkpoint['epoch']}")
print(f"val_loss:     {checkpoint['val_loss']:.4f}")
print(f"val_accuracy: {checkpoint['val_accuracy']:.4f}")
print(f"val_macro_f1: {checkpoint['val_macro_f1']:.4f}")
print()

# ----------------------------
# 3. Dataset utilities
# ----------------------------

def load_phase_tcn_features(motion_path, label_path):
    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(label_path, allow_pickle=False) as l:
        labels = np.asarray(l["labels"], dtype=np.int64)
        phase_speed = np.asarray(l["phase_speed"], dtype=np.float32).reshape(-1, 1)

    X = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X[~np.isfinite(X)] = 0.0

    return X, labels, phase_speed.reshape(-1)

class PhaseTCNDataset(Dataset):
    def __init__(self, df, feature_mean, feature_std):
        self.df = df.reset_index(drop=True)
        self.feature_mean = feature_mean.astype(np.float32)
        self.feature_std = feature_std.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        X, y, phase_speed = load_phase_tcn_features(
            row["motion_path"],
            row["phase_label_path"]
        )

        X = (X - self.feature_mean) / self.feature_std
        X[~np.isfinite(X)] = 0.0

        return {
            "x": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "phase_speed": torch.tensor(phase_speed, dtype=torch.float32),
            "length": int(X.shape[0]),
            "split": row["split"],
            "gloss": row["gloss"],
            "video_stem": row["video_stem"],
            "motion_path": row["motion_path"],
            "phase_label_path": row["phase_label_path"],
        }

def phase_tcn_collate(batch):
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    y_padded = torch.full(
        (batch_size, max_len),
        fill_value=-100,
        dtype=torch.long,
    )
    speed_padded = torch.zeros(batch_size, max_len, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        y_padded[i, :T] = item["y"]
        speed_padded[i, :T] = item["phase_speed"]
        mask[i, :T] = True

        meta.append(
            {
                "split": item["split"],
                "gloss": item["gloss"],
                "video_stem": item["video_stem"],
                "motion_path": item["motion_path"],
                "phase_label_path": item["phase_label_path"],
            }
        )

    return {
        "x": x_padded,
        "y": y_padded,
        "phase_speed": speed_padded,
        "mask": mask,
        "lengths": lengths,
        "meta": meta,
    }

# ----------------------------
# 4. Model definition
# ----------------------------

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class PhaseTCN(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes=4,
        hidden_dim=192,
        num_blocks=5,
        kernel_size=5,
        dropout=0.20,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.classifier = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.transpose(1, 2)
        h = self.tcn(x)
        logits = self.classifier(h)
        logits = logits.transpose(1, 2)
        return logits

model = PhaseTCN(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully.")
print()

# ----------------------------
# 5. Metrics
# ----------------------------

def compute_frame_metrics(y_true, y_pred, num_classes=4):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    accuracy = float(np.mean(y_true == y_pred)) if len(y_true) else 0.0

    per_class = {}
    f1s = []

    for c in range(num_classes):
        tp = int(np.sum((y_true == c) & (y_pred == c)))
        fp = int(np.sum((y_true != c) & (y_pred == c)))
        fn = int(np.sum((y_true == c) & (y_pred != c)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)

        support = int(np.sum(y_true == c))

        per_class[c] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": support,
        }

        f1s.append(f1)

    return {
        "accuracy": accuracy,
        "macro_f1": float(np.mean(f1s)),
        "per_class": per_class,
    }

def compute_confusion_matrix(y_true, y_pred, num_classes=4):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1
    return cm

# ----------------------------
# 6. Build test loader
# ----------------------------

test_df = dataset_df[dataset_df["split"] == "test"].copy()

test_dataset = PhaseTCNDataset(test_df, feature_mean, feature_std)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=phase_tcn_collate,
)

print(f"Test dataset size: {len(test_dataset)}")
print()

# ----------------------------
# 7. Evaluate on test set
# ----------------------------

all_true = []
all_pred = []
prediction_records = []

print("Running test evaluation...")

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        logits = model(x)
        preds = torch.argmax(logits, dim=-1)

        real_mask = y != -100

        y_true_np = y[real_mask].detach().cpu().numpy()
        y_pred_np = preds[real_mask].detach().cpu().numpy()

        all_true.append(y_true_np)
        all_pred.append(y_pred_np)

        # Per-sample summaries
        for i, meta in enumerate(batch["meta"]):
            T = int(batch["lengths"][i].item())

            true_i = y[i, :T].detach().cpu().numpy()
            pred_i = preds[i, :T].detach().cpu().numpy()

            acc_i = float(np.mean(true_i == pred_i))

            # Per-phase IoU-ish overlap.
            phase_acc = {}
            for c in range(4):
                true_c = true_i == c
                pred_c = pred_i == c
                inter = int(np.sum(true_c & pred_c))
                union = int(np.sum(true_c | pred_c))
                phase_acc[f"class_{c}_iou"] = inter / max(union, 1)

            prediction_records.append(
                {
                    "split": meta["split"],
                    "gloss": meta["gloss"],
                    "video_stem": meta["video_stem"],
                    "length": T,
                    "frame_accuracy": acc_i,
                    **phase_acc,
                    "motion_path": meta["motion_path"],
                    "phase_label_path": meta["phase_label_path"],
                }
            )

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)

test_metrics = compute_frame_metrics(all_true, all_pred, num_classes=4)
confusion_matrix = compute_confusion_matrix(all_true, all_pred, num_classes=4)

prediction_summary_df = pd.DataFrame(prediction_records)
prediction_summary_df.to_csv(TEST_PREDICTIONS_CSV_PATH, index=False)

metrics_to_save = {
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "validation_accuracy_at_checkpoint": float(checkpoint["val_accuracy"]),
    "validation_macro_f1_at_checkpoint": float(checkpoint["val_macro_f1"]),
    "test_accuracy": float(test_metrics["accuracy"]),
    "test_macro_f1": float(test_metrics["macro_f1"]),
    "per_class": test_metrics["per_class"],
    "confusion_matrix": confusion_matrix.tolist(),
    "num_test_sequences": int(len(test_dataset)),
    "num_test_frames": int(len(all_true)),
}

with open(TEST_METRICS_PATH, "w") as f:
    json.dump(metrics_to_save, f, indent=2)

# ----------------------------
# 8. Print test summary
# ----------------------------

print()
print("=" * 80)
print("TEST METRICS")
print("=" * 80)
print(f"test_accuracy: {test_metrics['accuracy']:.4f}")
print(f"test_macro_f1: {test_metrics['macro_f1']:.4f}")
print(f"test_frames:   {len(all_true)}")
print(f"test_sequences:{len(test_dataset)}")
print()

for c, name in enumerate(["background", "preparation", "stroke", "retraction"]):
    m = test_metrics["per_class"][c]
    print(
        f"class {c} ({name}): "
        f"precision={m['precision']:.4f}, "
        f"recall={m['recall']:.4f}, "
        f"f1={m['f1']:.4f}, "
        f"support={m['support']}"
    )

print()
print("Confusion matrix:")
print("Rows = true labels, columns = predicted labels")
print(pd.DataFrame(
    confusion_matrix,
    index=["true_bg", "true_prep", "true_stroke", "true_retract"],
    columns=["pred_bg", "pred_prep", "pred_stroke", "pred_retract"],
))
print()

print("=" * 80)
print("PER-SEQUENCE TEST ACCURACY SUMMARY")
print("=" * 80)
print(prediction_summary_df["frame_accuracy"].describe())
print()

print("Worst 10 sequences by frame accuracy:")
display(
    prediction_summary_df.sort_values("frame_accuracy", ascending=True)
    [
        [
            "gloss",
            "video_stem",
            "length",
            "frame_accuracy",
            "class_0_iou",
            "class_1_iou",
            "class_2_iou",
            "class_3_iou",
        ]
    ]
    .head(10)
)

print()
print("Best 10 sequences by frame accuracy:")
display(
    prediction_summary_df.sort_values("frame_accuracy", ascending=False)
    [
        [
            "gloss",
            "video_stem",
            "length",
            "frame_accuracy",
            "class_0_iou",
            "class_1_iou",
            "class_2_iou",
            "class_3_iou",
        ]
    ]
    .head(10)
)

# ----------------------------
# 9. Prediction visualization
# ----------------------------

PHASE_NAMES = {
    0: "background",
    1: "preparation",
    2: "stroke",
    3: "retraction",
}

PHASE_COLORS = {
    0: "lightgray",
    1: "gold",
    2: "tomato",
    3: "skyblue",
}

def label_segments(labels):
    labels = np.asarray(labels, dtype=np.int64)
    if len(labels) == 0:
        return []

    segments = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        val = int(labels[i])
        if val != current:
            segments.append((current, start, i - 1))
            start = i
            current = val

    segments.append((current, start, len(labels) - 1))
    return segments

def plot_true_vs_pred(row, save_dir, show=False):
    X, y_true, phase_speed = load_phase_tcn_features(
        row["motion_path"],
        row["phase_label_path"]
    )

    Xn = (X - feature_mean) / feature_std
    Xn[~np.isfinite(Xn)] = 0.0

    xt = torch.tensor(Xn, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(xt)
        pred = torch.argmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()

    T = len(y_true)
    frames = np.arange(T)

    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

    # Top: pseudo labels
    ax = axes[0]
    for phase_id, start, end in label_segments(y_true):
        ax.axvspan(
            start,
            end + 1,
            color=PHASE_COLORS.get(phase_id, "white"),
            alpha=0.35,
            label=PHASE_NAMES.get(phase_id, str(phase_id)),
        )
    ax.plot(frames, phase_speed, linewidth=2.0)
    ax.set_title("Pseudo-labels / weak labels")
    ax.set_ylabel("phase_speed")
    ax.set_ylim(bottom=0)

    # Bottom: model predictions
    ax = axes[1]
    for phase_id, start, end in label_segments(pred):
        ax.axvspan(
            start,
            end + 1,
            color=PHASE_COLORS.get(phase_id, "white"),
            alpha=0.35,
            label=PHASE_NAMES.get(phase_id, str(phase_id)),
        )
    ax.plot(frames, phase_speed, linewidth=2.0)
    ax.set_title("Phase TCN predictions")
    ax.set_xlabel("Frame")
    ax.set_ylabel("phase_speed")
    ax.set_ylim(bottom=0)

    # Unique legend
    handles, labels_legend = axes[0].get_legend_handles_labels()
    unique = {}
    for h, l in zip(handles, labels_legend):
        if l not in unique:
            unique[l] = h
    axes[0].legend(unique.values(), unique.keys(), loc="upper right", fontsize=8)

    acc = float(np.mean(y_true == pred))

    fig.suptitle(
        f"{row['gloss']} | {row['video_stem']} | T={T} | frame_acc={acc:.3f}",
        y=1.02,
    )

    plt.tight_layout()

    safe_name = str(row["video_stem"]).replace("/", "_").replace("\\", "_").replace(" ", "_")
    save_path = save_dir / f"pred_vs_true__{safe_name}.png"

    plt.savefig(save_path, dpi=140, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return save_path

# Select examples:
# - 5 random
# - 5 worst
# - 5 best
random_examples = prediction_summary_df.sample(
    n=min(5, len(prediction_summary_df)),
    random_state=42,
)

worst_examples = prediction_summary_df.sort_values("frame_accuracy", ascending=True).head(5)
best_examples = prediction_summary_df.sort_values("frame_accuracy", ascending=False).head(5)

plot_examples = pd.concat(
    [
        random_examples.assign(plot_bucket="random"),
        worst_examples.assign(plot_bucket="worst"),
        best_examples.assign(plot_bucket="best"),
    ],
    ignore_index=True,
).drop_duplicates(subset=["video_stem"])

saved_plot_paths = []

print()
print("=" * 80)
print("GENERATING PREDICTION VISUALIZATIONS")
print("=" * 80)

for idx, row in plot_examples.iterrows():
    show_now = idx < 5
    save_path = plot_true_vs_pred(row, PREDICTION_PLOT_DIR, show=show_now)
    saved_plot_paths.append(str(save_path))

plot_examples["prediction_plot_path"] = saved_plot_paths

PREDICTION_VIS_MANIFEST_PATH = MODEL_DIR / "phase_tcn_prediction_visualization_manifest.csv"
plot_examples.to_csv(PREDICTION_VIS_MANIFEST_PATH, index=False)

print(f"Saved prediction plots: {len(saved_plot_paths)}")
print(f"Prediction visualization manifest: {PREDICTION_VIS_MANIFEST_PATH}")
print()

print("Prediction plot examples:")
display(
    plot_examples[
        [
            "plot_bucket",
            "gloss",
            "video_stem",
            "length",
            "frame_accuracy",
            "prediction_plot_path",
        ]
    ]
)

print()
print("First saved plot paths:")
for p in saved_plot_paths[:10]:
    print(p)

print()
print("=" * 80)
print("PHASE 8C OUTPUT SAVED")
print("=" * 80)
print(f"Test metrics:        {TEST_METRICS_PATH}")
print(f"Prediction summary:  {TEST_PREDICTIONS_CSV_PATH}")
print(f"Prediction plots:    {PREDICTION_PLOT_DIR}")
print()

print("=" * 80)
print("PHASE 8C COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Test metrics")
print("2. Per-class test precision/recall/F1")
print("3. Confusion matrix")
print("4. Per-sequence accuracy summary")
print("5. Worst 10 sequences")
print("6. Prediction plot examples")
print()
print("Do NOT start Recognition TCN yet. Next step is Phase 9A: generate phase-aware segments.")

In [ ]:
# ============================================================
# PHASE 9A — GENERATE PHASE-AWARE SEGMENTS FROM PHASE TCN
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"
MODEL_DIR = BASE_DIR / "models" / "phase_tcn"

PHASE_TCN_DATASET_MANIFEST_PATH = MANIFEST_DIR / "phase_tcn_dataset_manifest.csv"
SAFE_BEST_MODEL_PATH = MODEL_DIR / "phase_tcn_best_safe_state.pt"

PHASE_SEGMENT_DIR = BASE_DIR / "phase_segments"
PHASE_SEGMENT_MANIFEST_PATH = MANIFEST_DIR / "phase_segments_manifest.csv"

PHASE_SEGMENT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 9A — GENERATE PHASE-AWARE SEGMENTS")
print("=" * 80)

if not PHASE_TCN_DATASET_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing dataset manifest: {PHASE_TCN_DATASET_MANIFEST_PATH}")

if not SAFE_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing safe checkpoint: {SAFE_BEST_MODEL_PATH}")

dataset_df = pd.read_csv(PHASE_TCN_DATASET_MANIFEST_PATH)
dataset_df = dataset_df[dataset_df["verify_status"] == "ok"].copy()

print(f"Dataset manifest: {PHASE_TCN_DATASET_MANIFEST_PATH}")
print(f"Rows: {len(dataset_df)}")
print(f"Safe checkpoint: {SAFE_BEST_MODEL_PATH}")
print(f"Segment output dir: {PHASE_SEGMENT_DIR}")
print()

print("Rows by split:")
print(dataset_df["split"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Device + checkpoint
# ----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

checkpoint = torch.load(SAFE_BEST_MODEL_PATH, map_location=device)

CONFIG = checkpoint["config"]
feature_mean = checkpoint["feature_mean"].detach().cpu().numpy().astype(np.float32)
feature_std = checkpoint["feature_std"].detach().cpu().numpy().astype(np.float32)

print("Loaded checkpoint:")
print(f"epoch:        {checkpoint['epoch']}")
print(f"val_accuracy: {checkpoint['val_accuracy']:.4f}")
print(f"val_macro_f1: {checkpoint['val_macro_f1']:.4f}")
print()

# ----------------------------
# 3. Model definition
# ----------------------------

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class PhaseTCN(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes=4,
        hidden_dim=192,
        num_blocks=5,
        kernel_size=5,
        dropout=0.20,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.classifier = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)

    def forward(self, x):
        """
        x: (B, T, F)
        returns logits: (B, T, C)
        """
        x = x.transpose(1, 2)
        h = self.tcn(x)
        logits = self.classifier(h)
        logits = logits.transpose(1, 2)
        return logits

model = PhaseTCN(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully.")
print()

# ----------------------------
# 4. Feature loading
# ----------------------------

def load_phase_tcn_features(motion_path, label_path):
    """
    Returns:
      X: (T, 444)
      pseudo_labels: (T,)
      phase_speed: (T,)
    """
    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(label_path, allow_pickle=False) as l:
        pseudo_labels = np.asarray(l["labels"], dtype=np.int64)
        phase_speed = np.asarray(l["phase_speed"], dtype=np.float32).reshape(-1, 1)

    X = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X[~np.isfinite(X)] = 0.0

    return X, pseudo_labels, phase_speed.reshape(-1)

def predict_phases(motion_path, label_path):
    X, pseudo_labels, phase_speed = load_phase_tcn_features(motion_path, label_path)

    Xn = (X - feature_mean) / feature_std
    Xn[~np.isfinite(Xn)] = 0.0

    xt = torch.tensor(Xn, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(xt)
        probs = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
        raw_pred = np.argmax(probs, axis=-1).astype(np.int64)

    return X, pseudo_labels, phase_speed, probs.astype(np.float32), raw_pred

# ----------------------------
# 5. Prediction post-processing helpers
# ----------------------------

def safe_stem(s):
    s = str(s)
    s = s.replace("/", "_").replace("\\", "_").replace(" ", "_")
    return s

def majority_filter_1d(labels, window=5):
    """
    Smooths single-frame glitches in predicted phase labels.
    """
    labels = np.asarray(labels, dtype=np.int64)
    T = len(labels)

    if T == 0 or window <= 1:
        return labels.copy()

    if window % 2 == 0:
        window += 1

    pad = window // 2
    padded = np.pad(labels, (pad, pad), mode="edge")
    out = labels.copy()

    for i in range(T):
        vals = padded[i:i + window]
        counts = np.bincount(vals, minlength=4)
        out[i] = int(np.argmax(counts))

    return out.astype(np.int64)

def get_segments(mask):
    mask = np.asarray(mask, dtype=bool)
    segments = []

    if len(mask) == 0:
        return segments

    in_seg = False
    start = 0

    for i, val in enumerate(mask):
        if val and not in_seg:
            start = i
            in_seg = True
        elif not val and in_seg:
            segments.append((start, i - 1))
            in_seg = False

    if in_seg:
        segments.append((start, len(mask) - 1))

    return segments

def fill_small_gaps(mask, max_gap=2):
    mask = np.asarray(mask, dtype=bool).copy()
    segs = get_segments(mask)

    if len(segs) <= 1:
        return mask

    for i in range(len(segs) - 1):
        end1 = segs[i][1]
        start2 = segs[i + 1][0]
        gap = start2 - end1 - 1

        if 0 < gap <= max_gap:
            mask[end1 + 1:start2] = True

    return mask

def remove_small_islands(mask, min_len=2):
    mask = np.asarray(mask, dtype=bool).copy()
    segs = get_segments(mask)

    for s, e in segs:
        if (e - s + 1) < min_len:
            mask[s:e + 1] = False

    return mask

def largest_segment(segments):
    if len(segments) == 0:
        return None
    return max(segments, key=lambda se: se[1] - se[0] + 1)

def extract_phase_segments(pred_labels, probs, phase_speed):
    """
    Converts frame-level predictions into one active segment and one ordered:
      preparation -> stroke -> retraction
    region.

    Returns boundaries and status flags.
    """
    pred_labels = np.asarray(pred_labels, dtype=np.int64)
    probs = np.asarray(probs, dtype=np.float32)
    phase_speed = np.asarray(phase_speed, dtype=np.float32)

    T = len(pred_labels)

    result = {
        "segment_status": "unknown",
        "segment_quality_flag": "",
        "active_start": -1,
        "active_end": -1,
        "prep_start": -1,
        "prep_end": -1,
        "stroke_start": -1,
        "stroke_end": -1,
        "retract_start": -1,
        "retract_end": -1,
        "background_len_pred": 0,
        "prep_len_pred": 0,
        "stroke_len_pred": 0,
        "retract_len_pred": 0,
        "active_len_pred": 0,
        "active_ratio_pred": 0.0,
        "stroke_active_ratio_pred": 0.0,
        "used_fallback": False,
    }

    if T < 8:
        result["segment_status"] = "reject"
        result["segment_quality_flag"] = "too_short"
        return result

    # Active = non-background prediction.
    active_mask = pred_labels != 0
    active_mask = fill_small_gaps(active_mask, max_gap=max(2, int(0.03 * T)))
    active_mask = remove_small_islands(active_mask, min_len=max(2, int(0.02 * T)))

    active_segments = get_segments(active_mask)

    if len(active_segments) == 0:
        # fallback: use motion curve central active region
        result["used_fallback"] = True

        if np.max(phase_speed) <= 1e-8:
            result["segment_status"] = "reject"
            result["segment_quality_flag"] = "no_active_prediction_and_flat_speed"
            return result

        threshold = 0.30 * float(np.max(phase_speed))
        active_mask = phase_speed >= threshold
        active_mask = fill_small_gaps(active_mask, max_gap=max(2, int(0.03 * T)))
        active_mask = remove_small_islands(active_mask, min_len=max(2, int(0.02 * T)))
        active_segments = get_segments(active_mask)

        if len(active_segments) == 0:
            result["segment_status"] = "reject"
            result["segment_quality_flag"] = "no_active_region"
            return result

    # Use first-to-last active span, not only largest island.
    active_start = active_segments[0][0]
    active_end = active_segments[-1][1]

    active_start = int(max(0, active_start))
    active_end = int(min(T - 1, active_end))

    active_len = active_end - active_start + 1

    if active_len < 8:
        result["segment_status"] = "reject"
        result["segment_quality_flag"] = "active_too_short"
        return result

    # Stroke region from class-2 predictions inside active.
    stroke_mask = np.zeros(T, dtype=bool)
    stroke_mask[active_start:active_end + 1] = pred_labels[active_start:active_end + 1] == 2

    stroke_mask = fill_small_gaps(stroke_mask, max_gap=max(1, int(0.02 * active_len)))
    stroke_mask = remove_small_islands(stroke_mask, min_len=max(2, int(0.03 * active_len)))

    stroke_segments = get_segments(stroke_mask)

    if len(stroke_segments) == 0:
        # fallback: choose window around max stroke probability inside active.
        result["used_fallback"] = True

        stroke_probs = probs[active_start:active_end + 1, 2]
        peak_local = int(np.argmax(stroke_probs))
        peak = active_start + peak_local

        stroke_len = max(4, int(round(0.20 * active_len)))
        stroke_len = min(stroke_len, active_len)

        stroke_start = peak - stroke_len // 2
        stroke_start = int(max(active_start, min(stroke_start, active_end - stroke_len + 1)))
        stroke_end = stroke_start + stroke_len - 1
    else:
        # Use largest stroke segment.
        stroke_start, stroke_end = largest_segment(stroke_segments)

    stroke_start = int(max(active_start, stroke_start))
    stroke_end = int(min(active_end, stroke_end))

    # Enforce at least tiny prep/retraction when possible.
    if stroke_start <= active_start and active_len >= 10:
        stroke_start = active_start + 1

    if stroke_end >= active_end and active_len >= 10:
        stroke_end = active_end - 1

    if stroke_start > stroke_end:
        result["segment_status"] = "reject"
        result["segment_quality_flag"] = "invalid_stroke_bounds"
        return result

    prep_start = active_start
    prep_end = stroke_start - 1

    retract_start = stroke_end + 1
    retract_end = active_end

    prep_len = max(0, prep_end - prep_start + 1)
    stroke_len = max(0, stroke_end - stroke_start + 1)
    retract_len = max(0, retract_end - retract_start + 1)
    background_len = T - active_len

    active_ratio = active_len / max(T, 1)
    stroke_active_ratio = stroke_len / max(active_len, 1)

    result.update(
        {
            "active_start": active_start,
            "active_end": active_end,
            "prep_start": prep_start,
            "prep_end": prep_end,
            "stroke_start": stroke_start,
            "stroke_end": stroke_end,
            "retract_start": retract_start,
            "retract_end": retract_end,
            "background_len_pred": int(background_len),
            "prep_len_pred": int(prep_len),
            "stroke_len_pred": int(stroke_len),
            "retract_len_pred": int(retract_len),
            "active_len_pred": int(active_len),
            "active_ratio_pred": float(active_ratio),
            "stroke_active_ratio_pred": float(stroke_active_ratio),
        }
    )

    # Quality rules for segment usefulness.
    if prep_len < 1:
        result["segment_status"] = "warning"
        result["segment_quality_flag"] = "prep_missing_or_tiny"
    elif stroke_len < 3:
        result["segment_status"] = "warning"
        result["segment_quality_flag"] = "stroke_too_short"
    elif retract_len < 1:
        result["segment_status"] = "warning"
        result["segment_quality_flag"] = "retract_missing_or_tiny"
    elif active_ratio > 0.95:
        result["segment_status"] = "warning"
        result["segment_quality_flag"] = "active_very_broad"
    elif stroke_active_ratio > 0.75:
        result["segment_status"] = "warning"
        result["segment_quality_flag"] = "stroke_very_broad"
    else:
        result["segment_status"] = "ok"
        result["segment_quality_flag"] = "good"

    return result

def make_segment_label_sequence(T, seg):
    """
    Creates clean ordered segment labels from extracted boundaries.
    0 background, 1 prep, 2 stroke, 3 retract.
    """
    labels = np.zeros(T, dtype=np.int64)

    if seg["segment_status"] == "reject":
        return labels

    ps, pe = seg["prep_start"], seg["prep_end"]
    ss, se = seg["stroke_start"], seg["stroke_end"]
    rs, re = seg["retract_start"], seg["retract_end"]

    if ps >= 0 and pe >= ps:
        labels[ps:pe + 1] = 1
    if ss >= 0 and se >= ss:
        labels[ss:se + 1] = 2
    if rs >= 0 and re >= rs:
        labels[rs:re + 1] = 3

    return labels

# ----------------------------
# 6. Generate segments
# ----------------------------

records = []

print("Generating phase-aware segments from Phase TCN predictions...")
print()

for i, row in dataset_df.iterrows():
    result = {
        "row_id": row["row_id"],
        "split": row["split"],
        "gloss": row["gloss"],
        "label_id": row["label_id"],
        "video_stem": row["video_stem"],
        "motion_path": row["motion_path"],
        "phase_label_path": row["phase_label_path"],
        "phase_segment_path": "",
        "n_frames": row["T_motion"],
        "frame_acc_vs_pseudo_raw": np.nan,
        "frame_acc_vs_pseudo_smooth": np.nan,
        "error": "",
    }

    try:
        X, pseudo_labels, phase_speed, probs, raw_pred = predict_phases(
            row["motion_path"],
            row["phase_label_path"]
        )

        smooth_pred = majority_filter_1d(raw_pred, window=5)

        frame_acc_raw = float(np.mean(raw_pred == pseudo_labels))
        frame_acc_smooth = float(np.mean(smooth_pred == pseudo_labels))

        seg = extract_phase_segments(
            pred_labels=smooth_pred,
            probs=probs,
            phase_speed=phase_speed
        )

        T = len(smooth_pred)
        ordered_segment_labels = make_segment_label_sequence(T, seg)

        segment_path = PHASE_SEGMENT_DIR / f"{safe_stem(row['video_stem'])}.npz"

        np.savez_compressed(
            segment_path,
            raw_pred=raw_pred.astype(np.int64),
            smooth_pred=smooth_pred.astype(np.int64),
            ordered_segment_labels=ordered_segment_labels.astype(np.int64),
            pred_probs=probs.astype(np.float32),
            pseudo_labels=pseudo_labels.astype(np.int64),
            phase_speed=phase_speed.astype(np.float32),
            active_start=np.int64(seg["active_start"]),
            active_end=np.int64(seg["active_end"]),
            prep_start=np.int64(seg["prep_start"]),
            prep_end=np.int64(seg["prep_end"]),
            stroke_start=np.int64(seg["stroke_start"]),
            stroke_end=np.int64(seg["stroke_end"]),
            retract_start=np.int64(seg["retract_start"]),
            retract_end=np.int64(seg["retract_end"]),
            video_stem=str(row["video_stem"]),
            gloss=str(row["gloss"]),
            split=str(row["split"]),
            label_id=np.int64(row["label_id"]),
            segment_status=str(seg["segment_status"]),
            segment_quality_flag=str(seg["segment_quality_flag"]),
        )

        result["phase_segment_path"] = str(segment_path)
        result["frame_acc_vs_pseudo_raw"] = frame_acc_raw
        result["frame_acc_vs_pseudo_smooth"] = frame_acc_smooth
        result.update(seg)

    except Exception as e:
        result["segment_status"] = "error"
        result["segment_quality_flag"] = "exception"
        result["error"] = repr(e)

    records.append(result)

    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1}/{len(dataset_df)} sequences...")

segments_df = pd.DataFrame(records)
segments_df.to_csv(PHASE_SEGMENT_MANIFEST_PATH, index=False)

# ----------------------------
# 7. Summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 9A OUTPUT SAVED")
print("=" * 80)
print(f"Phase segment directory: {PHASE_SEGMENT_DIR}")
print(f"Phase segment manifest:  {PHASE_SEGMENT_MANIFEST_PATH}")
print()

print("=" * 80)
print("PHASE SEGMENT SUMMARY")
print("=" * 80)

print("Segment status counts:")
print(segments_df["segment_status"].value_counts(dropna=False))
print()

print("Segment quality flag counts:")
print(segments_df["segment_quality_flag"].value_counts(dropna=False))
print()

print("Rows by split:")
print(segments_df["split"].value_counts(dropna=False))
print()

print("OK / warning / reject by split:")
print(pd.crosstab(segments_df["split"], segments_df["segment_status"]))
print()

print("Frame accuracy vs pseudo-labels:")
print("Raw predictions:")
print(segments_df["frame_acc_vs_pseudo_raw"].describe())
print()
print("Smoothed predictions:")
print(segments_df["frame_acc_vs_pseudo_smooth"].describe())
print()

print("Predicted segment length summaries:")
for col in [
    "background_len_pred",
    "prep_len_pred",
    "stroke_len_pred",
    "retract_len_pred",
    "active_len_pred",
    "active_ratio_pred",
    "stroke_active_ratio_pred",
]:
    print()
    print(col)
    print(segments_df[col].describe())

print()
print("Fallback usage:")
print(segments_df["used_fallback"].value_counts(dropna=False))
print()

# ----------------------------
# 8. Preview examples
# ----------------------------

print("=" * 80)
print("FIRST 10 OK SEGMENTS")
print("=" * 80)

ok_examples = segments_df[segments_df["segment_status"] == "ok"].head(10)

if len(ok_examples) == 0:
    print("No OK segments found.")
else:
    display_cols = [
        "split",
        "gloss",
        "video_stem",
        "n_frames",
        "segment_status",
        "segment_quality_flag",
        "active_start",
        "active_end",
        "prep_len_pred",
        "stroke_len_pred",
        "retract_len_pred",
        "active_ratio_pred",
        "stroke_active_ratio_pred",
        "frame_acc_vs_pseudo_smooth",
        "phase_segment_path",
    ]
    display(ok_examples[display_cols])

print()
print("=" * 80)
print("WARNING / REJECT PREVIEW")
print("=" * 80)

problem_df = segments_df[segments_df["segment_status"] != "ok"].copy()

if len(problem_df) == 0:
    print("No warning/reject segments.")
else:
    print(f"Problem segments: {len(problem_df)}")
    display_cols = [
        "split",
        "gloss",
        "video_stem",
        "n_frames",
        "segment_status",
        "segment_quality_flag",
        "prep_len_pred",
        "stroke_len_pred",
        "retract_len_pred",
        "active_ratio_pred",
        "stroke_active_ratio_pred",
        "frame_acc_vs_pseudo_smooth",
        "used_fallback",
        "error",
    ]
    display(problem_df[display_cols].head(30))

print()
print("=" * 80)
print("PHASE 9A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Segment status counts")
print("2. Segment quality flag counts")
print("3. OK / warning / reject by split")
print("4. Frame accuracy vs pseudo-labels")
print("5. Predicted segment length summaries")
print("6. Fallback usage")
print("7. First 10 OK segments")
print("8. Warning / reject preview")
print()
print("Do NOT start Recognition TCN yet. Next step will be Phase 9B: visualize generated segments.")

In [ ]:
# ============================================================
# PHASE 9B — VISUALIZE GENERATED PHASE-AWARE SEGMENTS
# Run this cell only, then paste the printed output and visual impression back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

PHASE_SEGMENT_MANIFEST_PATH = MANIFEST_DIR / "phase_segments_manifest.csv"

SEGMENT_PLOT_DIR = BASE_DIR / "plots" / "phase_segments"
SEGMENT_VIS_MANIFEST_PATH = MANIFEST_DIR / "phase_segments_visualization_manifest.csv"

SEGMENT_PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 9B — VISUALIZE GENERATED PHASE-AWARE SEGMENTS")
print("=" * 80)

if not PHASE_SEGMENT_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing phase segment manifest: {PHASE_SEGMENT_MANIFEST_PATH}")

segments_df = pd.read_csv(PHASE_SEGMENT_MANIFEST_PATH)

print(f"Loaded phase segment manifest: {PHASE_SEGMENT_MANIFEST_PATH}")
print(f"Rows: {len(segments_df)}")
print(f"Plot output directory: {SEGMENT_PLOT_DIR}")
print()

print("Segment status counts:")
print(segments_df["segment_status"].value_counts(dropna=False))
print()

print("Segment quality flag counts:")
print(segments_df["segment_quality_flag"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Select examples
# ----------------------------

random.seed(42)
np.random.seed(42)

selected_parts = []

def add_examples(df, bucket_name, n):
    if len(df) == 0:
        print(f"Warning: no rows for bucket: {bucket_name}")
        return

    n = min(n, len(df))
    sample = df.sample(n=n, random_state=42).copy()
    sample["plot_bucket"] = bucket_name
    selected_parts.append(sample)

ok_df = segments_df[segments_df["segment_status"] == "ok"].copy()
warning_df = segments_df[segments_df["segment_status"] == "warning"].copy()

high_acc_df = ok_df.sort_values("frame_acc_vs_pseudo_smooth", ascending=False).head(50)
low_acc_df = ok_df.sort_values("frame_acc_vs_pseudo_smooth", ascending=True).head(50)

broad_active_df = segments_df[segments_df["segment_quality_flag"] == "active_very_broad"].copy()
broad_stroke_df = segments_df[segments_df["segment_quality_flag"] == "stroke_very_broad"].copy()
fallback_df = segments_df[segments_df["used_fallback"] == True].copy()

# Buckets
add_examples(ok_df, "ok_random", 10)
add_examples(high_acc_df, "ok_high_accuracy", 5)
add_examples(low_acc_df, "ok_low_accuracy", 5)
add_examples(warning_df, "warning", 5)
add_examples(broad_active_df, "active_broad", 3)
add_examples(broad_stroke_df, "stroke_broad", 3)
add_examples(fallback_df, "fallback_used", 3)

selected_df = pd.concat(selected_parts, ignore_index=True)
selected_df = selected_df.drop_duplicates(subset=["phase_segment_path"]).reset_index(drop=True)

print(f"Selected examples to plot: {len(selected_df)}")
print()

# ----------------------------
# 3. Plot helpers
# ----------------------------

PHASE_NAMES = {
    0: "background",
    1: "preparation",
    2: "stroke",
    3: "retraction",
}

PHASE_COLORS = {
    0: "lightgray",
    1: "gold",
    2: "tomato",
    3: "skyblue",
}

def label_segments(labels):
    labels = np.asarray(labels, dtype=np.int64)

    if len(labels) == 0:
        return []

    segments = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        val = int(labels[i])
        if val != current:
            segments.append((current, start, i - 1))
            start = i
            current = val

    segments.append((current, start, len(labels) - 1))
    return segments

def draw_phase_panel(ax, frames, phase_speed, labels, title):
    labels = np.asarray(labels, dtype=np.int64)

    for phase_id, start, end in label_segments(labels):
        ax.axvspan(
            start,
            end + 1,
            color=PHASE_COLORS.get(phase_id, "white"),
            alpha=0.35,
            label=PHASE_NAMES.get(phase_id, str(phase_id)),
        )

    ax.plot(frames, phase_speed, linewidth=2.0)
    ax.set_title(title)
    ax.set_ylabel("phase_speed")
    ax.set_ylim(bottom=0)

    handles, labels_legend = ax.get_legend_handles_labels()
    unique = {}
    for h, l in zip(handles, labels_legend):
        if l not in unique:
            unique[l] = h
    ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize=8)

def plot_segment_example(row, show=False):
    segment_path = Path(row["phase_segment_path"])

    if not segment_path.exists():
        raise FileNotFoundError(f"Missing segment file: {segment_path}")

    with np.load(segment_path, allow_pickle=False) as npz:
        pseudo_labels = np.asarray(npz["pseudo_labels"], dtype=np.int64)
        raw_pred = np.asarray(npz["raw_pred"], dtype=np.int64)
        smooth_pred = np.asarray(npz["smooth_pred"], dtype=np.int64)
        ordered_segment_labels = np.asarray(npz["ordered_segment_labels"], dtype=np.int64)
        phase_speed = np.asarray(npz["phase_speed"], dtype=np.float32)

        active_start = int(np.asarray(npz["active_start"]).item())
        active_end = int(np.asarray(npz["active_end"]).item())
        prep_start = int(np.asarray(npz["prep_start"]).item())
        prep_end = int(np.asarray(npz["prep_end"]).item())
        stroke_start = int(np.asarray(npz["stroke_start"]).item())
        stroke_end = int(np.asarray(npz["stroke_end"]).item())
        retract_start = int(np.asarray(npz["retract_start"]).item())
        retract_end = int(np.asarray(npz["retract_end"]).item())

    phase_speed[~np.isfinite(phase_speed)] = 0.0

    T = len(phase_speed)
    frames = np.arange(T)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    draw_phase_panel(
        axes[0],
        frames,
        phase_speed,
        pseudo_labels,
        "Original pseudo-labels / weak labels",
    )

    draw_phase_panel(
        axes[1],
        frames,
        phase_speed,
        smooth_pred,
        "Smoothed Phase TCN frame predictions",
    )

    draw_phase_panel(
        axes[2],
        frames,
        phase_speed,
        ordered_segment_labels,
        "Final ordered phase-aware segment labels",
    )

    # Draw final boundary lines on all panels.
    for ax in axes:
        for x, name in [
            (active_start, "active_start"),
            (active_end, "active_end"),
            (stroke_start, "stroke_start"),
            (stroke_end, "stroke_end"),
        ]:
            if x >= 0:
                ax.axvline(x, linestyle="--", linewidth=1.0)

    axes[2].set_xlabel("Frame")

    fig.suptitle(
        f"{row['plot_bucket']} | {row['split']} | {row['gloss']} | {row['video_stem']} | "
        f"T={int(row['n_frames'])} | status={row['segment_status']} | "
        f"flag={row['segment_quality_flag']} | acc={float(row['frame_acc_vs_pseudo_smooth']):.3f}\n"
        f"active={active_start}-{active_end}, prep={prep_start}-{prep_end}, "
        f"stroke={stroke_start}-{stroke_end}, retract={retract_start}-{retract_end}",
        y=1.02,
    )

    plt.tight_layout()

    safe_name = str(row["video_stem"]).replace("/", "_").replace("\\", "_").replace(" ", "_")
    save_path = SEGMENT_PLOT_DIR / f"{row['plot_bucket']}__{safe_name}.png"

    plt.savefig(save_path, dpi=140, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return save_path

# ----------------------------
# 4. Generate plots
# ----------------------------

saved_paths = []

print("Generating segment visualizations...")
print("The first 8 plots will be displayed below.")
print()

for idx, row in selected_df.iterrows():
    show_now = idx < 8
    save_path = plot_segment_example(row, show=show_now)
    saved_paths.append(str(save_path))

selected_df["phase_segment_plot_path"] = saved_paths
selected_df.to_csv(SEGMENT_VIS_MANIFEST_PATH, index=False)

# ----------------------------
# 5. Summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 9B OUTPUT SAVED")
print("=" * 80)
print(f"Segment plots:                  {SEGMENT_PLOT_DIR}")
print(f"Segment visualization manifest: {SEGMENT_VIS_MANIFEST_PATH}")
print()

print("=" * 80)
print("VISUALIZATION SELECTION SUMMARY")
print("=" * 80)

print("Plot bucket counts:")
print(selected_df["plot_bucket"].value_counts(dropna=False))
print()

print("Selected segment status counts:")
print(selected_df["segment_status"].value_counts(dropna=False))
print()

print("Selected segment quality flag counts:")
print(selected_df["segment_quality_flag"].value_counts(dropna=False))
print()

print("=" * 80)
print("SELECTED EXAMPLES")
print("=" * 80)

display_cols = [
    "plot_bucket",
    "split",
    "gloss",
    "video_stem",
    "n_frames",
    "segment_status",
    "segment_quality_flag",
    "active_start",
    "active_end",
    "prep_len_pred",
    "stroke_len_pred",
    "retract_len_pred",
    "active_ratio_pred",
    "stroke_active_ratio_pred",
    "frame_acc_vs_pseudo_smooth",
    "used_fallback",
    "phase_segment_plot_path",
]

display(selected_df[display_cols])

print()
print("First 15 saved plot paths:")
for p in saved_paths[:15]:
    print(p)

print()
print("=" * 80)
print("PHASE 9B COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Plot bucket counts")
print("2. Selected segment status counts")
print("3. Selected segment quality flag counts")
print("4. Selected examples table")
print()
print("Also visually answer:")
print("A. Do final ordered segments look cleaner than raw/smoothed frame predictions?")
print("B. Does the stroke segment usually cover the main motion peak?")
print("C. Are warning examples obviously risky?")
print("D. Should we exclude warnings from Recognition TCN training?")
print()
print("Do NOT start Recognition TCN yet.")

In [ ]:
# ============================================================
# PHASE 10A — BUILD CLEAN RECOGNITION TCN MANIFEST
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

PHASE_SEGMENT_MANIFEST_PATH = MANIFEST_DIR / "phase_segments_manifest.csv"
MOTION_MANIFEST_PATH = MANIFEST_DIR / "motion_manifest.csv"

RECOGNITION_MANIFEST_PATH = MANIFEST_DIR / "recognition_manifest_clean.csv"
RECOGNITION_LABEL_MAP_PATH = MANIFEST_DIR / "recognition_label_map.csv"
RECOGNITION_MANIFEST_REJECTED_PATH = MANIFEST_DIR / "recognition_manifest_rejected.csv"

print("=" * 80)
print("PHASE 10A — BUILD CLEAN RECOGNITION TCN MANIFEST")
print("=" * 80)

if not PHASE_SEGMENT_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing phase segment manifest: {PHASE_SEGMENT_MANIFEST_PATH}")

if not MOTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing motion manifest: {MOTION_MANIFEST_PATH}")

segments_df = pd.read_csv(PHASE_SEGMENT_MANIFEST_PATH)
motion_df = pd.read_csv(MOTION_MANIFEST_PATH)

print(f"Loaded phase segment manifest: {PHASE_SEGMENT_MANIFEST_PATH}")
print(f"Segment rows: {len(segments_df)}")
print(f"Loaded motion manifest: {MOTION_MANIFEST_PATH}")
print(f"Motion rows: {len(motion_df)}")
print()

# ----------------------------
# 2. Merge motion info
# ----------------------------

motion_keep = motion_df[
    [
        "row_id",
        "video_stem",
        "motion_path",
        "status",
        "feature_dim",
        "n_frames",
    ]
].copy()

motion_keep = motion_keep.rename(
    columns={
        "status": "motion_status",
        "n_frames": "motion_n_frames",
    }
)

rec_df = segments_df.merge(
    motion_keep,
    on=["row_id", "video_stem"],
    how="left",
    suffixes=("", "_motion"),
)

print("After merge:")
print(f"Rows: {len(rec_df)}")
print()

# ----------------------------
# 3. Recognition-clean filtering rules
# ----------------------------

MIN_TOTAL_FRAMES = 24
MIN_ACTIVE_LEN = 16
MIN_PREP_LEN = 2
MIN_STROKE_LEN = 4
MIN_RETRACT_LEN = 2

MAX_ACTIVE_RATIO = 0.90
MAX_STROKE_ACTIVE_RATIO = 0.70

# This filters examples where Phase TCN predictions disagree strongly
# with the pseudo-labels. Keep this moderate, not too strict.
MIN_FRAME_ACC_VS_PSEUDO = 0.60

print("Recognition manifest filtering rules:")
print(f"MIN_TOTAL_FRAMES:          {MIN_TOTAL_FRAMES}")
print(f"MIN_ACTIVE_LEN:            {MIN_ACTIVE_LEN}")
print(f"MIN_PREP_LEN:              {MIN_PREP_LEN}")
print(f"MIN_STROKE_LEN:            {MIN_STROKE_LEN}")
print(f"MIN_RETRACT_LEN:           {MIN_RETRACT_LEN}")
print(f"MAX_ACTIVE_RATIO:          {MAX_ACTIVE_RATIO}")
print(f"MAX_STROKE_ACTIVE_RATIO:   {MAX_STROKE_ACTIVE_RATIO}")
print(f"MIN_FRAME_ACC_VS_PSEUDO:   {MIN_FRAME_ACC_VS_PSEUDO}")
print()

def check_file_exists(path_value):
    try:
        return Path(str(path_value)).exists()
    except Exception:
        return False

def recognition_filter_reason(row):
    """
    Returns:
      keep, reason
    """
    if row.get("motion_status") != "ok":
        return False, "motion_status_not_ok"

    if not check_file_exists(row.get("motion_path", "")):
        return False, "missing_motion_file"

    if not check_file_exists(row.get("phase_segment_path", "")):
        return False, "missing_phase_segment_file"

    if row.get("segment_status") != "ok":
        return False, f"segment_status_{row.get('segment_status')}"

    if row.get("segment_quality_flag") != "good":
        return False, f"segment_quality_{row.get('segment_quality_flag')}"

    if bool(row.get("used_fallback")) == True:
        return False, "used_fallback"

    if int(row.get("n_frames", 0)) < MIN_TOTAL_FRAMES:
        return False, "too_few_total_frames"

    if int(row.get("active_len_pred", 0)) < MIN_ACTIVE_LEN:
        return False, "active_too_short"

    if int(row.get("prep_len_pred", 0)) < MIN_PREP_LEN:
        return False, "prep_too_short"

    if int(row.get("stroke_len_pred", 0)) < MIN_STROKE_LEN:
        return False, "stroke_too_short"

    if int(row.get("retract_len_pred", 0)) < MIN_RETRACT_LEN:
        return False, "retract_too_short"

    if float(row.get("active_ratio_pred", 0.0)) > MAX_ACTIVE_RATIO:
        return False, "active_too_broad"

    if float(row.get("stroke_active_ratio_pred", 0.0)) > MAX_STROKE_ACTIVE_RATIO:
        return False, "stroke_too_broad"

    if float(row.get("frame_acc_vs_pseudo_smooth", 0.0)) < MIN_FRAME_ACC_VS_PSEUDO:
        return False, "low_phase_prediction_agreement"

    return True, "keep"

keep_flags = []
reasons = []

for _, row in rec_df.iterrows():
    keep, reason = recognition_filter_reason(row)
    keep_flags.append(keep)
    reasons.append(reason)

rec_df["usable_for_recognition"] = keep_flags
rec_df["recognition_filter_reason"] = reasons

clean_rec_df = rec_df[rec_df["usable_for_recognition"] == True].copy()
rejected_rec_df = rec_df[rec_df["usable_for_recognition"] != True].copy()

# ----------------------------
# 4. Create recognition class mapping
# ----------------------------

# Use gloss names as recognition classes.
# Sort for stable class IDs.
unique_glosses = sorted(clean_rec_df["gloss"].astype(str).unique())
gloss_to_rec_label = {g: i for i, g in enumerate(unique_glosses)}

clean_rec_df["rec_label_id"] = clean_rec_df["gloss"].astype(str).map(gloss_to_rec_label)

label_map_df = pd.DataFrame(
    [
        {"rec_label_id": idx, "gloss": gloss}
        for gloss, idx in gloss_to_rec_label.items()
    ]
).sort_values("rec_label_id")

# ----------------------------
# 5. Verify split class coverage
# ----------------------------

coverage_rows = []

for gloss in unique_glosses:
    sub = clean_rec_df[clean_rec_df["gloss"].astype(str) == gloss]

    coverage_rows.append(
        {
            "gloss": gloss,
            "rec_label_id": gloss_to_rec_label[gloss],
            "train_count": int((sub["split"] == "train").sum()),
            "val_count": int((sub["split"] == "val").sum()),
            "test_count": int((sub["split"] == "test").sum()),
            "total_count": int(len(sub)),
        }
    )

coverage_df = pd.DataFrame(coverage_rows)

# Add coverage information back into label map.
label_map_df = label_map_df.merge(
    coverage_df,
    on=["rec_label_id", "gloss"],
    how="left",
)

# If some classes have zero train examples, recognition training will break.
classes_missing_train = label_map_df[label_map_df["train_count"] == 0].copy()
classes_missing_val = label_map_df[label_map_df["val_count"] == 0].copy()
classes_missing_test = label_map_df[label_map_df["test_count"] == 0].copy()

# ----------------------------
# 6. Save outputs
# ----------------------------

clean_rec_df.to_csv(RECOGNITION_MANIFEST_PATH, index=False)
rejected_rec_df.to_csv(RECOGNITION_MANIFEST_REJECTED_PATH, index=False)
label_map_df.to_csv(RECOGNITION_LABEL_MAP_PATH, index=False)

# ----------------------------
# 7. Summary
# ----------------------------

print("=" * 80)
print("PHASE 10A OUTPUT SAVED")
print("=" * 80)
print(f"Clean recognition manifest:    {RECOGNITION_MANIFEST_PATH}")
print(f"Rejected recognition manifest: {RECOGNITION_MANIFEST_REJECTED_PATH}")
print(f"Recognition label map:         {RECOGNITION_LABEL_MAP_PATH}")
print()

print("=" * 80)
print("RECOGNITION MANIFEST SUMMARY")
print("=" * 80)

print("Original segment status counts:")
print(rec_df["segment_status"].value_counts(dropna=False))
print()

print("Recognition usable counts:")
print(rec_df["usable_for_recognition"].value_counts(dropna=False))
print()

print("Recognition filter reason counts:")
print(rec_df["recognition_filter_reason"].value_counts(dropna=False))
print()

print("Clean rows by split:")
print(clean_rec_df["split"].value_counts(dropna=False))
print()

print("Clean number of recognition classes:")
print(clean_rec_df["gloss"].nunique())
print()

print("Clean examples per class summary:")
print(coverage_df["total_count"].describe())
print()

print("Train examples per class summary:")
print(coverage_df["train_count"].describe())
print()

print("Val examples per class summary:")
print(coverage_df["val_count"].describe())
print()

print("Test examples per class summary:")
print(coverage_df["test_count"].describe())
print()

print("=" * 80)
print("CLASS COVERAGE WARNINGS")
print("=" * 80)

print(f"Classes missing train examples: {len(classes_missing_train)}")
if len(classes_missing_train) > 0:
    display(classes_missing_train)

print(f"Classes missing val examples: {len(classes_missing_val)}")
if len(classes_missing_val) > 0:
    display(classes_missing_val)

print(f"Classes missing test examples: {len(classes_missing_test)}")
if len(classes_missing_test) > 0:
    display(classes_missing_test)

print()
print("=" * 80)
print("FIRST 10 CLEAN RECOGNITION ROWS")
print("=" * 80)

display_cols = [
    "split",
    "gloss",
    "rec_label_id",
    "video_stem",
    "n_frames",
    "active_start",
    "active_end",
    "prep_len_pred",
    "stroke_len_pred",
    "retract_len_pred",
    "active_ratio_pred",
    "stroke_active_ratio_pred",
    "frame_acc_vs_pseudo_smooth",
    "motion_path",
    "phase_segment_path",
]

display(clean_rec_df[display_cols].head(10))

print()
print("=" * 80)
print("FIRST 20 REJECTED RECOGNITION ROWS")
print("=" * 80)

if len(rejected_rec_df) == 0:
    print("No rejected recognition rows.")
else:
    display_cols_rej = [
        "split",
        "gloss",
        "video_stem",
        "n_frames",
        "segment_status",
        "segment_quality_flag",
        "used_fallback",
        "prep_len_pred",
        "stroke_len_pred",
        "retract_len_pred",
        "active_ratio_pred",
        "stroke_active_ratio_pred",
        "frame_acc_vs_pseudo_smooth",
        "recognition_filter_reason",
    ]
    display(rejected_rec_df[display_cols_rej].head(20))

print()
print("=" * 80)
print("PHASE 10A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Recognition usable counts")
print("2. Recognition filter reason counts")
print("3. Clean rows by split")
print("4. Clean number of recognition classes")
print("5. Examples per class summaries")
print("6. Class coverage warnings")
print("7. First 10 clean recognition rows")
print("8. First 20 rejected recognition rows")
print()
print("Do NOT train Recognition TCN yet.")

In [ ]:
# ============================================================
# PHASE 10A-FIX — CREATE RECOGNITION-SPECIFIC SPLIT WITH FULL VAL COVERAGE
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

RECOGNITION_MANIFEST_PATH = MANIFEST_DIR / "recognition_manifest_clean.csv"
RECOGNITION_LABEL_MAP_PATH = MANIFEST_DIR / "recognition_label_map.csv"

RECOGNITION_SPLIT_MANIFEST_PATH = MANIFEST_DIR / "recognition_manifest_splits.csv"
RECOGNITION_LABEL_MAP_SPLITS_PATH = MANIFEST_DIR / "recognition_label_map_with_split_counts.csv"

print("=" * 80)
print("PHASE 10A-FIX — RECOGNITION SPLIT COVERAGE FIX")
print("=" * 80)

if not RECOGNITION_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing clean recognition manifest: {RECOGNITION_MANIFEST_PATH}")

if not RECOGNITION_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(f"Missing recognition label map: {RECOGNITION_LABEL_MAP_PATH}")

rec_df = pd.read_csv(RECOGNITION_MANIFEST_PATH)
label_map_df = pd.read_csv(RECOGNITION_LABEL_MAP_PATH)

print(f"Loaded clean recognition manifest: {RECOGNITION_MANIFEST_PATH}")
print(f"Rows: {len(rec_df)}")
print(f"Loaded label map: {RECOGNITION_LABEL_MAP_PATH}")
print(f"Classes: {len(label_map_df)}")
print()

# ----------------------------
# 2. Create recognition-specific split
# ----------------------------

# Keep original split untouched.
# rec_split will be used for Recognition TCN training/validation/test.
rec_df["original_split"] = rec_df["split"]
rec_df["rec_split"] = rec_df["split"]

print("Original clean rows by split:")
print(rec_df["split"].value_counts(dropna=False))
print()

# ----------------------------
# 3. Coverage helper
# ----------------------------

def compute_split_coverage(df, split_col="rec_split"):
    rows = []

    for _, label_row in label_map_df.iterrows():
        class_id = int(label_row["rec_label_id"])
        gloss = str(label_row["gloss"])

        sub = df[df["rec_label_id"] == class_id]

        rows.append(
            {
                "rec_label_id": class_id,
                "gloss": gloss,
                "train_count": int((sub[split_col] == "train").sum()),
                "val_count": int((sub[split_col] == "val").sum()),
                "test_count": int((sub[split_col] == "test").sum()),
                "total_count": int(len(sub)),
            }
        )

    return pd.DataFrame(rows).sort_values("rec_label_id").reset_index(drop=True)

coverage_before = compute_split_coverage(rec_df, "rec_split")

missing_val_before = coverage_before[coverage_before["val_count"] == 0].copy()
missing_train_before = coverage_before[coverage_before["train_count"] == 0].copy()
missing_test_before = coverage_before[coverage_before["test_count"] == 0].copy()

print("=" * 80)
print("COVERAGE BEFORE FIX")
print("=" * 80)
print(f"Classes missing train: {len(missing_train_before)}")
print(f"Classes missing val:   {len(missing_val_before)}")
print(f"Classes missing test:  {len(missing_test_before)}")
print()

if len(missing_val_before) > 0:
    print("Classes missing validation examples:")
    display(missing_val_before)

# ----------------------------
# 4. Move one clean train sample into rec_val for missing-val classes
# ----------------------------

moved_rows = []

for _, row in missing_val_before.iterrows():
    class_id = int(row["rec_label_id"])
    gloss = str(row["gloss"])

    candidates = rec_df[
        (rec_df["rec_label_id"] == class_id)
        & (rec_df["rec_split"] == "train")
    ].copy()

    if len(candidates) == 0:
        print(f"WARNING: No train candidate available for class {class_id} / {gloss}")
        continue

    # Pick the best-quality candidate:
    # high phase agreement, reasonable active ratio, enough frames.
    candidates = candidates.sort_values(
        by=[
            "frame_acc_vs_pseudo_smooth",
            "active_len_pred",
            "n_frames",
        ],
        ascending=[False, False, False],
    )

    chosen_index = candidates.index[0]

    rec_df.loc[chosen_index, "rec_split"] = "val"

    moved_rows.append(
        {
            "rec_label_id": class_id,
            "gloss": gloss,
            "video_stem": rec_df.loc[chosen_index, "video_stem"],
            "original_split": rec_df.loc[chosen_index, "original_split"],
            "new_rec_split": "val",
            "frame_acc_vs_pseudo_smooth": rec_df.loc[chosen_index, "frame_acc_vs_pseudo_smooth"],
            "n_frames": rec_df.loc[chosen_index, "n_frames"],
            "active_len_pred": rec_df.loc[chosen_index, "active_len_pred"],
        }
    )

moved_df = pd.DataFrame(moved_rows)

# ----------------------------
# 5. Recompute coverage after fix
# ----------------------------

coverage_after = compute_split_coverage(rec_df, "rec_split")

missing_val_after = coverage_after[coverage_after["val_count"] == 0].copy()
missing_train_after = coverage_after[coverage_after["train_count"] == 0].copy()
missing_test_after = coverage_after[coverage_after["test_count"] == 0].copy()

# Add original split counts too for transparency.
original_coverage = compute_split_coverage(
    rec_df.rename(columns={"original_split": "tmp_original_split"}),
    "tmp_original_split"
)

coverage_after = coverage_after.rename(
    columns={
        "train_count": "rec_train_count",
        "val_count": "rec_val_count",
        "test_count": "rec_test_count",
        "total_count": "rec_total_count",
    }
)

label_map_with_counts = label_map_df.merge(
    coverage_after,
    on=["rec_label_id", "gloss"],
    how="left",
)

# ----------------------------
# 6. Save outputs
# ----------------------------

rec_df.to_csv(RECOGNITION_SPLIT_MANIFEST_PATH, index=False)
label_map_with_counts.to_csv(RECOGNITION_LABEL_MAP_SPLITS_PATH, index=False)

# ----------------------------
# 7. Print summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 10A-FIX OUTPUT SAVED")
print("=" * 80)
print(f"Recognition split manifest: {RECOGNITION_SPLIT_MANIFEST_PATH}")
print(f"Label map with split counts: {RECOGNITION_LABEL_MAP_SPLITS_PATH}")
print()

print("=" * 80)
print("MOVED ROWS")
print("=" * 80)

if len(moved_df) == 0:
    print("No rows moved.")
else:
    display(moved_df)

print()
print("=" * 80)
print("RECOGNITION SPLIT SUMMARY AFTER FIX")
print("=" * 80)

print("Rows by original split:")
print(rec_df["original_split"].value_counts(dropna=False))
print()

print("Rows by recognition split:")
print(rec_df["rec_split"].value_counts(dropna=False))
print()

print("Classes missing train after fix:", len(missing_train_after))
if len(missing_train_after) > 0:
    display(missing_train_after)

print("Classes missing val after fix:", len(missing_val_after))
if len(missing_val_after) > 0:
    display(missing_val_after)

print("Classes missing test after fix:", len(missing_test_after))
if len(missing_test_after) > 0:
    display(missing_test_after)

print()
print("Recognition train examples per class:")
print(label_map_with_counts["rec_train_count"].describe())
print()

print("Recognition val examples per class:")
print(label_map_with_counts["rec_val_count"].describe())
print()

print("Recognition test examples per class:")
print(label_map_with_counts["rec_test_count"].describe())
print()

print("=" * 80)
print("FIRST 10 ROWS WITH rec_split")
print("=" * 80)

display_cols = [
    "original_split",
    "rec_split",
    "gloss",
    "rec_label_id",
    "video_stem",
    "n_frames",
    "frame_acc_vs_pseudo_smooth",
    "motion_path",
    "phase_segment_path",
]

display(rec_df[display_cols].head(10))

print()
print("=" * 80)
print("PHASE 10A-FIX COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Coverage before fix")
print("2. Moved rows")
print("3. Rows by recognition split")
print("4. Classes missing train/val/test after fix")
print("5. Recognition train/val/test examples per class summaries")
print()
print("Do NOT train Recognition TCN yet.")

In [ ]:
# ============================================================
# PHASE 10B — RECOGNITION TCN DATASET / DATALOADER SETUP
# Run this cell only, then paste the printed output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

RECOGNITION_SPLIT_MANIFEST_PATH = MANIFEST_DIR / "recognition_manifest_splits.csv"
RECOGNITION_LABEL_MAP_SPLITS_PATH = MANIFEST_DIR / "recognition_label_map_with_split_counts.csv"

RECOGNITION_DATASET_MANIFEST_PATH = MANIFEST_DIR / "recognition_dataset_verified.csv"
RECOGNITION_FEATURE_STATS_PATH = MANIFEST_DIR / "recognition_feature_stats_phase_aware.npz"
RECOGNITION_CLASS_WEIGHTS_PATH = MANIFEST_DIR / "recognition_class_weights.npy"
RECOGNITION_CONFIG_PATH = MANIFEST_DIR / "recognition_dataset_config.json"

print("=" * 80)
print("PHASE 10B — RECOGNITION TCN DATASET / DATALOADER SETUP")
print("=" * 80)

if not RECOGNITION_SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing recognition split manifest: {RECOGNITION_SPLIT_MANIFEST_PATH}")

if not RECOGNITION_LABEL_MAP_SPLITS_PATH.exists():
    raise FileNotFoundError(f"Missing recognition label map: {RECOGNITION_LABEL_MAP_SPLITS_PATH}")

rec_df = pd.read_csv(RECOGNITION_SPLIT_MANIFEST_PATH)
label_map_df = pd.read_csv(RECOGNITION_LABEL_MAP_SPLITS_PATH)

print(f"Loaded recognition split manifest: {RECOGNITION_SPLIT_MANIFEST_PATH}")
print(f"Rows: {len(rec_df)}")
print(f"Loaded recognition label map: {RECOGNITION_LABEL_MAP_SPLITS_PATH}")
print(f"Classes: {len(label_map_df)}")
print()

print("Rows by rec_split:")
print(rec_df["rec_split"].value_counts(dropna=False))
print()

print("Classes by split coverage:")
print("missing train:", int((label_map_df["rec_train_count"] == 0).sum()))
print("missing val:  ", int((label_map_df["rec_val_count"] == 0).sum()))
print("missing test: ", int((label_map_df["rec_test_count"] == 0).sum()))
print()

# ----------------------------
# 2. Recognition feature configuration
# ----------------------------

CONFIG = {
    "task": "phase_aware_sign_recognition",
    "num_classes": int(label_map_df["rec_label_id"].nunique()),
    "crop_mode": "active_region",
    "continuous_feature_dim": 444,
    "phase_onehot_dim": 4,
    "final_feature_dim": 448,
    "continuous_features": [
        "positions_147",
        "velocity_147",
        "acceleration_147",
        "global_speed_1",
        "hand_speed_1",
        "phase_speed_1",
    ],
    "phase_features": [
        "ordered_segment_phase_onehot_4"
    ],
    "label_source": "gloss / rec_label_id",
    "split_column": "rec_split",
    "padding_value": 0.0,
    "num_workers": 0,
    "batch_size": 16,
}

with open(RECOGNITION_CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Recognition dataset config:")
print(json.dumps(CONFIG, indent=2))
print()

# ----------------------------
# 3. Feature loading helpers
# ----------------------------

def load_recognition_features(row):
    """
    Builds one phase-aware recognition sample.

    Continuous feature dim:
      positions:      147
      velocity:       147
      acceleration:   147
      global_speed:     1
      hand_speed:       1
      phase_speed:      1
      --------------------
      total:          444

    Extra phase-aware features:
      ordered phase one-hot: 4

    Final feature dim:
      444 + 4 = 448
    """
    motion_path = Path(str(row["motion_path"]))
    segment_path = Path(str(row["phase_segment_path"]))

    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(segment_path, allow_pickle=False) as s:
        ordered_phase = np.asarray(s["ordered_segment_labels"], dtype=np.int64)
        phase_speed = np.asarray(s["phase_speed"], dtype=np.float32).reshape(-1, 1)

        active_start = int(np.asarray(s["active_start"]).item())
        active_end = int(np.asarray(s["active_end"]).item())

    T = positions.shape[0]

    # Safety checks
    if velocity.shape[0] != T or acceleration.shape[0] != T:
        raise ValueError("Motion array length mismatch.")

    if global_speed.shape[0] != T or hand_speed.shape[0] != T:
        raise ValueError("Speed array length mismatch.")

    if len(ordered_phase) != T or phase_speed.shape[0] != T:
        raise ValueError("Segment/phase length mismatch.")

    if active_start < 0 or active_end < active_start or active_end >= T:
        raise ValueError(f"Invalid active bounds: {active_start}, {active_end}, T={T}")

    # Build continuous features first.
    X_cont = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X_cont[~np.isfinite(X_cont)] = 0.0

    if X_cont.shape[1] != CONFIG["continuous_feature_dim"]:
        raise ValueError(f"Expected continuous dim 444, got {X_cont.shape[1]}")

    # Crop active signing region only.
    X_cont = X_cont[active_start:active_end + 1]
    phase_crop = ordered_phase[active_start:active_end + 1]

    phase_crop = np.clip(phase_crop, 0, 3).astype(np.int64)

    # One-hot phase labels from final ordered segments.
    phase_onehot = np.eye(4, dtype=np.float32)[phase_crop]

    X = np.concatenate([X_cont, phase_onehot], axis=1).astype(np.float32)
    X[~np.isfinite(X)] = 0.0

    if X.shape[1] != CONFIG["final_feature_dim"]:
        raise ValueError(f"Expected final dim 448, got {X.shape[1]}")

    y = int(row["rec_label_id"])

    return X, X_cont, phase_crop, y

def verify_one_row(row):
    result = {
        "row_id": row["row_id"],
        "rec_split": row["rec_split"],
        "original_split": row["original_split"],
        "gloss": row["gloss"],
        "rec_label_id": row["rec_label_id"],
        "video_stem": row["video_stem"],
        "motion_path": row["motion_path"],
        "phase_segment_path": row["phase_segment_path"],
        "verify_status": "unknown",
        "T_active": np.nan,
        "feature_dim": np.nan,
        "continuous_feature_dim": np.nan,
        "phase_0_count": 0,
        "phase_1_count": 0,
        "phase_2_count": 0,
        "phase_3_count": 0,
        "error": "",
    }

    try:
        if not Path(str(row["motion_path"])).exists():
            result["verify_status"] = "missing_motion_file"
            return result

        if not Path(str(row["phase_segment_path"])).exists():
            result["verify_status"] = "missing_phase_segment_file"
            return result

        X, X_cont, phase_crop, y = load_recognition_features(row)

        if X.shape[0] < 1:
            result["verify_status"] = "empty_active_crop"
            return result

        if np.isnan(X).any() or np.isinf(X).any():
            result["verify_status"] = "nan_inf_features"
            return result

        if y < 0 or y >= CONFIG["num_classes"]:
            result["verify_status"] = "bad_rec_label_id"
            return result

        result["T_active"] = int(X.shape[0])
        result["feature_dim"] = int(X.shape[1])
        result["continuous_feature_dim"] = int(X_cont.shape[1])

        for c in range(4):
            result[f"phase_{c}_count"] = int(np.sum(phase_crop == c))

        result["verify_status"] = "ok"
        return result

    except Exception as e:
        result["verify_status"] = "exception"
        result["error"] = repr(e)
        return result

# ----------------------------
# 4. Verify all recognition rows
# ----------------------------

print("Verifying recognition dataset rows...")
verify_records = []

for i, row in rec_df.iterrows():
    verify_records.append(verify_one_row(row))

    if (i + 1) % 500 == 0:
        print(f"Verified {i + 1}/{len(rec_df)} rows...")

verify_df = pd.DataFrame(verify_records)

verified_df = rec_df.merge(
    verify_df[
        [
            "row_id",
            "verify_status",
            "T_active",
            "feature_dim",
            "continuous_feature_dim",
            "phase_0_count",
            "phase_1_count",
            "phase_2_count",
            "phase_3_count",
            "error",
        ]
    ],
    on="row_id",
    how="left",
    suffixes=("", "_verify"),
)

verified_df.to_csv(RECOGNITION_DATASET_MANIFEST_PATH, index=False)

ok_df = verified_df[verified_df["verify_status"] == "ok"].copy()

print()
print("=" * 80)
print("RECOGNITION DATASET VERIFICATION SUMMARY")
print("=" * 80)

print("Verify status counts:")
print(verified_df["verify_status"].value_counts(dropna=False))
print()

print("OK rows by rec_split:")
print(ok_df["rec_split"].value_counts(dropna=False))
print()

print("Feature dim summary:")
print(ok_df["feature_dim"].describe())
print()

print("Active sequence length summary:")
print(ok_df["T_active"].describe())
print()

print("Phase counts across active crops:")
print("phase 0 background:", int(ok_df["phase_0_count"].sum()))
print("phase 1 preparation:", int(ok_df["phase_1_count"].sum()))
print("phase 2 stroke:", int(ok_df["phase_2_count"].sum()))
print("phase 3 retraction:", int(ok_df["phase_3_count"].sum()))
print()

if len(ok_df) == 0:
    raise ValueError("No verified recognition rows. Stop.")

# ----------------------------
# 5. Check split/class coverage after verification
# ----------------------------

coverage_rows = []

for _, label_row in label_map_df.iterrows():
    class_id = int(label_row["rec_label_id"])
    gloss = str(label_row["gloss"])

    sub = ok_df[ok_df["rec_label_id"] == class_id]

    coverage_rows.append(
        {
            "rec_label_id": class_id,
            "gloss": gloss,
            "train_count": int((sub["rec_split"] == "train").sum()),
            "val_count": int((sub["rec_split"] == "val").sum()),
            "test_count": int((sub["rec_split"] == "test").sum()),
            "total_count": int(len(sub)),
        }
    )

coverage_verified_df = pd.DataFrame(coverage_rows)

print("=" * 80)
print("VERIFIED CLASS COVERAGE")
print("=" * 80)

print("Classes missing train:", int((coverage_verified_df["train_count"] == 0).sum()))
print("Classes missing val:", int((coverage_verified_df["val_count"] == 0).sum()))
print("Classes missing test:", int((coverage_verified_df["test_count"] == 0).sum()))
print()

print("Verified train examples per class:")
print(coverage_verified_df["train_count"].describe())
print()

print("Verified val examples per class:")
print(coverage_verified_df["val_count"].describe())
print()

print("Verified test examples per class:")
print(coverage_verified_df["test_count"].describe())
print()

# ----------------------------
# 6. Compute train feature normalization stats
# ----------------------------

train_df = ok_df[ok_df["rec_split"] == "train"].copy()

if len(train_df) == 0:
    raise ValueError("No train rows after verification.")

print("=" * 80)
print("COMPUTING TRAIN FEATURE STATS")
print("=" * 80)
print("Stats are computed only on the 444 continuous features.")
print("The 4 phase one-hot features are NOT standardized.")
print()

feature_sum = None
feature_sq_sum = None
frame_count = 0

for i, row in train_df.iterrows():
    X, X_cont, phase_crop, y = load_recognition_features(row)

    if feature_sum is None:
        feature_sum = np.zeros(X_cont.shape[1], dtype=np.float64)
        feature_sq_sum = np.zeros(X_cont.shape[1], dtype=np.float64)

    feature_sum += X_cont.sum(axis=0)
    feature_sq_sum += (X_cont.astype(np.float64) ** 2).sum(axis=0)
    frame_count += X_cont.shape[0]

feature_mean = feature_sum / max(frame_count, 1)
feature_var = feature_sq_sum / max(frame_count, 1) - feature_mean ** 2
feature_var = np.maximum(feature_var, 1e-8)
feature_std = np.sqrt(feature_var)

feature_mean = feature_mean.astype(np.float32)
feature_std = feature_std.astype(np.float32)

np.savez_compressed(
    RECOGNITION_FEATURE_STATS_PATH,
    mean=feature_mean,
    std=feature_std,
    frame_count=np.int64(frame_count),
    continuous_feature_dim=np.int64(CONFIG["continuous_feature_dim"]),
    final_feature_dim=np.int64(CONFIG["final_feature_dim"]),
)

print(f"Saved recognition feature stats: {RECOGNITION_FEATURE_STATS_PATH}")
print(f"Train active frames used for stats: {frame_count}")
print()

print("Recognition continuous feature mean summary:")
print(pd.Series(feature_mean).describe())
print()

print("Recognition continuous feature std summary:")
print(pd.Series(feature_std).describe())
print()

# ----------------------------
# 7. Compute recognition class weights
# ----------------------------

print("=" * 80)
print("COMPUTING RECOGNITION CLASS WEIGHTS")
print("=" * 80)

train_class_counts = np.zeros(CONFIG["num_classes"], dtype=np.int64)

for _, row in train_df.iterrows():
    train_class_counts[int(row["rec_label_id"])] += 1

total_train = int(train_class_counts.sum())

class_weights = total_train / (CONFIG["num_classes"] * np.maximum(train_class_counts, 1))
class_weights = class_weights / np.mean(class_weights)
class_weights = class_weights.astype(np.float32)

np.save(RECOGNITION_CLASS_WEIGHTS_PATH, class_weights)

print("Train class count summary:")
print(pd.Series(train_class_counts).describe())
print()

print("Recognition class weight summary:")
print(pd.Series(class_weights).describe())
print()

print(f"Saved recognition class weights: {RECOGNITION_CLASS_WEIGHTS_PATH}")
print()

# ----------------------------
# 8. PyTorch Dataset and Dataloader
# ----------------------------

class RecognitionDataset(Dataset):
    def __init__(self, df, feature_mean, feature_std):
        self.df = df.reset_index(drop=True)
        self.feature_mean = feature_mean.astype(np.float32)
        self.feature_std = feature_std.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        X, X_cont, phase_crop, y = load_recognition_features(row)

        # Standardize only continuous features.
        X_cont_norm = (X_cont - self.feature_mean) / self.feature_std
        X_cont_norm[~np.isfinite(X_cont_norm)] = 0.0

        # Recreate raw phase one-hot.
        phase_crop = np.clip(phase_crop, 0, 3).astype(np.int64)
        phase_onehot = np.eye(4, dtype=np.float32)[phase_crop]

        X_final = np.concatenate([X_cont_norm, phase_onehot], axis=1).astype(np.float32)
        X_final[~np.isfinite(X_final)] = 0.0

        return {
            "x": torch.tensor(X_final, dtype=torch.float32),  # (T_active, 448)
            "y": torch.tensor(int(y), dtype=torch.long),      # sign class
            "length": int(X_final.shape[0]),
            "rec_split": row["rec_split"],
            "gloss": row["gloss"],
            "rec_label_id": int(row["rec_label_id"]),
            "video_stem": row["video_stem"],
        }

def recognition_collate(batch):
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    y = torch.zeros(batch_size, dtype=torch.long)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        mask[i, :T] = True
        y[i] = item["y"]

        meta.append(
            {
                "rec_split": item["rec_split"],
                "gloss": item["gloss"],
                "rec_label_id": item["rec_label_id"],
                "video_stem": item["video_stem"],
            }
        )

    return {
        "x": x_padded,
        "y": y,
        "mask": mask,
        "lengths": lengths,
        "meta": meta,
    }

train_ok_df = ok_df[ok_df["rec_split"] == "train"].copy()
val_ok_df = ok_df[ok_df["rec_split"] == "val"].copy()
test_ok_df = ok_df[ok_df["rec_split"] == "test"].copy()

train_dataset = RecognitionDataset(train_ok_df, feature_mean, feature_std)
val_dataset = RecognitionDataset(val_ok_df, feature_mean, feature_std)
test_dataset = RecognitionDataset(test_ok_df, feature_mean, feature_std)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

batch = next(iter(train_loader))

print("=" * 80)
print("RECOGNITION DATALOADER SANITY CHECK")
print("=" * 80)

print("Dataset sizes:")
print(f"train_dataset: {len(train_dataset)}")
print(f"val_dataset:   {len(val_dataset)}")
print(f"test_dataset:  {len(test_dataset)}")
print()

print("One train batch:")
print(f"x shape:       {tuple(batch['x'].shape)}   # (B, T_max, F=448)")
print(f"y shape:       {tuple(batch['y'].shape)}   # (B,)")
print(f"mask shape:    {tuple(batch['mask'].shape)}")
print(f"lengths shape: {tuple(batch['lengths'].shape)}")
print(f"lengths:       {batch['lengths'].tolist()}")
print()

print("Labels in this batch:")
unique, counts = torch.unique(batch["y"], return_counts=True)
for u, c in zip(unique.tolist(), counts.tolist()):
    print(f"class {u}: {c}")
print()

print("Example metadata from batch:")
for m in batch["meta"][:5]:
    print(m)

print()
print("=" * 80)
print("PHASE 10B OUTPUT SAVED")
print("=" * 80)
print(f"Verified recognition dataset manifest: {RECOGNITION_DATASET_MANIFEST_PATH}")
print(f"Recognition feature stats:             {RECOGNITION_FEATURE_STATS_PATH}")
print(f"Recognition class weights:             {RECOGNITION_CLASS_WEIGHTS_PATH}")
print(f"Recognition config:                    {RECOGNITION_CONFIG_PATH}")
print()

print("=" * 80)
print("PHASE 10B COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Verify status counts")
print("2. OK rows by rec_split")
print("3. Feature dim summary")
print("4. Active sequence length summary")
print("5. Verified class coverage")
print("6. Feature mean/std summaries")
print("7. Train class count summary")
print("8. Class weight summary")
print("9. Dataloader sanity check")
print()
print("Do NOT train Recognition TCN yet.")

In [ ]:
# ============================================================
# PHASE 10B-FIX — CLEAN RECOGNITION DATASET MANIFEST COLUMNS
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

RECOGNITION_DATASET_MANIFEST_PATH = MANIFEST_DIR / "recognition_dataset_verified.csv"
RECOGNITION_DATASET_MANIFEST_FIXED_PATH = MANIFEST_DIR / "recognition_dataset_verified_fixed.csv"

print("=" * 80)
print("PHASE 10B-FIX — CLEAN RECOGNITION DATASET MANIFEST COLUMNS")
print("=" * 80)

if not RECOGNITION_DATASET_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing: {RECOGNITION_DATASET_MANIFEST_PATH}")

df = pd.read_csv(RECOGNITION_DATASET_MANIFEST_PATH)

print(f"Loaded: {RECOGNITION_DATASET_MANIFEST_PATH}")
print(f"Rows: {len(df)}")
print()
print("Columns containing 'feature_dim':")
print([c for c in df.columns if "feature_dim" in c])
print()

# The original 'feature_dim' came from motion_manifest and means 147 position-only features.
# Rename it clearly if present.
if "feature_dim" in df.columns:
    df = df.rename(columns={"feature_dim": "motion_position_feature_dim"})

# The verification-created column may have been suffixed during merge.
# Detect it and make a clear final column.
if "feature_dim_verify" in df.columns:
    df["recognition_final_feature_dim"] = df["feature_dim_verify"]
elif "feature_dim_verified" in df.columns:
    df["recognition_final_feature_dim"] = df["feature_dim_verified"]
else:
    # If missing, infer from the known setup.
    df["recognition_final_feature_dim"] = 448

# Also make the expected continuous/phase dimensions explicit.
df["recognition_continuous_feature_dim"] = 444
df["recognition_phase_onehot_dim"] = 4

# Save fixed manifest.
df.to_csv(RECOGNITION_DATASET_MANIFEST_FIXED_PATH, index=False)

print("=" * 80)
print("FIXED MANIFEST SAVED")
print("=" * 80)
print(RECOGNITION_DATASET_MANIFEST_FIXED_PATH)
print()

print("Rows by rec_split:")
print(df["rec_split"].value_counts(dropna=False))
print()

print("Verify status counts:")
print(df["verify_status"].value_counts(dropna=False))
print()

print("Motion position feature dim summary:")
print(df["motion_position_feature_dim"].describe())
print()

print("Recognition continuous feature dim summary:")
print(df["recognition_continuous_feature_dim"].describe())
print()

print("Recognition phase one-hot dim summary:")
print(df["recognition_phase_onehot_dim"].describe())
print()

print("Recognition final feature dim summary:")
print(df["recognition_final_feature_dim"].describe())
print()

print("Class coverage after fixed manifest:")
coverage = df[df["verify_status"] == "ok"].groupby("rec_label_id")["rec_split"].value_counts().unstack(fill_value=0)

for split in ["train", "val", "test"]:
    if split not in coverage.columns:
        coverage[split] = 0

print("classes missing train:", int((coverage["train"] == 0).sum()))
print("classes missing val:", int((coverage["val"] == 0).sum()))
print("classes missing test:", int((coverage["test"] == 0).sum()))
print()

print("=" * 80)
print("PHASE 10B-FIX COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Rows by rec_split")
print("2. Verify status counts")
print("3. Motion position feature dim summary")
print("4. Recognition final feature dim summary")
print("5. Class coverage after fixed manifest")
print()
print("Do NOT train Recognition TCN yet.")

In [ ]:
# ============================================================
# PHASE 10C — TRAIN PHASE-AWARE RECOGNITION TCN + ATTENTION
# Run this cell only, then paste the final output back.
# ============================================================

from pathlib import Path
import time
import json
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

RECOGNITION_DATASET_MANIFEST_PATH = MANIFEST_DIR / "recognition_dataset_verified_fixed.csv"
RECOGNITION_FEATURE_STATS_PATH = MANIFEST_DIR / "recognition_feature_stats_phase_aware.npz"
RECOGNITION_CLASS_WEIGHTS_PATH = MANIFEST_DIR / "recognition_class_weights.npy"
RECOGNITION_LABEL_MAP_PATH = MANIFEST_DIR / "recognition_label_map_with_split_counts.csv"

MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "recognition_tcn_attention_best.pt"
LAST_MODEL_PATH = MODEL_DIR / "recognition_tcn_attention_last.pt"
TRAINING_LOG_PATH = MODEL_DIR / "recognition_training_log.csv"
CONFIG_PATH = MODEL_DIR / "recognition_tcn_attention_config.json"

print("=" * 80)
print("PHASE 10C — TRAIN PHASE-AWARE RECOGNITION TCN + ATTENTION")
print("=" * 80)

for p in [
    RECOGNITION_DATASET_MANIFEST_PATH,
    RECOGNITION_FEATURE_STATS_PATH,
    RECOGNITION_CLASS_WEIGHTS_PATH,
    RECOGNITION_LABEL_MAP_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

df = pd.read_csv(RECOGNITION_DATASET_MANIFEST_PATH)
df = df[df["verify_status"] == "ok"].copy()

label_map_df = pd.read_csv(RECOGNITION_LABEL_MAP_PATH)

stats = np.load(RECOGNITION_FEATURE_STATS_PATH)
feature_mean = stats["mean"].astype(np.float32)
feature_std = stats["std"].astype(np.float32)

class_weights_np = np.load(RECOGNITION_CLASS_WEIGHTS_PATH).astype(np.float32)

print(f"Loaded dataset manifest: {RECOGNITION_DATASET_MANIFEST_PATH}")
print(f"Rows: {len(df)}")
print(f"Classes: {len(label_map_df)}")
print()

print("Rows by rec_split:")
print(df["rec_split"].value_counts(dropna=False))
print()

print("Recognition final feature dim summary:")
print(df["recognition_final_feature_dim"].describe())
print()

# ----------------------------
# 2. Device + seed
# ----------------------------

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# ----------------------------
# 3. Training config
# ----------------------------

CONFIG = {
    "seed": SEED,
    "input_dim": 448,
    "continuous_dim": 444,
    "phase_onehot_dim": 4,
    "num_classes": 100,
    "hidden_dim": 256,
    "num_blocks": 4,
    "kernel_size": 5,
    "dropout": 0.30,
    "attention_dropout": 0.10,
    "batch_size": 16,
    "epochs": 50,
    "learning_rate": 8e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "patience": 10,
    "num_workers": 0,
}

with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Training config:")
print(json.dumps(CONFIG, indent=2))
print()

# ----------------------------
# 4. Feature loading
# ----------------------------

def load_recognition_features(row):
    """
    Returns:
      X_final: (T_active, 448)
      y: int class id
    """
    motion_path = Path(str(row["motion_path"]))
    segment_path = Path(str(row["phase_segment_path"]))

    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(segment_path, allow_pickle=False) as s:
        ordered_phase = np.asarray(s["ordered_segment_labels"], dtype=np.int64)
        phase_speed = np.asarray(s["phase_speed"], dtype=np.float32).reshape(-1, 1)
        active_start = int(np.asarray(s["active_start"]).item())
        active_end = int(np.asarray(s["active_end"]).item())

    X_cont = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X_cont[~np.isfinite(X_cont)] = 0.0

    # Active-region crop.
    X_cont = X_cont[active_start:active_end + 1]
    phase_crop = ordered_phase[active_start:active_end + 1]
    phase_crop = np.clip(phase_crop, 0, 3).astype(np.int64)

    # Standardize continuous features only.
    X_cont = (X_cont - feature_mean) / feature_std
    X_cont[~np.isfinite(X_cont)] = 0.0

    # Add phase one-hot without standardizing it.
    phase_onehot = np.eye(4, dtype=np.float32)[phase_crop]

    X_final = np.concatenate([X_cont, phase_onehot], axis=1).astype(np.float32)
    X_final[~np.isfinite(X_final)] = 0.0

    y = int(row["rec_label_id"])

    return X_final, y

class RecognitionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        X, y = load_recognition_features(row)

        return {
            "x": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "length": int(X.shape[0]),
            "rec_split": row["rec_split"],
            "gloss": row["gloss"],
            "rec_label_id": int(row["rec_label_id"]),
            "video_stem": row["video_stem"],
        }

def recognition_collate(batch):
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    y = torch.zeros(batch_size, dtype=torch.long)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        mask[i, :T] = True
        y[i] = item["y"]

        meta.append(
            {
                "rec_split": item["rec_split"],
                "gloss": item["gloss"],
                "rec_label_id": item["rec_label_id"],
                "video_stem": item["video_stem"],
            }
        )

    return {
        "x": x_padded,
        "y": y,
        "mask": mask,
        "lengths": lengths,
        "meta": meta,
    }

# ----------------------------
# 5. DataLoaders
# ----------------------------

train_df = df[df["rec_split"] == "train"].copy()
val_df = df[df["rec_split"] == "val"].copy()
test_df = df[df["rec_split"] == "test"].copy()

train_dataset = RecognitionDataset(train_df)
val_dataset = RecognitionDataset(val_df)
test_dataset = RecognitionDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

print("Dataset sizes:")
print(f"train: {len(train_dataset)}")
print(f"val:   {len(val_dataset)}")
print(f"test:  {len(test_dataset)}")
print()

batch = next(iter(train_loader))
print("Sanity batch:")
print(f"x shape: {tuple(batch['x'].shape)}")
print(f"y shape: {tuple(batch['y'].shape)}")
print(f"mask shape: {tuple(batch['mask'].shape)}")
print(f"lengths: {batch['lengths'].tolist()}")
print()

# ----------------------------
# 6. Model: TCN + attention pooling
# ----------------------------

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        # x: (B, C, T)
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class MaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        """
        h:    (B, T, H)
        mask: (B, T), True for real frames

        returns:
          pooled: (B, H)
          attn_weights: (B, T)
        """
        scores = self.attn(h).squeeze(-1)  # (B, T)

        scores = scores.masked_fill(~mask, -1e9)

        attn_weights = torch.softmax(scores, dim=1)
        attn_weights = attn_weights.masked_fill(~mask, 0.0)

        pooled = torch.sum(h * attn_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights

class RecognitionTCNAttention(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = MaskedAttentionPooling(hidden_dim, dropout=attention_dropout)

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        """
        x:    (B, T, F)
        mask: (B, T)

        returns:
          logits: (B, num_classes)
          attn_weights: (B, T)
        """
        h = x.transpose(1, 2)  # (B, F, T)
        h = self.tcn(h)        # (B, H, T)
        h = h.transpose(1, 2)  # (B, T, H)

        pooled, attn_weights = self.attention_pool(h, mask)
        logits = self.classifier(pooled)

        return logits, attn_weights

model = RecognitionTCNAttention(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
    attention_dropout=CONFIG["attention_dropout"],
).to(device)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model:")
print(model)
print()
print(f"Total parameters:     {num_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()

# ----------------------------
# 7. Loss / optimizer / scheduler
# ----------------------------

class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("Class weight summary:")
print(pd.Series(class_weights_np).describe())
print()

# ----------------------------
# 8. Metrics
# ----------------------------

def compute_classification_metrics(y_true, logits, num_classes=100, topk=(1, 5)):
    """
    y_true: numpy array (N,)
    logits: numpy array (N, C)
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    logits = np.asarray(logits, dtype=np.float32)

    pred = np.argmax(logits, axis=1)

    top1 = float(np.mean(pred == y_true)) if len(y_true) else 0.0

    top5 = 0.0
    if logits.shape[1] >= 5 and len(y_true):
        top5_idx = np.argsort(logits, axis=1)[:, -5:]
        top5 = float(np.mean([y_true[i] in top5_idx[i] for i in range(len(y_true))]))

    per_class_f1 = []
    per_class = {}

    for c in range(num_classes):
        tp = int(np.sum((y_true == c) & (pred == c)))
        fp = int(np.sum((y_true != c) & (pred == c)))
        fn = int(np.sum((y_true == c) & (pred != c)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        support = int(np.sum(y_true == c))

        per_class[c] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": support,
        }

        per_class_f1.append(f1)

    macro_f1 = float(np.mean(per_class_f1))

    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": macro_f1,
        "per_class": per_class,
    }

def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None

    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_samples = 0

    all_y = []
    all_logits = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits, attn_weights = model(x, mask)
                loss = criterion(logits, y)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()

        bs = y.shape[0]
        total_loss += float(loss.detach().cpu().item()) * bs
        total_samples += bs

        all_y.append(y.detach().cpu().numpy())
        all_logits.append(logits.detach().cpu().numpy())

    all_y = np.concatenate(all_y) if all_y else np.array([], dtype=np.int64)
    all_logits = np.concatenate(all_logits) if all_logits else np.zeros((0, CONFIG["num_classes"]), dtype=np.float32)

    avg_loss = total_loss / max(total_samples, 1)
    metrics = compute_classification_metrics(all_y, all_logits, num_classes=CONFIG["num_classes"])

    return avg_loss, metrics

# ----------------------------
# 9. Training loop
# ----------------------------

print("=" * 80)
print("START RECOGNITION TRAINING")
print("=" * 80)

best_val_top1 = -1.0
best_val_macro_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0

log_rows = []
start_time = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_start = time.time()

    train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_metrics = run_one_epoch(model, val_loader, optimizer=None)

    # Main selection metric: validation top-1 accuracy.
    scheduler.step(val_metrics["top1"])

    current_lr = optimizer.param_groups[0]["lr"]

    improved = val_metrics["top1"] > best_val_top1

    if improved:
        best_val_top1 = val_metrics["top1"]
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        safe_checkpoint = {
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "config": CONFIG,
            "feature_mean": torch.tensor(feature_mean, dtype=torch.float32),
            "feature_std": torch.tensor(feature_std, dtype=torch.float32),
            "class_weights": torch.tensor(class_weights_np, dtype=torch.float32),
            "val_loss": float(val_loss),
            "val_top1": float(val_metrics["top1"]),
            "val_top5": float(val_metrics["top5"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "num_params": int(num_params),
        }

        torch.save(safe_checkpoint, BEST_MODEL_PATH)

    else:
        epochs_without_improvement += 1

    # Save last every epoch.
    torch.save(
        {
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "config": CONFIG,
            "feature_mean": torch.tensor(feature_mean, dtype=torch.float32),
            "feature_std": torch.tensor(feature_std, dtype=torch.float32),
            "class_weights": torch.tensor(class_weights_np, dtype=torch.float32),
            "val_loss": float(val_loss),
            "val_top1": float(val_metrics["top1"]),
            "val_top5": float(val_metrics["top5"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "num_params": int(num_params),
        },
        LAST_MODEL_PATH,
    )

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_loss,
        "train_top1": train_metrics["top1"],
        "train_top5": train_metrics["top5"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_loss,
        "val_top1": val_metrics["top1"],
        "val_top5": val_metrics["top5"],
        "val_macro_f1": val_metrics["macro_f1"],
        "best_val_top1": best_val_top1,
        "best_val_macro_f1": best_val_macro_f1,
        "best_epoch": best_epoch,
        "epoch_seconds": time.time() - epoch_start,
    }

    log_rows.append(row)
    pd.DataFrame(log_rows).to_csv(TRAINING_LOG_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
        f"lr={current_lr:.2e} | "
        f"train_loss={train_loss:.4f} | train_top1={train_metrics['top1']:.4f} | train_top5={train_metrics['top5']:.4f} | "
        f"val_loss={val_loss:.4f} | val_top1={val_metrics['top1']:.4f} | val_top5={val_metrics['top5']:.4f} | "
        f"val_f1={val_metrics['macro_f1']:.4f} | "
        f"best_top1={best_val_top1:.4f}@{best_epoch} | "
        f"{'BEST' if improved else ''}"
    )

    if epochs_without_improvement >= CONFIG["patience"]:
        print()
        print(f"Early stopping triggered after {CONFIG['patience']} epochs without improvement.")
        break

total_time = time.time() - start_time

# ----------------------------
# 10. Final validation summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 10C TRAINING COMPLETE")
print("=" * 80)
print(f"Best epoch:        {best_epoch}")
print(f"Best val top-1:    {best_val_top1:.4f}")
print(f"Best val macro F1: {best_val_macro_f1:.4f}")
print(f"Best model path:   {BEST_MODEL_PATH}")
print(f"Last model path:   {LAST_MODEL_PATH}")
print(f"Training log path: {TRAINING_LOG_PATH}")
print(f"Config path:       {CONFIG_PATH}")
print(f"Total train time:  {total_time / 60:.2f} minutes")
print()

# Reload best and evaluate validation once more.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

best_val_loss, best_val_metrics = run_one_epoch(model, val_loader, optimizer=None)

print("=" * 80)
print("BEST MODEL VALIDATION METRICS")
print("=" * 80)
print(f"val_loss:     {best_val_loss:.4f}")
print(f"val_top1:     {best_val_metrics['top1']:.4f}")
print(f"val_top5:     {best_val_metrics['top5']:.4f}")
print(f"val_macro_f1: {best_val_metrics['macro_f1']:.4f}")
print()

print("Worst validation classes by F1:")
per_class_rows = []
for c, m in best_val_metrics["per_class"].items():
    gloss = label_map_df[label_map_df["rec_label_id"] == c]["gloss"].values
    gloss = str(gloss[0]) if len(gloss) else str(c)
    per_class_rows.append(
        {
            "rec_label_id": c,
            "gloss": gloss,
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "support": m["support"],
        }
    )

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df.sort_values(["f1", "support"], ascending=[True, False]).head(15))

print()
print("=" * 80)
print("PHASE 10C COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Training epoch logs")
print("2. Best epoch")
print("3. Best val top-1")
print("4. Best val top-5")
print("5. Best val macro F1")
print("6. Worst validation classes table")
print()
print("Do NOT run test evaluation yet. Next step is Phase 10D: test evaluation + attention visualization.")

In [ ]:
# ============================================================
# PHASE 10C-FIX — SAFE ATTENTION MASK + RESTART RECOGNITION TRAINING
# Run this cell only. This fixes the FP16 attention masking error.
# ============================================================

from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 80)
print("PHASE 10C-FIX — SAFE ATTENTION MASK + RESTART TRAINING")
print("=" * 80)

# ----------------------------
# 1. Check required objects from Phase 10C exist
# ----------------------------

required_objects = [
    "CONFIG",
    "TemporalBlock",
    "train_loader",
    "val_loader",
    "test_loader",
    "device",
    "class_weights_np",
    "feature_mean",
    "feature_std",
    "label_map_df",
    "MODEL_DIR",
    "BEST_MODEL_PATH",
    "LAST_MODEL_PATH",
    "TRAINING_LOG_PATH",
    "CONFIG_PATH",
]

missing = [name for name in required_objects if name not in globals()]

if missing:
    raise RuntimeError(
        "Some Phase 10C objects are missing from the runtime:\n"
        f"{missing}\n\n"
        "This usually means the runtime was restarted. Tell ChatGPT this output, "
        "and I will give you a full standalone corrected Phase 10C cell."
    )

print("Required Phase 10C objects found.")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# ----------------------------
# 2. Safe attention pooling
# ----------------------------

class SafeMaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        """
        h:    (B, T, H)
        mask: (B, T), True for real frames

        Fix:
        - do attention scoring in float32
        - use -1e4 instead of -1e9 to avoid FP16 overflow
        """
        scores = self.attn(h.float()).squeeze(-1)  # force float32 scores

        mask = mask.bool()

        # -1e4 is safe for float16/float32 and still makes softmax ~0.
        scores = scores.masked_fill(~mask, -1e4)

        attn_weights = torch.softmax(scores, dim=1)
        attn_weights = attn_weights.masked_fill(~mask, 0.0)

        pooled = torch.sum(h.float() * attn_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights

class RecognitionTCNAttentionSafe(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = SafeMaskedAttentionPooling(
            hidden_dim,
            dropout=attention_dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        """
        x:    (B, T, F)
        mask: (B, T)
        """
        h = x.transpose(1, 2)  # (B, F, T)
        h = self.tcn(h)        # (B, H, T)
        h = h.transpose(1, 2)  # (B, T, H)

        pooled, attn_weights = self.attention_pool(h, mask)

        # classifier expects float32 pooled features
        logits = self.classifier(pooled.float())

        return logits, attn_weights

# ----------------------------
# 3. Recreate model, optimizer, scheduler
# ----------------------------

torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

model = RecognitionTCNAttentionSafe(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
    attention_dropout=CONFIG["attention_dropout"],
).to(device)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Safe model recreated.")
print(f"Total parameters:     {num_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()

class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("Optimizer/scheduler/scaler recreated.")
print(f"use_amp: {use_amp}")
print()

# ----------------------------
# 4. Metrics and epoch runner
# ----------------------------

def compute_classification_metrics(y_true, logits, num_classes=100):
    y_true = np.asarray(y_true, dtype=np.int64)
    logits = np.asarray(logits, dtype=np.float32)

    pred = np.argmax(logits, axis=1)

    top1 = float(np.mean(pred == y_true)) if len(y_true) else 0.0

    top5 = 0.0
    if logits.shape[1] >= 5 and len(y_true):
        top5_idx = np.argsort(logits, axis=1)[:, -5:]
        top5 = float(np.mean([y_true[i] in top5_idx[i] for i in range(len(y_true))]))

    per_class_f1 = []
    per_class = {}

    for c in range(num_classes):
        tp = int(np.sum((y_true == c) & (pred == c)))
        fp = int(np.sum((y_true != c) & (pred == c)))
        fn = int(np.sum((y_true == c) & (pred != c)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        support = int(np.sum(y_true == c))

        per_class[c] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": support,
        }

        per_class_f1.append(f1)

    macro_f1 = float(np.mean(per_class_f1))

    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": macro_f1,
        "per_class": per_class,
    }

def run_one_epoch_safe(model, loader, optimizer=None):
    is_train = optimizer is not None

    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_samples = 0

    all_y = []
    all_logits = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits, attn_weights = model(x, mask)
                loss = criterion(logits, y)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()

        bs = y.shape[0]
        total_loss += float(loss.detach().cpu().item()) * bs
        total_samples += bs

        all_y.append(y.detach().cpu().numpy())
        all_logits.append(logits.detach().float().cpu().numpy())

    all_y = np.concatenate(all_y) if all_y else np.array([], dtype=np.int64)
    all_logits = np.concatenate(all_logits) if all_logits else np.zeros((0, CONFIG["num_classes"]), dtype=np.float32)

    avg_loss = total_loss / max(total_samples, 1)
    metrics = compute_classification_metrics(
        all_y,
        all_logits,
        num_classes=CONFIG["num_classes"],
    )

    return avg_loss, metrics

# ----------------------------
# 5. Training loop
# ----------------------------

print("=" * 80)
print("START FIXED RECOGNITION TRAINING")
print("=" * 80)

best_val_top1 = -1.0
best_val_macro_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0

log_rows = []
start_time = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_start = time.time()

    train_loss, train_metrics = run_one_epoch_safe(model, train_loader, optimizer=optimizer)
    val_loss, val_metrics = run_one_epoch_safe(model, val_loader, optimizer=None)

    scheduler.step(val_metrics["top1"])

    current_lr = optimizer.param_groups[0]["lr"]
    improved = val_metrics["top1"] > best_val_top1

    if improved:
        best_val_top1 = val_metrics["top1"]
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": int(epoch),
                "model_state_dict": model.state_dict(),
                "config": CONFIG,
                "feature_mean": torch.tensor(feature_mean, dtype=torch.float32),
                "feature_std": torch.tensor(feature_std, dtype=torch.float32),
                "class_weights": torch.tensor(class_weights_np, dtype=torch.float32),
                "val_loss": float(val_loss),
                "val_top1": float(val_metrics["top1"]),
                "val_top5": float(val_metrics["top5"]),
                "val_macro_f1": float(val_metrics["macro_f1"]),
                "num_params": int(num_params),
                "attention_mask_fix": "scores_float32_mask_-1e4",
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    torch.save(
        {
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "config": CONFIG,
            "feature_mean": torch.tensor(feature_mean, dtype=torch.float32),
            "feature_std": torch.tensor(feature_std, dtype=torch.float32),
            "class_weights": torch.tensor(class_weights_np, dtype=torch.float32),
            "val_loss": float(val_loss),
            "val_top1": float(val_metrics["top1"]),
            "val_top5": float(val_metrics["top5"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "num_params": int(num_params),
            "attention_mask_fix": "scores_float32_mask_-1e4",
        },
        LAST_MODEL_PATH,
    )

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_loss,
        "train_top1": train_metrics["top1"],
        "train_top5": train_metrics["top5"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_loss,
        "val_top1": val_metrics["top1"],
        "val_top5": val_metrics["top5"],
        "val_macro_f1": val_metrics["macro_f1"],
        "best_val_top1": best_val_top1,
        "best_val_macro_f1": best_val_macro_f1,
        "best_epoch": best_epoch,
        "epoch_seconds": time.time() - epoch_start,
    }

    log_rows.append(row)
    pd.DataFrame(log_rows).to_csv(TRAINING_LOG_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
        f"lr={current_lr:.2e} | "
        f"train_loss={train_loss:.4f} | train_top1={train_metrics['top1']:.4f} | train_top5={train_metrics['top5']:.4f} | "
        f"val_loss={val_loss:.4f} | val_top1={val_metrics['top1']:.4f} | val_top5={val_metrics['top5']:.4f} | "
        f"val_f1={val_metrics['macro_f1']:.4f} | "
        f"best_top1={best_val_top1:.4f}@{best_epoch} | "
        f"{'BEST' if improved else ''}"
    )

    if epochs_without_improvement >= CONFIG["patience"]:
        print()
        print(f"Early stopping triggered after {CONFIG['patience']} epochs without improvement.")
        break

total_time = time.time() - start_time

# ----------------------------
# 6. Final validation summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 10C-FIX TRAINING COMPLETE")
print("=" * 80)
print(f"Best epoch:        {best_epoch}")
print(f"Best val top-1:    {best_val_top1:.4f}")
print(f"Best val macro F1: {best_val_macro_f1:.4f}")
print(f"Best model path:   {BEST_MODEL_PATH}")
print(f"Last model path:   {LAST_MODEL_PATH}")
print(f"Training log path: {TRAINING_LOG_PATH}")
print(f"Config path:       {CONFIG_PATH}")
print(f"Total train time:  {total_time / 60:.2f} minutes")
print()

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

best_val_loss, best_val_metrics = run_one_epoch_safe(model, val_loader, optimizer=None)

print("=" * 80)
print("BEST MODEL VALIDATION METRICS")
print("=" * 80)
print(f"val_loss:     {best_val_loss:.4f}")
print(f"val_top1:     {best_val_metrics['top1']:.4f}")
print(f"val_top5:     {best_val_metrics['top5']:.4f}")
print(f"val_macro_f1: {best_val_metrics['macro_f1']:.4f}")
print()

print("Worst validation classes by F1:")
per_class_rows = []
for c, m in best_val_metrics["per_class"].items():
    gloss_values = label_map_df[label_map_df["rec_label_id"] == c]["gloss"].values
    gloss = str(gloss_values[0]) if len(gloss_values) else str(c)

    per_class_rows.append(
        {
            "rec_label_id": c,
            "gloss": gloss,
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "support": m["support"],
        }
    )

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df.sort_values(["f1", "support"], ascending=[True, False]).head(15))

print()
print("=" * 80)
print("PHASE 10C-FIX COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Training epoch logs")
print("2. Best epoch")
print("3. Best val top-1")
print("4. Best val top-5")
print("5. Best val macro F1")
print("6. Worst validation classes table")
print()
print("Do NOT run test evaluation yet.")

In [ ]:
# ============================================================
# PHASE 10D — TEST EVALUATION + ATTENTION VISUALIZATION
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"
MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention"

RECOGNITION_DATASET_MANIFEST_PATH = MANIFEST_DIR / "recognition_dataset_verified_fixed.csv"
RECOGNITION_LABEL_MAP_PATH = MANIFEST_DIR / "recognition_label_map_with_split_counts.csv"
BEST_MODEL_PATH = MODEL_DIR / "recognition_tcn_attention_best.pt"

TEST_METRICS_PATH = MODEL_DIR / "recognition_test_metrics.json"
TEST_PREDICTIONS_PATH = MODEL_DIR / "recognition_test_predictions.csv"
PER_CLASS_TEST_PATH = MODEL_DIR / "recognition_test_per_class_metrics.csv"
CONFUSION_MATRIX_PATH = MODEL_DIR / "recognition_test_confusion_matrix.csv"

ATTENTION_PLOT_DIR = BASE_DIR / "plots" / "recognition_attention"
ATTENTION_PLOT_DIR.mkdir(parents=True, exist_ok=True)

ATTENTION_VIS_MANIFEST_PATH = MODEL_DIR / "recognition_attention_visualization_manifest.csv"

print("=" * 80)
print("PHASE 10D — TEST EVALUATION + ATTENTION VISUALIZATION")
print("=" * 80)

for p in [
    RECOGNITION_DATASET_MANIFEST_PATH,
    RECOGNITION_LABEL_MAP_PATH,
    BEST_MODEL_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

df = pd.read_csv(RECOGNITION_DATASET_MANIFEST_PATH)
df = df[df["verify_status"] == "ok"].copy()

label_map_df = pd.read_csv(RECOGNITION_LABEL_MAP_PATH)
id_to_gloss = {
    int(row["rec_label_id"]): str(row["gloss"])
    for _, row in label_map_df.iterrows()
}

print(f"Loaded dataset: {RECOGNITION_DATASET_MANIFEST_PATH}")
print(f"Rows: {len(df)}")
print(f"Loaded label map: {RECOGNITION_LABEL_MAP_PATH}")
print(f"Classes: {len(label_map_df)}")
print(f"Best model: {BEST_MODEL_PATH}")
print()

print("Rows by rec_split:")
print(df["rec_split"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Device + checkpoint
# ----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# This checkpoint was created by your own Colab training code.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)

CONFIG = checkpoint["config"]
feature_mean = checkpoint["feature_mean"].detach().cpu().numpy().astype(np.float32)
feature_std = checkpoint["feature_std"].detach().cpu().numpy().astype(np.float32)

print("Loaded checkpoint:")
print(f"epoch:        {checkpoint['epoch']}")
print(f"val_loss:     {checkpoint['val_loss']:.4f}")
print(f"val_top1:     {checkpoint['val_top1']:.4f}")
print(f"val_top5:     {checkpoint['val_top5']:.4f}")
print(f"val_macro_f1: {checkpoint['val_macro_f1']:.4f}")
print()

# ----------------------------
# 3. Feature loading
# ----------------------------

def load_recognition_features(row):
    """
    Returns:
      X_final: (T_active, 448)
      y: int class id
      phase_crop: (T_active,)
      phase_speed_crop: (T_active,)
    """
    motion_path = Path(str(row["motion_path"]))
    segment_path = Path(str(row["phase_segment_path"]))

    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(segment_path, allow_pickle=False) as s:
        ordered_phase = np.asarray(s["ordered_segment_labels"], dtype=np.int64)
        phase_speed = np.asarray(s["phase_speed"], dtype=np.float32).reshape(-1, 1)
        active_start = int(np.asarray(s["active_start"]).item())
        active_end = int(np.asarray(s["active_end"]).item())

    X_cont = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X_cont[~np.isfinite(X_cont)] = 0.0

    # Active-region crop.
    X_cont = X_cont[active_start:active_end + 1]
    phase_crop = ordered_phase[active_start:active_end + 1]
    phase_speed_crop = phase_speed[active_start:active_end + 1].reshape(-1)

    phase_crop = np.clip(phase_crop, 0, 3).astype(np.int64)

    # Standardize continuous features.
    X_cont = (X_cont - feature_mean) / feature_std
    X_cont[~np.isfinite(X_cont)] = 0.0

    # Add phase one-hot.
    phase_onehot = np.eye(4, dtype=np.float32)[phase_crop]

    X_final = np.concatenate([X_cont, phase_onehot], axis=1).astype(np.float32)
    X_final[~np.isfinite(X_final)] = 0.0

    y = int(row["rec_label_id"])

    return X_final, y, phase_crop, phase_speed_crop

class RecognitionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        X, y, phase_crop, phase_speed_crop = load_recognition_features(row)

        return {
            "x": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "phase_crop": torch.tensor(phase_crop, dtype=torch.long),
            "phase_speed": torch.tensor(phase_speed_crop, dtype=torch.float32),
            "length": int(X.shape[0]),
            "rec_split": row["rec_split"],
            "gloss": row["gloss"],
            "rec_label_id": int(row["rec_label_id"]),
            "video_stem": row["video_stem"],
            "motion_path": row["motion_path"],
            "phase_segment_path": row["phase_segment_path"],
        }

def recognition_collate(batch):
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    y = torch.zeros(batch_size, dtype=torch.long)

    phase_padded = torch.zeros(batch_size, max_len, dtype=torch.long)
    speed_padded = torch.zeros(batch_size, max_len, dtype=torch.float32)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        mask[i, :T] = True
        y[i] = item["y"]
        phase_padded[i, :T] = item["phase_crop"]
        speed_padded[i, :T] = item["phase_speed"]

        meta.append(
            {
                "rec_split": item["rec_split"],
                "gloss": item["gloss"],
                "rec_label_id": item["rec_label_id"],
                "video_stem": item["video_stem"],
                "motion_path": item["motion_path"],
                "phase_segment_path": item["phase_segment_path"],
            }
        )

    return {
        "x": x_padded,
        "y": y,
        "mask": mask,
        "phase_crop": phase_padded,
        "phase_speed": speed_padded,
        "lengths": lengths,
        "meta": meta,
    }

# ----------------------------
# 4. Model definition
# ----------------------------

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class SafeMaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        scores = self.attn(h.float()).squeeze(-1)
        mask = mask.bool()
        scores = scores.masked_fill(~mask, -1e4)

        attn_weights = torch.softmax(scores, dim=1)
        attn_weights = attn_weights.masked_fill(~mask, 0.0)

        pooled = torch.sum(h.float() * attn_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights

class RecognitionTCNAttentionSafe(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = SafeMaskedAttentionPooling(
            hidden_dim,
            dropout=attention_dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        h = x.transpose(1, 2)
        h = self.tcn(h)
        h = h.transpose(1, 2)

        pooled, attn_weights = self.attention_pool(h, mask)
        logits = self.classifier(pooled.float())

        return logits, attn_weights

model = RecognitionTCNAttentionSafe(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
    attention_dropout=CONFIG["attention_dropout"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully.")
print()

# ----------------------------
# 5. Metrics helpers
# ----------------------------

def compute_metrics(y_true, logits, num_classes=100):
    y_true = np.asarray(y_true, dtype=np.int64)
    logits = np.asarray(logits, dtype=np.float32)

    pred = np.argmax(logits, axis=1)

    top1 = float(np.mean(pred == y_true)) if len(y_true) else 0.0

    top5_idx = np.argsort(logits, axis=1)[:, -5:]
    top5 = float(np.mean([y_true[i] in top5_idx[i] for i in range(len(y_true))])) if len(y_true) else 0.0

    per_class_rows = []
    f1s = []

    for c in range(num_classes):
        tp = int(np.sum((y_true == c) & (pred == c)))
        fp = int(np.sum((y_true != c) & (pred == c)))
        fn = int(np.sum((y_true == c) & (pred != c)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        support = int(np.sum(y_true == c))

        per_class_rows.append(
            {
                "rec_label_id": c,
                "gloss": id_to_gloss.get(c, str(c)),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "support": support,
            }
        )

        f1s.append(f1)

    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": float(np.mean(f1s)),
        "pred": pred,
        "per_class_df": pd.DataFrame(per_class_rows),
    }

def confusion_matrix(y_true, y_pred, num_classes=100):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1

    return cm

# ----------------------------
# 6. Test evaluation
# ----------------------------

test_df = df[df["rec_split"] == "test"].copy()
test_dataset = RecognitionDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=recognition_collate,
)

print(f"Test dataset size: {len(test_dataset)}")
print("Running test evaluation...")
print()

all_y = []
all_logits = []
prediction_records = []
attention_records = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        logits, attn_weights = model(x, mask)

        probs = torch.softmax(logits, dim=-1)

        logits_np = logits.detach().cpu().numpy()
        probs_np = probs.detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()
        attn_np = attn_weights.detach().cpu().numpy()

        all_y.append(y_np)
        all_logits.append(logits_np)

        for i, meta in enumerate(batch["meta"]):
            true_id = int(y_np[i])
            pred_id = int(np.argmax(logits_np[i]))
            top5_ids = np.argsort(logits_np[i])[-5:][::-1].tolist()

            T = int(batch["lengths"][i].item())
            attn_i = attn_np[i, :T]

            phase_i = batch["phase_crop"][i, :T].detach().cpu().numpy()
            speed_i = batch["phase_speed"][i, :T].detach().cpu().numpy()

            # Attention mass by phase.
            phase_attention = {}
            for c in range(4):
                phase_attention[f"attn_phase_{c}"] = float(attn_i[phase_i == c].sum()) if np.any(phase_i == c) else 0.0

            prediction_records.append(
                {
                    "gloss_true": id_to_gloss.get(true_id, str(true_id)),
                    "gloss_pred": id_to_gloss.get(pred_id, str(pred_id)),
                    "true_id": true_id,
                    "pred_id": pred_id,
                    "correct": bool(pred_id == true_id),
                    "top5_correct": bool(true_id in top5_ids),
                    "top5_ids": top5_ids,
                    "top5_glosses": [id_to_gloss.get(k, str(k)) for k in top5_ids],
                    "top1_confidence": float(probs_np[i, pred_id]),
                    "true_class_confidence": float(probs_np[i, true_id]),
                    "length": T,
                    "video_stem": meta["video_stem"],
                    "motion_path": meta["motion_path"],
                    "phase_segment_path": meta["phase_segment_path"],
                    **phase_attention,
                }
            )

all_y = np.concatenate(all_y)
all_logits = np.concatenate(all_logits)

metrics = compute_metrics(all_y, all_logits, num_classes=CONFIG["num_classes"])
y_pred = metrics["pred"]

cm = confusion_matrix(all_y, y_pred, num_classes=CONFIG["num_classes"])

prediction_df = pd.DataFrame(prediction_records)
per_class_df = metrics["per_class_df"]

prediction_df.to_csv(TEST_PREDICTIONS_PATH, index=False)
per_class_df.to_csv(PER_CLASS_TEST_PATH, index=False)

cm_df = pd.DataFrame(
    cm,
    index=[id_to_gloss.get(i, str(i)) for i in range(CONFIG["num_classes"])],
    columns=[id_to_gloss.get(i, str(i)) for i in range(CONFIG["num_classes"])],
)
cm_df.to_csv(CONFUSION_MATRIX_PATH)

metrics_to_save = {
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "checkpoint_val_top1": float(checkpoint["val_top1"]),
    "checkpoint_val_top5": float(checkpoint["val_top5"]),
    "checkpoint_val_macro_f1": float(checkpoint["val_macro_f1"]),
    "test_top1": float(metrics["top1"]),
    "test_top5": float(metrics["top5"]),
    "test_macro_f1": float(metrics["macro_f1"]),
    "num_test_sequences": int(len(test_dataset)),
    "num_classes": int(CONFIG["num_classes"]),
}

with open(TEST_METRICS_PATH, "w") as f:
    json.dump(metrics_to_save, f, indent=2)

# ----------------------------
# 7. Print metrics
# ----------------------------

print("=" * 80)
print("TEST METRICS")
print("=" * 80)
print(f"test_top1:     {metrics['top1']:.4f}")
print(f"test_top5:     {metrics['top5']:.4f}")
print(f"test_macro_f1: {metrics['macro_f1']:.4f}")
print(f"test_sequences:{len(test_dataset)}")
print()

print("Prediction correctness counts:")
print(prediction_df["correct"].value_counts(dropna=False))
print()

print("Top-5 correctness counts:")
print(prediction_df["top5_correct"].value_counts(dropna=False))
print()

print("=" * 80)
print("PER-CLASS TEST METRICS SUMMARY")
print("=" * 80)
print(per_class_df[["precision", "recall", "f1", "support"]].describe())
print()

print("Worst 20 classes by test F1:")
display(
    per_class_df.sort_values(["f1", "support"], ascending=[True, False]).head(20)
)

print("Best 20 classes by test F1:")
display(
    per_class_df.sort_values(["f1", "support"], ascending=[False, False]).head(20)
)

print("=" * 80)
print("TOP CONFUSIONS")
print("=" * 80)

confusions = []

for true_id in range(CONFIG["num_classes"]):
    for pred_id in range(CONFIG["num_classes"]):
        if true_id != pred_id and cm[true_id, pred_id] > 0:
            confusions.append(
                {
                    "true_id": true_id,
                    "true_gloss": id_to_gloss.get(true_id, str(true_id)),
                    "pred_id": pred_id,
                    "pred_gloss": id_to_gloss.get(pred_id, str(pred_id)),
                    "count": int(cm[true_id, pred_id]),
                }
            )

confusion_long_df = pd.DataFrame(confusions)

if len(confusion_long_df) == 0:
    print("No confusions found.")
else:
    display(confusion_long_df.sort_values("count", ascending=False).head(20))

print("=" * 80)
print("ATTENTION MASS BY PHASE")
print("=" * 80)

phase_attn_cols = ["attn_phase_0", "attn_phase_1", "attn_phase_2", "attn_phase_3"]
print(prediction_df[phase_attn_cols].describe())
print()

print("Average attention mass by phase:")
for col, name in zip(
    phase_attn_cols,
    ["background", "preparation", "stroke", "retraction"]
):
    print(f"{name}: {prediction_df[col].mean():.4f}")
print()

# ----------------------------
# 8. Attention visualization
# ----------------------------

PHASE_COLORS = {
    0: "lightgray",
    1: "gold",
    2: "tomato",
    3: "skyblue",
}

PHASE_NAMES = {
    0: "background",
    1: "preparation",
    2: "stroke",
    3: "retraction",
}

def label_segments(labels):
    labels = np.asarray(labels, dtype=np.int64)
    if len(labels) == 0:
        return []

    segments = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        val = int(labels[i])
        if val != current:
            segments.append((current, start, i - 1))
            start = i
            current = val

    segments.append((current, start, len(labels) - 1))
    return segments

def predict_one_for_plot(row):
    X, y, phase_crop, phase_speed = load_recognition_features(row)

    xt = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
    mask = torch.ones(1, X.shape[0], dtype=torch.bool).to(device)

    with torch.no_grad():
        logits, attn = model(xt, mask)
        probs = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
        attn = attn.squeeze(0).detach().cpu().numpy()

    pred_id = int(np.argmax(probs))
    top5_ids = np.argsort(probs)[-5:][::-1].tolist()

    return {
        "X": X,
        "y": y,
        "phase_crop": phase_crop,
        "phase_speed": phase_speed,
        "probs": probs,
        "attn": attn,
        "pred_id": pred_id,
        "top5_ids": top5_ids,
    }

def plot_attention_example(row, bucket, show=False):
    out = predict_one_for_plot(row)

    phase_crop = out["phase_crop"]
    phase_speed = out["phase_speed"]
    attn = out["attn"]
    y = out["y"]
    pred_id = out["pred_id"]
    top5_ids = out["top5_ids"]

    T = len(phase_crop)
    frames = np.arange(T)

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # Panel 1: phase speed + phase regions
    ax = axes[0]
    for phase_id, start, end in label_segments(phase_crop):
        ax.axvspan(
            start,
            end + 1,
            color=PHASE_COLORS.get(phase_id, "white"),
            alpha=0.35,
            label=PHASE_NAMES.get(phase_id, str(phase_id)),
        )

    ax.plot(frames, phase_speed, linewidth=2.0, label="phase_speed")
    ax.set_title("Phase-aware active crop")
    ax.set_ylabel("phase_speed")

    handles, labels_legend = ax.get_legend_handles_labels()
    unique = {}
    for h, l in zip(handles, labels_legend):
        if l not in unique:
            unique[l] = h
    ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize=8)

    # Panel 2: attention weights
    ax = axes[1]
    for phase_id, start, end in label_segments(phase_crop):
        ax.axvspan(
            start,
            end + 1,
            color=PHASE_COLORS.get(phase_id, "white"),
            alpha=0.25,
        )

    ax.plot(frames, attn, linewidth=2.0, label="attention_weight")
    ax.set_title("Recognition attention over active frames")
    ax.set_xlabel("Active-crop frame")
    ax.set_ylabel("attention")
    ax.legend(loc="upper right")

    true_gloss = id_to_gloss.get(y, str(y))
    pred_gloss = id_to_gloss.get(pred_id, str(pred_id))
    correct = pred_id == y

    top5_text = ", ".join([id_to_gloss.get(k, str(k)) for k in top5_ids])

    fig.suptitle(
        f"{bucket} | true={true_gloss} | pred={pred_gloss} | correct={correct} | "
        f"video={row['video_stem']}\nTop-5: {top5_text}",
        y=1.03,
    )

    plt.tight_layout()

    safe_name = str(row["video_stem"]).replace("/", "_").replace("\\", "_").replace(" ", "_")
    save_path = ATTENTION_PLOT_DIR / f"{bucket}__{safe_name}.png"

    plt.savefig(save_path, dpi=140, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return save_path

# Select examples:
# - 5 random correct
# - 5 random incorrect
# - 5 high-confidence correct
# - 5 high-confidence wrong if available

test_with_preds = test_df.merge(
    prediction_df[
        [
            "video_stem",
            "correct",
            "top5_correct",
            "gloss_pred",
            "top1_confidence",
            "true_class_confidence",
        ]
    ],
    on="video_stem",
    how="left",
)

plot_parts = []

correct_df = test_with_preds[test_with_preds["correct"] == True].copy()
wrong_df = test_with_preds[test_with_preds["correct"] == False].copy()

if len(correct_df) > 0:
    plot_parts.append(correct_df.sample(n=min(5, len(correct_df)), random_state=42).assign(plot_bucket="random_correct"))

if len(wrong_df) > 0:
    plot_parts.append(wrong_df.sample(n=min(5, len(wrong_df)), random_state=42).assign(plot_bucket="random_wrong"))

if len(correct_df) > 0:
    plot_parts.append(
        correct_df.sort_values("top1_confidence", ascending=False)
        .head(5)
        .assign(plot_bucket="high_conf_correct")
    )

if len(wrong_df) > 0:
    plot_parts.append(
        wrong_df.sort_values("top1_confidence", ascending=False)
        .head(5)
        .assign(plot_bucket="high_conf_wrong")
    )

plot_df = pd.concat(plot_parts, ignore_index=True)
plot_df = plot_df.drop_duplicates(subset=["video_stem"]).reset_index(drop=True)

saved_paths = []

print("=" * 80)
print("GENERATING ATTENTION VISUALIZATIONS")
print("=" * 80)

for idx, row in plot_df.iterrows():
    show_now = idx < 6
    save_path = plot_attention_example(row, row["plot_bucket"], show=show_now)
    saved_paths.append(str(save_path))

plot_df["attention_plot_path"] = saved_paths
plot_df.to_csv(ATTENTION_VIS_MANIFEST_PATH, index=False)

print(f"Saved attention plots: {len(saved_paths)}")
print(f"Attention visualization manifest: {ATTENTION_VIS_MANIFEST_PATH}")
print()

print("Attention plot examples:")
display(
    plot_df[
        [
            "plot_bucket",
            "gloss",
            "video_stem",
            "correct",
            "top5_correct",
            "gloss_pred",
            "top1_confidence",
            "true_class_confidence",
            "attention_plot_path",
        ]
    ]
)

print()
print("First saved attention plot paths:")
for p in saved_paths[:10]:
    print(p)

# ----------------------------
# 9. Output summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 10D OUTPUT SAVED")
print("=" * 80)
print(f"Test metrics:          {TEST_METRICS_PATH}")
print(f"Test predictions:      {TEST_PREDICTIONS_PATH}")
print(f"Per-class metrics:     {PER_CLASS_TEST_PATH}")
print(f"Confusion matrix:      {CONFUSION_MATRIX_PATH}")
print(f"Attention plots:       {ATTENTION_PLOT_DIR}")
print()

print("=" * 80)
print("PHASE 10D COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste these sections:")
print("1. Test metrics")
print("2. Prediction correctness counts")
print("3. Top-5 correctness counts")
print("4. Per-class test metrics summary")
print("5. Worst 20 classes by test F1")
print("6. Top confusions")
print("7. Attention mass by phase")
print("8. Attention plot examples")
print()
print("Do NOT start new training yet. Next step is Phase 11: baseline/ablation comparison.")

In [ ]:
# ============================================================
# PHASE 11A — BASELINE RECOGNITION TCN WITHOUT PHASE ONE-HOT
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import time
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

RECOGNITION_DATASET_MANIFEST_PATH = MANIFEST_DIR / "recognition_dataset_verified_fixed.csv"
RECOGNITION_FEATURE_STATS_PATH = MANIFEST_DIR / "recognition_feature_stats_phase_aware.npz"
RECOGNITION_CLASS_WEIGHTS_PATH = MANIFEST_DIR / "recognition_class_weights.npy"
RECOGNITION_LABEL_MAP_PATH = MANIFEST_DIR / "recognition_label_map_with_split_counts.csv"

BASELINE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention_baseline_no_phase"
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_best.pt"
LAST_MODEL_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_last.pt"
TRAINING_LOG_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_training_log.csv"
CONFIG_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_config.json"

print("=" * 80)
print("PHASE 11A — BASELINE RECOGNITION TCN WITHOUT PHASE ONE-HOT")
print("=" * 80)

for p in [
    RECOGNITION_DATASET_MANIFEST_PATH,
    RECOGNITION_FEATURE_STATS_PATH,
    RECOGNITION_CLASS_WEIGHTS_PATH,
    RECOGNITION_LABEL_MAP_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

df = pd.read_csv(RECOGNITION_DATASET_MANIFEST_PATH)
df = df[df["verify_status"] == "ok"].copy()

label_map_df = pd.read_csv(RECOGNITION_LABEL_MAP_PATH)

stats = np.load(RECOGNITION_FEATURE_STATS_PATH)
feature_mean = stats["mean"].astype(np.float32)
feature_std = stats["std"].astype(np.float32)

class_weights_np = np.load(RECOGNITION_CLASS_WEIGHTS_PATH).astype(np.float32)

print(f"Loaded dataset manifest: {RECOGNITION_DATASET_MANIFEST_PATH}")
print(f"Rows: {len(df)}")
print(f"Classes: {len(label_map_df)}")
print()

print("Rows by rec_split:")
print(df["rec_split"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Device + seed
# ----------------------------

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# ----------------------------
# 3. Config
# ----------------------------

CONFIG = {
    "seed": SEED,
    "experiment": "baseline_no_phase_onehot",
    "input_dim": 444,
    "continuous_dim": 444,
    "phase_onehot_dim": 0,
    "num_classes": 100,
    "hidden_dim": 256,
    "num_blocks": 4,
    "kernel_size": 5,
    "dropout": 0.30,
    "attention_dropout": 0.10,
    "batch_size": 16,
    "epochs": 50,
    "learning_rate": 8e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "patience": 10,
    "num_workers": 0,
    "crop_mode": "active_region",
    "ablation_note": "same active crop and continuous features, but no ordered phase one-hot features",
}

with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Baseline config:")
print(json.dumps(CONFIG, indent=2))
print()

# ----------------------------
# 4. Feature loading
# ----------------------------

def load_baseline_features(row):
    """
    Baseline features:
      positions      147
      velocity       147
      acceleration   147
      global_speed     1
      hand_speed       1
      phase_speed      1
      -----------------
      total          444

    No phase one-hot features.
    """
    motion_path = Path(str(row["motion_path"]))
    segment_path = Path(str(row["phase_segment_path"]))

    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(segment_path, allow_pickle=False) as s:
        phase_speed = np.asarray(s["phase_speed"], dtype=np.float32).reshape(-1, 1)
        active_start = int(np.asarray(s["active_start"]).item())
        active_end = int(np.asarray(s["active_end"]).item())

    X_cont = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X_cont[~np.isfinite(X_cont)] = 0.0

    # Same active crop as phase-aware model.
    X_cont = X_cont[active_start:active_end + 1]

    # Same standardization.
    X_cont = (X_cont - feature_mean) / feature_std
    X_cont[~np.isfinite(X_cont)] = 0.0

    if X_cont.shape[1] != CONFIG["input_dim"]:
        raise ValueError(f"Expected input_dim 444, got {X_cont.shape[1]}")

    y = int(row["rec_label_id"])

    return X_cont.astype(np.float32), y

class BaselineRecognitionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        X, y = load_baseline_features(row)

        return {
            "x": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "length": int(X.shape[0]),
            "rec_split": row["rec_split"],
            "gloss": row["gloss"],
            "rec_label_id": int(row["rec_label_id"]),
            "video_stem": row["video_stem"],
        }

def recognition_collate(batch):
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    y = torch.zeros(batch_size, dtype=torch.long)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        mask[i, :T] = True
        y[i] = item["y"]

        meta.append(
            {
                "rec_split": item["rec_split"],
                "gloss": item["gloss"],
                "rec_label_id": item["rec_label_id"],
                "video_stem": item["video_stem"],
            }
        )

    return {
        "x": x_padded,
        "y": y,
        "mask": mask,
        "lengths": lengths,
        "meta": meta,
    }

# ----------------------------
# 5. DataLoaders
# ----------------------------

train_df = df[df["rec_split"] == "train"].copy()
val_df = df[df["rec_split"] == "val"].copy()
test_df = df[df["rec_split"] == "test"].copy()

train_dataset = BaselineRecognitionDataset(train_df)
val_dataset = BaselineRecognitionDataset(val_df)
test_dataset = BaselineRecognitionDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=recognition_collate,
)

print("Dataset sizes:")
print(f"train: {len(train_dataset)}")
print(f"val:   {len(val_dataset)}")
print(f"test:  {len(test_dataset)}")
print()

batch = next(iter(train_loader))
print("Sanity batch:")
print(f"x shape: {tuple(batch['x'].shape)}")
print(f"y shape: {tuple(batch['y'].shape)}")
print(f"mask shape: {tuple(batch['mask'].shape)}")
print(f"lengths: {batch['lengths'].tolist()}")
print()

# ----------------------------
# 6. Model
# ----------------------------

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class SafeMaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        scores = self.attn(h.float()).squeeze(-1)
        mask = mask.bool()
        scores = scores.masked_fill(~mask, -1e4)

        attn_weights = torch.softmax(scores, dim=1)
        attn_weights = attn_weights.masked_fill(~mask, 0.0)

        pooled = torch.sum(h.float() * attn_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights

class BaselineTCNAttention(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = SafeMaskedAttentionPooling(
            hidden_dim,
            dropout=attention_dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        h = x.transpose(1, 2)
        h = self.tcn(h)
        h = h.transpose(1, 2)

        pooled, attn_weights = self.attention_pool(h, mask)
        logits = self.classifier(pooled.float())

        return logits, attn_weights

model = BaselineTCNAttention(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
    attention_dropout=CONFIG["attention_dropout"],
).to(device)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Baseline model:")
print(model)
print()
print(f"Total parameters:     {num_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()

# ----------------------------
# 7. Loss / optimizer / scheduler
# ----------------------------

class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# ----------------------------
# 8. Metrics
# ----------------------------

def compute_classification_metrics(y_true, logits, num_classes=100):
    y_true = np.asarray(y_true, dtype=np.int64)
    logits = np.asarray(logits, dtype=np.float32)

    pred = np.argmax(logits, axis=1)

    top1 = float(np.mean(pred == y_true)) if len(y_true) else 0.0

    top5 = 0.0
    if logits.shape[1] >= 5 and len(y_true):
        top5_idx = np.argsort(logits, axis=1)[:, -5:]
        top5 = float(np.mean([y_true[i] in top5_idx[i] for i in range(len(y_true))]))

    per_class_f1 = []
    per_class = {}

    for c in range(num_classes):
        tp = int(np.sum((y_true == c) & (pred == c)))
        fp = int(np.sum((y_true != c) & (pred == c)))
        fn = int(np.sum((y_true == c) & (pred != c)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        support = int(np.sum(y_true == c))

        per_class[c] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": support,
        }

        per_class_f1.append(f1)

    macro_f1 = float(np.mean(per_class_f1))

    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": macro_f1,
        "per_class": per_class,
    }

def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None

    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_samples = 0

    all_y = []
    all_logits = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits, attn_weights = model(x, mask)
                loss = criterion(logits, y)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()

        bs = y.shape[0]
        total_loss += float(loss.detach().cpu().item()) * bs
        total_samples += bs

        all_y.append(y.detach().cpu().numpy())
        all_logits.append(logits.detach().float().cpu().numpy())

    all_y = np.concatenate(all_y) if all_y else np.array([], dtype=np.int64)
    all_logits = np.concatenate(all_logits) if all_logits else np.zeros((0, CONFIG["num_classes"]), dtype=np.float32)

    avg_loss = total_loss / max(total_samples, 1)
    metrics = compute_classification_metrics(
        all_y,
        all_logits,
        num_classes=CONFIG["num_classes"],
    )

    return avg_loss, metrics

# ----------------------------
# 9. Training loop
# ----------------------------

print("=" * 80)
print("START BASELINE TRAINING")
print("=" * 80)

best_val_top1 = -1.0
best_val_macro_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0

log_rows = []
start_time = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_start = time.time()

    train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_metrics = run_one_epoch(model, val_loader, optimizer=None)

    scheduler.step(val_metrics["top1"])

    current_lr = optimizer.param_groups[0]["lr"]
    improved = val_metrics["top1"] > best_val_top1

    if improved:
        best_val_top1 = val_metrics["top1"]
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": int(epoch),
                "model_state_dict": model.state_dict(),
                "config": CONFIG,
                "feature_mean": torch.tensor(feature_mean, dtype=torch.float32),
                "feature_std": torch.tensor(feature_std, dtype=torch.float32),
                "class_weights": torch.tensor(class_weights_np, dtype=torch.float32),
                "val_loss": float(val_loss),
                "val_top1": float(val_metrics["top1"]),
                "val_top5": float(val_metrics["top5"]),
                "val_macro_f1": float(val_metrics["macro_f1"]),
                "num_params": int(num_params),
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    torch.save(
        {
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "config": CONFIG,
            "feature_mean": torch.tensor(feature_mean, dtype=torch.float32),
            "feature_std": torch.tensor(feature_std, dtype=torch.float32),
            "class_weights": torch.tensor(class_weights_np, dtype=torch.float32),
            "val_loss": float(val_loss),
            "val_top1": float(val_metrics["top1"]),
            "val_top5": float(val_metrics["top5"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "num_params": int(num_params),
        },
        LAST_MODEL_PATH,
    )

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_loss,
        "train_top1": train_metrics["top1"],
        "train_top5": train_metrics["top5"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_loss,
        "val_top1": val_metrics["top1"],
        "val_top5": val_metrics["top5"],
        "val_macro_f1": val_metrics["macro_f1"],
        "best_val_top1": best_val_top1,
        "best_val_macro_f1": best_val_macro_f1,
        "best_epoch": best_epoch,
        "epoch_seconds": time.time() - epoch_start,
    }

    log_rows.append(row)
    pd.DataFrame(log_rows).to_csv(TRAINING_LOG_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
        f"lr={current_lr:.2e} | "
        f"train_loss={train_loss:.4f} | train_top1={train_metrics['top1']:.4f} | train_top5={train_metrics['top5']:.4f} | "
        f"val_loss={val_loss:.4f} | val_top1={val_metrics['top1']:.4f} | val_top5={val_metrics['top5']:.4f} | "
        f"val_f1={val_metrics['macro_f1']:.4f} | "
        f"best_top1={best_val_top1:.4f}@{best_epoch} | "
        f"{'BEST' if improved else ''}"
    )

    if epochs_without_improvement >= CONFIG["patience"]:
        print()
        print(f"Early stopping triggered after {CONFIG['patience']} epochs without improvement.")
        break

total_time = time.time() - start_time

# ----------------------------
# 10. Final validation summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 11A BASELINE TRAINING COMPLETE")
print("=" * 80)
print(f"Best epoch:        {best_epoch}")
print(f"Best val top-1:    {best_val_top1:.4f}")
print(f"Best val macro F1: {best_val_macro_f1:.4f}")
print(f"Best model path:   {BEST_MODEL_PATH}")
print(f"Last model path:   {LAST_MODEL_PATH}")
print(f"Training log path: {TRAINING_LOG_PATH}")
print(f"Config path:       {CONFIG_PATH}")
print(f"Total train time:  {total_time / 60:.2f} minutes")
print()

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

best_val_loss, best_val_metrics = run_one_epoch(model, val_loader, optimizer=None)

print("=" * 80)
print("BASELINE BEST MODEL VALIDATION METRICS")
print("=" * 80)
print(f"val_loss:     {best_val_loss:.4f}")
print(f"val_top1:     {best_val_metrics['top1']:.4f}")
print(f"val_top5:     {best_val_metrics['top5']:.4f}")
print(f"val_macro_f1: {best_val_metrics['macro_f1']:.4f}")
print()

print("=" * 80)
print("PHASE 11A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Training epoch logs")
print("2. Best epoch")
print("3. Baseline best val top-1")
print("4. Baseline best val top-5")
print("5. Baseline best val macro F1")
print()
print("Do NOT run baseline test evaluation yet.")

In [ ]:
# ============================================================
# PHASE 11B — BASELINE TEST EVALUATION + COMPARISON
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

RECOGNITION_DATASET_MANIFEST_PATH = MANIFEST_DIR / "recognition_dataset_verified_fixed.csv"
RECOGNITION_LABEL_MAP_PATH = MANIFEST_DIR / "recognition_label_map_with_split_counts.csv"

BASELINE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention_baseline_no_phase"
BASELINE_BEST_MODEL_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_best.pt"

PHASE_AWARE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention"
PHASE_AWARE_TEST_METRICS_PATH = PHASE_AWARE_MODEL_DIR / "recognition_test_metrics.json"

BASELINE_TEST_METRICS_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_metrics.json"
BASELINE_TEST_PREDICTIONS_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_predictions.csv"
BASELINE_PER_CLASS_TEST_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_per_class_metrics.csv"
BASELINE_CONFUSION_MATRIX_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_confusion_matrix.csv"
COMPARISON_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_comparison.csv"

print("=" * 80)
print("PHASE 11B — BASELINE TEST EVALUATION + COMPARISON")
print("=" * 80)

for p in [
    RECOGNITION_DATASET_MANIFEST_PATH,
    RECOGNITION_LABEL_MAP_PATH,
    BASELINE_BEST_MODEL_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

df = pd.read_csv(RECOGNITION_DATASET_MANIFEST_PATH)
df = df[df["verify_status"] == "ok"].copy()

label_map_df = pd.read_csv(RECOGNITION_LABEL_MAP_PATH)
id_to_gloss = {
    int(row["rec_label_id"]): str(row["gloss"])
    for _, row in label_map_df.iterrows()
}

print(f"Loaded dataset: {RECOGNITION_DATASET_MANIFEST_PATH}")
print(f"Rows: {len(df)}")
print(f"Loaded baseline checkpoint: {BASELINE_BEST_MODEL_PATH}")
print()

print("Rows by rec_split:")
print(df["rec_split"].value_counts(dropna=False))
print()

# ----------------------------
# 2. Device + checkpoint
# ----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

checkpoint = torch.load(BASELINE_BEST_MODEL_PATH, map_location=device, weights_only=False)

CONFIG = checkpoint["config"]
feature_mean = checkpoint["feature_mean"].detach().cpu().numpy().astype(np.float32)
feature_std = checkpoint["feature_std"].detach().cpu().numpy().astype(np.float32)

print("Loaded baseline checkpoint:")
print(f"epoch:        {checkpoint['epoch']}")
print(f"val_loss:     {checkpoint['val_loss']:.4f}")
print(f"val_top1:     {checkpoint['val_top1']:.4f}")
print(f"val_top5:     {checkpoint['val_top5']:.4f}")
print(f"val_macro_f1: {checkpoint['val_macro_f1']:.4f}")
print()

# ----------------------------
# 3. Feature loading
# ----------------------------

def load_baseline_features(row):
    """
    Baseline input:
      444 continuous features only
      no ordered phase one-hot
    """
    motion_path = Path(str(row["motion_path"]))
    segment_path = Path(str(row["phase_segment_path"]))

    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(segment_path, allow_pickle=False) as s:
        phase_speed = np.asarray(s["phase_speed"], dtype=np.float32).reshape(-1, 1)
        active_start = int(np.asarray(s["active_start"]).item())
        active_end = int(np.asarray(s["active_end"]).item())

    X = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X[~np.isfinite(X)] = 0.0

    # Same active crop as phase-aware model.
    X = X[active_start:active_end + 1]

    # Same continuous-feature standardization.
    X = (X - feature_mean) / feature_std
    X[~np.isfinite(X)] = 0.0

    if X.shape[1] != CONFIG["input_dim"]:
        raise ValueError(f"Expected input dim {CONFIG['input_dim']}, got {X.shape[1]}")

    y = int(row["rec_label_id"])

    return X.astype(np.float32), y

class BaselineRecognitionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        X, y = load_baseline_features(row)

        return {
            "x": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "length": int(X.shape[0]),
            "rec_split": row["rec_split"],
            "gloss": row["gloss"],
            "rec_label_id": int(row["rec_label_id"]),
            "video_stem": row["video_stem"],
        }

def recognition_collate(batch):
    batch_size = len(batch)
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    feat_dim = batch[0]["x"].shape[1]

    x_padded = torch.zeros(batch_size, max_len, feat_dim, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    y = torch.zeros(batch_size, dtype=torch.long)

    meta = []

    for i, item in enumerate(batch):
        T = item["length"]
        x_padded[i, :T] = item["x"]
        mask[i, :T] = True
        y[i] = item["y"]

        meta.append(
            {
                "rec_split": item["rec_split"],
                "gloss": item["gloss"],
                "rec_label_id": item["rec_label_id"],
                "video_stem": item["video_stem"],
            }
        )

    return {
        "x": x_padded,
        "y": y,
        "mask": mask,
        "lengths": lengths,
        "meta": meta,
    }

# ----------------------------
# 4. Model definition
# ----------------------------

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class SafeMaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        scores = self.attn(h.float()).squeeze(-1)
        mask = mask.bool()
        scores = scores.masked_fill(~mask, -1e4)

        attn_weights = torch.softmax(scores, dim=1)
        attn_weights = attn_weights.masked_fill(~mask, 0.0)

        pooled = torch.sum(h.float() * attn_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights

class BaselineTCNAttention(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = SafeMaskedAttentionPooling(
            hidden_dim,
            dropout=attention_dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        h = x.transpose(1, 2)
        h = self.tcn(h)
        h = h.transpose(1, 2)

        pooled, attn_weights = self.attention_pool(h, mask)
        logits = self.classifier(pooled.float())

        return logits, attn_weights

model = BaselineTCNAttention(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
    attention_dropout=CONFIG["attention_dropout"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Baseline model loaded successfully.")
print()

# ----------------------------
# 5. Metrics helpers
# ----------------------------

def compute_metrics(y_true, logits, num_classes=100):
    y_true = np.asarray(y_true, dtype=np.int64)
    logits = np.asarray(logits, dtype=np.float32)

    pred = np.argmax(logits, axis=1)

    top1 = float(np.mean(pred == y_true)) if len(y_true) else 0.0

    top5_idx = np.argsort(logits, axis=1)[:, -5:]
    top5 = float(np.mean([y_true[i] in top5_idx[i] for i in range(len(y_true))])) if len(y_true) else 0.0

    per_class_rows = []
    f1s = []

    for c in range(num_classes):
        tp = int(np.sum((y_true == c) & (pred == c)))
        fp = int(np.sum((y_true != c) & (pred == c)))
        fn = int(np.sum((y_true == c) & (pred != c)))

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-8)
        support = int(np.sum(y_true == c))

        per_class_rows.append(
            {
                "rec_label_id": c,
                "gloss": id_to_gloss.get(c, str(c)),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "support": support,
            }
        )

        f1s.append(f1)

    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": float(np.mean(f1s)),
        "pred": pred,
        "per_class_df": pd.DataFrame(per_class_rows),
    }

def confusion_matrix(y_true, y_pred, num_classes=100):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1

    return cm

# ----------------------------
# 6. Test evaluation
# ----------------------------

test_df = df[df["rec_split"] == "test"].copy()
test_dataset = BaselineRecognitionDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=recognition_collate,
)

print(f"Baseline test dataset size: {len(test_dataset)}")
print("Running baseline test evaluation...")
print()

all_y = []
all_logits = []
prediction_records = []

with torch.no_grad():
    for batch in test_loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        mask = batch["mask"].to(device)

        logits, attn_weights = model(x, mask)
        probs = torch.softmax(logits, dim=-1)

        logits_np = logits.detach().cpu().numpy()
        probs_np = probs.detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()

        all_y.append(y_np)
        all_logits.append(logits_np)

        for i, meta in enumerate(batch["meta"]):
            true_id = int(y_np[i])
            pred_id = int(np.argmax(logits_np[i]))
            top5_ids = np.argsort(logits_np[i])[-5:][::-1].tolist()

            prediction_records.append(
                {
                    "gloss_true": id_to_gloss.get(true_id, str(true_id)),
                    "gloss_pred": id_to_gloss.get(pred_id, str(pred_id)),
                    "true_id": true_id,
                    "pred_id": pred_id,
                    "correct": bool(pred_id == true_id),
                    "top5_correct": bool(true_id in top5_ids),
                    "top5_ids": top5_ids,
                    "top5_glosses": [id_to_gloss.get(k, str(k)) for k in top5_ids],
                    "top1_confidence": float(probs_np[i, pred_id]),
                    "true_class_confidence": float(probs_np[i, true_id]),
                    "length": int(batch["lengths"][i].item()),
                    "video_stem": meta["video_stem"],
                }
            )

all_y = np.concatenate(all_y)
all_logits = np.concatenate(all_logits)

metrics = compute_metrics(all_y, all_logits, num_classes=CONFIG["num_classes"])
y_pred = metrics["pred"]

cm = confusion_matrix(all_y, y_pred, num_classes=CONFIG["num_classes"])

prediction_df = pd.DataFrame(prediction_records)
per_class_df = metrics["per_class_df"]

prediction_df.to_csv(BASELINE_TEST_PREDICTIONS_PATH, index=False)
per_class_df.to_csv(BASELINE_PER_CLASS_TEST_PATH, index=False)

cm_df = pd.DataFrame(
    cm,
    index=[id_to_gloss.get(i, str(i)) for i in range(CONFIG["num_classes"])],
    columns=[id_to_gloss.get(i, str(i)) for i in range(CONFIG["num_classes"])],
)
cm_df.to_csv(BASELINE_CONFUSION_MATRIX_PATH)

baseline_metrics_to_save = {
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "checkpoint_val_top1": float(checkpoint["val_top1"]),
    "checkpoint_val_top5": float(checkpoint["val_top5"]),
    "checkpoint_val_macro_f1": float(checkpoint["val_macro_f1"]),
    "test_top1": float(metrics["top1"]),
    "test_top5": float(metrics["top5"]),
    "test_macro_f1": float(metrics["macro_f1"]),
    "num_test_sequences": int(len(test_dataset)),
    "num_classes": int(CONFIG["num_classes"]),
}

with open(BASELINE_TEST_METRICS_PATH, "w") as f:
    json.dump(baseline_metrics_to_save, f, indent=2)

# ----------------------------
# 7. Comparison with phase-aware model
# ----------------------------

phase_aware_metrics = None

if PHASE_AWARE_TEST_METRICS_PATH.exists():
    with open(PHASE_AWARE_TEST_METRICS_PATH, "r") as f:
        phase_aware_metrics = json.load(f)

comparison_rows = []

comparison_rows.append(
    {
        "model": "baseline_no_phase_onehot",
        "val_top1": float(checkpoint["val_top1"]),
        "val_top5": float(checkpoint["val_top5"]),
        "val_macro_f1": float(checkpoint["val_macro_f1"]),
        "test_top1": float(metrics["top1"]),
        "test_top5": float(metrics["top5"]),
        "test_macro_f1": float(metrics["macro_f1"]),
        "test_sequences": int(len(test_dataset)),
    }
)

if phase_aware_metrics is not None:
    comparison_rows.append(
        {
            "model": "phase_aware_plus_phase_onehot",
            "val_top1": float(phase_aware_metrics["checkpoint_val_top1"]),
            "val_top5": float(phase_aware_metrics["checkpoint_val_top5"]),
            "val_macro_f1": float(phase_aware_metrics["checkpoint_val_macro_f1"]),
            "test_top1": float(phase_aware_metrics["test_top1"]),
            "test_top5": float(phase_aware_metrics["test_top5"]),
            "test_macro_f1": float(phase_aware_metrics["test_macro_f1"]),
            "test_sequences": int(phase_aware_metrics["num_test_sequences"]),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)

# Add delta row if possible.
if phase_aware_metrics is not None:
    base = comparison_df[comparison_df["model"] == "baseline_no_phase_onehot"].iloc[0]
    phase = comparison_df[comparison_df["model"] == "phase_aware_plus_phase_onehot"].iloc[0]

    delta_row = {
        "model": "delta_phase_aware_minus_baseline",
        "val_top1": phase["val_top1"] - base["val_top1"],
        "val_top5": phase["val_top5"] - base["val_top5"],
        "val_macro_f1": phase["val_macro_f1"] - base["val_macro_f1"],
        "test_top1": phase["test_top1"] - base["test_top1"],
        "test_top5": phase["test_top5"] - base["test_top5"],
        "test_macro_f1": phase["test_macro_f1"] - base["test_macro_f1"],
        "test_sequences": int(phase["test_sequences"]),
    }

    comparison_df = pd.concat(
        [comparison_df, pd.DataFrame([delta_row])],
        ignore_index=True,
    )

comparison_df.to_csv(COMPARISON_PATH, index=False)

# ----------------------------
# 8. Print summary
# ----------------------------

print("=" * 80)
print("BASELINE TEST METRICS")
print("=" * 80)
print(f"baseline_test_top1:     {metrics['top1']:.4f}")
print(f"baseline_test_top5:     {metrics['top5']:.4f}")
print(f"baseline_test_macro_f1: {metrics['macro_f1']:.4f}")
print(f"test_sequences:         {len(test_dataset)}")
print()

print("Baseline prediction correctness counts:")
print(prediction_df["correct"].value_counts(dropna=False))
print()

print("Baseline top-5 correctness counts:")
print(prediction_df["top5_correct"].value_counts(dropna=False))
print()

print("=" * 80)
print("BASELINE PER-CLASS TEST METRICS SUMMARY")
print("=" * 80)
print(per_class_df[["precision", "recall", "f1", "support"]].describe())
print()

print("Worst 20 baseline classes by test F1:")
display(
    per_class_df.sort_values(["f1", "support"], ascending=[True, False]).head(20)
)

print("=" * 80)
print("BASELINE TOP CONFUSIONS")
print("=" * 80)

confusions = []

for true_id in range(CONFIG["num_classes"]):
    for pred_id in range(CONFIG["num_classes"]):
        if true_id != pred_id and cm[true_id, pred_id] > 0:
            confusions.append(
                {
                    "true_id": true_id,
                    "true_gloss": id_to_gloss.get(true_id, str(true_id)),
                    "pred_id": pred_id,
                    "pred_gloss": id_to_gloss.get(pred_id, str(pred_id)),
                    "count": int(cm[true_id, pred_id]),
                }
            )

confusion_long_df = pd.DataFrame(confusions)

if len(confusion_long_df) == 0:
    print("No confusions found.")
else:
    display(confusion_long_df.sort_values("count", ascending=False).head(20))

print("=" * 80)
print("PHASE-AWARE VS BASELINE COMPARISON")
print("=" * 80)

if phase_aware_metrics is None:
    print(f"Phase-aware metrics file not found: {PHASE_AWARE_TEST_METRICS_PATH}")
    print("Baseline metrics were saved, but comparison is incomplete.")
else:
    display(comparison_df)

print()
print("=" * 80)
print("PHASE 11B OUTPUT SAVED")
print("=" * 80)
print(f"Baseline test metrics:      {BASELINE_TEST_METRICS_PATH}")
print(f"Baseline test predictions:  {BASELINE_TEST_PREDICTIONS_PATH}")
print(f"Baseline per-class metrics: {BASELINE_PER_CLASS_TEST_PATH}")
print(f"Baseline confusion matrix:  {BASELINE_CONFUSION_MATRIX_PATH}")
print(f"Comparison CSV:             {COMPARISON_PATH}")
print()

print("=" * 80)
print("PHASE 11B COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Baseline test metrics")
print("2. Baseline correctness counts")
print("3. Baseline per-class test metrics summary")
print("4. Worst 20 baseline classes")
print("5. Baseline top confusions")
print("6. Phase-aware vs baseline comparison table")
print()
print("Do NOT start new training yet.")

In [ ]:
# ============================================================
# PHASE 11C — PAIRED PHASE-AWARE VS BASELINE ERROR ANALYSIS
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")

PHASE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention"
BASELINE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention_baseline_no_phase"

PHASE_PRED_PATH = PHASE_MODEL_DIR / "recognition_test_predictions.csv"
BASELINE_PRED_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_predictions.csv"

PHASE_PER_CLASS_PATH = PHASE_MODEL_DIR / "recognition_test_per_class_metrics.csv"
BASELINE_PER_CLASS_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_per_class_metrics.csv"

COMPARISON_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_comparison.csv"

PAIRWISE_OUTPUT_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_pairwise_predictions.csv"
PER_CLASS_DELTA_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_per_class_delta.csv"
SUMMARY_JSON_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_pairwise_summary.json"

print("=" * 80)
print("PHASE 11C — PAIRED PHASE-AWARE VS BASELINE ERROR ANALYSIS")
print("=" * 80)

for p in [
    PHASE_PRED_PATH,
    BASELINE_PRED_PATH,
    PHASE_PER_CLASS_PATH,
    BASELINE_PER_CLASS_PATH,
    COMPARISON_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

phase_df = pd.read_csv(PHASE_PRED_PATH)
base_df = pd.read_csv(BASELINE_PRED_PATH)

phase_pc = pd.read_csv(PHASE_PER_CLASS_PATH)
base_pc = pd.read_csv(BASELINE_PER_CLASS_PATH)

comparison_df = pd.read_csv(COMPARISON_PATH)

print(f"Loaded phase-aware predictions: {PHASE_PRED_PATH}")
print(f"Phase-aware rows: {len(phase_df)}")
print(f"Loaded baseline predictions:    {BASELINE_PRED_PATH}")
print(f"Baseline rows: {len(base_df)}")
print()

# ----------------------------
# 2. Merge per-sample predictions
# ----------------------------

phase_keep = phase_df[
    [
        "video_stem",
        "gloss_true",
        "gloss_pred",
        "true_id",
        "pred_id",
        "correct",
        "top5_correct",
        "top1_confidence",
        "true_class_confidence",
        "top5_glosses",
        "length",
    ]
].copy()

base_keep = base_df[
    [
        "video_stem",
        "gloss_true",
        "gloss_pred",
        "true_id",
        "pred_id",
        "correct",
        "top5_correct",
        "top1_confidence",
        "true_class_confidence",
        "top5_glosses",
        "length",
    ]
].copy()

merged = phase_keep.merge(
    base_keep,
    on="video_stem",
    how="inner",
    suffixes=("_phase", "_base"),
)

print("Merged paired rows:", len(merged))
print()

if len(merged) != len(phase_df) or len(merged) != len(base_df):
    print("WARNING: Merge row count differs from input counts. Check duplicate/missing video_stem values.")

# Safety check
true_mismatch = (merged["true_id_phase"] != merged["true_id_base"]).sum()
print("True-label mismatches after merge:", int(true_mismatch))
print()

# ----------------------------
# 3. Top-1 paired comparison
# ----------------------------

merged["both_correct_top1"] = (merged["correct_phase"] == True) & (merged["correct_base"] == True)
merged["phase_only_correct_top1"] = (merged["correct_phase"] == True) & (merged["correct_base"] == False)
merged["baseline_only_correct_top1"] = (merged["correct_phase"] == False) & (merged["correct_base"] == True)
merged["both_wrong_top1"] = (merged["correct_phase"] == False) & (merged["correct_base"] == False)

both_correct = int(merged["both_correct_top1"].sum())
phase_only = int(merged["phase_only_correct_top1"].sum())
baseline_only = int(merged["baseline_only_correct_top1"].sum())
both_wrong = int(merged["both_wrong_top1"].sum())

# McNemar's test approximate with continuity correction.
# b = baseline wrong, phase correct
# c = baseline correct, phase wrong
b = phase_only
c = baseline_only

if b + c > 0:
    mcnemar_chi2 = ((abs(b - c) - 1) ** 2) / (b + c)
    # chi-square df=1 survival function = erfc(sqrt(x/2))
    mcnemar_p_approx = math.erfc(math.sqrt(mcnemar_chi2 / 2))
else:
    mcnemar_chi2 = 0.0
    mcnemar_p_approx = 1.0

# ----------------------------
# 4. Top-5 paired comparison
# ----------------------------

merged["both_correct_top5"] = (merged["top5_correct_phase"] == True) & (merged["top5_correct_base"] == True)
merged["phase_only_correct_top5"] = (merged["top5_correct_phase"] == True) & (merged["top5_correct_base"] == False)
merged["baseline_only_correct_top5"] = (merged["top5_correct_phase"] == False) & (merged["top5_correct_base"] == True)
merged["both_wrong_top5"] = (merged["top5_correct_phase"] == False) & (merged["top5_correct_base"] == False)

both_correct_top5 = int(merged["both_correct_top5"].sum())
phase_only_top5 = int(merged["phase_only_correct_top5"].sum())
baseline_only_top5 = int(merged["baseline_only_correct_top5"].sum())
both_wrong_top5 = int(merged["both_wrong_top5"].sum())

# ----------------------------
# 5. Per-class delta comparison
# ----------------------------

phase_pc_keep = phase_pc.rename(
    columns={
        "precision": "precision_phase",
        "recall": "recall_phase",
        "f1": "f1_phase",
        "support": "support_phase",
    }
)

base_pc_keep = base_pc.rename(
    columns={
        "precision": "precision_base",
        "recall": "recall_base",
        "f1": "f1_base",
        "support": "support_base",
    }
)

per_class_delta = phase_pc_keep.merge(
    base_pc_keep,
    on=["rec_label_id", "gloss"],
    how="inner",
)

per_class_delta["delta_precision"] = per_class_delta["precision_phase"] - per_class_delta["precision_base"]
per_class_delta["delta_recall"] = per_class_delta["recall_phase"] - per_class_delta["recall_base"]
per_class_delta["delta_f1"] = per_class_delta["f1_phase"] - per_class_delta["f1_base"]

# ----------------------------
# 6. Save outputs
# ----------------------------

merged.to_csv(PAIRWISE_OUTPUT_PATH, index=False)
per_class_delta.to_csv(PER_CLASS_DELTA_PATH, index=False)

summary = {
    "n_paired": int(len(merged)),
    "top1": {
        "both_correct": both_correct,
        "phase_only_correct": phase_only,
        "baseline_only_correct": baseline_only,
        "both_wrong": both_wrong,
        "net_phase_gain": phase_only - baseline_only,
        "mcnemar_chi2_approx": float(mcnemar_chi2),
        "mcnemar_p_approx": float(mcnemar_p_approx),
    },
    "top5": {
        "both_correct": both_correct_top5,
        "phase_only_correct": phase_only_top5,
        "baseline_only_correct": baseline_only_top5,
        "both_wrong": both_wrong_top5,
        "net_phase_gain": phase_only_top5 - baseline_only_top5,
    },
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary, f, indent=2)

# ----------------------------
# 7. Print comparison
# ----------------------------

print("=" * 80)
print("OVERALL METRIC COMPARISON")
print("=" * 80)
display(comparison_df)
print()

print("=" * 80)
print("PAIRED TOP-1 COMPARISON")
print("=" * 80)
print(f"Both correct:             {both_correct}")
print(f"Phase-aware only correct: {phase_only}")
print(f"Baseline only correct:    {baseline_only}")
print(f"Both wrong:               {both_wrong}")
print(f"Net phase-aware gain:     {phase_only - baseline_only}")
print()
print(f"McNemar chi2 approx:      {mcnemar_chi2:.4f}")
print(f"McNemar p approx:         {mcnemar_p_approx:.4f}")
print()

if mcnemar_p_approx < 0.05:
    print("Interpretation: paired Top-1 difference is statistically noticeable under approximate McNemar test.")
else:
    print("Interpretation: paired Top-1 difference is NOT statistically strong under approximate McNemar test.")

print()
print("=" * 80)
print("PAIRED TOP-5 COMPARISON")
print("=" * 80)
print(f"Both Top-5 correct:             {both_correct_top5}")
print(f"Phase-aware only Top-5 correct: {phase_only_top5}")
print(f"Baseline only Top-5 correct:    {baseline_only_top5}")
print(f"Both Top-5 wrong:               {both_wrong_top5}")
print(f"Net phase-aware Top-5 gain:     {phase_only_top5 - baseline_only_top5}")
print()

print("=" * 80)
print("PER-CLASS F1 DELTA SUMMARY")
print("=" * 80)
print(per_class_delta["delta_f1"].describe())
print()

print("Classes most improved by phase-aware features:")
display(
    per_class_delta.sort_values("delta_f1", ascending=False)
    [
        [
            "rec_label_id",
            "gloss",
            "support_phase",
            "f1_base",
            "f1_phase",
            "delta_f1",
            "precision_base",
            "precision_phase",
            "recall_base",
            "recall_phase",
        ]
    ]
    .head(20)
)

print("Classes most hurt by phase-aware features:")
display(
    per_class_delta.sort_values("delta_f1", ascending=True)
    [
        [
            "rec_label_id",
            "gloss",
            "support_phase",
            "f1_base",
            "f1_phase",
            "delta_f1",
            "precision_base",
            "precision_phase",
            "recall_base",
            "recall_phase",
        ]
    ]
    .head(20)
)

print("=" * 80)
print("EXAMPLES FIXED BY PHASE-AWARE MODEL")
print("=" * 80)

fixed_by_phase = merged[merged["phase_only_correct_top1"] == True].copy()

if len(fixed_by_phase) == 0:
    print("No examples fixed by phase-aware model.")
else:
    display(
        fixed_by_phase[
            [
                "video_stem",
                "gloss_true_phase",
                "gloss_pred_base",
                "gloss_pred_phase",
                "top1_confidence_base",
                "top1_confidence_phase",
                "top5_correct_base",
                "top5_correct_phase",
            ]
        ].head(25)
    )

print("=" * 80)
print("EXAMPLES HURT BY PHASE-AWARE MODEL")
print("=" * 80)

hurt_by_phase = merged[merged["baseline_only_correct_top1"] == True].copy()

if len(hurt_by_phase) == 0:
    print("No examples hurt by phase-aware model.")
else:
    display(
        hurt_by_phase[
            [
                "video_stem",
                "gloss_true_phase",
                "gloss_pred_base",
                "gloss_pred_phase",
                "top1_confidence_base",
                "top1_confidence_phase",
                "top5_correct_base",
                "top5_correct_phase",
            ]
        ].head(25)
    )

print()
print("=" * 80)
print("PHASE 11C OUTPUT SAVED")
print("=" * 80)
print(f"Pairwise predictions: {PAIRWISE_OUTPUT_PATH}")
print(f"Per-class deltas:     {PER_CLASS_DELTA_PATH}")
print(f"Summary JSON:         {SUMMARY_JSON_PATH}")
print()

print("=" * 80)
print("PHASE 11C COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Paired Top-1 comparison")
print("2. Paired Top-5 comparison")
print("3. McNemar interpretation")
print("4. Per-class F1 delta summary")
print("5. Classes most improved")
print("6. Classes most hurt")
print("7. Examples fixed/hurt by phase-aware model")
print()
print("Do NOT train anything else yet.")

In [ ]:
# ============================================================
# PHASE 12A — FINAL RESULTS SUMMARY PACKAGE
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")

REPORT_DIR = BASE_DIR / "reports" / "final_results"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Phase TCN outputs
PHASE_TCN_MODEL_DIR = BASE_DIR / "models" / "phase_tcn"
PHASE_TCN_TEST_METRICS_PATH = PHASE_TCN_MODEL_DIR / "phase_tcn_test_metrics.json"

# Recognition phase-aware outputs
PHASE_AWARE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention"
PHASE_AWARE_TEST_METRICS_PATH = PHASE_AWARE_MODEL_DIR / "recognition_test_metrics.json"
PHASE_AWARE_PER_CLASS_PATH = PHASE_AWARE_MODEL_DIR / "recognition_test_per_class_metrics.csv"
PHASE_AWARE_PREDICTIONS_PATH = PHASE_AWARE_MODEL_DIR / "recognition_test_predictions.csv"

# Baseline outputs
BASELINE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention_baseline_no_phase"
BASELINE_TEST_METRICS_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_metrics.json"
BASELINE_PER_CLASS_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_per_class_metrics.csv"
COMPARISON_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_comparison.csv"
PAIRWISE_SUMMARY_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_pairwise_summary.json"
PER_CLASS_DELTA_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_per_class_delta.csv"

# Dataset manifests
RECOGNITION_DATASET_MANIFEST_PATH = BASE_DIR / "manifests" / "recognition_dataset_verified_fixed.csv"
PHASE_SEGMENT_MANIFEST_PATH = BASE_DIR / "manifests" / "phase_segments_manifest.csv"
PHASE_LABEL_CLEAN_MANIFEST_PATH = BASE_DIR / "manifests" / "phase_label_manifest_clean.csv"

# Report outputs
FINAL_SUMMARY_JSON_PATH = REPORT_DIR / "final_results_summary.json"
FINAL_TABLES_MD_PATH = REPORT_DIR / "final_results_tables.md"
FINAL_KEY_POINTS_MD_PATH = REPORT_DIR / "final_report_key_points.md"
PHASE_STATUS_CSV_PATH = REPORT_DIR / "project_phase_status.csv"

print("=" * 80)
print("PHASE 12A — FINAL RESULTS SUMMARY PACKAGE")
print("=" * 80)
print(f"Report output directory: {REPORT_DIR}")
print()

required_paths = [
    PHASE_TCN_TEST_METRICS_PATH,
    PHASE_AWARE_TEST_METRICS_PATH,
    BASELINE_TEST_METRICS_PATH,
    COMPARISON_PATH,
    PAIRWISE_SUMMARY_PATH,
    PER_CLASS_DELTA_PATH,
    RECOGNITION_DATASET_MANIFEST_PATH,
    PHASE_SEGMENT_MANIFEST_PATH,
    PHASE_LABEL_CLEAN_MANIFEST_PATH,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Missing required files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Some required final-result files are missing.")

print("All required files found.")
print()

# ----------------------------
# 2. Load files
# ----------------------------

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

phase_tcn_metrics = load_json(PHASE_TCN_TEST_METRICS_PATH)
phase_aware_metrics = load_json(PHASE_AWARE_TEST_METRICS_PATH)
baseline_metrics = load_json(BASELINE_TEST_METRICS_PATH)
pairwise_summary = load_json(PAIRWISE_SUMMARY_PATH)

comparison_df = pd.read_csv(COMPARISON_PATH)
per_class_delta_df = pd.read_csv(PER_CLASS_DELTA_PATH)

rec_df = pd.read_csv(RECOGNITION_DATASET_MANIFEST_PATH)
segments_df = pd.read_csv(PHASE_SEGMENT_MANIFEST_PATH)
phase_clean_df = pd.read_csv(PHASE_LABEL_CLEAN_MANIFEST_PATH)

print("Loaded final result files.")
print()

# ----------------------------
# 3. Build phase status table
# ----------------------------

phase_status_rows = [
    {
        "phase": "Phase 0",
        "name": "Reference repo / project path planning",
        "status": "Done",
        "main_output": "Project direction finalized",
    },
    {
        "phase": "Phase 1",
        "name": "MediaPipe keypoint extraction",
        "status": "Done",
        "main_output": "Pose/hand keypoints extracted",
    },
    {
        "phase": "Phase 2",
        "name": "Keypoint manifest + normalization",
        "status": "Done",
        "main_output": "Normalized 147D landmark representation",
    },
    {
        "phase": "Phase 3",
        "name": "Motion features",
        "status": "Done",
        "main_output": "Velocity, acceleration, speed curves",
    },
    {
        "phase": "Phase 4",
        "name": "Motion curve analysis",
        "status": "Done",
        "main_output": "Improved phase signal",
    },
    {
        "phase": "Phase 5",
        "name": "Weak phase pseudo-label generation",
        "status": "Done",
        "main_output": "Preparation/stroke/retraction pseudo-labels",
    },
    {
        "phase": "Phase 6",
        "name": "Phase label visualization",
        "status": "Done",
        "main_output": "Visual sanity checks",
    },
    {
        "phase": "Phase 7",
        "name": "Clean phase-label manifest",
        "status": "Done",
        "main_output": "Strict validated phase-label set",
    },
    {
        "phase": "Phase 8",
        "name": "Phase Detection TCN",
        "status": "Done",
        "main_output": "Frame-level phase classifier",
    },
    {
        "phase": "Phase 9",
        "name": "Phase-aware segment extraction",
        "status": "Done",
        "main_output": "Ordered active/prep/stroke/retraction segments",
    },
    {
        "phase": "Phase 10",
        "name": "Phase-aware Recognition TCN + Attention",
        "status": "Done",
        "main_output": "100-class sign recognizer",
    },
    {
        "phase": "Phase 11",
        "name": "Baseline + paired error analysis",
        "status": "Done",
        "main_output": "Ablation comparison against no-phase baseline",
    },
    {
        "phase": "Phase 12",
        "name": "Final report packaging",
        "status": "Current",
        "main_output": "Report-ready results and summary tables",
    },
    {
        "phase": "Phase 13",
        "name": "Offline / real-time demo",
        "status": "Next",
        "main_output": "Saved-video inference, then webcam demo",
    },
]

phase_status_df = pd.DataFrame(phase_status_rows)
phase_status_df.to_csv(PHASE_STATUS_CSV_PATH, index=False)

# ----------------------------
# 4. Dataset summaries
# ----------------------------

recognition_split_counts = rec_df["rec_split"].value_counts(dropna=False).to_dict()
recognition_num_classes = int(rec_df["rec_label_id"].nunique())
recognition_num_rows = int(len(rec_df))

segment_status_counts = segments_df["segment_status"].value_counts(dropna=False).to_dict()
segment_quality_counts = segments_df["segment_quality_flag"].value_counts(dropna=False).to_dict()

clean_phase_split_counts = phase_clean_df["split"].value_counts(dropna=False).to_dict()
clean_phase_rows = int(len(phase_clean_df))

# ----------------------------
# 5. Final metric summaries
# ----------------------------

phase_tcn_summary = {
    "test_accuracy": float(phase_tcn_metrics.get("test_accuracy", np.nan)),
    "test_macro_f1": float(phase_tcn_metrics.get("test_macro_f1", np.nan)),
    "num_test_sequences": int(phase_tcn_metrics.get("num_test_sequences", -1)),
    "num_test_frames": int(phase_tcn_metrics.get("num_test_frames", -1)),
}

phase_aware_summary = {
    "checkpoint_epoch": int(phase_aware_metrics.get("checkpoint_epoch", -1)),
    "val_top1": float(phase_aware_metrics.get("checkpoint_val_top1", np.nan)),
    "val_top5": float(phase_aware_metrics.get("checkpoint_val_top5", np.nan)),
    "val_macro_f1": float(phase_aware_metrics.get("checkpoint_val_macro_f1", np.nan)),
    "test_top1": float(phase_aware_metrics.get("test_top1", np.nan)),
    "test_top5": float(phase_aware_metrics.get("test_top5", np.nan)),
    "test_macro_f1": float(phase_aware_metrics.get("test_macro_f1", np.nan)),
    "num_test_sequences": int(phase_aware_metrics.get("num_test_sequences", -1)),
}

baseline_summary = {
    "checkpoint_epoch": int(baseline_metrics.get("checkpoint_epoch", -1)),
    "val_top1": float(baseline_metrics.get("checkpoint_val_top1", np.nan)),
    "val_top5": float(baseline_metrics.get("checkpoint_val_top5", np.nan)),
    "val_macro_f1": float(baseline_metrics.get("checkpoint_val_macro_f1", np.nan)),
    "test_top1": float(baseline_metrics.get("test_top1", np.nan)),
    "test_top5": float(baseline_metrics.get("test_top5", np.nan)),
    "test_macro_f1": float(baseline_metrics.get("test_macro_f1", np.nan)),
    "num_test_sequences": int(baseline_metrics.get("num_test_sequences", -1)),
}

delta_summary = {
    "delta_test_top1": phase_aware_summary["test_top1"] - baseline_summary["test_top1"],
    "delta_test_top5": phase_aware_summary["test_top5"] - baseline_summary["test_top5"],
    "delta_test_macro_f1": phase_aware_summary["test_macro_f1"] - baseline_summary["test_macro_f1"],
    "delta_val_top1": phase_aware_summary["val_top1"] - baseline_summary["val_top1"],
    "delta_val_top5": phase_aware_summary["val_top5"] - baseline_summary["val_top5"],
    "delta_val_macro_f1": phase_aware_summary["val_macro_f1"] - baseline_summary["val_macro_f1"],
}

pairwise_top1 = pairwise_summary["top1"]
pairwise_top5 = pairwise_summary["top5"]

# ----------------------------
# 6. Per-class delta summaries
# ----------------------------

most_improved = (
    per_class_delta_df.sort_values("delta_f1", ascending=False)
    [
        [
            "rec_label_id",
            "gloss",
            "support_phase",
            "f1_base",
            "f1_phase",
            "delta_f1",
            "precision_base",
            "precision_phase",
            "recall_base",
            "recall_phase",
        ]
    ]
    .head(10)
)

most_hurt = (
    per_class_delta_df.sort_values("delta_f1", ascending=True)
    [
        [
            "rec_label_id",
            "gloss",
            "support_phase",
            "f1_base",
            "f1_phase",
            "delta_f1",
            "precision_base",
            "precision_phase",
            "recall_base",
            "recall_phase",
        ]
    ]
    .head(10)
)

# ----------------------------
# 7. Save JSON summary
# ----------------------------

final_summary = {
    "dataset_summary": {
        "recognition_rows": recognition_num_rows,
        "recognition_num_classes": recognition_num_classes,
        "recognition_split_counts": recognition_split_counts,
        "clean_phase_rows": clean_phase_rows,
        "clean_phase_split_counts": clean_phase_split_counts,
        "segment_status_counts": segment_status_counts,
        "segment_quality_counts": segment_quality_counts,
    },
    "phase_tcn_summary": phase_tcn_summary,
    "phase_aware_recognition_summary": phase_aware_summary,
    "baseline_recognition_summary": baseline_summary,
    "delta_phase_aware_minus_baseline": delta_summary,
    "paired_analysis": {
        "top1": pairwise_top1,
        "top5": pairwise_top5,
    },
    "interpretation": {
        "main_claim": (
            "The phase-aware model achieved a small but consistent improvement over the no-phase baseline "
            "across test Top-1 accuracy, Top-5 accuracy, and Macro F1."
        ),
        "statistical_caution": (
            "The paired McNemar approximation did not show a statistically strong Top-1 difference, "
            "so the improvement should be interpreted as modest and not conclusive."
        ),
    },
}

with open(FINAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(final_summary, f, indent=2)

# ----------------------------
# 8. Save Markdown results tables
# ----------------------------

comparison_markdown = comparison_df.to_markdown(index=False)
most_improved_markdown = most_improved.to_markdown(index=False)
most_hurt_markdown = most_hurt.to_markdown(index=False)
phase_status_markdown = phase_status_df.to_markdown(index=False)

tables_md = f"""# Final Results Tables

## Project Phase Status

{phase_status_markdown}

## Recognition Dataset Summary

| Item | Value |
|---|---:|
| Recognition samples | {recognition_num_rows} |
| Recognition classes | {recognition_num_classes} |
| Train samples | {recognition_split_counts.get("train", 0)} |
| Validation samples | {recognition_split_counts.get("val", 0)} |
| Test samples | {recognition_split_counts.get("test", 0)} |

## Phase Detection TCN Test Summary

| Metric | Value |
|---|---:|
| Test accuracy | {phase_tcn_summary["test_accuracy"]:.4f} |
| Test macro F1 | {phase_tcn_summary["test_macro_f1"]:.4f} |
| Test sequences | {phase_tcn_summary["num_test_sequences"]} |
| Test frames | {phase_tcn_summary["num_test_frames"]} |

## Recognition Model Comparison

{comparison_markdown}

## Paired Top-1 Comparison

| Item | Value |
|---|---:|
| Both correct | {pairwise_top1["both_correct"]} |
| Phase-aware only correct | {pairwise_top1["phase_only_correct"]} |
| Baseline only correct | {pairwise_top1["baseline_only_correct"]} |
| Both wrong | {pairwise_top1["both_wrong"]} |
| Net phase-aware gain | {pairwise_top1["net_phase_gain"]} |
| McNemar chi2 approx | {pairwise_top1["mcnemar_chi2_approx"]:.4f} |
| McNemar p approx | {pairwise_top1["mcnemar_p_approx"]:.4f} |

## Paired Top-5 Comparison

| Item | Value |
|---|---:|
| Both Top-5 correct | {pairwise_top5["both_correct"]} |
| Phase-aware only Top-5 correct | {pairwise_top5["phase_only_correct"]} |
| Baseline only Top-5 correct | {pairwise_top5["baseline_only_correct"]} |
| Both Top-5 wrong | {pairwise_top5["both_wrong"]} |
| Net phase-aware Top-5 gain | {pairwise_top5["net_phase_gain"]} |

## Classes Most Improved by Phase-Aware Features

{most_improved_markdown}

## Classes Most Hurt by Phase-Aware Features

{most_hurt_markdown}
"""

with open(FINAL_TABLES_MD_PATH, "w") as f:
    f.write(tables_md)

# ----------------------------
# 9. Save report key points
# ----------------------------

key_points_md = f"""# Final Report Key Points

## Main Result

The final phase-aware Recognition TCN with attention achieved:

- Test Top-1 accuracy: {phase_aware_summary["test_top1"]:.4f}
- Test Top-5 accuracy: {phase_aware_summary["test_top5"]:.4f}
- Test Macro F1: {phase_aware_summary["test_macro_f1"]:.4f}

The no-phase baseline achieved:

- Test Top-1 accuracy: {baseline_summary["test_top1"]:.4f}
- Test Top-5 accuracy: {baseline_summary["test_top5"]:.4f}
- Test Macro F1: {baseline_summary["test_macro_f1"]:.4f}

The phase-aware model improved over the baseline by:

- Top-1: {delta_summary["delta_test_top1"]:.4f}
- Top-5: {delta_summary["delta_test_top5"]:.4f}
- Macro F1: {delta_summary["delta_test_macro_f1"]:.4f}

## Correct Interpretation

The phase-aware model produced a small but consistent improvement over the no-phase baseline. However, the paired McNemar test approximation gave p = {pairwise_top1["mcnemar_p_approx"]:.4f}, meaning the Top-1 improvement was not statistically strong. Therefore, the contribution should be framed as a promising phase-aware design rather than a definitive performance breakthrough.

## Recommended Report Claim

This project demonstrates a complete phase-aware isolated sign recognition pipeline using MediaPipe landmarks, motion-derived weak phase pseudo-labels, a Phase Detection TCN, phase-aware segment extraction, and a TCN-attention recognition model. The proposed phase-aware representation slightly improved recognition performance compared with a no-phase baseline, but the paired comparison suggests the improvement is modest and not statistically conclusive.

## Limitations

1. Phase labels are weak pseudo-labels derived from motion curves, not human linguistic annotations.
2. The recognition dataset is relatively small after filtering.
3. The model shows a validation-to-test generalization gap.
4. Some classes benefit from phase features while others are hurt.
5. Real-time webcam testing may perform worse because of camera angle, lighting, signer differences, and MediaPipe tracking instability.

## Future Work

1. Add more data per class.
2. Use signer-independent validation/test splits if signer metadata is available.
3. Improve phase pseudo-labeling with manual correction or semi-supervised refinement.
4. Add real-time webcam inference.
5. Add YOLO signer localization before MediaPipe for cluttered camera scenes.
6. Compare against stronger sequence models such as BiLSTM, Transformer encoder, or ST-GCN.
"""

with open(FINAL_KEY_POINTS_MD_PATH, "w") as f:
    f.write(key_points_md)

# ----------------------------
# 10. Print final summary
# ----------------------------

print("=" * 80)
print("PHASE 12A OUTPUT SAVED")
print("=" * 80)
print(f"Final summary JSON:      {FINAL_SUMMARY_JSON_PATH}")
print(f"Final tables markdown:   {FINAL_TABLES_MD_PATH}")
print(f"Final key points:        {FINAL_KEY_POINTS_MD_PATH}")
print(f"Project phase status:    {PHASE_STATUS_CSV_PATH}")
print()

print("=" * 80)
print("FINAL DATASET SUMMARY")
print("=" * 80)
print(f"Recognition samples: {recognition_num_rows}")
print(f"Recognition classes: {recognition_num_classes}")
print("Recognition split counts:")
print(pd.Series(recognition_split_counts))
print()

print("Segment status counts:")
print(pd.Series(segment_status_counts))
print()

print("=" * 80)
print("FINAL MODEL RESULTS")
print("=" * 80)
print("Phase TCN:")
print(f"  test_accuracy: {phase_tcn_summary['test_accuracy']:.4f}")
print(f"  test_macro_f1: {phase_tcn_summary['test_macro_f1']:.4f}")
print()

print("Recognition baseline without phase one-hot:")
print(f"  test_top1:     {baseline_summary['test_top1']:.4f}")
print(f"  test_top5:     {baseline_summary['test_top5']:.4f}")
print(f"  test_macro_f1: {baseline_summary['test_macro_f1']:.4f}")
print()

print("Recognition phase-aware + phase one-hot:")
print(f"  test_top1:     {phase_aware_summary['test_top1']:.4f}")
print(f"  test_top5:     {phase_aware_summary['test_top5']:.4f}")
print(f"  test_macro_f1: {phase_aware_summary['test_macro_f1']:.4f}")
print()

print("Delta phase-aware minus baseline:")
print(f"  test_top1:     {delta_summary['delta_test_top1']:.4f}")
print(f"  test_top5:     {delta_summary['delta_test_top5']:.4f}")
print(f"  test_macro_f1: {delta_summary['delta_test_macro_f1']:.4f}")
print()

print("=" * 80)
print("PAIRED ANALYSIS SUMMARY")
print("=" * 80)
print(f"Both correct:             {pairwise_top1['both_correct']}")
print(f"Phase-aware only correct: {pairwise_top1['phase_only_correct']}")
print(f"Baseline only correct:    {pairwise_top1['baseline_only_correct']}")
print(f"Both wrong:               {pairwise_top1['both_wrong']}")
print(f"Net phase-aware gain:     {pairwise_top1['net_phase_gain']}")
print(f"McNemar p approx:         {pairwise_top1['mcnemar_p_approx']:.4f}")
print()

print("=" * 80)
print("MOST IMPROVED CLASSES")
print("=" * 80)
display(most_improved)

print("=" * 80)
print("MOST HURT CLASSES")
print("=" * 80)
display(most_hurt)

print()
print("=" * 80)
print("PHASE 12A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Final dataset summary")
print("2. Final model results")
print("3. Paired analysis summary")
print("4. Most improved / most hurt classes")
print("5. Saved output paths")
print()
print("Next step: Phase 12B — generate final plots and report-ready figures.")

In [ ]:
# ============================================================
# PHASE 12B — FINAL PLOTS AND REPORT-READY FIGURES
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")

REPORT_DIR = BASE_DIR / "reports" / "final_results"
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SUMMARY_JSON_PATH = REPORT_DIR / "final_results_summary.json"
FINAL_TABLES_MD_PATH = REPORT_DIR / "final_results_tables.md"
PROJECT_PHASE_STATUS_PATH = REPORT_DIR / "project_phase_status.csv"

PHASE_AWARE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention"
BASELINE_MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention_baseline_no_phase"
PHASE_TCN_MODEL_DIR = BASE_DIR / "models" / "phase_tcn"

COMPARISON_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_comparison.csv"
PAIRWISE_SUMMARY_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_pairwise_summary.json"
PER_CLASS_DELTA_PATH = BASELINE_MODEL_DIR / "phase_aware_vs_baseline_per_class_delta.csv"

PHASE_AWARE_PREDICTIONS_PATH = PHASE_AWARE_MODEL_DIR / "recognition_test_predictions.csv"
PHASE_AWARE_PER_CLASS_PATH = PHASE_AWARE_MODEL_DIR / "recognition_test_per_class_metrics.csv"
BASELINE_PER_CLASS_PATH = BASELINE_MODEL_DIR / "baseline_no_phase_test_per_class_metrics.csv"

FIGURE_MANIFEST_PATH = FIGURE_DIR / "final_figures_manifest.csv"
FIGURE_CAPTIONS_PATH = FIGURE_DIR / "figure_captions.md"

print("=" * 80)
print("PHASE 12B — FINAL PLOTS AND REPORT-READY FIGURES")
print("=" * 80)
print(f"Figure output directory: {FIGURE_DIR}")
print()

required_paths = [
    FINAL_SUMMARY_JSON_PATH,
    COMPARISON_PATH,
    PAIRWISE_SUMMARY_PATH,
    PER_CLASS_DELTA_PATH,
    PHASE_AWARE_PREDICTIONS_PATH,
    PHASE_AWARE_PER_CLASS_PATH,
    BASELINE_PER_CLASS_PATH,
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Some required files are missing.")

print("All required files found.")
print()

# ----------------------------
# 2. Load data
# ----------------------------

with open(FINAL_SUMMARY_JSON_PATH, "r") as f:
    final_summary = json.load(f)

with open(PAIRWISE_SUMMARY_PATH, "r") as f:
    pairwise_summary = json.load(f)

comparison_df = pd.read_csv(COMPARISON_PATH)
delta_df = pd.read_csv(PER_CLASS_DELTA_PATH)
phase_pred_df = pd.read_csv(PHASE_AWARE_PREDICTIONS_PATH)
phase_per_class_df = pd.read_csv(PHASE_AWARE_PER_CLASS_PATH)
baseline_per_class_df = pd.read_csv(BASELINE_PER_CLASS_PATH)

print("Loaded final summary and result tables.")
print()

saved_figures = []

def save_current_fig(filename, title, description):
    path = FIGURE_DIR / filename
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    saved_figures.append(
        {
            "filename": filename,
            "path": str(path),
            "title": title,
            "description": description,
        }
    )
    return path

# ----------------------------
# 3. Figure 1 — Recognition model comparison
# ----------------------------

comparison_plot_df = comparison_df[
    comparison_df["model"].isin(
        ["baseline_no_phase_onehot", "phase_aware_plus_phase_onehot"]
    )
].copy()

label_map = {
    "baseline_no_phase_onehot": "Baseline\nNo Phase One-Hot",
    "phase_aware_plus_phase_onehot": "Phase-Aware\n+ Phase One-Hot",
}

comparison_plot_df["model_label"] = comparison_plot_df["model"].map(label_map)

metrics = ["test_top1", "test_top5", "test_macro_f1"]
metric_labels = ["Top-1", "Top-5", "Macro F1"]

x = np.arange(len(metric_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

baseline_values = comparison_plot_df[
    comparison_plot_df["model"] == "baseline_no_phase_onehot"
][metrics].iloc[0].values

phase_values = comparison_plot_df[
    comparison_plot_df["model"] == "phase_aware_plus_phase_onehot"
][metrics].iloc[0].values

ax.bar(x - width/2, baseline_values, width, label="Baseline")
ax.bar(x + width/2, phase_values, width, label="Phase-aware")

ax.set_ylabel("Score")
ax.set_title("Recognition Test Performance: Phase-Aware vs Baseline")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.0)
ax.legend()

for i, v in enumerate(baseline_values):
    ax.text(i - width/2, v + 0.015, f"{v:.3f}", ha="center", fontsize=9)

for i, v in enumerate(phase_values):
    ax.text(i + width/2, v + 0.015, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
save_current_fig(
    "fig_01_recognition_model_comparison.png",
    "Recognition Model Comparison",
    "Comparison of baseline TCN-attention recognition model and phase-aware TCN-attention recognition model on the test set.",
)

# ----------------------------
# 4. Figure 2 — Delta improvement
# ----------------------------

delta_row = comparison_df[comparison_df["model"] == "delta_phase_aware_minus_baseline"].iloc[0]

delta_values = [
    float(delta_row["test_top1"]),
    float(delta_row["test_top5"]),
    float(delta_row["test_macro_f1"]),
]

fig, ax = plt.subplots(figsize=(8, 4.5))

bars = ax.bar(metric_labels, delta_values)
ax.axhline(0, linewidth=1)
ax.set_ylabel("Absolute improvement")
ax.set_title("Phase-Aware Improvement Over Baseline")

for i, v in enumerate(delta_values):
    ax.text(i, v + 0.0008, f"+{v:.4f}", ha="center", fontsize=10)

plt.tight_layout()
save_current_fig(
    "fig_02_phase_aware_delta.png",
    "Phase-Aware Improvement",
    "Absolute test-set improvement of the phase-aware model over the no-phase baseline.",
)

# ----------------------------
# 5. Figure 3 — Paired Top-1 and Top-5 comparison
# ----------------------------

top1 = pairwise_summary["top1"]
top5 = pairwise_summary["top5"]

paired_data = pd.DataFrame(
    {
        "category": [
            "Both correct",
            "Phase-aware only",
            "Baseline only",
            "Both wrong",
        ],
        "Top-1": [
            top1["both_correct"],
            top1["phase_only_correct"],
            top1["baseline_only_correct"],
            top1["both_wrong"],
        ],
        "Top-5": [
            top5["both_correct"],
            top5["phase_only_correct"],
            top5["baseline_only_correct"],
            top5["both_wrong"],
        ],
    }
)

x = np.arange(len(paired_data))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(x - width/2, paired_data["Top-1"], width, label="Top-1")
ax.bar(x + width/2, paired_data["Top-5"], width, label="Top-5")

ax.set_ylabel("Number of test samples")
ax.set_title("Paired Prediction Comparison")
ax.set_xticks(x)
ax.set_xticklabels(paired_data["category"], rotation=15, ha="right")
ax.legend()

for i, v in enumerate(paired_data["Top-1"]):
    ax.text(i - width/2, v + 5, str(v), ha="center", fontsize=9)

for i, v in enumerate(paired_data["Top-5"]):
    ax.text(i + width/2, v + 5, str(v), ha="center", fontsize=9)

plt.tight_layout()
save_current_fig(
    "fig_03_paired_prediction_comparison.png",
    "Paired Prediction Comparison",
    "Paired test-set comparison showing which samples were correct under both models, only the phase-aware model, only the baseline, or neither.",
)

# ----------------------------
# 6. Figure 4 — Per-class F1 delta top/bottom
# ----------------------------

top_improved = delta_df.sort_values("delta_f1", ascending=False).head(10).copy()
top_hurt = delta_df.sort_values("delta_f1", ascending=True).head(10).copy()

plot_delta = pd.concat([top_improved, top_hurt], ignore_index=True)
plot_delta = plot_delta.sort_values("delta_f1", ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))

ax.barh(plot_delta["gloss"], plot_delta["delta_f1"])
ax.axvline(0, linewidth=1)
ax.set_xlabel("F1 delta: phase-aware minus baseline")
ax.set_title("Classes Most Helped and Hurt by Phase-Aware Features")

for y, v in enumerate(plot_delta["delta_f1"]):
    label = f"{v:+.3f}"
    if v >= 0:
        ax.text(v + 0.01, y, label, va="center", fontsize=8)
    else:
        ax.text(v - 0.01, y, label, va="center", ha="right", fontsize=8)

plt.tight_layout()
save_current_fig(
    "fig_04_per_class_f1_delta.png",
    "Per-Class F1 Delta",
    "Classes most improved and most hurt by adding ordered phase one-hot features.",
)

# ----------------------------
# 7. Figure 5 — Attention mass by phase
# ----------------------------

attn_cols = ["attn_phase_0", "attn_phase_1", "attn_phase_2", "attn_phase_3"]

if all(c in phase_pred_df.columns for c in attn_cols):
    phase_names = ["Background", "Preparation", "Stroke", "Retraction"]
    attn_means = [phase_pred_df[c].mean() for c in attn_cols]

    fig, ax = plt.subplots(figsize=(8, 4.5))

    ax.bar(phase_names, attn_means)
    ax.set_ylabel("Average attention mass")
    ax.set_title("Average Recognition Attention Mass by Phase")
    ax.set_ylim(0, max(attn_means) + 0.15)

    for i, v in enumerate(attn_means):
        ax.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=10)

    plt.tight_layout()
    save_current_fig(
        "fig_05_attention_mass_by_phase.png",
        "Attention Mass by Phase",
        "Average attention mass assigned by the recognition model to background, preparation, stroke, and retraction regions.",
    )
else:
    print("Attention phase columns not found. Skipping attention mass figure.")

# ----------------------------
# 8. Figure 6 — Phase TCN and recognition summary
# ----------------------------

phase_tcn = final_summary["phase_tcn_summary"]
phase_rec = final_summary["phase_aware_recognition_summary"]
baseline = final_summary["baseline_recognition_summary"]

summary_items = [
    ("Phase TCN\nAccuracy", phase_tcn["test_accuracy"]),
    ("Phase TCN\nMacro F1", phase_tcn["test_macro_f1"]),
    ("Baseline\nTop-1", baseline["test_top1"]),
    ("Phase-Aware\nTop-1", phase_rec["test_top1"]),
    ("Baseline\nMacro F1", baseline["test_macro_f1"]),
    ("Phase-Aware\nMacro F1", phase_rec["test_macro_f1"]),
]

fig, ax = plt.subplots(figsize=(10, 5))

labels = [x[0] for x in summary_items]
values = [x[1] for x in summary_items]

ax.bar(labels, values)
ax.set_ylabel("Score")
ax.set_title("Final Pipeline Performance Summary")
ax.set_ylim(0, 1.0)

for i, v in enumerate(values):
    ax.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=9)

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_current_fig(
    "fig_06_final_pipeline_summary.png",
    "Final Pipeline Summary",
    "Combined summary of the Phase Detection TCN and final recognition models.",
)

# ----------------------------
# 9. Save manifest and captions
# ----------------------------

fig_manifest_df = pd.DataFrame(saved_figures)
fig_manifest_df.to_csv(FIGURE_MANIFEST_PATH, index=False)

captions_lines = ["# Final Figure Captions\n"]

for i, row in fig_manifest_df.iterrows():
    captions_lines.append(f"## Figure {i+1}. {row['title']}\n")
    captions_lines.append(f"**File:** `{row['filename']}`\n")
    captions_lines.append(f"{row['description']}\n")

with open(FIGURE_CAPTIONS_PATH, "w") as f:
    f.write("\n".join(captions_lines))

# ----------------------------
# 10. Print summary
# ----------------------------

print()
print("=" * 80)
print("PHASE 12B OUTPUT SAVED")
print("=" * 80)
print(f"Figure directory:      {FIGURE_DIR}")
print(f"Figure manifest:       {FIGURE_MANIFEST_PATH}")
print(f"Figure captions file:  {FIGURE_CAPTIONS_PATH}")
print()

print("Saved figures:")
for row in saved_figures:
    print(f"- {row['filename']}: {row['title']}")

print()
print("=" * 80)
print("PHASE 12B COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Saved figures list")
print("2. Figure manifest path")
print("3. Figure captions path")
print()
print("Next step: Phase 12C — final report text / methodology write-up.")

In [ ]:
# ============================================================
# PHASE 12C — FINAL REPORT TEXT / METHODOLOGY WRITE-UP
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import json
import pandas as pd

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")

REPORT_DIR = BASE_DIR / "reports" / "final_results"
TEXT_DIR = REPORT_DIR / "report_text"
TEXT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SUMMARY_JSON_PATH = REPORT_DIR / "final_results_summary.json"
FINAL_TABLES_MD_PATH = REPORT_DIR / "final_results_tables.md"
FIGURE_CAPTIONS_PATH = REPORT_DIR / "figures" / "figure_captions.md"

ABSTRACT_PATH = TEXT_DIR / "01_abstract.md"
INTRODUCTION_PATH = TEXT_DIR / "02_introduction.md"
METHODOLOGY_PATH = TEXT_DIR / "03_methodology.md"
EXPERIMENTS_PATH = TEXT_DIR / "04_experiments_and_results.md"
DISCUSSION_PATH = TEXT_DIR / "05_discussion.md"
LIMITATIONS_PATH = TEXT_DIR / "06_limitations_and_future_work.md"
CONCLUSION_PATH = TEXT_DIR / "07_conclusion.md"
FULL_DRAFT_PATH = TEXT_DIR / "final_report_draft.md"

print("=" * 80)
print("PHASE 12C — FINAL REPORT TEXT / METHODOLOGY WRITE-UP")
print("=" * 80)
print(f"Text output directory: {TEXT_DIR}")
print()

required_paths = [
    FINAL_SUMMARY_JSON_PATH,
    FINAL_TABLES_MD_PATH,
    FIGURE_CAPTIONS_PATH,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Missing required files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Some required report files are missing.")

with open(FINAL_SUMMARY_JSON_PATH, "r") as f:
    summary = json.load(f)

with open(FINAL_TABLES_MD_PATH, "r") as f:
    tables_md = f.read()

with open(FIGURE_CAPTIONS_PATH, "r") as f:
    captions_md = f.read()

dataset = summary["dataset_summary"]
phase_tcn = summary["phase_tcn_summary"]
phase_aware = summary["phase_aware_recognition_summary"]
baseline = summary["baseline_recognition_summary"]
delta = summary["delta_phase_aware_minus_baseline"]
paired = summary["paired_analysis"]

# ----------------------------
# 2. Abstract
# ----------------------------

abstract_md = f"""# Abstract

This project presents a phase-aware isolated sign recognition pipeline using pose and hand landmarks extracted from video data. The system first converts sign videos into normalized landmark sequences, computes temporal motion features, generates weak phase pseudo-labels corresponding to preparation, stroke, and retraction phases, and trains a Phase Detection Temporal Convolutional Network (TCN) to estimate frame-level signing phases. These predicted phases are then used to extract active signing segments and to construct phase-aware features for a final sign recognition model.

The final recognition model uses a TCN encoder with attention pooling over active signing frames. The phase-aware version includes ordered phase one-hot features, while the baseline uses the same active-region crop and continuous motion features but removes the phase one-hot representation. On the 100-class recognition task, the phase-aware model achieved a test Top-1 accuracy of {phase_aware["test_top1"]:.4f}, Top-5 accuracy of {phase_aware["test_top5"]:.4f}, and Macro F1 score of {phase_aware["test_macro_f1"]:.4f}. The no-phase baseline achieved a test Top-1 accuracy of {baseline["test_top1"]:.4f}, Top-5 accuracy of {baseline["test_top5"]:.4f}, and Macro F1 score of {baseline["test_macro_f1"]:.4f}. The phase-aware representation therefore produced a small but consistent improvement across all three metrics. However, paired analysis showed that the Top-1 improvement was not statistically strong, so the contribution should be interpreted as a promising phase-aware design rather than a conclusive performance breakthrough.
"""

# ----------------------------
# 3. Introduction
# ----------------------------

introduction_md = f"""# Introduction

Sign language recognition is a challenging computer vision and sequence modeling problem because signs are not defined by single static hand poses alone. A sign usually contains temporal structure, including movement onset, the main expressive motion, and movement offset. In sign linguistics, these temporal regions are commonly understood as preparation, stroke, and retraction phases. Many recognition systems treat a sign video as one continuous sequence and attempt to classify the entire motion directly. This project investigates whether explicitly modeling internal signing phases can improve isolated sign recognition.

The goal of this project is to build a complete phase-aware isolated sign recognition pipeline. Instead of relying only on raw video frames, the system uses MediaPipe-based pose and hand landmarks as a compact skeleton representation. Motion features such as velocity, acceleration, and speed curves are computed from the landmark sequences. These motion cues are used to generate weak phase pseudo-labels, which are then used to train a Phase Detection TCN. The predicted phase structure is finally incorporated into a TCN-attention sign recognition model.

The main research question is:

**Does adding phase-aware temporal information improve isolated sign recognition compared with a comparable model that does not use explicit phase features?**

To answer this, the final phase-aware model is compared against a no-phase baseline using the same active-region crops, same continuous motion features, same train/validation/test split, and same TCN-attention architecture. The only major difference is the inclusion or removal of ordered phase one-hot features.
"""

# ----------------------------
# 4. Methodology
# ----------------------------

methodology_md = f"""# Methodology

## Dataset Preparation

The recognition dataset contains {dataset["recognition_rows"]} verified samples across {dataset["recognition_num_classes"]} sign classes. The final recognition split contains:

- Training samples: {dataset["recognition_split_counts"].get("train", 0)}
- Validation samples: {dataset["recognition_split_counts"].get("val", 0)}
- Test samples: {dataset["recognition_split_counts"].get("test", 0)}

Only clean samples were used for recognition training. Warning segments, fallback-generated segments, overly short phase regions, overly broad active regions, and low-agreement phase predictions were excluded from the strict recognition dataset.

## Landmark Extraction

The system uses MediaPipe pose and hand landmarks as the base representation. For each frame, the extracted representation consists of selected pose landmarks and left/right hand landmarks. These are combined into a 147-dimensional position vector.

The position representation is then normalized to reduce the effect of subject position and scale. This makes the model focus more on relative movement patterns instead of absolute image location.

## Motion Feature Generation

From the normalized landmark sequence, the system computes temporal motion features:

- Position features
- Velocity features
- Acceleration features
- Global speed
- Hand speed
- Phase speed

The continuous feature vector used for temporal modeling has 444 dimensions:

- 147 position features
- 147 velocity features
- 147 acceleration features
- 1 global speed feature
- 1 hand speed feature
- 1 phase speed feature

## Weak Phase Pseudo-Label Generation

The project does not use manually annotated linguistic phase labels. Instead, weak pseudo-labels are generated from motion curves. The phase pseudo-labels divide each sequence into four frame-level classes:

- 0: background
- 1: preparation
- 2: stroke
- 3: retraction

The pseudo-labels are derived from the active motion region and the dominant motion peak. The goal is not to create perfect linguistic annotations, but to create useful weak supervision for a phase detection model.

## Phase Detection TCN

A Phase Detection TCN is trained to predict frame-level phase labels from the 444-dimensional motion feature sequence. The model predicts one of four phase classes for every frame.

The Phase TCN achieved:

- Test accuracy: {phase_tcn["test_accuracy"]:.4f}
- Test Macro F1: {phase_tcn["test_macro_f1"]:.4f}

This result shows that the model can learn the motion-derived phase structure with high agreement.

## Phase-Aware Segment Extraction

The trained Phase TCN is used to generate phase predictions for each sequence. These predictions are smoothed and converted into ordered segment boundaries:

- active signing region
- preparation region
- stroke region
- retraction region

The final recognition model uses only the active signing region. This reduces background-heavy frames and focuses the recognition model on the meaningful signing motion.

The phase segment generation produced:

- OK segments: {dataset["segment_status_counts"].get("ok", 0)}
- Warning segments: {dataset["segment_status_counts"].get("warning", 0)}

Warning segments were excluded from strict recognition training.

## Recognition TCN with Attention

The final recognition model uses a TCN encoder followed by attention pooling. The input is the active-region crop of the sequence.

The baseline model uses only the 444 continuous motion features.

The phase-aware model uses:

- 444 continuous motion features
- 4 ordered phase one-hot features

This gives a final phase-aware input size of 448 dimensions per frame.

The attention layer learns to assign importance weights to frames inside the active signing region. The pooled sequence representation is passed to a classifier to predict one of 100 sign classes.
"""

# ----------------------------
# 5. Experiments and Results
# ----------------------------

experiments_md = f"""# Experiments and Results

## Phase Detection Experiment

The Phase Detection TCN was evaluated on the clean test set. It achieved:

| Metric | Value |
|---|---:|
| Test accuracy | {phase_tcn["test_accuracy"]:.4f} |
| Test Macro F1 | {phase_tcn["test_macro_f1"]:.4f} |
| Test sequences | {phase_tcn["num_test_sequences"]} |
| Test frames | {phase_tcn["num_test_frames"]} |

This confirms that the motion-derived weak phase labels can be learned reliably by a temporal convolutional model.

## Recognition Experiment

Two recognition models were compared:

1. **Baseline TCN-Attention model:** active-region crop + 444 continuous features.
2. **Phase-aware TCN-Attention model:** active-region crop + 444 continuous features + 4 ordered phase one-hot features.

| Model | Test Top-1 | Test Top-5 | Test Macro F1 |
|---|---:|---:|---:|
| Baseline without phase one-hot | {baseline["test_top1"]:.4f} | {baseline["test_top5"]:.4f} | {baseline["test_macro_f1"]:.4f} |
| Phase-aware with phase one-hot | {phase_aware["test_top1"]:.4f} | {phase_aware["test_top5"]:.4f} | {phase_aware["test_macro_f1"]:.4f} |
| Delta | {delta["delta_test_top1"]:.4f} | {delta["delta_test_top5"]:.4f} | {delta["delta_test_macro_f1"]:.4f} |

The phase-aware model improved all three test metrics:

- Top-1 improvement: {delta["delta_test_top1"]:.4f}
- Top-5 improvement: {delta["delta_test_top5"]:.4f}
- Macro F1 improvement: {delta["delta_test_macro_f1"]:.4f}

## Paired Error Analysis

A paired comparison was performed to check whether the same test samples were corrected or worsened by the phase-aware model.

| Category | Count |
|---|---:|
| Both models correct | {paired["top1"]["both_correct"]} |
| Phase-aware only correct | {paired["top1"]["phase_only_correct"]} |
| Baseline only correct | {paired["top1"]["baseline_only_correct"]} |
| Both models wrong | {paired["top1"]["both_wrong"]} |
| Net phase-aware gain | {paired["top1"]["net_phase_gain"]} |

The phase-aware model fixed {paired["top1"]["phase_only_correct"]} samples that the baseline missed, but it also hurt {paired["top1"]["baseline_only_correct"]} samples that the baseline predicted correctly. The net gain was therefore {paired["top1"]["net_phase_gain"]} test samples.

The approximate McNemar p-value was {paired["top1"]["mcnemar_p_approx"]:.4f}. This means the paired Top-1 improvement was not statistically strong.

## Attention Analysis

The recognition attention mechanism showed that the model does not only focus on the stroke phase. The average attention mass was distributed across preparation, stroke, and retraction. This suggests that the temporal context around the main sign motion contributes to recognition.
"""

# ----------------------------
# 6. Discussion
# ----------------------------

discussion_md = f"""# Discussion

The final results show that the phase-aware representation provides a small but consistent improvement over the no-phase baseline. The phase-aware model improved Top-1 accuracy, Top-5 accuracy, and Macro F1. This supports the idea that explicit temporal phase information can help isolated sign recognition.

However, the improvement is modest. The paired analysis showed that the phase-aware model fixed 44 samples that the baseline missed, but also caused 37 samples to become incorrect. The net Top-1 gain was only 7 test samples out of 777. The approximate McNemar p-value was {paired["top1"]["mcnemar_p_approx"]:.4f}, so the improvement should not be described as statistically significant.

The results suggest that phase-aware features are useful for some signs but harmful for others. Some classes improved substantially when phase information was added, while other classes lost performance. This may happen because weak phase pseudo-labels do not always correspond perfectly to the linguistically meaningful parts of a sign. In some cases, the model may over-rely on phase boundaries that are noisy or not discriminative for that class.

The strongest part of the project is the complete end-to-end pipeline. The system successfully moves from landmark extraction to motion features, phase pseudo-labeling, phase detection, segment extraction, recognition, baseline comparison, and paired error analysis. This gives the project a clear experimental structure and makes the final conclusion defensible.
"""

# ----------------------------
# 7. Limitations and Future Work
# ----------------------------

limitations_md = f"""# Limitations and Future Work

## Limitations

1. **Weak phase labels:** The preparation, stroke, and retraction labels were generated from motion curves rather than manually annotated by sign language experts. Therefore, they should be described as motion-derived weak pseudo-labels, not ground-truth linguistic phase labels.

2. **Small number of samples per class:** The final recognition dataset contains 100 classes but only {dataset["recognition_split_counts"].get("train", 0)} training samples. This creates a difficult learning problem and increases the risk of overfitting.

3. **Generalization gap:** The validation performance was higher than the test performance. This suggests that the model learned the training/validation distribution well but had more difficulty generalizing to the test set.

4. **Phase features are not always helpful:** Some classes improved with phase-aware features, but others got worse. This means phase-aware modeling is promising but not universally beneficial in the current form.

5. **No real-time deployment yet:** The current project is evaluated offline using processed sequences. Real-time webcam inference remains a future deployment step.

## Future Work

1. **Increase dataset size:** More samples per class would likely improve recognition stability and reduce overfitting.

2. **Manual or semi-automatic phase correction:** Human-corrected phase labels could improve the Phase TCN and make phase boundaries more meaningful.

3. **Signer-independent evaluation:** If signer metadata is available, future experiments should use signer-independent train/test splits to better measure generalization.

4. **Stronger sequence models:** The TCN-attention model can be compared with BiLSTM, Transformer encoder, ST-GCN, or hybrid graph-temporal models.

5. **Real-time webcam demo:** The model can be adapted to real-time inference using a rolling landmark buffer.

6. **YOLO-assisted signer localization:** YOLO can be added before MediaPipe in the live pipeline to localize the signer and crop the person region, especially in cluttered scenes.
"""

# ----------------------------
# 8. Conclusion
# ----------------------------

conclusion_md = f"""# Conclusion

This project developed a complete phase-aware isolated sign recognition pipeline using MediaPipe landmarks, motion-derived weak phase pseudo-labels, a Phase Detection TCN, phase-aware segment extraction, and a TCN-attention recognition model. The Phase TCN successfully learned the weak phase labels, achieving a test accuracy of {phase_tcn["test_accuracy"]:.4f} and a test Macro F1 of {phase_tcn["test_macro_f1"]:.4f}.

For the final 100-class recognition task, the phase-aware model achieved a test Top-1 accuracy of {phase_aware["test_top1"]:.4f}, Top-5 accuracy of {phase_aware["test_top5"]:.4f}, and Macro F1 of {phase_aware["test_macro_f1"]:.4f}. The no-phase baseline achieved a test Top-1 accuracy of {baseline["test_top1"]:.4f}, Top-5 accuracy of {baseline["test_top5"]:.4f}, and Macro F1 of {baseline["test_macro_f1"]:.4f}. The phase-aware model therefore produced a small improvement across all metrics.

The paired comparison showed that this improvement was modest and not statistically strong. Therefore, the final contribution should be framed carefully: the project demonstrates that phase-aware modeling is feasible and can slightly improve recognition performance, but stronger data, better phase annotations, and more robust architectures are needed to prove a larger and more conclusive benefit.
"""

# ----------------------------
# 9. Write files
# ----------------------------

sections = [
    (ABSTRACT_PATH, abstract_md),
    (INTRODUCTION_PATH, introduction_md),
    (METHODOLOGY_PATH, methodology_md),
    (EXPERIMENTS_PATH, experiments_md),
    (DISCUSSION_PATH, discussion_md),
    (LIMITATIONS_PATH, limitations_md),
    (CONCLUSION_PATH, conclusion_md),
]

for path, content in sections:
    with open(path, "w") as f:
        f.write(content)

full_draft = "\n\n".join([
    abstract_md,
    introduction_md,
    methodology_md,
    experiments_md,
    discussion_md,
    limitations_md,
    conclusion_md,
    "\n\n# Result Tables\n\n",
    tables_md,
    "\n\n# Figure Captions\n\n",
    captions_md,
])

with open(FULL_DRAFT_PATH, "w") as f:
    f.write(full_draft)

# ----------------------------
# 10. Print summary
# ----------------------------

print("=" * 80)
print("PHASE 12C OUTPUT SAVED")
print("=" * 80)
print(f"Abstract:                 {ABSTRACT_PATH}")
print(f"Introduction:             {INTRODUCTION_PATH}")
print(f"Methodology:              {METHODOLOGY_PATH}")
print(f"Experiments and Results:  {EXPERIMENTS_PATH}")
print(f"Discussion:               {DISCUSSION_PATH}")
print(f"Limitations/Future Work:  {LIMITATIONS_PATH}")
print(f"Conclusion:               {CONCLUSION_PATH}")
print(f"Full report draft:        {FULL_DRAFT_PATH}")
print()

print("=" * 80)
print("REPORT CLAIM CHECK")
print("=" * 80)
print("Use this claim:")
print("The phase-aware model produced a small but consistent improvement over the no-phase baseline.")
print()
print("Do NOT claim:")
print("The phase-aware model significantly outperformed the baseline.")
print()

print("=" * 80)
print("KEY FINAL NUMBERS")
print("=" * 80)
print(f"Phase TCN test accuracy:       {phase_tcn['test_accuracy']:.4f}")
print(f"Phase TCN test macro F1:       {phase_tcn['test_macro_f1']:.4f}")
print()
print(f"Baseline test Top-1:           {baseline['test_top1']:.4f}")
print(f"Baseline test Top-5:           {baseline['test_top5']:.4f}")
print(f"Baseline test Macro F1:        {baseline['test_macro_f1']:.4f}")
print()
print(f"Phase-aware test Top-1:        {phase_aware['test_top1']:.4f}")
print(f"Phase-aware test Top-5:        {phase_aware['test_top5']:.4f}")
print(f"Phase-aware test Macro F1:     {phase_aware['test_macro_f1']:.4f}")
print()
print(f"Delta Top-1:                   {delta['delta_test_top1']:.4f}")
print(f"Delta Top-5:                   {delta['delta_test_top5']:.4f}")
print(f"Delta Macro F1:                {delta['delta_test_macro_f1']:.4f}")
print(f"McNemar p approx:              {paired['top1']['mcnemar_p_approx']:.4f}")
print()

print("=" * 80)
print("PHASE 12C COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Saved report text paths")
print("2. Key final numbers")
print("3. Report claim check")
print()
print("Next step: Phase 13A — offline inference demo on saved samples.")

In [ ]:
# ============================================================
# PHASE 13A — OFFLINE INFERENCE DEMO ON SAVED TEST SAMPLES
# Run this cell only, then paste the output back.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# 1. Paths
# ----------------------------

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
MANIFEST_DIR = BASE_DIR / "manifests"

RECOGNITION_DATASET_MANIFEST_PATH = MANIFEST_DIR / "recognition_dataset_verified_fixed.csv"
RECOGNITION_LABEL_MAP_PATH = MANIFEST_DIR / "recognition_label_map_with_split_counts.csv"

MODEL_DIR = BASE_DIR / "models" / "recognition_tcn_attention"
BEST_MODEL_PATH = MODEL_DIR / "recognition_tcn_attention_best.pt"

DEMO_DIR = BASE_DIR / "reports" / "final_results" / "offline_demo"
DEMO_PLOT_DIR = DEMO_DIR / "attention_plots"

DEMO_DIR.mkdir(parents=True, exist_ok=True)
DEMO_PLOT_DIR.mkdir(parents=True, exist_ok=True)

DEMO_PREDICTIONS_PATH = DEMO_DIR / "offline_demo_predictions.csv"
DEMO_SUMMARY_PATH = DEMO_DIR / "offline_demo_summary.md"
DEMO_CONFIG_PATH = DEMO_DIR / "offline_demo_config.json"

print("=" * 80)
print("PHASE 13A — OFFLINE INFERENCE DEMO ON SAVED TEST SAMPLES")
print("=" * 80)

for p in [
    RECOGNITION_DATASET_MANIFEST_PATH,
    RECOGNITION_LABEL_MAP_PATH,
    BEST_MODEL_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print(f"Dataset manifest: {RECOGNITION_DATASET_MANIFEST_PATH}")
print(f"Label map:        {RECOGNITION_LABEL_MAP_PATH}")
print(f"Best model:       {BEST_MODEL_PATH}")
print(f"Demo output dir:  {DEMO_DIR}")
print()

# ----------------------------
# 2. Load dataset, label map, checkpoint
# ----------------------------

df = pd.read_csv(RECOGNITION_DATASET_MANIFEST_PATH)
df = df[df["verify_status"] == "ok"].copy()

label_map_df = pd.read_csv(RECOGNITION_LABEL_MAP_PATH)

id_to_gloss = {
    int(row["rec_label_id"]): str(row["gloss"])
    for _, row in label_map_df.iterrows()
}

gloss_to_id = {
    str(row["gloss"]): int(row["rec_label_id"])
    for _, row in label_map_df.iterrows()
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)

CONFIG = checkpoint["config"]
feature_mean = checkpoint["feature_mean"].detach().cpu().numpy().astype(np.float32)
feature_std = checkpoint["feature_std"].detach().cpu().numpy().astype(np.float32)

print("Loaded checkpoint:")
print(f"epoch:        {checkpoint['epoch']}")
print(f"val_top1:     {checkpoint['val_top1']:.4f}")
print(f"val_top5:     {checkpoint['val_top5']:.4f}")
print(f"val_macro_f1: {checkpoint['val_macro_f1']:.4f}")
print()

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

print("Rows by rec_split:")
print(df["rec_split"].value_counts(dropna=False))
print()

# ----------------------------
# 3. Model definition
# ----------------------------

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)

class SafeMaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        scores = self.attn(h.float()).squeeze(-1)
        mask = mask.bool()

        scores = scores.masked_fill(~mask, -1e4)

        attn_weights = torch.softmax(scores, dim=1)
        attn_weights = attn_weights.masked_fill(~mask, 0.0)

        pooled = torch.sum(h.float() * attn_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights

class RecognitionTCNAttentionSafe(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = SafeMaskedAttentionPooling(
            hidden_dim,
            dropout=attention_dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        h = x.transpose(1, 2)
        h = self.tcn(h)
        h = h.transpose(1, 2)

        pooled, attn_weights = self.attention_pool(h, mask)
        logits = self.classifier(pooled.float())

        return logits, attn_weights

model = RecognitionTCNAttentionSafe(
    input_dim=CONFIG["input_dim"],
    num_classes=CONFIG["num_classes"],
    hidden_dim=CONFIG["hidden_dim"],
    num_blocks=CONFIG["num_blocks"],
    kernel_size=CONFIG["kernel_size"],
    dropout=CONFIG["dropout"],
    attention_dropout=CONFIG["attention_dropout"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully.")
print()

# ----------------------------
# 4. Feature loading for one saved sample
# ----------------------------

def load_saved_sample_features(row):
    """
    Loads one already-processed sample from:
      motion_path + phase_segment_path

    Returns:
      X_final: (T_active, 448)
      y: true class id
      phase_crop: ordered phase labels inside active crop
      phase_speed_crop: phase speed inside active crop
    """
    motion_path = Path(str(row["motion_path"]))
    segment_path = Path(str(row["phase_segment_path"]))

    if not motion_path.exists():
        raise FileNotFoundError(f"Missing motion file: {motion_path}")

    if not segment_path.exists():
        raise FileNotFoundError(f"Missing phase segment file: {segment_path}")

    with np.load(motion_path, allow_pickle=False) as m:
        positions = np.asarray(m["positions"], dtype=np.float32)
        velocity = np.asarray(m["velocity"], dtype=np.float32)
        acceleration = np.asarray(m["acceleration"], dtype=np.float32)
        global_speed = np.asarray(m["global_speed"], dtype=np.float32).reshape(-1, 1)
        hand_speed = np.asarray(m["hand_speed"], dtype=np.float32).reshape(-1, 1)

    with np.load(segment_path, allow_pickle=False) as s:
        ordered_phase = np.asarray(s["ordered_segment_labels"], dtype=np.int64)
        phase_speed = np.asarray(s["phase_speed"], dtype=np.float32).reshape(-1, 1)
        active_start = int(np.asarray(s["active_start"]).item())
        active_end = int(np.asarray(s["active_end"]).item())

    X_cont = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X_cont[~np.isfinite(X_cont)] = 0.0

    # Active crop.
    X_cont = X_cont[active_start:active_end + 1]
    phase_crop = ordered_phase[active_start:active_end + 1]
    phase_speed_crop = phase_speed[active_start:active_end + 1].reshape(-1)

    phase_crop = np.clip(phase_crop, 0, 3).astype(np.int64)

    # Standardize continuous features.
    X_cont = (X_cont - feature_mean) / feature_std
    X_cont[~np.isfinite(X_cont)] = 0.0

    # Add phase one-hot.
    phase_onehot = np.eye(4, dtype=np.float32)[phase_crop]

    X_final = np.concatenate([X_cont, phase_onehot], axis=1).astype(np.float32)
    X_final[~np.isfinite(X_final)] = 0.0

    if X_final.shape[1] != CONFIG["input_dim"]:
        raise ValueError(f"Expected input dim {CONFIG['input_dim']}, got {X_final.shape[1]}")

    y = int(row["rec_label_id"])

    return X_final, y, phase_crop, phase_speed_crop

def predict_saved_sample(row, top_k=5):
    X, y, phase_crop, phase_speed_crop = load_saved_sample_features(row)

    xt = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
    mask = torch.ones(1, X.shape[0], dtype=torch.bool).to(device)

    with torch.no_grad():
        logits, attn = model(xt, mask)
        probs = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
        attn = attn.squeeze(0).detach().cpu().numpy()

    top_ids = np.argsort(probs)[-top_k:][::-1].tolist()

    pred_id = int(top_ids[0])
    true_id = int(y)

    result = {
        "video_stem": str(row["video_stem"]),
        "true_id": true_id,
        "true_gloss": id_to_gloss.get(true_id, str(true_id)),
        "pred_id": pred_id,
        "pred_gloss": id_to_gloss.get(pred_id, str(pred_id)),
        "correct": bool(pred_id == true_id),
        "top5_correct": bool(true_id in top_ids),
        "top_ids": top_ids,
        "top_glosses": [id_to_gloss.get(i, str(i)) for i in top_ids],
        "top_probs": [float(probs[i]) for i in top_ids],
        "true_prob": float(probs[true_id]),
        "T_active": int(X.shape[0]),
        "phase_attention_background": float(attn[phase_crop == 0].sum()) if np.any(phase_crop == 0) else 0.0,
        "phase_attention_preparation": float(attn[phase_crop == 1].sum()) if np.any(phase_crop == 1) else 0.0,
        "phase_attention_stroke": float(attn[phase_crop == 2].sum()) if np.any(phase_crop == 2) else 0.0,
        "phase_attention_retraction": float(attn[phase_crop == 3].sum()) if np.any(phase_crop == 3) else 0.0,
        "motion_path": str(row["motion_path"]),
        "phase_segment_path": str(row["phase_segment_path"]),
    }

    aux = {
        "X": X,
        "phase_crop": phase_crop,
        "phase_speed": phase_speed_crop,
        "attention": attn,
        "probs": probs,
    }

    return result, aux

# ----------------------------
# 5. Plot helper
# ----------------------------

PHASE_COLORS = {
    0: "lightgray",
    1: "gold",
    2: "tomato",
    3: "skyblue",
}

PHASE_NAMES = {
    0: "background",
    1: "preparation",
    2: "stroke",
    3: "retraction",
}

def label_segments(labels):
    labels = np.asarray(labels, dtype=np.int64)
    if len(labels) == 0:
        return []

    segments = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        val = int(labels[i])
        if val != current:
            segments.append((current, start, i - 1))
            start = i
            current = val

    segments.append((current, start, len(labels) - 1))
    return segments

def plot_demo_prediction(result, aux, show=False):
    phase_crop = aux["phase_crop"]
    phase_speed = aux["phase_speed"]
    attention = aux["attention"]

    T = len(phase_crop)
    frames = np.arange(T)

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # Panel 1: phase speed with phase regions.
    ax = axes[0]
    for phase_id, start, end in label_segments(phase_crop):
        ax.axvspan(
            start,
            end + 1,
            color=PHASE_COLORS.get(phase_id, "white"),
            alpha=0.35,
            label=PHASE_NAMES.get(phase_id, str(phase_id)),
        )

    ax.plot(frames, phase_speed, linewidth=2.0, label="phase_speed")
    ax.set_ylabel("phase_speed")
    ax.set_title("Active signing crop with ordered phase regions")

    handles, labels_legend = ax.get_legend_handles_labels()
    unique = {}
    for h, l in zip(handles, labels_legend):
        if l not in unique:
            unique[l] = h
    ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize=8)

    # Panel 2: attention.
    ax = axes[1]
    for phase_id, start, end in label_segments(phase_crop):
        ax.axvspan(
            start,
            end + 1,
            color=PHASE_COLORS.get(phase_id, "white"),
            alpha=0.25,
        )

    ax.plot(frames, attention, linewidth=2.0, label="attention")
    ax.set_ylabel("attention")
    ax.set_xlabel("Active-crop frame")
    ax.set_title("Recognition attention over active frames")
    ax.legend(loc="upper right")

    top5_text = ", ".join(
        [
            f"{g} ({p:.2f})"
            for g, p in zip(result["top_glosses"], result["top_probs"])
        ]
    )

    fig.suptitle(
        f"true={result['true_gloss']} | pred={result['pred_gloss']} | "
        f"correct={result['correct']} | video={result['video_stem']}\n"
        f"Top-5: {top5_text}",
        y=1.03,
    )

    plt.tight_layout()

    safe_name = str(result["video_stem"]).replace("/", "_").replace("\\", "_").replace(" ", "_")
    save_path = DEMO_PLOT_DIR / f"offline_demo__{safe_name}.png"

    plt.savefig(save_path, dpi=160, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close()

    return save_path

# ----------------------------
# 6. Select demo samples
# ----------------------------

test_df = df[df["rec_split"] == "test"].copy()

# Use a reproducible random test subset.
N_DEMO = 15
demo_df = test_df.sample(n=min(N_DEMO, len(test_df)), random_state=13).reset_index(drop=True)

print(f"Selected demo samples: {len(demo_df)}")
print()

# ----------------------------
# 7. Run demo predictions
# ----------------------------

records = []
plot_paths = []

print("Running offline inference demo...")
print("First 6 plots will display below.")
print()

for i, row in demo_df.iterrows():
    result, aux = predict_saved_sample(row, top_k=5)

    show_now = i < 6
    plot_path = plot_demo_prediction(result, aux, show=show_now)

    result["attention_plot_path"] = str(plot_path)

    records.append(result)
    plot_paths.append(str(plot_path))

demo_pred_df = pd.DataFrame(records)
demo_pred_df.to_csv(DEMO_PREDICTIONS_PATH, index=False)

# ----------------------------
# 8. Save summary
# ----------------------------

demo_top1 = float(demo_pred_df["correct"].mean()) if len(demo_pred_df) else 0.0
demo_top5 = float(demo_pred_df["top5_correct"].mean()) if len(demo_pred_df) else 0.0

summary_md = f"""# Offline Inference Demo Summary

## Purpose

This demo loads the trained phase-aware Recognition TCN + Attention model from disk and runs inference on saved test samples. It does not retrain any model.

## Model

- Model: Phase-aware Recognition TCN + Attention
- Checkpoint: `{BEST_MODEL_PATH}`
- Input: active-region cropped sequence with 448 features per frame
  - 444 continuous motion/keypoint features
  - 4 ordered phase one-hot features

## Demo Results

- Demo samples: {len(demo_pred_df)}
- Demo Top-1 accuracy: {demo_top1:.4f}
- Demo Top-5 accuracy: {demo_top5:.4f}

## Output Files

- Predictions CSV: `{DEMO_PREDICTIONS_PATH}`
- Attention plots directory: `{DEMO_PLOT_DIR}`

## Note

This is an offline saved-sample demo. It uses precomputed motion features and precomputed phase segment files. Real-time webcam testing requires a live frame-to-keypoint pipeline, rolling sequence buffer, online motion feature calculation, Phase TCN inference, and Recognition TCN inference.
"""

with open(DEMO_SUMMARY_PATH, "w") as f:
    f.write(summary_md)

demo_config = {
    "demo_samples": int(len(demo_pred_df)),
    "checkpoint": str(BEST_MODEL_PATH),
    "output_predictions": str(DEMO_PREDICTIONS_PATH),
    "output_summary": str(DEMO_SUMMARY_PATH),
    "plot_dir": str(DEMO_PLOT_DIR),
    "top1": demo_top1,
    "top5": demo_top5,
}

with open(DEMO_CONFIG_PATH, "w") as f:
    json.dump(demo_config, f, indent=2)

# ----------------------------
# 9. Print output
# ----------------------------

print()
print("=" * 80)
print("OFFLINE DEMO RESULTS")
print("=" * 80)
print(f"Demo samples: {len(demo_pred_df)}")
print(f"Demo Top-1:   {demo_top1:.4f}")
print(f"Demo Top-5:   {demo_top5:.4f}")
print()

print("Prediction table:")
display(
    demo_pred_df[
        [
            "video_stem",
            "true_gloss",
            "pred_gloss",
            "correct",
            "top5_correct",
            "top_glosses",
            "top_probs",
            "true_prob",
            "T_active",
            "phase_attention_preparation",
            "phase_attention_stroke",
            "phase_attention_retraction",
            "attention_plot_path",
        ]
    ]
)

print()
print("=" * 80)
print("PHASE 13A OUTPUT SAVED")
print("=" * 80)
print(f"Demo predictions: {DEMO_PREDICTIONS_PATH}")
print(f"Demo summary:     {DEMO_SUMMARY_PATH}")
print(f"Demo config:      {DEMO_CONFIG_PATH}")
print(f"Demo plots:       {DEMO_PLOT_DIR}")
print()

print("First saved plot paths:")
for p in plot_paths[:10]:
    print(p)

print()
print("=" * 80)
print("PHASE 13A COMPLETE — WHAT TO SEND BACK")
print("=" * 80)
print("Paste:")
print("1. Demo Top-1 and Top-5")
print("2. Prediction table")
print("3. Output paths")
print()
print("Next step: Phase 13B — real-time webcam prototype without YOLO first.")

In [ ]:
# ============================================================
# PHASE 13B — GENERATE REAL-TIME WEBCAM DEMO SCRIPT WITHOUT YOLO
# Run this cell in Colab. It writes the local webcam demo script.
# You will run the generated script locally in VS Code / PowerShell.
# ============================================================

from pathlib import Path
import textwrap
import json

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
REALTIME_DIR = BASE_DIR / "realtime_demo"
REALTIME_DIR.mkdir(parents=True, exist_ok=True)

SCRIPT_PATH = REALTIME_DIR / "webcam_realtime_phase_aware_demo.py"
REQ_PATH = REALTIME_DIR / "requirements_realtime.txt"
README_PATH = REALTIME_DIR / "README_realtime_demo.md"
CONFIG_PATH = REALTIME_DIR / "realtime_demo_file_manifest.json"

print("=" * 80)
print("PHASE 13B — GENERATE REAL-TIME WEBCAM DEMO SCRIPT WITHOUT YOLO")
print("=" * 80)
print(f"Realtime demo directory: {REALTIME_DIR}")
print()

script_code = r'''
import argparse
import time
from pathlib import Path
from collections import deque

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import mediapipe as mp
except ImportError as e:
    raise ImportError(
        "mediapipe is not installed. Run: pip install mediapipe"
    ) from e


# ============================================================
# Model definitions
# ============================================================

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)

        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.dropout2(out)

        residual = x if self.downsample is None else self.downsample(x)

        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)


class PhaseTCN(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes=4,
        hidden_dim=192,
        num_blocks=5,
        kernel_size=5,
        dropout=0.20,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.classifier = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.transpose(1, 2)
        h = self.tcn(x)
        logits = self.classifier(h)
        logits = logits.transpose(1, 2)
        return logits


class SafeMaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        scores = self.attn(h.float()).squeeze(-1)
        mask = mask.bool()

        scores = scores.masked_fill(~mask, -1e4)

        attn_weights = torch.softmax(scores, dim=1)
        attn_weights = attn_weights.masked_fill(~mask, 0.0)

        pooled = torch.sum(h.float() * attn_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights


class RecognitionTCNAttentionSafe(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = SafeMaskedAttentionPooling(
            hidden_dim,
            dropout=attention_dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        h = x.transpose(1, 2)
        h = self.tcn(h)
        h = h.transpose(1, 2)

        pooled, attn_weights = self.attention_pool(h, mask)
        logits = self.classifier(pooled.float())

        return logits, attn_weights


# ============================================================
# Keypoint extraction and feature utilities
# ============================================================

POSE_IDXS = [0, 11, 12, 13, 14, 15, 16]
# 7 pose landmarks + 21 left hand + 21 right hand = 49 landmarks
# 49 * 3 = 147 features


def landmark_to_xyz(landmark):
    return np.array([landmark.x, landmark.y, landmark.z], dtype=np.float32)


def extract_147_positions_from_mediapipe(results):
    """
    Returns one normalized 147D vector.

    Assumption:
    - 7 pose landmarks: nose, shoulders, elbows, wrists
    - 21 left hand landmarks
    - 21 right hand landmarks

    Important:
    This must match the training-time landmark order as closely as possible.
    If your earlier Phase 2 used a different landmark order, edit POSE_IDXS/order here.
    """
    points = []

    # Pose landmarks.
    if results.pose_landmarks is not None:
        pose_lms = results.pose_landmarks.landmark

        for idx in POSE_IDXS:
            points.append(landmark_to_xyz(pose_lms[idx]))

        left_shoulder = landmark_to_xyz(pose_lms[11])
        right_shoulder = landmark_to_xyz(pose_lms[12])
        origin = (left_shoulder + right_shoulder) / 2.0
        scale = float(np.linalg.norm(left_shoulder - right_shoulder))
    else:
        for _ in POSE_IDXS:
            points.append(np.zeros(3, dtype=np.float32))

        origin = np.array([0.5, 0.5, 0.0], dtype=np.float32)
        scale = 1.0

    if scale < 1e-6:
        scale = 1.0

    # Left hand landmarks.
    if results.left_hand_landmarks is not None:
        for lm in results.left_hand_landmarks.landmark:
            points.append(landmark_to_xyz(lm))
    else:
        for _ in range(21):
            points.append(np.zeros(3, dtype=np.float32))

    # Right hand landmarks.
    if results.right_hand_landmarks is not None:
        for lm in results.right_hand_landmarks.landmark:
            points.append(landmark_to_xyz(lm))
    else:
        for _ in range(21):
            points.append(np.zeros(3, dtype=np.float32))

    arr = np.stack(points, axis=0).astype(np.float32)  # (49, 3)

    # Normalize around shoulder center and shoulder scale.
    # Missing hand points remain near zero after this; this is a prototype choice.
    missing_mask = np.linalg.norm(arr, axis=1) < 1e-8
    arr_norm = (arr - origin.reshape(1, 3)) / scale
    arr_norm[missing_mask] = 0.0

    out = arr_norm.reshape(-1).astype(np.float32)

    if out.shape[0] != 147:
        raise ValueError(f"Expected 147D position vector, got {out.shape[0]}")

    out[~np.isfinite(out)] = 0.0
    return out


def compute_motion_features_from_positions(positions_seq):
    """
    positions_seq: (T, 147)

    Returns:
      X_cont: (T, 444)
    """
    positions = np.asarray(positions_seq, dtype=np.float32)

    T = positions.shape[0]

    if T < 2:
        velocity = np.zeros_like(positions)
        acceleration = np.zeros_like(positions)
    else:
        velocity = np.zeros_like(positions)
        velocity[1:] = positions[1:] - positions[:-1]

        acceleration = np.zeros_like(positions)
        acceleration[1:] = velocity[1:] - velocity[:-1]

    # Global speed from full 147D velocity.
    global_speed = np.linalg.norm(velocity, axis=1, keepdims=True).astype(np.float32)

    # Hand speed from the hand part only:
    # pose = first 7 landmarks = first 21 values
    # hands = remaining 126 values
    hand_velocity = velocity[:, 21:]
    hand_speed = np.linalg.norm(hand_velocity, axis=1, keepdims=True).astype(np.float32)

    # Phase speed: blended normalized motion signal.
    raw_phase_speed = 0.5 * global_speed + 0.5 * hand_speed
    max_val = float(np.max(raw_phase_speed))

    if max_val > 1e-8:
        phase_speed = raw_phase_speed / max_val
    else:
        phase_speed = raw_phase_speed

    X_cont = np.concatenate(
        [
            positions,
            velocity,
            acceleration,
            global_speed,
            hand_speed,
            phase_speed,
        ],
        axis=1,
    ).astype(np.float32)

    X_cont[~np.isfinite(X_cont)] = 0.0

    if X_cont.shape[1] != 444:
        raise ValueError(f"Expected 444D continuous features, got {X_cont.shape[1]}")

    return X_cont, phase_speed.reshape(-1)


def majority_filter_1d(labels, window=5):
    labels = np.asarray(labels, dtype=np.int64)
    T = len(labels)

    if T == 0 or window <= 1:
        return labels.copy()

    if window % 2 == 0:
        window += 1

    pad = window // 2
    padded = np.pad(labels, (pad, pad), mode="edge")
    out = labels.copy()

    for i in range(T):
        vals = padded[i:i + window]
        counts = np.bincount(vals, minlength=4)
        out[i] = int(np.argmax(counts))

    return out.astype(np.int64)


def get_segments(mask):
    mask = np.asarray(mask, dtype=bool)
    segments = []

    if len(mask) == 0:
        return segments

    in_seg = False
    start = 0

    for i, val in enumerate(mask):
        if val and not in_seg:
            start = i
            in_seg = True
        elif not val and in_seg:
            segments.append((start, i - 1))
            in_seg = False

    if in_seg:
        segments.append((start, len(mask) - 1))

    return segments


def build_ordered_phase_labels_from_pred(pred_labels):
    """
    Converts raw/smoothed phase predictions to one active ordered sequence.

    Output labels:
      0 background
      1 preparation
      2 stroke
      3 retraction
    """
    pred_labels = np.asarray(pred_labels, dtype=np.int64)
    T = len(pred_labels)

    ordered = np.zeros(T, dtype=np.int64)

    active_mask = pred_labels != 0
    active_segments = get_segments(active_mask)

    if len(active_segments) == 0:
        # fallback: use the whole sequence
        active_start, active_end = 0, T - 1
    else:
        active_start = active_segments[0][0]
        active_end = active_segments[-1][1]

    active_len = active_end - active_start + 1

    if active_len < 3:
        return ordered, active_start, active_end

    stroke_mask = np.zeros(T, dtype=bool)
    stroke_mask[active_start:active_end + 1] = pred_labels[active_start:active_end + 1] == 2
    stroke_segments = get_segments(stroke_mask)

    if len(stroke_segments) > 0:
        stroke_start, stroke_end = max(stroke_segments, key=lambda se: se[1] - se[0] + 1)
    else:
        # fallback: central 20 percent of active segment
        stroke_len = max(2, int(round(0.20 * active_len)))
        center = (active_start + active_end) // 2
        stroke_start = max(active_start, center - stroke_len // 2)
        stroke_end = min(active_end, stroke_start + stroke_len - 1)

    # Ensure prep/retraction exist when possible.
    if stroke_start <= active_start and active_len >= 8:
        stroke_start = active_start + 1
    if stroke_end >= active_end and active_len >= 8:
        stroke_end = active_end - 1

    prep_start = active_start
    prep_end = max(active_start, stroke_start - 1)

    retract_start = min(active_end, stroke_end + 1)
    retract_end = active_end

    if prep_end >= prep_start:
        ordered[prep_start:prep_end + 1] = 1
    if stroke_end >= stroke_start:
        ordered[stroke_start:stroke_end + 1] = 2
    if retract_end >= retract_start:
        ordered[retract_start:retract_end + 1] = 3

    return ordered, active_start, active_end


# ============================================================
# Inference wrapper
# ============================================================

class RealtimeRecognizer:
    def __init__(self, base_dir, device):
        self.base_dir = Path(base_dir)
        self.device = device

        self.phase_ckpt_path = self.base_dir / "models" / "phase_tcn" / "phase_tcn_best_safe_state.pt"
        self.rec_ckpt_path = self.base_dir / "models" / "recognition_tcn_attention" / "recognition_tcn_attention_best.pt"
        self.label_map_path = self.base_dir / "manifests" / "recognition_label_map_with_split_counts.csv"

        for p in [self.phase_ckpt_path, self.rec_ckpt_path, self.label_map_path]:
            if not p.exists():
                raise FileNotFoundError(f"Missing required file: {p}")

        self.label_map_df = pd.read_csv(self.label_map_path)
        self.id_to_gloss = {
            int(row["rec_label_id"]): str(row["gloss"])
            for _, row in self.label_map_df.iterrows()
        }

        self.phase_ckpt = torch.load(self.phase_ckpt_path, map_location=device, weights_only=False)
        self.rec_ckpt = torch.load(self.rec_ckpt_path, map_location=device, weights_only=False)

        self.phase_config = self.phase_ckpt["config"]
        self.rec_config = self.rec_ckpt["config"]

        self.phase_mean = self.phase_ckpt["feature_mean"].detach().cpu().numpy().astype(np.float32)
        self.phase_std = self.phase_ckpt["feature_std"].detach().cpu().numpy().astype(np.float32)

        self.rec_mean = self.rec_ckpt["feature_mean"].detach().cpu().numpy().astype(np.float32)
        self.rec_std = self.rec_ckpt["feature_std"].detach().cpu().numpy().astype(np.float32)

        self.phase_model = PhaseTCN(
            input_dim=self.phase_config["input_dim"],
            num_classes=self.phase_config["num_classes"],
            hidden_dim=self.phase_config["hidden_dim"],
            num_blocks=self.phase_config["num_blocks"],
            kernel_size=self.phase_config["kernel_size"],
            dropout=self.phase_config["dropout"],
        ).to(device)

        self.phase_model.load_state_dict(self.phase_ckpt["model_state_dict"])
        self.phase_model.eval()

        self.rec_model = RecognitionTCNAttentionSafe(
            input_dim=self.rec_config["input_dim"],
            num_classes=self.rec_config["num_classes"],
            hidden_dim=self.rec_config["hidden_dim"],
            num_blocks=self.rec_config["num_blocks"],
            kernel_size=self.rec_config["kernel_size"],
            dropout=self.rec_config["dropout"],
            attention_dropout=self.rec_config["attention_dropout"],
        ).to(device)

        self.rec_model.load_state_dict(self.rec_ckpt["model_state_dict"])
        self.rec_model.eval()

    def predict_from_position_buffer(self, positions_buffer, top_k=5):
        positions_seq = np.asarray(positions_buffer, dtype=np.float32)

        if positions_seq.ndim != 2 or positions_seq.shape[1] != 147:
            raise ValueError(f"Expected positions buffer shape (T, 147), got {positions_seq.shape}")

        X_cont, phase_speed = compute_motion_features_from_positions(positions_seq)

        # Phase TCN inference.
        X_phase = (X_cont - self.phase_mean) / self.phase_std
        X_phase[~np.isfinite(X_phase)] = 0.0

        xt = torch.tensor(X_phase, dtype=torch.float32).unsqueeze(0).to(self.device)

        with torch.no_grad():
            phase_logits = self.phase_model(xt)
            phase_probs = torch.softmax(phase_logits, dim=-1).squeeze(0).detach().cpu().numpy()
            raw_phase = np.argmax(phase_probs, axis=-1).astype(np.int64)

        smooth_phase = majority_filter_1d(raw_phase, window=5)
        ordered_phase, active_start, active_end = build_ordered_phase_labels_from_pred(smooth_phase)

        # Recognition active crop.
        X_cont_crop = X_cont[active_start:active_end + 1]
        phase_crop = ordered_phase[active_start:active_end + 1]

        X_rec_cont = (X_cont_crop - self.rec_mean) / self.rec_std
        X_rec_cont[~np.isfinite(X_rec_cont)] = 0.0

        phase_crop = np.clip(phase_crop, 0, 3).astype(np.int64)
        phase_onehot = np.eye(4, dtype=np.float32)[phase_crop]

        X_rec = np.concatenate([X_rec_cont, phase_onehot], axis=1).astype(np.float32)
        X_rec[~np.isfinite(X_rec)] = 0.0

        xrt = torch.tensor(X_rec, dtype=torch.float32).unsqueeze(0).to(self.device)
        mask = torch.ones(1, X_rec.shape[0], dtype=torch.bool).to(self.device)

        with torch.no_grad():
            logits, attn = self.rec_model(xrt, mask)
            probs = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
            attn = attn.squeeze(0).detach().cpu().numpy()

        top_ids = np.argsort(probs)[-top_k:][::-1].tolist()

        result = {
            "top_ids": top_ids,
            "top_glosses": [self.id_to_gloss.get(i, str(i)) for i in top_ids],
            "top_probs": [float(probs[i]) for i in top_ids],
            "pred_id": int(top_ids[0]),
            "pred_gloss": self.id_to_gloss.get(int(top_ids[0]), str(top_ids[0])),
            "active_start": int(active_start),
            "active_end": int(active_end),
            "buffer_T": int(len(positions_seq)),
            "active_T": int(X_rec.shape[0]),
            "phase_counts": {
                "background": int(np.sum(ordered_phase == 0)),
                "preparation": int(np.sum(ordered_phase == 1)),
                "stroke": int(np.sum(ordered_phase == 2)),
                "retraction": int(np.sum(ordered_phase == 3)),
            },
            "attention_phase_mass": {
                "preparation": float(attn[phase_crop == 1].sum()) if np.any(phase_crop == 1) else 0.0,
                "stroke": float(attn[phase_crop == 2].sum()) if np.any(phase_crop == 2) else 0.0,
                "retraction": float(attn[phase_crop == 3].sum()) if np.any(phase_crop == 3) else 0.0,
            },
        }

        return result


# ============================================================
# Main webcam loop
# ============================================================

def draw_text_block(frame, lines, x=20, y=30, line_h=28):
    for i, line in enumerate(lines):
        yy = y + i * line_h
        cv2.putText(
            frame,
            line,
            (x, yy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--base_dir",
        type=str,
        required=True,
        help="Path to CAPSTONE_ASL folder containing models/ and manifests/",
    )
    parser.add_argument("--camera", type=int, default=0)
    parser.add_argument("--buffer_size", type=int, default=90)
    parser.add_argument("--min_buffer", type=int, default=45)
    parser.add_argument("--predict_every", type=int, default=8)
    parser.add_argument("--width", type=int, default=960)
    parser.add_argument("--height", type=int, default=540)
    parser.add_argument("--top_k", type=int, default=5)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 80)
    print("REAL-TIME PHASE-AWARE SIGN RECOGNITION DEMO")
    print("=" * 80)
    print(f"Base dir:       {args.base_dir}")
    print(f"Device:         {device}")
    print(f"Camera index:   {args.camera}")
    print(f"Buffer size:    {args.buffer_size}")
    print(f"Min buffer:     {args.min_buffer}")
    print(f"Predict every:  {args.predict_every}")
    print()
    print("Controls:")
    print("  q = quit")
    print("  c = clear rolling buffer")
    print()

    recognizer = RealtimeRecognizer(args.base_dir, device=device)
    print("Models loaded.")

    mp_holistic = mp.solutions.holistic
    mp_drawing = mp.solutions.drawing_utils

    cap = cv2.VideoCapture(args.camera)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, args.width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, args.height)

    if not cap.isOpened():
        raise RuntimeError("Could not open webcam. Try --camera 1 or check camera permissions.")

    position_buffer = deque(maxlen=args.buffer_size)
    last_prediction = None
    frame_idx = 0
    fps_time = time.time()
    fps = 0.0

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as holistic:

        while True:
            ok, frame = cap.read()
            if not ok:
                print("Failed to read frame.")
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)

            # Draw landmarks for user feedback.
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    results.pose_landmarks,
                    mp_holistic.POSE_CONNECTIONS,
                )

            if results.left_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    results.left_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS,
                )

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    results.right_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS,
                )

            # Extract 147D normalized positions.
            try:
                pos147 = extract_147_positions_from_mediapipe(results)
                position_buffer.append(pos147)
                extraction_ok = True
            except Exception as e:
                extraction_ok = False
                print(f"Keypoint extraction error: {e}")

            # FPS estimate.
            now = time.time()
            dt = now - fps_time
            if dt > 0:
                fps = 0.95 * fps + 0.05 * (1.0 / dt)
            fps_time = now

            # Predict periodically once buffer is ready.
            if (
                extraction_ok
                and len(position_buffer) >= args.min_buffer
                and frame_idx % args.predict_every == 0
            ):
                try:
                    last_prediction = recognizer.predict_from_position_buffer(
                        np.array(position_buffer),
                        top_k=args.top_k,
                    )
                except Exception as e:
                    print(f"Prediction error: {e}")
                    last_prediction = {
                        "pred_gloss": "ERROR",
                        "top_glosses": ["ERROR"],
                        "top_probs": [0.0],
                    }

            lines = [
                f"FPS: {fps:.1f}",
                f"Buffer: {len(position_buffer)}/{args.buffer_size}",
            ]

            if last_prediction is None:
                lines.append("Prediction: collecting buffer...")
            else:
                lines.append(
                    f"Prediction: {last_prediction['pred_gloss']} "
                    f"({last_prediction['top_probs'][0]:.2f})"
                )

                top_pairs = list(zip(
                    last_prediction["top_glosses"],
                    last_prediction["top_probs"],
                ))

                for rank, (gloss, prob) in enumerate(top_pairs[:5], start=1):
                    lines.append(f"{rank}. {gloss}: {prob:.2f}")

                if "attention_phase_mass" in last_prediction:
                    attn = last_prediction["attention_phase_mass"]
                    lines.append(
                        "Attn prep/stroke/retract: "
                        f"{attn.get('preparation', 0.0):.2f} / "
                        f"{attn.get('stroke', 0.0):.2f} / "
                        f"{attn.get('retraction', 0.0):.2f}"
                    )

            draw_text_block(frame, lines)

            cv2.imshow("Phase-Aware Sign Recognition Demo", frame)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break

            if key == ord("c"):
                position_buffer.clear()
                last_prediction = None
                print("Buffer cleared.")

            frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
'''

requirements_text = """\
torch
torchvision
numpy
pandas
opencv-python
mediapipe
"""

readme_text = f"""# Real-Time Phase-Aware Sign Recognition Demo

This folder contains a local webcam prototype for the capstone sign recognition project.

## Generated Files

- `webcam_realtime_phase_aware_demo.py`
- `requirements_realtime.txt`
- `README_realtime_demo.md`

## What This Demo Does

The script opens your webcam and runs this live pipeline:

```text
webcam frame
→ MediaPipe pose/hand keypoints
→ normalized 147D landmark vector
→ rolling sequence buffer
→ velocity / acceleration / speed features
→ Phase TCN prediction
→ active signing crop
→ phase-aware Recognition TCN + attention
→ Top-5 sign predictions on screen

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
REALTIME_DIR = BASE_DIR / "realtime_demo"
REALTIME_DIR.mkdir(parents=True, exist_ok=True)

print("Realtime demo folder ready:")
print(REALTIME_DIR)

In [ ]:
%%writefile /content/drive/MyDrive/CAPSTONE_ASL/realtime_demo/webcam_realtime_phase_aware_demo.py
import argparse
import time
from pathlib import Path
from collections import deque

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import mediapipe as mp


POSE_IDXS = [0, 11, 12, 13, 14, 15, 16]


def safe_torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.drop2 = nn.Dropout(dropout)

        self.downsample = None
        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        out = self.conv1(x)
        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn1(out)
        out = F.relu(out)
        out = self.drop1(out)

        out = self.conv2(out)
        if out.shape[-1] > x.shape[-1]:
            out = out[:, :, :x.shape[-1]]
        elif out.shape[-1] < x.shape[-1]:
            out = F.pad(out, (0, x.shape[-1] - out.shape[-1]))

        out = self.bn2(out)
        out = F.relu(out)
        out = self.drop2(out)

        residual = x if self.downsample is None else self.downsample(x)
        if residual.shape[-1] > out.shape[-1]:
            residual = residual[:, :, :out.shape[-1]]
        elif residual.shape[-1] < out.shape[-1]:
            residual = F.pad(residual, (0, out.shape[-1] - residual.shape[-1]))

        return F.relu(out + residual)


class PhaseTCN(nn.Module):
    def __init__(self, input_dim, num_classes=4, hidden_dim=192, num_blocks=5, kernel_size=5, dropout=0.20):
        super().__init__()
        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=2 ** i,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.classifier = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.transpose(1, 2)
        h = self.tcn(x)
        logits = self.classifier(h)
        return logits.transpose(1, 2)


class SafeMaskedAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.10):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h, mask):
        scores = self.attn(h.float()).squeeze(-1)
        scores = scores.masked_fill(~mask.bool(), -1e4)
        weights = torch.softmax(scores, dim=1)
        weights = weights.masked_fill(~mask.bool(), 0.0)
        pooled = torch.sum(h.float() * weights.unsqueeze(-1), dim=1)
        return pooled, weights


class RecognitionTCNAttentionSafe(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        hidden_dim=256,
        num_blocks=4,
        kernel_size=5,
        dropout=0.30,
        attention_dropout=0.10,
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim

        for i in range(num_blocks):
            blocks.append(
                TemporalBlock(
                    in_channels=in_ch,
                    out_channels=hidden_dim,
                    kernel_size=kernel_size,
                    dilation=2 ** i,
                    dropout=dropout,
                )
            )
            in_ch = hidden_dim

        self.tcn = nn.Sequential(*blocks)
        self.attention_pool = SafeMaskedAttentionPooling(hidden_dim, dropout=attention_dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, mask):
        h = x.transpose(1, 2)
        h = self.tcn(h)
        h = h.transpose(1, 2)
        pooled, attn = self.attention_pool(h, mask)
        logits = self.classifier(pooled.float())
        return logits, attn


def lm_xyz(lm):
    return np.array([lm.x, lm.y, lm.z], dtype=np.float32)


def extract_147_positions(results):
    points = []

    if results.pose_landmarks is not None:
        pose = results.pose_landmarks.landmark

        for idx in POSE_IDXS:
            points.append(lm_xyz(pose[idx]))

        left_shoulder = lm_xyz(pose[11])
        right_shoulder = lm_xyz(pose[12])
        origin = (left_shoulder + right_shoulder) / 2.0
        scale = float(np.linalg.norm(left_shoulder - right_shoulder))
    else:
        for _ in POSE_IDXS:
            points.append(np.zeros(3, dtype=np.float32))
        origin = np.array([0.5, 0.5, 0.0], dtype=np.float32)
        scale = 1.0

    if scale < 1e-6:
        scale = 1.0

    if results.left_hand_landmarks is not None:
        for lm in results.left_hand_landmarks.landmark:
            points.append(lm_xyz(lm))
    else:
        for _ in range(21):
            points.append(np.zeros(3, dtype=np.float32))

    if results.right_hand_landmarks is not None:
        for lm in results.right_hand_landmarks.landmark:
            points.append(lm_xyz(lm))
    else:
        for _ in range(21):
            points.append(np.zeros(3, dtype=np.float32))

    arr = np.stack(points, axis=0).astype(np.float32)
    missing = np.linalg.norm(arr, axis=1) < 1e-8

    arr = (arr - origin.reshape(1, 3)) / scale
    arr[missing] = 0.0

    out = arr.reshape(-1).astype(np.float32)
    out[~np.isfinite(out)] = 0.0

    if out.shape[0] != 147:
        raise ValueError(f"Expected 147D vector, got {out.shape[0]}")

    return out


def compute_motion_features(positions_seq):
    positions = np.asarray(positions_seq, dtype=np.float32)
    T = positions.shape[0]

    velocity = np.zeros_like(positions)
    acceleration = np.zeros_like(positions)

    if T > 1:
        velocity[1:] = positions[1:] - positions[:-1]
        acceleration[1:] = velocity[1:] - velocity[:-1]

    global_speed = np.linalg.norm(velocity, axis=1, keepdims=True).astype(np.float32)

    hand_velocity = velocity[:, 21:]
    hand_speed = np.linalg.norm(hand_velocity, axis=1, keepdims=True).astype(np.float32)

    raw_phase_speed = 0.5 * global_speed + 0.5 * hand_speed
    max_val = float(np.max(raw_phase_speed))

    if max_val > 1e-8:
        phase_speed = raw_phase_speed / max_val
    else:
        phase_speed = raw_phase_speed

    X = np.concatenate(
        [positions, velocity, acceleration, global_speed, hand_speed, phase_speed],
        axis=1,
    ).astype(np.float32)

    X[~np.isfinite(X)] = 0.0

    if X.shape[1] != 444:
        raise ValueError(f"Expected 444D features, got {X.shape[1]}")

    return X


def majority_filter(labels, window=5):
    labels = np.asarray(labels, dtype=np.int64)

    if len(labels) == 0:
        return labels

    if window % 2 == 0:
        window += 1

    pad = window // 2
    padded = np.pad(labels, (pad, pad), mode="edge")
    out = labels.copy()

    for i in range(len(labels)):
        vals = padded[i:i + window]
        counts = np.bincount(vals, minlength=4)
        out[i] = int(np.argmax(counts))

    return out


def get_segments(mask):
    mask = np.asarray(mask, dtype=bool)
    segs = []
    start = None

    for i, v in enumerate(mask):
        if v and start is None:
            start = i
        elif not v and start is not None:
            segs.append((start, i - 1))
            start = None

    if start is not None:
        segs.append((start, len(mask) - 1))

    return segs


def ordered_phase_from_predictions(pred):
    pred = np.asarray(pred, dtype=np.int64)
    T = len(pred)
    ordered = np.zeros(T, dtype=np.int64)

    active = pred != 0
    active_segs = get_segments(active)

    if len(active_segs) == 0:
        active_start, active_end = 0, T - 1
    else:
        active_start = active_segs[0][0]
        active_end = active_segs[-1][1]

    active_len = active_end - active_start + 1

    stroke_mask = np.zeros(T, dtype=bool)
    stroke_mask[active_start:active_end + 1] = pred[active_start:active_end + 1] == 2
    stroke_segs = get_segments(stroke_mask)

    if len(stroke_segs) > 0:
        stroke_start, stroke_end = max(stroke_segs, key=lambda se: se[1] - se[0] + 1)
    else:
        stroke_len = max(2, int(round(0.20 * max(active_len, 1))))
        center = (active_start + active_end) // 2
        stroke_start = max(active_start, center - stroke_len // 2)
        stroke_end = min(active_end, stroke_start + stroke_len - 1)

    if stroke_start <= active_start and active_len >= 8:
        stroke_start = active_start + 1
    if stroke_end >= active_end and active_len >= 8:
        stroke_end = active_end - 1

    prep_start = active_start
    prep_end = max(active_start, stroke_start - 1)

    retract_start = min(active_end, stroke_end + 1)
    retract_end = active_end

    ordered[prep_start:prep_end + 1] = 1
    ordered[stroke_start:stroke_end + 1] = 2
    ordered[retract_start:retract_end + 1] = 3

    return ordered, active_start, active_end


class RealtimeRecognizer:
    def __init__(self, base_dir, device):
        self.base_dir = Path(base_dir)
        self.device = device

        self.phase_path = self.base_dir / "models" / "phase_tcn" / "phase_tcn_best_safe_state.pt"
        self.rec_path = self.base_dir / "models" / "recognition_tcn_attention" / "recognition_tcn_attention_best.pt"
        self.label_path = self.base_dir / "manifests" / "recognition_label_map_with_split_counts.csv"

        for p in [self.phase_path, self.rec_path, self.label_path]:
            if not p.exists():
                raise FileNotFoundError(f"Missing required file: {p}")

        label_df = pd.read_csv(self.label_path)
        self.id_to_gloss = {
            int(row["rec_label_id"]): str(row["gloss"])
            for _, row in label_df.iterrows()
        }

        self.phase_ckpt = safe_torch_load(self.phase_path, device)
        self.rec_ckpt = safe_torch_load(self.rec_path, device)

        pcfg = self.phase_ckpt["config"]
        rcfg = self.rec_ckpt["config"]

        self.phase_mean = self.phase_ckpt["feature_mean"].detach().cpu().numpy().astype(np.float32)
        self.phase_std = self.phase_ckpt["feature_std"].detach().cpu().numpy().astype(np.float32)

        self.rec_mean = self.rec_ckpt["feature_mean"].detach().cpu().numpy().astype(np.float32)
        self.rec_std = self.rec_ckpt["feature_std"].detach().cpu().numpy().astype(np.float32)

        self.phase_model = PhaseTCN(
            input_dim=pcfg["input_dim"],
            num_classes=pcfg["num_classes"],
            hidden_dim=pcfg["hidden_dim"],
            num_blocks=pcfg["num_blocks"],
            kernel_size=pcfg["kernel_size"],
            dropout=pcfg["dropout"],
        ).to(device)

        self.phase_model.load_state_dict(self.phase_ckpt["model_state_dict"])
        self.phase_model.eval()

        self.rec_model = RecognitionTCNAttentionSafe(
            input_dim=rcfg["input_dim"],
            num_classes=rcfg["num_classes"],
            hidden_dim=rcfg["hidden_dim"],
            num_blocks=rcfg["num_blocks"],
            kernel_size=rcfg["kernel_size"],
            dropout=rcfg["dropout"],
            attention_dropout=rcfg["attention_dropout"],
        ).to(device)

        self.rec_model.load_state_dict(self.rec_ckpt["model_state_dict"])
        self.rec_model.eval()

    def predict(self, position_buffer, top_k=5):
        positions = np.asarray(position_buffer, dtype=np.float32)

        X_cont = compute_motion_features(positions)

        X_phase = (X_cont - self.phase_mean) / self.phase_std
        X_phase[~np.isfinite(X_phase)] = 0.0

        xt = torch.tensor(X_phase, dtype=torch.float32).unsqueeze(0).to(self.device)

        with torch.no_grad():
            phase_logits = self.phase_model(xt)
            phase_probs = torch.softmax(phase_logits, dim=-1).squeeze(0).cpu().numpy()
            raw_phase = np.argmax(phase_probs, axis=-1).astype(np.int64)

        smooth_phase = majority_filter(raw_phase, window=5)
        ordered_phase, active_start, active_end = ordered_phase_from_predictions(smooth_phase)

        X_crop = X_cont[active_start:active_end + 1]
        phase_crop = ordered_phase[active_start:active_end + 1]

        X_rec_cont = (X_crop - self.rec_mean) / self.rec_std
        X_rec_cont[~np.isfinite(X_rec_cont)] = 0.0

        phase_crop = np.clip(phase_crop, 0, 3).astype(np.int64)
        phase_onehot = np.eye(4, dtype=np.float32)[phase_crop]

        X_rec = np.concatenate([X_rec_cont, phase_onehot], axis=1).astype(np.float32)
        X_rec[~np.isfinite(X_rec)] = 0.0

        xrt = torch.tensor(X_rec, dtype=torch.float32).unsqueeze(0).to(self.device)
        mask = torch.ones(1, X_rec.shape[0], dtype=torch.bool).to(self.device)

        with torch.no_grad():
            logits, attn = self.rec_model(xrt, mask)
            probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
            attn = attn.squeeze(0).cpu().numpy()

        top_ids = np.argsort(probs)[-top_k:][::-1].tolist()

        return {
            "pred_gloss": self.id_to_gloss.get(int(top_ids[0]), str(top_ids[0])),
            "top_glosses": [self.id_to_gloss.get(int(i), str(i)) for i in top_ids],
            "top_probs": [float(probs[i]) for i in top_ids],
            "active_start": int(active_start),
            "active_end": int(active_end),
            "attention_preparation": float(attn[phase_crop == 1].sum()) if np.any(phase_crop == 1) else 0.0,
            "attention_stroke": float(attn[phase_crop == 2].sum()) if np.any(phase_crop == 2) else 0.0,
            "attention_retraction": float(attn[phase_crop == 3].sum()) if np.any(phase_crop == 3) else 0.0,
        }


def draw_lines(frame, lines, x=20, y=30, line_h=27):
    for i, line in enumerate(lines):
        cv2.putText(
            frame,
            line,
            (x, y + i * line_h),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base_dir", type=str, required=True)
    parser.add_argument("--camera", type=int, default=0)
    parser.add_argument("--buffer_size", type=int, default=90)
    parser.add_argument("--min_buffer", type=int, default=45)
    parser.add_argument("--predict_every", type=int, default=8)
    parser.add_argument("--top_k", type=int, default=5)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 80)
    print("REAL-TIME PHASE-AWARE SIGN RECOGNITION DEMO")
    print("=" * 80)
    print("q = quit")
    print("c = clear buffer")
    print(f"Device: {device}")
    print(f"Base dir: {args.base_dir}")

    recognizer = RealtimeRecognizer(args.base_dir, device=device)
    print("Models loaded successfully.")

    mp_holistic = mp.solutions.holistic
    mp_drawing = mp.solutions.drawing_utils

    cap = cv2.VideoCapture(args.camera)

    if not cap.isOpened():
        raise RuntimeError("Could not open webcam. Try --camera 1.")

    buffer = deque(maxlen=args.buffer_size)
    last_pred = None
    frame_idx = 0
    last_time = time.time()
    fps = 0.0

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as holistic:

        while True:
            ok, frame = cap.read()
            if not ok:
                print("Camera frame read failed.")
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)

            if results.pose_landmarks:
                mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

            if results.left_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

            try:
                pos = extract_147_positions(results)
                buffer.append(pos)
            except Exception as e:
                print("Extraction error:", e)

            now = time.time()
            dt = now - last_time
            if dt > 0:
                fps = 0.95 * fps + 0.05 * (1.0 / dt)
            last_time = now

            if len(buffer) >= args.min_buffer and frame_idx % args.predict_every == 0:
                try:
                    last_pred = recognizer.predict(np.array(buffer), top_k=args.top_k)
                except Exception as e:
                    print("Prediction error:", e)
                    last_pred = None

            lines = [
                f"FPS: {fps:.1f}",
                f"Buffer: {len(buffer)}/{args.buffer_size}",
            ]

            if last_pred is None:
                lines.append("Prediction: collecting buffer...")
            else:
                lines.append(f"Prediction: {last_pred['pred_gloss']} ({last_pred['top_probs'][0]:.2f})")
                for i, (g, p) in enumerate(zip(last_pred["top_glosses"], last_pred["top_probs"]), start=1):
                    lines.append(f"{i}. {g}: {p:.2f}")

                lines.append(
                    "Attn prep/stroke/retract: "
                    f"{last_pred['attention_preparation']:.2f} / "
                    f"{last_pred['attention_stroke']:.2f} / "
                    f"{last_pred['attention_retraction']:.2f}"
                )

            draw_lines(frame, lines)

            cv2.imshow("Phase-Aware Sign Recognition Demo", frame)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break

            if key == ord("c"):
                buffer.clear()
                last_pred = None
                print("Buffer cleared.")

            frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
REALTIME_DIR = BASE_DIR / "realtime_demo"
REALTIME_DIR.mkdir(parents=True, exist_ok=True)

REQ_PATH = REALTIME_DIR / "requirements_realtime.txt"
README_PATH = REALTIME_DIR / "README_realtime_demo.md"

REQ_PATH.write_text(
    "\n".join([
        "torch",
        "torchvision",
        "numpy",
        "pandas",
        "opencv-python",
        "mediapipe",
    ]),
    encoding="utf-8"
)

README_PATH.write_text(
"""# Real-Time Webcam Demo

This folder contains the local webcam prototype for the phase-aware sign recognition project.

## Files

- webcam_realtime_phase_aware_demo.py
- requirements_realtime.txt
- README_realtime_demo.md

## Required project structure

The local CAPSTONE_ASL folder must contain:

CAPSTONE_ASL/
- models/phase_tcn/phase_tcn_best_safe_state.pt
- models/recognition_tcn_attention/recognition_tcn_attention_best.pt
- manifests/recognition_label_map_with_split_counts.csv
- realtime_demo/webcam_realtime_phase_aware_demo.py

## Setup on Windows PowerShell

Open PowerShell inside the realtime_demo folder.

python -m venv .venv
.\\.venv\\Scripts\\activate
python -m pip install --upgrade pip
pip install -r requirements_realtime.txt

## Run

python webcam_realtime_phase_aware_demo.py --base_dir "C:\\path\\to\\CAPSTONE_ASL" --camera 0

If camera 0 does not work:

python webcam_realtime_phase_aware_demo.py --base_dir "C:\\path\\to\\CAPSTONE_ASL" --camera 1

## Controls

q = quit
c = clear rolling buffer

## Important warning

This is a prototype. Accuracy may be lower than offline testing because webcam lighting, camera angle, distance, signer style, and MediaPipe tracking are different from the dataset videos.

## YOLO

YOLO is not used yet. Add YOLO only after this MediaPipe-only webcam version opens successfully and predictions update.
""",
    encoding="utf-8"
)

print("PHASE 13B-FIX files saved:")
print("Script:", REALTIME_DIR / "webcam_realtime_phase_aware_demo.py")
print("Requirements:", REQ_PATH)
print("README:", README_PATH)

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
REALTIME_DIR = BASE_DIR / "realtime_demo"

files = [
    REALTIME_DIR / "webcam_realtime_phase_aware_demo.py",
    REALTIME_DIR / "requirements_realtime.txt",
    REALTIME_DIR / "README_realtime_demo.md",
]

print("=" * 80)
print("PHASE 13B-FIX VERIFICATION")
print("=" * 80)

for p in files:
    print(p, "EXISTS =", p.exists(), "SIZE =", p.stat().st_size if p.exists() else 0)

print()
print("If all files exist, copy/sync CAPSTONE_ASL locally and run the webcam script in VS Code/PowerShell.")

In [ ]:
from pathlib import Path
import shutil

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
OUT_DIR = Path("/content/capstone-asl-github-ready")

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Create folders
(OUT_DIR / "realtime_demo").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models" / "phase_tcn").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models" / "recognition_tcn_attention").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "manifests").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports" / "final_results").mkdir(parents=True, exist_ok=True)

# Copy realtime demo
for name in [
    "webcam_realtime_phase_aware_demo.py",
    "requirements_realtime.txt",
    "README_realtime_demo.md",
]:
    shutil.copy2(
        BASE_DIR / "realtime_demo" / name,
        OUT_DIR / "realtime_demo" / name
    )

# Copy model checkpoints
shutil.copy2(
    BASE_DIR / "models" / "phase_tcn" / "phase_tcn_best_safe_state.pt",
    OUT_DIR / "models" / "phase_tcn" / "phase_tcn_best_safe_state.pt"
)

shutil.copy2(
    BASE_DIR / "models" / "recognition_tcn_attention" / "recognition_tcn_attention_best.pt",
    OUT_DIR / "models" / "recognition_tcn_attention" / "recognition_tcn_attention_best.pt"
)

# Copy label map
shutil.copy2(
    BASE_DIR / "manifests" / "recognition_label_map_with_split_counts.csv",
    OUT_DIR / "manifests" / "recognition_label_map_with_split_counts.csv"
)

# Copy report results if available
report_src = BASE_DIR / "reports" / "final_results"
report_dst = OUT_DIR / "reports" / "final_results"

for item in [
    "final_results_summary.json",
    "final_results_tables.md",
    "final_report_key_points.md",
    "project_phase_status.csv",
]:
    src = report_src / item
    if src.exists():
        shutil.copy2(src, report_dst / item)

# Copy report text folder
if (report_src / "report_text").exists():
    shutil.copytree(
        report_src / "report_text",
        report_dst / "report_text",
        dirs_exist_ok=True
    )

# Copy figures folder
if (report_src / "figures").exists():
    shutil.copytree(
        report_src / "figures",
        report_dst / "figures",
        dirs_exist_ok=True
    )

print("Created GitHub-ready folder:")
print(OUT_DIR)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

OUT_DIR = Path("/content/capstone-asl-github-ready")

gitignore = """
# Python
__pycache__/
*.pyc
.venv/
venv/
.env

# Large/raw project data - do not push
videos/
raw_videos/
datasets/
keypoints/
motion_features/
phase_labels/
phase_segments/
subset_top100/
*.mp4
*.avi
*.mov
*.mkv

# Colab / notebooks checkpoints
.ipynb_checkpoints/

# OS
.DS_Store
Thumbs.db
"""

readme = """# Phase-Aware ASL Recognition Capstone

This repository contains the final demo and report package for a phase-aware isolated sign recognition capstone project.

## Main Pipeline

MediaPipe landmarks → motion features → weak phase pseudo-labels → Phase Detection TCN → phase-aware segment extraction → Recognition TCN + Attention.

## Final Results

Phase-aware Recognition TCN + Attention:

- Test Top-1: 0.6885
- Test Top-5: 0.8700
- Test Macro F1: 0.6695

Baseline without phase one-hot:

- Test Top-1: 0.6795
- Test Top-5: 0.8597
- Test Macro F1: 0.6574

The phase-aware model produced a small but consistent improvement over the no-phase baseline. The paired comparison showed that the improvement was modest and not statistically strong.

## Realtime Demo

See:

```text
realtime_demo/README_realtime_demo.md

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
REALTIME_DIR = BASE_DIR / "realtime_demo"

print("BASE_DIR exists:", BASE_DIR.exists())
print("REALTIME_DIR exists:", REALTIME_DIR.exists())

if REALTIME_DIR.exists():
    print("\nFiles inside realtime_demo:")
    for p in REALTIME_DIR.iterdir():
        print("-", p.name, "| size:", p.stat().st_size if p.is_file() else "folder")
else:
    print("realtime_demo folder does not exist.")

In [ ]:
from pathlib import Path
import shutil

BASE_DIR = Path("/content/drive/MyDrive/CAPSTONE_ASL")
OUT_DIR = Path("/content/capstone-asl-github-ready")

required_files = [
    BASE_DIR / "realtime_demo" / "webcam_realtime_phase_aware_demo.py",
    BASE_DIR / "realtime_demo" / "requirements_realtime.txt",
    BASE_DIR / "realtime_demo" / "README_realtime_demo.md",
    BASE_DIR / "models" / "phase_tcn" / "phase_tcn_best_safe_state.pt",
    BASE_DIR / "models" / "recognition_tcn_attention" / "recognition_tcn_attention_best.pt",
    BASE_DIR / "manifests" / "recognition_label_map_with_split_counts.csv",
]

missing = [p for p in required_files if not p.exists()]

print("Checking required files...")
if missing:
    print("❌ Missing files:")
    for p in missing:
        print("-", p)
    raise FileNotFoundError("Fix the missing files before packaging.")
else:
    print("✅ All required files found.")

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

# Create output folders
(OUT_DIR / "realtime_demo").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models" / "phase_tcn").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models" / "recognition_tcn_attention").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "manifests").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports" / "final_results").mkdir(parents=True, exist_ok=True)

# Copy realtime demo
for name in [
    "webcam_realtime_phase_aware_demo.py",
    "requirements_realtime.txt",
    "README_realtime_demo.md",
]:
    shutil.copy2(
        BASE_DIR / "realtime_demo" / name,
        OUT_DIR / "realtime_demo" / name
    )

# Copy model checkpoints
shutil.copy2(
    BASE_DIR / "models" / "phase_tcn" / "phase_tcn_best_safe_state.pt",
    OUT_DIR / "models" / "phase_tcn" / "phase_tcn_best_safe_state.pt"
)

shutil.copy2(
    BASE_DIR / "models" / "recognition_tcn_attention" / "recognition_tcn_attention_best.pt",
    OUT_DIR / "models" / "recognition_tcn_attention" / "recognition_tcn_attention_best.pt"
)

# Copy label map
shutil.copy2(
    BASE_DIR / "manifests" / "recognition_label_map_with_split_counts.csv",
    OUT_DIR / "manifests" / "recognition_label_map_with_split_counts.csv"
)

# Copy report files if available
report_src = BASE_DIR / "reports" / "final_results"
report_dst = OUT_DIR / "reports" / "final_results"

for item in [
    "final_results_summary.json",
    "final_results_tables.md",
    "final_report_key_points.md",
    "project_phase_status.csv",
]:
    src = report_src / item
    if src.exists():
        shutil.copy2(src, report_dst / item)

if (report_src / "report_text").exists():
    shutil.copytree(
        report_src / "report_text",
        report_dst / "report_text",
        dirs_exist_ok=True
    )

if (report_src / "figures").exists():
    shutil.copytree(
        report_src / "figures",
        report_dst / "figures",
        dirs_exist_ok=True
    )

print("✅ Created GitHub-ready folder:")
print(OUT_DIR)

In [ ]:
from pathlib import Path

OUT_DIR = Path("/content/capstone-asl-github-ready")

gitignore = """
# Python
__pycache__/
*.pyc
.venv/
venv/
.env

# Large/raw project data - do not push
videos/
raw_videos/
datasets/
keypoints/
motion_features/
phase_labels/
phase_segments/
subset_top100/
*.mp4
*.avi
*.mov
*.mkv

# Colab / notebooks checkpoints
.ipynb_checkpoints/

# OS
.DS_Store
Thumbs.db
"""

readme = """# Phase-Aware ASL Recognition Capstone

This repository contains the final demo and report package for a phase-aware isolated sign recognition capstone project.

## Main Pipeline

MediaPipe landmarks → motion features → weak phase pseudo-labels → Phase Detection TCN → phase-aware segment extraction → Recognition TCN + Attention.

## Final Results

Phase-aware Recognition TCN + Attention:

- Test Top-1: 0.6885
- Test Top-5: 0.8700
- Test Macro F1: 0.6695

Baseline without phase one-hot:

- Test Top-1: 0.6795
- Test Top-5: 0.8597
- Test Macro F1: 0.6574

The phase-aware model produced a small but consistent improvement over the no-phase baseline. The paired comparison showed that the improvement was modest and not statistically strong.

## Realtime Demo

See:

```text
realtime_demo/README_realtime_demo.md

In [ ]:
from pathlib import Path

OUT_DIR = Path("/content/capstone-asl-github-ready")

gitignore_lines = [
    "# Python",
    "__pycache__/",
    "*.pyc",
    ".venv/",
    "venv/",
    ".env",
    "",
    "# Large/raw project data - do not push",
    "videos/",
    "raw_videos/",
    "datasets/",
    "keypoints/",
    "motion_features/",
    "phase_labels/",
    "phase_segments/",
    "subset_top100/",
    "*.mp4",
    "*.avi",
    "*.mov",
    "*.mkv",
    "",
    "# Colab / notebooks checkpoints",
    ".ipynb_checkpoints/",
    "",
    "# OS",
    ".DS_Store",
    "Thumbs.db",
]

readme_lines = [
    "# Phase-Aware ASL Recognition Capstone",
    "",
    "This repository contains the final demo and report package for a phase-aware isolated sign recognition capstone project.",
    "",
    "## Main Pipeline",
    "",
    "MediaPipe landmarks -> motion features -> weak phase pseudo-labels -> Phase Detection TCN -> phase-aware segment extraction -> Recognition TCN + Attention.",
    "",
    "## Final Results",
    "",
    "Phase-aware Recognition TCN + Attention:",
    "",
    "- Test Top-1: 0.6885",
    "- Test Top-5: 0.8700",
    "- Test Macro F1: 0.6695",
    "",
    "Baseline without phase one-hot:",
    "",
    "- Test Top-1: 0.6795",
    "- Test Top-5: 0.8597",
    "- Test Macro F1: 0.6574",
    "",
    "The phase-aware model produced a small but consistent improvement over the no-phase baseline. The paired comparison showed that the improvement was modest and not statistically strong.",
    "",
    "## Realtime Demo",
    "",
    "See:",
    "",
    "realtime_demo/README_realtime_demo.md",
    "",
    "Run locally from PowerShell:",
    "",
    "cd realtime_demo",
    "python -m venv .venv",
    ".\\.venv\\Scripts\\activate",
    "python -m pip install --upgrade pip",
    "pip install -r requirements_realtime.txt",
    "python webcam_realtime_phase_aware_demo.py --base_dir \"C:\\path\\to\\capstone-asl\" --camera 0",
    "",
    "## Important",
    "",
    "This repo is a deployment/report package. It does not include the full raw dataset.",
]

(OUT_DIR / ".gitignore").write_text("\n".join(gitignore_lines) + "\n", encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join(readme_lines) + "\n", encoding="utf-8")

print("Wrote:")
print(OUT_DIR / ".gitignore")
print(OUT_DIR / "README.md")

print("\nPreview README:")
print((OUT_DIR / "README.md").read_text(encoding="utf-8")[:1000])

In [ ]:
from pathlib import Path

OUT_DIR = Path("/content/capstone-asl-github-ready")

large_files = []

for p in OUT_DIR.rglob("*"):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        if size_mb > 90:
            large_files.append((str(p), size_mb))

print("Files larger than 90 MB:")
if not large_files:
    print("None")
else:
    for path, size in large_files:
        print(f"{size:.2f} MB - {path}")

In [ ]:
# ============================================================
# PHASE 13B-COLAB-1-FIX — CLEAN MEDIAPIPE INSTALL
# Run this cell, then RESTART RUNTIME.
# ============================================================

!pip -q uninstall -y mediapipe protobuf numpy opencv-python opencv-python-headless opencv-contrib-python

!pip -q install --no-cache-dir "numpy==1.26.4"
!pip -q install --no-cache-dir "opencv-python==4.10.0.84"
!pip -q install --no-cache-dir "mediapipe==0.10.21"

# Force protobuf new enough to provide runtime_version.
!pip -q install --no-cache-dir --force-reinstall "protobuf==5.29.5"

print("=" * 80)
print("INSTALL COMPLETE")
print("=" * 80)
print("Now do: Runtime → Restart session")
print("After restart, run the check cell I give next.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 219.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
grain 0.2.16 requires protobuf>=5.28.3, which is not installed.
tensorflow-datasets 4.9.9 requires protobuf>=3.20, which is not installed.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, which is not installed.
tensorboard 2.20.0 requires protobuf!=4.24.0,>=3.19.6, which is not installed.
tensorflow-hub 0.16.1 requires protobuf>=3.19.6, which is not installed.
tensorflow 2.20.0 requires protobuf>=5.28.0, which is not installed.
orbax-checkpoint 0.11.36 requires protobuf, which is not installed.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, which is not installed.
albu

In [ ]:
# ============================================================
# CLEAN COLAB STACK FOR MEDIAPIPE LEGACY SOLUTIONS
# ============================================================

!pip -q uninstall -y mediapipe protobuf numpy pandas opencv-python opencv-python-headless opencv-contrib-python

!pip -q install --no-cache-dir "numpy==1.26.4"
!pip -q install --no-cache-dir "pandas==2.2.2"
!pip -q install --no-cache-dir "protobuf==4.25.9"
!pip -q install --no-cache-dir "opencv-python==4.10.0.84"
!pip -q install --no-cache-dir "mediapipe==0.10.21"

print("Install complete. Now restart runtime one more time.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 142.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
access 1.1.10.post3 requires pandas>=2.1.0, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
dopamine-rl 4.1.2 requires pandas>=0.24.2, which is not installed.
datasets 4.0.0 requires pandas, which is not installed.
inequality 1.1.2 requires pandas>=2.1, which is not installed.
shap 0.51.0 requires pandas, which is not installed.
grain 0.2.16 requires protobuf>=5.28.3, which is not installed.
plotnine 0.14.5 requires pandas>=2.2.0, which is not installed.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
tensorflow-datasets 4.9.9 requires protobuf>=3.20, which is not installed.
libpysal 4.14

In [ ]:
!pip install mediapipe -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.0 MB/s eta 0:00:00


In [ ]:
import sys
import numpy as np
import pandas as pd
import google.protobuf
import mediapipe as mp

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("mp.solutions exists:", hasattr(mp, "solutions"))

if hasattr(mp, "solutions"):
    print("Holistic exists:", hasattr(mp.solutions, "holistic"))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.0.2
pandas: 2.2.2
protobuf: 5.29.6
MediaPipe: 0.10.35
mp.solutions exists: False


In [ ]:
import mediapipe as mp
print(dir(mp))
try:
    from mediapipe import solutions
    print("solutions imported OK, holistic available:", hasattr(solutions, 'holistic'))
except Exception as e:
    print("Import failed:", type(e).__name__, e)

try:
    holistic = mp.solutions.holistic.Holistic()
    print("Holistic instantiated OK")
except Exception as e:
    print("Holistic failed:", type(e).__name__, e)

['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']
Import failed: ImportError cannot import name 'solutions' from 'mediapipe' (/usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py)
Holistic failed: AttributeError module 'mediapipe' has no attribute 'solutions'


In [ ]:
##For a 100-class problem with ~18 samples per class average, dropping 33% means some classes might be severely under-represented in train now. Run this check

import pandas as pd

manifest = pd.read_csv('/content/drive/MyDrive/CAPSTONE_ASL/manifests/recognition_manifest_splits.csv')

print(f"Total rows: {len(manifest)}")
print()

# === What determines usable_for_recognition? ===
print("=" * 60)
print("WHO GOT FILTERED OUT AND WHY")
print("=" * 60)

# Cross-tab of usable vs filter reason
print("\nrecognition_filter_reason value counts:")
print(manifest['recognition_filter_reason'].value_counts())

print("\nusable_for_recognition value counts:")
print(manifest['usable_for_recognition'].value_counts())

# === Was filtering applied symmetrically across splits? ===
print()
print("=" * 60)
print("FILTERING BY ORIGINAL SPLIT (most important question)")
print("=" * 60)

cross = pd.crosstab(
    manifest['original_split'],
    manifest['usable_for_recognition'],
    margins=True
)
print("\nKept vs dropped per original split:")
print(cross)

# Percentage dropped per split
print("\nPercentage DROPPED per split:")
for split in ['train', 'val', 'test']:
    sub = manifest[manifest['original_split'] == split]
    n_total = len(sub)
    n_dropped = (~sub['usable_for_recognition']).sum()
    if n_total > 0:
        print(f"  {split}: {n_dropped}/{n_total} dropped ({n_dropped/n_total:.1%})")

# === What rec_split do the kept clips end up in? ===
print()
print("=" * 60)
print("FINAL RECOGNITION SPLITS (after filtering)")
print("=" * 60)
kept = manifest[manifest['usable_for_recognition'] == True]
print(kept['rec_split'].value_counts())

# === Reasons broken down per split ===
print()
print("=" * 60)
print("FILTER REASONS PER SPLIT")
print("=" * 60)
for split in ['train', 'val', 'test']:
    sub = manifest[manifest['original_split'] == split]
    print(f"\n{split}:")
    print(sub['recognition_filter_reason'].value_counts())

Total rows: 2234

WHO GOT FILTERED OUT AND WHY

recognition_filter_reason value counts:
recognition_filter_reason
keep    2234
Name: count, dtype: int64

usable_for_recognition value counts:
usable_for_recognition
True    2234
Name: count, dtype: int64

FILTERING BY ORIGINAL SPLIT (most important question)

Kept vs dropped per original split:
usable_for_recognition  True   All
original_split                    
test                     777   777
train                   1217  1217
val                      240   240
All                     2234  2234

Percentage DROPPED per split:
  train: 0/1217 dropped (0.0%)
  val: 0/240 dropped (0.0%)
  test: 0/777 dropped (0.0%)

FINAL RECOGNITION SPLITS (after filtering)
rec_split
train    1215
test      777
val       242
Name: count, dtype: int64

FILTER REASONS PER SPLIT

train:
recognition_filter_reason
keep    1217
Name: count, dtype: int64

val:
recognition_filter_reason
keep    240
Name: count, dtype: int64

test:
recognition_filter_reason
ke

In [ ]:
manifest = pd.read_csv('/content/drive/MyDrive/CAPSTONE_ASL/manifests/recognition_manifest_splits.csv')

print("Sample of motion_status / segment_status / segment_quality_flag / error:")
print(manifest[['motion_status', 'segment_status', 'segment_quality_flag', 'error']].head(10))
print()
print("Value counts for each:")
for col in ['motion_status', 'segment_status', 'segment_quality_flag', 'error']:
    print(f"\n{col}:")
    print(manifest[col].value_counts(dropna=False))

Sample of motion_status / segment_status / segment_quality_flag / error:
  motion_status segment_status segment_quality_flag  error
0            ok             ok                 good    NaN
1            ok             ok                 good    NaN
2            ok             ok                 good    NaN
3            ok             ok                 good    NaN
4            ok             ok                 good    NaN
5            ok             ok                 good    NaN
6            ok             ok                 good    NaN
7            ok             ok                 good    NaN
8            ok             ok                 good    NaN
9            ok             ok                 good    NaN

Value counts for each:

motion_status:
motion_status
ok    2234
Name: count, dtype: int64

segment_status:
segment_status
ok    2234
Name: count, dtype: int64

segment_quality_flag:
segment_quality_flag
good    2234
Name: count, dtype: int64

error:
error
NaN    2234
Name: cou

In [ ]:
from pathlib import Path
import pandas as pd

KEYPOINT_DIR = Path('/content/drive/MyDrive/CAPSTONE_ASL/keypoints')
SUBSET_DIR   = Path('/content/drive/MyDrive/CAPSTONE_ASL/subset_top100')

# Original clips per split
orig_train = pd.read_csv(SUBSET_DIR / 'train.csv')
orig_val   = pd.read_csv(SUBSET_DIR / 'val.csv')
orig_test  = pd.read_csv(SUBSET_DIR / 'test.csv')

orig_stems = {}
for split, df in [('train', orig_train), ('val', orig_val), ('test', orig_test)]:
    stems = set(Path(f).stem for f in df['Video file'])
    orig_stems[split] = stems

# Stems that survived to the manifest
manifest = pd.read_csv('/content/drive/MyDrive/CAPSTONE_ASL/manifests/recognition_manifest_splits.csv')
manifest_stems = set(manifest['video_stem'])

print("Drop rate per original split:")
print(f"{'split':<8} {'original':>10} {'kept':>8} {'dropped':>10} {'drop %':>10}")
for split, stems in orig_stems.items():
    dropped = stems - manifest_stems
    kept = stems & manifest_stems
    drop_pct = len(dropped) / len(stems) if len(stems) > 0 else 0
    print(f"{split:<8} {len(stems):>10} {len(kept):>8} {len(dropped):>10} {drop_pct:>9.1%}")

Drop rate per original split:
split      original     kept    dropped     drop %
train          1800     1217        583     32.4%
val             367      240        127     34.6%
test           1286      777        509     39.6%


In [ ]:
import pandas as pd
from pathlib import Path

SUBSET_DIR = Path('/content/drive/MyDrive/CAPSTONE_ASL/subset_top100')
orig_train = pd.read_csv(SUBSET_DIR / 'train.csv')
orig_val   = pd.read_csv(SUBSET_DIR / 'val.csv')
orig_test  = pd.read_csv(SUBSET_DIR / 'test.csv')
orig_all = pd.concat([orig_train, orig_val, orig_test])
orig_all['stem'] = orig_all['Video file'].apply(lambda f: Path(f).stem)

manifest = pd.read_csv('/content/drive/MyDrive/CAPSTONE_ASL/manifests/recognition_manifest_splits.csv')
manifest_stems = set(manifest['video_stem'])

# Mark which clips survived
orig_all['kept'] = orig_all['stem'].isin(manifest_stems)

# Drop rate per class
per_class = orig_all.groupby('Gloss').agg(
    total=('stem', 'count'),
    kept=('kept', 'sum'),
)
per_class['dropped'] = per_class['total'] - per_class['kept']
per_class['drop_pct'] = per_class['dropped'] / per_class['total']
per_class = per_class.sort_values('drop_pct', ascending=False)

print("10 classes with HIGHEST drop rate:")
print(per_class.head(10))
print()
print("10 classes with LOWEST drop rate:")
print(per_class.tail(10))
print()
print(f"Classes where more than 50% of clips were dropped: {(per_class['drop_pct'] > 0.5).sum()}")
print(f"Classes where more than 70% of clips were dropped: {(per_class['drop_pct'] > 0.7).sum()}")

# Are there classes with very few surviving training clips?
train_only = orig_all[orig_all['stem'].isin(set(Path(f).stem for f in orig_train['Video file']))]
train_only_kept = train_only[train_only['kept']]
train_counts_kept = train_only_kept['Gloss'].value_counts()
print()
print(f"After filtering, training samples per class:")
print(f"  Min:    {train_counts_kept.min()}")
print(f"  Median: {train_counts_kept.median():.0f}")
print(f"  Max:    {train_counts_kept.max()}")
print()
print(f"Classes with fewer than 8 training samples after filtering:")
low = train_counts_kept[train_counts_kept < 8]
print(low if len(low) > 0 else "  None")

10 classes with HIGHEST drop rate:
         total  kept  dropped  drop_pct
Gloss                                  
TWINS1      37    14       23  0.621622
COMB2       31    13       18  0.580645
PIPE2       31    13       18  0.580645
DEMAND1     39    17       22  0.564103
PONDER      32    14       18  0.562500
GRAMMAR     31    14       17  0.548387
PEG2        31    14       17  0.548387
DOG1        45    21       24  0.533333
CANCER1     37    18       19  0.513514
EAT1        39    20       19  0.487179

10 classes with LOWEST drop rate:
            total  kept  dropped  drop_pct
Gloss                                     
SHARK1         31    25        6  0.193548
FLOAT1         31    25        6  0.193548
TOMATO         31    25        6  0.193548
IMPOSSIBLE     31    25        6  0.193548
CALENDAR1      37    30        7  0.189189
ERUPT2         31    26        5  0.161290
SWEATER2       31    26        5  0.161290
RIVER1         38    32        6  0.157895
TAIL1          29   